In [ ]:
from pathlib import Path
import os
import json
import re
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# -----------------------------
# Notebook-local utilities
# -----------------------------

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)


def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def _jsonable(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    return obj


def save_json(obj, path, indent=2):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(json.dumps(_jsonable(obj), indent=indent, sort_keys=True))


# -----------------------------
# RASTA manifest scanning
# -----------------------------
LAYER_ALIASES = {
    'sup': ('sup', 'super', 'superficial', 'svc'),
    'deep': ('deep', 'dvc'),
    'cc': ('cc', 'choriocap', 'choriocapillaris'),
}
EYES = ('OD', 'OS')
IMAGE_EXTS = {'.bmp', '.png', '.jpg', '.jpeg', '.tif', '.tiff'}

# Stable eight-class schema supplied by the project guide. Folder numbers define labels 0-7.
COHORT_FOLDER_TO_LABEL = {
    '1_RETINORM': 0,
    '2_ORNET': 1,
    '3_FAMILIPO': 2,
    '4_MRCC': 3,
    '5_GIANTS': 4,
    '6_AWARD': 5,
    '7_EVIRED': 6,
    '8_ORNET_TEMOINS': 7,
}
EXPECTED_COHORT_FOLDERS = tuple(COHORT_FOLDER_TO_LABEL)


def _normalise_name(name):
    return re.sub(r'[^a-z0-9]+', ' ', str(name).lower()).strip()


def infer_eye(filename):
    tokens = set(_normalise_name(filename).upper().split())
    for eye in EYES:
        if eye in tokens:
            return eye
    m = re.search(r'(?:^|[^A-Z])(OD|OS)(?:[^A-Z]|$)', str(filename).upper())
    return m.group(1) if m else None


def infer_layer(filename):
    norm = _normalise_name(filename)
    tokens = set(norm.split())
    for layer, aliases in LAYER_ALIASES.items():
        if any(alias in tokens or alias in norm for alias in aliases):
            return layer
    return None


def cohort_label_from_folder(folder_name):
    m = re.match(r'(\d+)[_\-\s]*(.*)', str(folder_name))
    if not m:
        return str(folder_name), None
    idx = int(m.group(1)) - 1
    name = m.group(2) or str(folder_name)
    return name, idx


def scan_image_manifest(root, require_all_layers=True, expected_cohort_folders=EXPECTED_COHORT_FOLDERS):
    """Scan RASTA-style folders into one row per patient-eye.

    Expected structure: root/cohort_folder/patient_id/*.bmp.
    Naming is matched case-insensitively and tolerates spaces/underscores,
    e.g. `cc OD.bmp`, `Cc_OD.bmp`, `deep OS.bmp`.
    """
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f'Dataset root does not exist: {root}')

    records = []
    available_dirs = {p.name: p for p in root.iterdir() if p.is_dir() and not p.name.startswith('.')}
    if expected_cohort_folders is not None:
        missing = [name for name in expected_cohort_folders if name not in available_dirs]
        if missing:
            raise FileNotFoundError(
                'The expanded eight-class dataset is incomplete. Missing cohort folder(s): ' + ', '.join(missing)
            )
        cohort_dirs = [available_dirs[name] for name in expected_cohort_folders]
    else:
        cohort_dirs = sorted(available_dirs.values(), key=lambda p: p.name)
    label_map = {}

    for fallback_label, cohort_dir in enumerate(cohort_dirs):
        cohort_name, parsed_label = cohort_label_from_folder(cohort_dir.name)
        label = COHORT_FOLDER_TO_LABEL.get(
            cohort_dir.name, parsed_label if parsed_label is not None else fallback_label
        )
        label_map[cohort_name] = label

        patient_dirs = sorted([p for p in cohort_dir.iterdir() if p.is_dir() and not p.name.startswith('.')], key=lambda p: p.name)
        for patient_dir in patient_dirs:
            per_eye = {eye: {'sup_path': None, 'deep_path': None, 'cc_path': None} for eye in EYES}
            for file in patient_dir.iterdir():
                if not file.is_file() or file.name.startswith('._') or file.suffix.lower() not in IMAGE_EXTS:
                    continue
                eye = infer_eye(file.name)
                layer = infer_layer(file.name)
                if eye is None or layer is None:
                    continue
                per_eye[eye][f'{layer}_path'] = str(file)

            for eye, paths in per_eye.items():
                has_any = any(paths.values())
                has_all = all(paths.values())
                if not has_any:
                    continue
                if require_all_layers and not has_all:
                    continue
                records.append({
                    'patient_id': patient_dir.name,
                    'cohort': cohort_name,
                    'cohort_folder': cohort_dir.name,
                    'label': label,
                    'eye': eye,
                    **paths,
                    'has_all_layers': has_all,
                })

    df = pd.DataFrame(records)
    if not df.empty:
        df = df.sort_values(['label', 'patient_id', 'eye']).reset_index(drop=True)
    df.attrs['label_map'] = label_map
    df.attrs['cohort_folders'] = [p.name for p in cohort_dirs]
    return df


# -----------------------------
# Clinical metadata helpers
# -----------------------------

def load_clinical_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)
    df = df.rename(columns={c: str(c).strip() for c in df.columns})
    if 'ID' in df.columns and 'patient_id' not in df.columns:
        df = df.rename(columns={'ID': 'patient_id'})
    return df


def attach_clinical(manifest, clinical):
    if clinical is None or clinical.empty:
        return manifest.copy()
    if 'patient_id' not in clinical.columns:
        raise ValueError('Clinical table must contain ID or patient_id column')
    return manifest.merge(clinical, on='patient_id', how='left', validate='many_to_one')


def default_clinical_columns(df):
    preferred = [
        'Age', 'Sex', 'Congestive heart failure', 'Hypertension',
        'Diabetes mellitus', 'Stroke', 'Vascular disease', 'Body mass index',
        'CHA2DS2-VASc', 'Obstructive sleep apnea syndrom', 'Smoking', 'Dyslipidemia',
    ]
    stripped = {str(c).strip(): c for c in df.columns}
    cols = [stripped[c] for c in preferred if c in stripped]
    if cols:
        return cols
    exclude = {
        'patient_id', 'cohort', 'cohort_folder', 'label', 'eye',
        'sup_path', 'deep_path', 'cc_path', 'fold', 'has_all_layers'
    }
    return [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]


def make_age_decade(age):
    age = pd.to_numeric(age, errors='coerce')
    decade = (age.fillna(-1) // 10).astype(int) * 10
    return decade.astype(str).where(age.notna(), 'age_missing')


def dataset_audit(df, clinical_cols=None):
    audit = {}
    audit['n_eye_samples'] = int(len(df))
    audit['n_patients'] = int(df['patient_id'].nunique()) if 'patient_id' in df else 0

    if 'cohort' in df:
        audit['class_counts_eye'] = df['cohort'].value_counts().to_dict()
        patient_counts = df.drop_duplicates('patient_id')['cohort'].value_counts().to_dict()
        audit['class_counts_patient'] = patient_counts
        counts = np.array(list(patient_counts.values()), dtype=float)
        audit['class_imbalance_ratio_patient'] = float(counts.max() / max(counts.min(), 1)) if len(counts) else np.nan

    if {'patient_id', 'eye'}.issubset(df.columns):
        eyes_per_patient = df.groupby('patient_id')['eye'].agg(lambda x: '+'.join(sorted(set(x))))
        audit['bilateral_availability'] = eyes_per_patient.value_counts().to_dict()

    layer_cols = [c for c in ['sup_path', 'deep_path', 'cc_path'] if c in df]
    audit['layer_missing_counts'] = {c: int(df[c].isna().sum()) for c in layer_cols}

    if clinical_cols:
        missing = df[list(clinical_cols)].isna().sum().sort_values(ascending=False)
        audit['clinical_missing_counts'] = missing.astype(int).to_dict()
        if 'cohort' in df:
            by_cohort = {}
            for cohort, group in df.groupby('cohort'):
                by_cohort[cohort] = group[list(clinical_cols)].isna().sum().astype(int).to_dict()
            audit['clinical_missing_by_cohort'] = by_cohort

    if 'patient_id' in df and 'cohort' in df:
        leakage = df.groupby('patient_id')['cohort'].nunique()
        audit['patients_in_multiple_cohorts'] = leakage[leakage > 1].index.tolist()

    return audit


# -----------------------------
# Patient-level fold helpers
# -----------------------------

def make_stratification_key(df, age_col='Age', sex_col='Sex'):
    label = df['label'].astype(str)
    key = label.copy()
    if age_col and age_col in df.columns:
        key = key + '_' + make_age_decade(df[age_col])
    if sex_col and sex_col in df.columns:
        sex = df[sex_col].astype('object').where(df[sex_col].notna(), 'sex_missing').astype(str)
        key = key + '_' + sex
    counts = key.value_counts()
    rare = key.map(counts) < 2
    return key.where(~rare, label)


def assign_patient_folds(df, n_splits=5, seed=42, age_col='Age', sex_col='Sex'):
    if not {'patient_id', 'label'}.issubset(df.columns):
        raise ValueError('df must include patient_id and label columns')

    out = df.copy().reset_index(drop=True)
    patient_df = out.drop_duplicates('patient_id').reset_index(drop=True)
    y = make_stratification_key(patient_df, age_col=age_col, sex_col=sex_col)
    groups = patient_df['patient_id'].astype(str)
    folds = np.full(len(patient_df), -1, dtype=int)

    try:
        from sklearn.model_selection import StratifiedGroupKFold
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for fold, (_, val_idx) in enumerate(splitter.split(patient_df, y, groups)):
            folds[val_idx] = fold
    except Exception:
        # Deterministic label/stratum-balanced fallback at patient level.
        rng = np.random.default_rng(seed)
        fold_sizes = np.zeros(n_splits, dtype=int)
        for _, grp in patient_df.assign(_key=y.values).groupby('_key', sort=False):
            idx = grp.index.to_numpy()
            rng.shuffle(idx)
            for i in idx:
                f = int(np.argmin(fold_sizes))
                folds[i] = f
                fold_sizes[f] += 1

    patient_to_fold = dict(zip(patient_df['patient_id'], folds.astype(int)))
    out['fold'] = out['patient_id'].map(patient_to_fold).astype(int)
    validate_patient_folds(out, n_splits=n_splits)
    return out


def validate_patient_folds(df, n_splits=None):
    per_patient = df.groupby('patient_id')['fold'].nunique()
    bad = per_patient[per_patient > 1]
    if len(bad):
        raise AssertionError(f'Patient leakage across folds: {bad.index.tolist()[:10]}')
    if n_splits is not None:
        got = set(df['fold'].dropna().astype(int).unique())
        expected = set(range(n_splits))
        if not got.issubset(expected):
            raise AssertionError(f'Unexpected fold ids: {got - expected}')


seed_everything(42)
sns.set_theme(style='whitegrid', context='notebook')
PROJECT_ROOT = Path.cwd()

In [ ]:
# ============================================================
# EDIT THESE PATHS ON THE WINDOWS MACHINE
# ============================================================

# 1) RASTA image dataset folder. Example Windows path:
# DATA_ROOT_STR = r"D:\RASTA\Rasta dataset"
DATA_ROOT_STR = r"E:\Rasta dataset"

# 2) Clinical spreadsheet path. If it is in the same folder as this notebook,
#    the default below should work. Example Windows path:
# CLINICAL_PATH_STR = r"D:\RASTA\table_for_publication .xlsx"
CLINICAL_PATH_STR = r"E:\Rasta dataset\table_for_publication.xlsx"

# 3) Fresh output folder for the expanded eight-class run. The old four-class outputs are never deleted or overwritten.
OUTPUT_DIR_STR = r"outputs/phase1_dataset_analysis_8class"

# Optional runtime settings
REQUIRE_ALL_LAYERS_FOR_MODEL = True
QC_MAX_IMAGES_PER_LAYER = None  # set e.g. 500 for a quicker smoke-test audit


def choose_path(user_value, env_name, default_value):
    """Choose path in order: explicit notebook value -> environment variable -> default."""
    value = str(user_value).strip() if user_value is not None else ""
    if not value:
        value = os.environ.get(env_name, default_value)
    return Path(value).expanduser()


DATA_ROOT = choose_path(DATA_ROOT_STR, 'RASTA_DATA_ROOT', 'Rasta dataset')
CLINICAL_PATH = choose_path(CLINICAL_PATH_STR, 'RASTA_CLINICAL_PATH', 'table_for_publication .xlsx')
OUTPUT_DIR = ensure_dir(Path(OUTPUT_DIR_STR).expanduser())

print('DATA_ROOT      =', DATA_ROOT.resolve() if DATA_ROOT.exists() else DATA_ROOT)
print('CLINICAL_PATH  =', CLINICAL_PATH.resolve() if CLINICAL_PATH.exists() else CLINICAL_PATH)
print('OUTPUT_DIR     =', OUTPUT_DIR.resolve())
print()
print('To change paths on Windows, edit DATA_ROOT_STR and CLINICAL_PATH_STR in this cell only.')

In [ ]:
clinical_df = load_clinical_table(CLINICAL_PATH)
print('Clinical shape:', clinical_df.shape)
display(clinical_df.head())
print('Columns:')
print(list(clinical_df.columns))

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f'DATA_ROOT does not exist: {DATA_ROOT}\n\n'
        'Edit DATA_ROOT_STR in the Configuration cell above. '
        'For Windows, use a raw string like: r"D:\\RASTA\\Rasta dataset"'
    )

manifest_all = scan_image_manifest(
    DATA_ROOT, require_all_layers=False, expected_cohort_folders=EXPECTED_COHORT_FOLDERS
)
manifest = scan_image_manifest(
    DATA_ROOT, require_all_layers=REQUIRE_ALL_LAYERS_FOR_MODEL, expected_cohort_folders=EXPECTED_COHORT_FOLDERS
)
manifest_clin = attach_clinical(manifest, clinical_df)

manifest_all.to_csv(OUTPUT_DIR / 'manifest_all_detected_eyes.csv', index=False)
manifest_clin.to_csv(OUTPUT_DIR / 'manifest_model_ready_with_clinical.csv', index=False)

print('Expanded cohort folders:', list(EXPECTED_COHORT_FOLDERS))
print('Label map:', manifest.attrs.get('label_map', {}))
print('All detected eye rows:', manifest_all.shape)
print('Model-ready rows with all layers:', manifest.shape)
display(manifest_clin.head())

In [ ]:
clinical_cols = default_clinical_columns(manifest_clin)
audit = dataset_audit(manifest_clin, clinical_cols=clinical_cols)
save_json(audit, OUTPUT_DIR / 'dataset_audit.json')

print(json.dumps(audit, indent=2, default=str))

class_eye = manifest_clin['cohort'].value_counts().rename_axis('cohort').reset_index(name='eye_count')
class_patient = manifest_clin.drop_duplicates('patient_id')['cohort'].value_counts().rename_axis('cohort').reset_index(name='patient_count')
class_table = class_patient.merge(class_eye, on='cohort', how='outer')
class_table.to_csv(OUTPUT_DIR / 'class_distribution.csv', index=False)
display(class_table)

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=class_table, x='cohort', y='patient_count', color='#4C72B0')
plt.xticks(rotation=30, ha='right')
plt.title('Patient count per cohort')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution_patients.png', dpi=200)
plt.show()

eyes_per_patient = manifest_clin.groupby('patient_id')['eye'].agg(lambda s: '+'.join(sorted(set(s))))
bilat = eyes_per_patient.value_counts().rename_axis('eye_availability').reset_index(name='patient_count')
bilat.to_csv(OUTPUT_DIR / 'bilateral_availability.csv', index=False)
display(bilat)

In [ ]:
missing_overall = manifest_clin[clinical_cols].isna().mean().sort_values(ascending=False).rename('missing_fraction')
missing_by_cohort = manifest_clin.groupby('cohort')[clinical_cols].apply(lambda g: g.isna().mean()).reset_index()
missing_overall.to_csv(OUTPUT_DIR / 'clinical_missing_overall.csv')
missing_by_cohort.to_csv(OUTPUT_DIR / 'clinical_missing_by_cohort.csv', index=False)
display(missing_overall.to_frame())
display(missing_by_cohort)

plt.figure(figsize=(12, max(4, 0.025 * len(manifest_clin))))
sns.heatmap(manifest_clin[clinical_cols].isna(), cbar=False, yticklabels=False)
plt.title('Missing clinical data heatmap (eye rows × features)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'clinical_missingness_heatmap.png', dpi=200)
plt.show()

In [ ]:
demo_cols = [c for c in ['Age', 'Sex', 'Body mass index'] if c in manifest_clin.columns]
demo_summary = manifest_clin.drop_duplicates('patient_id').groupby('cohort')[demo_cols].agg(['count', 'mean', 'std', 'median', 'min', 'max'])
demo_summary.to_csv(OUTPUT_DIR / 'demographic_summary_by_cohort.csv')
display(demo_summary)

if 'Age' in manifest_clin.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=manifest_clin.drop_duplicates('patient_id'), x='cohort', y='Age')
    plt.xticks(rotation=30, ha='right')
    plt.title('Age distribution per cohort')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'age_distribution_by_cohort.png', dpi=200)
    plt.show()

if 'Body mass index' in manifest_clin.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=manifest_clin.drop_duplicates('patient_id'), x='cohort', y='Body mass index')
    plt.xticks(rotation=30, ha='right')
    plt.title('BMI distribution per cohort')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'bmi_distribution_by_cohort.png', dpi=200)
    plt.show()

In [ ]:
def audit_image_paths(df, max_per_layer=None):
    rows = []
    for layer, col in [('sup', 'sup_path'), ('deep', 'deep_path'), ('cc', 'cc_path')]:
        paths = df[col].dropna().tolist()
        if max_per_layer is not None:
            paths = paths[:max_per_layer]
        for p in paths:
            rec = {'layer': layer, 'path': p}
            try:
                with Image.open(p) as img:
                    arr = np.asarray(img.convert('L'), dtype=np.float32)
                rec.update({
                    'height': int(arr.shape[0]), 'width': int(arr.shape[1]),
                    'mean': float(arr.mean()), 'std': float(arr.std()),
                    'min': float(arr.min()), 'max': float(arr.max()),
                    'zero_variance': bool(arr.std() <= 1e-6),
                    'readable': True,
                })
            except Exception as e:
                rec.update({'readable': False, 'error': repr(e)})
            rows.append(rec)
    return pd.DataFrame(rows)

image_qc = audit_image_paths(manifest_clin, max_per_layer=QC_MAX_IMAGES_PER_LAYER)
image_qc.to_csv(OUTPUT_DIR / 'image_quality_audit.csv', index=False)
display(image_qc.head())

qc_summary = image_qc.groupby('layer').agg(
    n=('path', 'count'),
    readable=('readable', 'sum'),
    zero_variance=('zero_variance', 'sum'),
    median_height=('height', 'median'),
    median_width=('width', 'median'),
    median_std=('std', 'median'),
).reset_index()
qc_summary.to_csv(OUTPUT_DIR / 'image_quality_summary.csv', index=False)
display(qc_summary)

if not image_qc.empty and 'std' in image_qc:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=image_qc[image_qc['readable']], x='layer', y='std')
    plt.title('Image intensity standard deviation by OCTA layer')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'image_std_by_layer.png', dpi=200)
    plt.show()

In [ ]:
biomarker_pattern = re.compile(r'^(FAZ|Faz|DENS_|PERF_)')
biomarker_cols = [c for c in manifest_clin.columns if biomarker_pattern.search(str(c))]
print('Detected biomarker columns:', len(biomarker_cols))
print(biomarker_cols[:20])

if biomarker_cols:
    biomarker_summary = manifest_clin.groupby('cohort')[biomarker_cols].agg(['count', 'mean', 'std', 'median'])
    biomarker_summary.to_csv(OUTPUT_DIR / 'biomarker_summary_by_cohort.csv')
    display(biomarker_summary.iloc[:, :min(16, biomarker_summary.shape[1])])

    plot_cols = biomarker_cols[:12]
    for col in plot_cols:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=manifest_clin, x='cohort', y=col)
        plt.xticks(rotation=30, ha='right')
        plt.title(f'{col} by cohort')
        plt.tight_layout()
        safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', col)
        plt.savefig(OUTPUT_DIR / f'biomarker_boxplot_{safe}.png', dpi=200)
        plt.show()

In [ ]:
patient_level = manifest_clin.drop_duplicates('patient_id').copy()
leakage = patient_level.groupby('patient_id')['cohort'].nunique()
leaked_patients = leakage[leakage > 1].index.tolist()
print('Patients appearing in multiple cohorts:', len(leaked_patients))
print(leaked_patients[:20])

confounder_tables = {}
if 'Age' in patient_level.columns:
    confounder_tables['age_by_cohort'] = patient_level.groupby('cohort')['Age'].describe()
    display(confounder_tables['age_by_cohort'])
if 'Sex' in patient_level.columns:
    confounder_tables['sex_by_cohort'] = pd.crosstab(patient_level['cohort'], patient_level['Sex'], normalize='index')
    display(confounder_tables['sex_by_cohort'])

for name, table in confounder_tables.items():
    table.to_csv(OUTPUT_DIR / f'{name}.csv')

fold_df = assign_patient_folds(manifest_clin, n_splits=5, seed=42, age_col='Age' if 'Age' in manifest_clin else None, sex_col='Sex' if 'Sex' in manifest_clin else None)
validate_patient_folds(fold_df, n_splits=5)
fold_df[['patient_id', 'eye', 'cohort', 'label', 'fold']].to_csv(OUTPUT_DIR / 'preliminary_patient_level_folds.csv', index=False)
display(pd.crosstab(fold_df['fold'], fold_df['cohort']))
print('Zero patient overlap across folds verified.')

In [ ]:
# ============================================================
# Phase 2 configuration and optional dependencies
# ============================================================

PHASE2_OUTPUT_DIR = ensure_dir('outputs/phase2_preprocessing_8class')
PHASE2_N_SPLITS = 5
PHASE2_SEED = 42
IMAGE_SIZE = 224

# QC thresholds are intentionally conservative; review Phase 1 image-quality plots before tightening.
QC_MIN_RAW_STD = 1.0
QC_MIN_ROBUST_RANGE = 5.0      # p98 - p2 on raw grayscale [0, 255]
QC_MIN_SNR = 0.05              # mean / std; mainly catches degenerate images
CNN_QUALITY_THRESHOLD = 0.50   # used only if a quality_model_fn is supplied

# Computing fold-wise OCTA mean/std requires reading all training images once per fold.
COMPUTE_PHASE2_PIXEL_STATS = True

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset
except ImportError as e:
    torch = None
    nn = None
    Dataset = object
    print('PyTorch is required for the Dataset/model-facing parts of Phase 2:', e)

try:
    import cv2
except ImportError:
    cv2 = None

try:
    from skimage import exposure
except ImportError:
    exposure = None

try:
    import torchvision.transforms.functional as TVF
    from torchvision.transforms import InterpolationMode
except Exception:
    TVF = None
    InterpolationMode = None

from PIL import ImageEnhance, ImageOps

seed_everything(PHASE2_SEED)
print('PHASE2_OUTPUT_DIR =', PHASE2_OUTPUT_DIR.resolve())

In [ ]:

# ============================================================
# OCTA image QC and deterministic preprocessing
# ============================================================

LAYER_PATH_COLS = {'sup': 'sup_path', 'deep': 'deep_path', 'cc': 'cc_path'}


def read_grayscale_array(path):
    """Read an OCTA image as float32 grayscale in raw 0-255 intensity space."""
    with Image.open(path) as img:
        arr = np.asarray(img.convert('L'), dtype=np.float32)
    return arr


def quick_image_qc(path, min_raw_std=QC_MIN_RAW_STD, min_robust_range=QC_MIN_ROBUST_RANGE,
                   min_snr=QC_MIN_SNR, quality_model_fn=None, cnn_threshold=CNN_QUALITY_THRESHOLD):
    """Lightweight image-quality gate.

    Includes corruption checks, low-contrast/SNR checks, and an optional CNN-quality hook.
    `quality_model_fn(path)` should return a probability-like acceptable-quality score.
    """
    rec = {
        'path': str(path), 'readable': False, 'accepted': False,
        'height': np.nan, 'width': np.nan, 'mean': np.nan, 'std': np.nan,
        'p2': np.nan, 'p98': np.nan, 'robust_range': np.nan, 'snr': np.nan,
        'all_black': False, 'zero_variance': False, 'low_dynamic_range': False,
        'low_snr': False, 'cnn_quality_score': np.nan, 'failed_reason': '',
    }
    try:
        arr = read_grayscale_array(path)
        p2, p98 = np.percentile(arr, [2, 98])
        std = float(arr.std())
        mean = float(arr.mean())
        robust_range = float(p98 - p2)
        snr = float(mean / (std + 1e-6))
        rec.update({
            'readable': True, 'height': int(arr.shape[0]), 'width': int(arr.shape[1]),
            'mean': mean, 'std': std, 'p2': float(p2), 'p98': float(p98),
            'robust_range': robust_range, 'snr': snr,
            'all_black': bool(arr.max() <= 1),
            'zero_variance': bool(std <= min_raw_std),
            'low_dynamic_range': bool(robust_range < min_robust_range),
            'low_snr': bool(snr < min_snr),
        })
        if quality_model_fn is not None:
            score = float(quality_model_fn(path))
            rec['cnn_quality_score'] = score
            if score < cnn_threshold:
                rec['failed_reason'] = 'cnn_quality'
        if not rec['failed_reason']:
            for key in ['all_black', 'zero_variance', 'low_dynamic_range', 'low_snr']:
                if rec[key]:
                    rec['failed_reason'] = key
                    break
        rec['accepted'] = rec['readable'] and not bool(rec['failed_reason'])
    except Exception as e:
        rec['failed_reason'] = f'unreadable: {repr(e)}'
    return rec


def percentile_normalise(arr, lower=2, upper=98):
    lo, hi = np.percentile(arr, [lower, upper])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(arr, dtype=np.float32)
    out = np.clip((arr - lo) / (hi - lo), 0, 1)
    return out.astype(np.float32)


def apply_clahe(arr01, clip_limit=2.0, tile_grid_size=(8, 8)):
    """Apply CLAHE to an image already scaled to [0, 1]."""
    arr01 = np.clip(arr01, 0, 1).astype(np.float32)
    if cv2 is not None:
        u8 = (arr01 * 255).astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
        return clahe.apply(u8).astype(np.float32) / 255.0
    if exposure is not None:
        return exposure.equalize_adapthist(arr01, clip_limit=clip_limit / 40.0, nbins=256).astype(np.float32)
    return arr01


def resize_array(arr01, image_size=IMAGE_SIZE, resample=Image.Resampling.BICUBIC):
    img = Image.fromarray((np.clip(arr01, 0, 1) * 255).astype(np.uint8), mode='L')
    img = img.resize((image_size, image_size), resample=resample)
    return np.asarray(img, dtype=np.float32) / 255.0


def preprocess_octa_image(path, image_size=IMAGE_SIZE, use_clahe=True):
    """Raw image -> percentile normalisation -> CLAHE -> resize; output float32 [0, 1]."""
    arr = read_grayscale_array(path)
    arr = percentile_normalise(arr, lower=2, upper=98)
    if use_clahe:
        arr = apply_clahe(arr, clip_limit=2.0, tile_grid_size=(8, 8))
    arr = resize_array(arr, image_size=image_size, resample=Image.Resampling.BICUBIC)
    return arr.astype(np.float32)


def preprocess_mask_image(path, image_size=IMAGE_SIZE):
    if path is None or (isinstance(path, float) and np.isnan(path)) or not Path(path).exists():
        return np.zeros((image_size, image_size), dtype=np.float32)
    arr = read_grayscale_array(path)
    arr = (arr > 0).astype(np.float32)
    return resize_array(arr, image_size=image_size, resample=Image.Resampling.NEAREST)

In [ ]:
# ============================================================
# Clinical metadata preprocessing and learnable missing-token layer
# ============================================================

SMOKING_MAP = {
    'never': 0, 'no': 0, 'none': 0, 'non-smoker': 0, 'nonsmoker': 0, '0': 0,
    'former': 1, 'ex': 1, 'past': 1, '1': 1,
    'current': 2, 'yes': 2, 'smoker': 2, 'active': 2, '2': 2,
}
SEX_MAP = {
    'female': 0, 'f': 0, 'woman': 0, '0': 0,
    'male': 1, 'm': 1, 'man': 1, '1': 1,
}


def canonical_feature_name(name):
    return re.sub(r'[^a-z0-9]+', ' ', str(name).strip().lower()).strip()


def infer_clinical_specs(df, clinical_cols):
    """Classify clinical features as continuous, binary, or ordinal."""
    specs = {}
    for col in clinical_cols:
        key = canonical_feature_name(col)
        observed = df[col].dropna() if col in df else pd.Series(dtype='float64')
        unique = set(pd.to_numeric(observed, errors='coerce').dropna().unique().tolist())
        if key in {'age', 'body mass index', 'bmi'} or 'age' == key or 'body mass' in key:
            specs[col] = 'continuous'
        elif 'smoking' in key:
            specs[col] = 'ordinal'
        elif len(unique) <= 2:
            specs[col] = 'binary'
        else:
            specs[col] = 'continuous'
    return specs


class ClinicalPreprocessor:
    """Fold-fitted clinical encoder.

    Continuous variables are z-scored using training-fold statistics.
    Binary variables are encoded as 0/1.
    Ordinal variables (e.g. smoking) are integer-coded for a later learned ordinal token layer.
    Missingness is not imputed statistically; missing values are set to 0 and accompanied by mask=0.
    """
    def __init__(self, clinical_cols, specs=None):
        self.clinical_cols = list(clinical_cols)
        self.specs = dict(specs) if specs is not None else None
        self.continuous_stats = {}
        self.category_maps = {}
        self.ordinal_cardinalities = {}

    def fit(self, df):
        if self.specs is None:
            self.specs = infer_clinical_specs(df, self.clinical_cols)
        for col in self.clinical_cols:
            kind = self.specs[col]
            if kind == 'continuous':
                vals = pd.to_numeric(df[col], errors='coerce')
                mean = float(vals.mean()) if vals.notna().any() else 0.0
                std = float(vals.std(ddof=0)) if vals.notna().any() else 1.0
                if not np.isfinite(std) or std < 1e-6:
                    std = 1.0
                self.continuous_stats[col] = {'mean': mean, 'std': std}
            elif kind == 'ordinal':
                vals = df[col].dropna().map(self._encode_ordinal_value)
                max_code = int(pd.to_numeric(vals, errors='coerce').max()) if len(vals) else 2
                self.ordinal_cardinalities[col] = max(max_code + 1, 3)
            else:
                self.category_maps[col] = self._make_binary_map(df[col])
        return self

    @staticmethod
    def _make_binary_map(series):
        values = [v for v in series.dropna().unique().tolist()]
        if not values:
            return {}
        mapping = {}
        for v in values:
            if isinstance(v, str):
                key = canonical_feature_name(v)
                if key in SEX_MAP:
                    mapping[v] = SEX_MAP[key]
                else:
                    mapping[v] = len(mapping)
            else:
                mapping[v] = int(v)
        return mapping

    @staticmethod
    def _encode_ordinal_value(value):
        if pd.isna(value):
            return np.nan
        if isinstance(value, str):
            key = canonical_feature_name(value)
            return SMOKING_MAP.get(key, np.nan)
        return int(value)

    def transform_row(self, row):
        values, mask = [], []
        for col in self.clinical_cols:
            value = row[col] if col in row else np.nan
            if pd.isna(value):
                values.append(0.0)
                mask.append(0.0)
                continue
            kind = self.specs[col]
            if kind == 'continuous':
                value = float(pd.to_numeric(value, errors='coerce'))
                stats = self.continuous_stats[col]
                encoded = (value - stats['mean']) / stats['std'] if np.isfinite(value) else 0.0
            elif kind == 'ordinal':
                encoded = self._encode_ordinal_value(value)
                encoded = 0.0 if pd.isna(encoded) else float(encoded)
            else:
                encoded = self.category_maps.get(col, {}).get(value, value)
                if isinstance(encoded, str):
                    encoded = SEX_MAP.get(canonical_feature_name(encoded), 0)
                encoded = float(encoded)
            values.append(float(encoded))
            mask.append(1.0)
        return np.asarray(values, dtype=np.float32), np.asarray(mask, dtype=np.float32)

    def transform_dataframe(self, df):
        vals, masks = [], []
        for _, row in df.iterrows():
            v, m = self.transform_row(row)
            vals.append(v)
            masks.append(m)
        return np.vstack(vals).astype(np.float32), np.vstack(masks).astype(np.float32)

    @property
    def ordinal_indices(self):
        return [i for i, c in enumerate(self.clinical_cols) if self.specs.get(c) == 'ordinal']

    def to_dict(self):
        return {
            'clinical_cols': self.clinical_cols,
            'specs': self.specs,
            'continuous_stats': self.continuous_stats,
            'category_maps': {str(k): {str(kk): vv for kk, vv in v.items()} for k, v in self.category_maps.items()},
            'ordinal_cardinalities': self.ordinal_cardinalities,
        }

    @classmethod
    def from_dict(cls, state):
        obj = cls(state['clinical_cols'], specs=state['specs'])
        obj.continuous_stats = state.get('continuous_stats', {})
        obj.category_maps = state.get('category_maps', {})
        obj.ordinal_cardinalities = state.get('ordinal_cardinalities', {})
        return obj


if nn is not None:
    class ClinicalMaskTokenImputer(nn.Module):
        """Model-facing learnable imputation layer.

        It converts `(clinical_features, clinical_mask)` into a single D-dimensional vector.
        Missing features receive a trainable scalar token per feature. Ordinal observed
        values can also be passed through a learned scalar embedding instead of being used
        as raw integers.
        """
        def __init__(self, num_features, ordinal_indices=None, ordinal_cardinalities=None):
            super().__init__()
            self.num_features = int(num_features)
            self.missing_tokens = nn.Parameter(torch.zeros(self.num_features))
            self.ordinal_indices = [int(i) for i in (ordinal_indices or [])]
            self.ordinal_embeddings = nn.ModuleDict()
            ordinal_cardinalities = ordinal_cardinalities or {}
            for idx in self.ordinal_indices:
                card = int(ordinal_cardinalities.get(idx, 3))
                self.ordinal_embeddings[str(idx)] = nn.Embedding(card, 1)

        def forward(self, values, mask):
            x = values.float().clone()
            mask = mask.float()
            for idx in self.ordinal_indices:
                if str(idx) in self.ordinal_embeddings:
                    codes = values[:, idx].round().long().clamp(min=0, max=self.ordinal_embeddings[str(idx)].num_embeddings - 1)
                    x[:, idx] = self.ordinal_embeddings[str(idx)](codes).squeeze(-1)
            tokens = self.missing_tokens.view(1, -1).expand_as(x)
            return torch.where(mask > 0.5, x, tokens)
else:
    ClinicalMaskTokenImputer = None


def make_clinical_mask_token_imputer(clinical_preprocessor):
    """Create the model-facing learned missing-token layer for a fitted ClinicalPreprocessor."""
    if ClinicalMaskTokenImputer is None:
        raise ImportError('PyTorch is required for ClinicalMaskTokenImputer')
    ordinal_cardinalities = {
        i: clinical_preprocessor.ordinal_cardinalities.get(col, 3)
        for i, col in enumerate(clinical_preprocessor.clinical_cols)
        if clinical_preprocessor.specs.get(col) == 'ordinal'
    }
    return ClinicalMaskTokenImputer(
        num_features=len(clinical_preprocessor.clinical_cols),
        ordinal_indices=list(ordinal_cardinalities.keys()),
        ordinal_cardinalities=ordinal_cardinalities,
    )

In [ ]:
# ============================================================
# Phase 2 fold/QC/statistics utilities
# ============================================================


def build_phase2_manifest(manifest_df, output_dir=PHASE2_OUTPUT_DIR, quality_model_fn=None):
    """Apply per-layer QC and return model-ready rows plus an exclusion log."""
    accepted_rows = []
    qc_rows = []
    for idx, row in manifest_df.reset_index(drop=True).iterrows():
        row_ok = True
        reasons = []
        for layer, col in LAYER_PATH_COLS.items():
            rec = quick_image_qc(row[col], quality_model_fn=quality_model_fn)
            rec.update({'row_index': int(idx), 'patient_id': row['patient_id'], 'eye': row['eye'], 'cohort': row['cohort'], 'layer': layer})
            qc_rows.append(rec)
            if not rec['accepted']:
                row_ok = False
                reasons.append(f'{layer}:{rec["failed_reason"]}')
        if row_ok:
            accepted_rows.append(row.to_dict())
    phase2_df = pd.DataFrame(accepted_rows)
    qc_df = pd.DataFrame(qc_rows)
    exclusions = qc_df[~qc_df['accepted']].copy() if not qc_df.empty else pd.DataFrame()
    phase2_df.to_csv(Path(output_dir) / 'phase2_manifest_qc_pass.csv', index=False)
    qc_df.to_csv(Path(output_dir) / 'phase2_image_qc_full_log.csv', index=False)
    exclusions.to_csv(Path(output_dir) / 'phase2_image_qc_exclusions.csv', index=False)
    return phase2_df, exclusions, qc_df


def compute_layer_stats(df, image_size=IMAGE_SIZE):
    """Compute per-layer mean/std after deterministic preprocessing, using a training fold only."""
    stats = {}
    for layer, col in LAYER_PATH_COLS.items():
        n_pixels = 0
        total = 0.0
        total_sq = 0.0
        for p in df[col].dropna().tolist():
            arr = preprocess_octa_image(p, image_size=image_size, use_clahe=True)
            total += float(arr.sum())
            total_sq += float((arr ** 2).sum())
            n_pixels += int(arr.size)
        if n_pixels == 0:
            mean, std = 0.0, 1.0
        else:
            mean = total / n_pixels
            var = max(total_sq / n_pixels - mean ** 2, 1e-8)
            std = float(np.sqrt(var))
        stats[layer] = {'mean': float(mean), 'std': float(std), 'n_pixels': int(n_pixels)}
    return stats


def compute_clinical_mutual_information(df, clinical_preprocessor, label_col='label'):
    try:
        from sklearn.feature_selection import mutual_info_classif
    except Exception as e:
        print('Skipping mutual information; sklearn is unavailable:', e)
        return pd.DataFrame()
    values, masks = clinical_preprocessor.transform_dataframe(df)
    # Include missingness indicators in MI because missingness can itself be clinically informative.
    x = np.concatenate([values, masks], axis=1)
    y = df[label_col].astype(int).to_numpy()
    mi = mutual_info_classif(x, y, discrete_features=False, random_state=PHASE2_SEED)
    feature_names = clinical_preprocessor.clinical_cols + [f'{c}__observed_mask' for c in clinical_preprocessor.clinical_cols]
    out = pd.DataFrame({'feature': feature_names, 'mutual_information': mi})
    out['low_information_flag'] = out['mutual_information'] < 0.01
    return out.sort_values('mutual_information', ascending=False).reset_index(drop=True)


def validate_no_patient_overlap_between_splits(train_df, val_df):
    train_patients = set(train_df['patient_id'].astype(str))
    val_patients = set(val_df['patient_id'].astype(str))
    overlap = sorted(train_patients & val_patients)
    if overlap:
        raise AssertionError(f'Patient overlap between train/val: {overlap[:10]}')
    return True

In [ ]:
# ============================================================
# PyTorch Dataset with synchronized OCTA augmentation
# ============================================================


def _array_to_pil(arr01):
    return Image.fromarray((np.clip(arr01, 0, 1) * 255).astype(np.uint8), mode='L')


def _pil_to_array(img):
    return np.asarray(img.convert('L'), dtype=np.float32) / 255.0


def sample_octa_augmentation_params(rng):
    return {
        'hflip': bool(rng.random() < 0.5),
        'vflip': bool(rng.random() < 0.3),
        'angle': float(rng.uniform(-15, 15)),
        'translate': (int(rng.integers(-11, 12)), int(rng.integers(-11, 12))),  # ~5% of 224
        'scale': float(rng.uniform(0.95, 1.05)),
        'shear': float(rng.uniform(-5, 5)),
        'brightness': float(rng.uniform(0.9, 1.1)),
        'contrast': float(rng.uniform(0.9, 1.1)),
        'noise': bool(rng.random() < 0.2),
        'noise_sigma': 0.01,
    }


def apply_octa_group_augmentation(arrays, params):
    """Apply the same geometric transform to Sup/Deep/CC to preserve correspondence."""
    out = []
    for arr in arrays:
        img = _array_to_pil(arr)
        if params['hflip']:
            img = ImageOps.mirror(img)
        if params['vflip']:
            img = ImageOps.flip(img)
        if TVF is not None:
            img = TVF.affine(
                img, angle=params['angle'], translate=params['translate'], scale=params['scale'],
                shear=[params['shear'], 0.0], interpolation=InterpolationMode.BICUBIC, fill=0
            )
        else:
            img = img.rotate(params['angle'], resample=Image.Resampling.BICUBIC, fillcolor=0)
            tx, ty = params['translate']
            img = img.transform(img.size, Image.Transform.AFFINE, (1, 0, -tx, 0, 1, -ty),
                                resample=Image.Resampling.BICUBIC, fillcolor=0)
        img = ImageEnhance.Brightness(img).enhance(params['brightness'])
        img = ImageEnhance.Contrast(img).enhance(params['contrast'])
        aug = _pil_to_array(img)
        if params['noise']:
            aug = np.clip(aug + np.random.normal(0, params['noise_sigma'], size=aug.shape), 0, 1)
        out.append(aug.astype(np.float32))
    return out


class RastaPhase2Dataset(Dataset):
    """RASTA eye-level Dataset.

    Returns the dictionary specified in Phase 2. Missing clinical values are represented by
    `clinical_mask=0` and should be converted to learned mask tokens inside the model via
    `ClinicalMaskTokenImputer`.
    """
    def __init__(self, df, clinical_preprocessor, layer_stats, train=False, image_size=IMAGE_SIZE,
                 clinical_mask_dropout=0.10, mask_path_cols=None, seed=PHASE2_SEED):
        if torch is None:
            raise ImportError('PyTorch is required to instantiate RastaPhase2Dataset')
        self.df = df.reset_index(drop=True).copy()
        self.clinical_preprocessor = clinical_preprocessor
        self.layer_stats = layer_stats
        self.train = bool(train)
        self.image_size = int(image_size)
        self.clinical_mask_dropout = float(clinical_mask_dropout)
        self.mask_path_cols = mask_path_cols or {'vessel_mask': None, 'faz_mask': None, 'flow_mask': None}
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.df)

    def _load_layer(self, row, layer):
        arr = preprocess_octa_image(row[LAYER_PATH_COLS[layer]], image_size=self.image_size, use_clahe=True)
        return arr

    def _standardize_layer(self, arr, layer):
        mean = float(self.layer_stats[layer]['mean'])
        std = float(self.layer_stats[layer]['std'])
        return ((arr - mean) / max(std, 1e-6)).astype(np.float32)

    def _load_optional_mask(self, row, key):
        col = self.mask_path_cols.get(key)
        path = row[col] if col and col in row and pd.notna(row[col]) else None
        arr = preprocess_mask_image(path, image_size=self.image_size)
        return torch.from_numpy(arr[None, :, :].astype(np.float32))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        arrays = [self._load_layer(row, layer) for layer in ['sup', 'deep', 'cc']]
        if self.train:
            params = sample_octa_augmentation_params(self.rng)
            arrays = apply_octa_group_augmentation(arrays, params)
        sup, deep, cc = [self._standardize_layer(arr, layer) for arr, layer in zip(arrays, ['sup', 'deep', 'cc'])]

        clinical_values, clinical_mask = self.clinical_preprocessor.transform_row(row)
        if self.train and self.clinical_mask_dropout > 0:
            observed = np.where(clinical_mask > 0.5)[0]
            if len(observed):
                drop = observed[self.rng.random(len(observed)) < self.clinical_mask_dropout]
                clinical_values[drop] = 0.0
                clinical_mask[drop] = 0.0

        sample = {
            'sup_image': torch.from_numpy(sup[None, :, :]),
            'deep_image': torch.from_numpy(deep[None, :, :]),
            'cc_image': torch.from_numpy(cc[None, :, :]),
            'clinical_features': torch.from_numpy(clinical_values.astype(np.float32)),
            'clinical_mask': torch.from_numpy(clinical_mask.astype(np.float32)),
            'vessel_mask': self._load_optional_mask(row, 'vessel_mask'),
            'faz_mask': self._load_optional_mask(row, 'faz_mask'),
            'flow_mask': self._load_optional_mask(row, 'flow_mask'),
            'label': torch.tensor(int(row['label']), dtype=torch.long),
            'patient_id': str(row['patient_id']),
            'eye': str(row['eye']),
            'fold': int(row['fold']) if 'fold' in row else -1,
        }
        return sample


def make_phase2_datasets_for_fold(fold_df, fold, clinical_preprocessor, layer_stats, **dataset_kwargs):
    train_df = fold_df[fold_df['fold'] != fold].copy()
    val_df = fold_df[fold_df['fold'] == fold].copy()
    validate_no_patient_overlap_between_splits(train_df, val_df)
    train_ds = RastaPhase2Dataset(train_df, clinical_preprocessor, layer_stats, train=True, **dataset_kwargs)
    val_ds = RastaPhase2Dataset(val_df, clinical_preprocessor, layer_stats, train=False, **dataset_kwargs)
    return train_ds, val_ds

In [ ]:
# ============================================================
# Build Phase 2 artifacts: QC-passed manifest, patient folds, clinical processors, fold stats
# ============================================================

if 'manifest_clin' not in globals():
    raise RuntimeError('Run Phase 1 cells first so manifest_clin is available.')

phase2_manifest, phase2_exclusions, phase2_qc_log = build_phase2_manifest(manifest_clin, output_dir=PHASE2_OUTPUT_DIR)
print('QC-passed eye rows:', phase2_manifest.shape)
print('Excluded layer images:', phase2_exclusions.shape)

phase2_folds = assign_patient_folds(
    phase2_manifest,
    n_splits=PHASE2_N_SPLITS,
    seed=PHASE2_SEED,
    age_col='Age' if 'Age' in phase2_manifest.columns else None,
    sex_col='Sex' if 'Sex' in phase2_manifest.columns else None,
)
validate_patient_folds(phase2_folds, n_splits=PHASE2_N_SPLITS)
phase2_folds.to_csv(PHASE2_OUTPUT_DIR / 'phase2_patient_stratified_folds.csv', index=False)

display(pd.crosstab(phase2_folds['fold'], phase2_folds['cohort']))
print('Zero patient overlap across folds verified for Phase 2.')

clinical_cols_phase2 = default_clinical_columns(phase2_folds)
save_json({'clinical_cols': clinical_cols_phase2}, PHASE2_OUTPUT_DIR / 'phase2_clinical_columns.json')

clinical_preprocessors = {}
layer_stats_by_fold = {}
mi_tables = []

for fold in range(PHASE2_N_SPLITS):
    train_df = phase2_folds[phase2_folds['fold'] != fold].copy()
    val_df = phase2_folds[phase2_folds['fold'] == fold].copy()
    validate_no_patient_overlap_between_splits(train_df, val_df)

    clinical_pre = ClinicalPreprocessor(clinical_cols_phase2).fit(train_df)
    clinical_preprocessors[fold] = clinical_pre
    save_json(clinical_pre.to_dict(), PHASE2_OUTPUT_DIR / f'clinical_preprocessor_fold{fold}.json')

    mi = compute_clinical_mutual_information(train_df, clinical_pre)
    if not mi.empty:
        mi.insert(0, 'fold', fold)
        mi.to_csv(PHASE2_OUTPUT_DIR / f'clinical_mutual_information_fold{fold}.csv', index=False)
        mi_tables.append(mi)

    if COMPUTE_PHASE2_PIXEL_STATS:
        stats = compute_layer_stats(train_df, image_size=IMAGE_SIZE)
    else:
        stats = {layer: {'mean': 0.0, 'std': 1.0, 'n_pixels': 0} for layer in LAYER_PATH_COLS}
    layer_stats_by_fold[fold] = stats
    save_json(stats, PHASE2_OUTPUT_DIR / f'octa_train_stats_fold{fold}.json')

if mi_tables:
    pd.concat(mi_tables, ignore_index=True).to_csv(PHASE2_OUTPUT_DIR / 'clinical_mutual_information_all_folds.csv', index=False)

save_json(layer_stats_by_fold, PHASE2_OUTPUT_DIR / 'octa_train_stats_all_folds.json')

print('Phase 2 artifacts saved to:', PHASE2_OUTPUT_DIR.resolve())
print('Example fold 0 layer stats:')
print(json.dumps(layer_stats_by_fold.get(0, {}), indent=2))

if torch is not None and len(phase2_folds):
    # Smoke-test construction only; indexing loads images, so leave actual sample loading to the GPU/data machine.
    train_ds0, val_ds0 = make_phase2_datasets_for_fold(
        phase2_folds, fold=0, clinical_preprocessor=clinical_preprocessors[0], layer_stats=layer_stats_by_fold[0]
    )
    print('Fold 0 dataset sizes:', len(train_ds0), 'train /', len(val_ds0), 'val')
    print('Clinical feature dimension:', len(clinical_cols_phase2))

In [ ]:
# ============================================================
# SUMMARIZE / FAST RELOAD: Phase 1 + Phase 2 state needed by Phase 3
# ============================================================
# Use this after restarting the notebook kernel when Phase 1/2 outputs already exist.
# Run only this cell, then continue with the Phase 3 cells. It restores Phase 3's
# in-memory prerequisites from disk and never regenerates Phase 1 or Phase 2 outputs.

from pathlib import Path
import os
import json
import re
import random

import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageOps

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset
except ImportError as e:
    torch = None
    nn = None
    Dataset = object
    print('PyTorch unavailable; Phase 3 / Dataset cells will need PyTorch:', e)

try:
    import cv2
except ImportError:
    cv2 = None

try:
    from skimage import exposure
except ImportError:
    exposure = None

try:
    import torchvision.transforms.functional as TVF
    from torchvision.transforms import InterpolationMode
except Exception:
    TVF = None
    InterpolationMode = None


def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def _jsonable(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    return obj


def save_json(obj, path, indent=2):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(json.dumps(_jsonable(obj), indent=indent, sort_keys=True))


def choose_path(user_value, env_name, default_value):
    value = str(user_value).strip() if user_value is not None else ''
    if not value:
        value = os.environ.get(env_name, default_value)
    return Path(value).expanduser()


# Keep these constants aligned with Phase 1 / Phase 2 config cells.
PROJECT_ROOT = Path.cwd()
DATA_ROOT_STR = r'J:\Rasta dataset'
CLINICAL_PATH_STR = r'J:\Rasta dataset\table_for_publication.xlsx'
OUTPUT_DIR_STR = r'outputs/phase1_dataset_analysis_8class'
DATA_ROOT = choose_path(DATA_ROOT_STR, 'RASTA_DATA_ROOT', 'Rasta dataset')
CLINICAL_PATH = choose_path(CLINICAL_PATH_STR, 'RASTA_CLINICAL_PATH', 'table_for_publication .xlsx')
OUTPUT_DIR = Path(OUTPUT_DIR_STR).expanduser()
REQUIRE_ALL_LAYERS_FOR_MODEL = True
QC_MAX_IMAGES_PER_LAYER = None

PHASE2_OUTPUT_DIR = Path('outputs/phase2_preprocessing_8class')
PHASE2_N_SPLITS = 5
PHASE2_SEED = 42
IMAGE_SIZE = 224
QC_MIN_RAW_STD = 1.0
QC_MIN_ROBUST_RANGE = 5.0
QC_MIN_SNR = 0.05
CNN_QUALITY_THRESHOLD = 0.50
COMPUTE_PHASE2_PIXEL_STATS = True
LAYER_PATH_COLS = {'sup': 'sup_path', 'deep': 'deep_path', 'cc': 'cc_path'}
EYES = ('OD', 'OS')
COHORT_FOLDER_TO_LABEL = {
    '1_RETINORM': 0, '2_ORNET': 1, '3_FAMILIPO': 2, '4_MRCC': 3,
    '5_GIANTS': 4, '6_AWARD': 5, '7_EVIRED': 6, '8_ORNET_TEMOINS': 7,
}
EXPECTED_COHORT_FOLDERS = tuple(COHORT_FOLDER_TO_LABEL)

SMOKING_MAP = {
    'never': 0, 'no': 0, 'none': 0, 'non-smoker': 0, 'nonsmoker': 0, '0': 0,
    'former': 1, 'ex': 1, 'past': 1, '1': 1,
    'current': 2, 'yes': 2, 'smoker': 2, 'active': 2, '2': 2,
}
SEX_MAP = {
    'female': 0, 'f': 0, 'woman': 0, '0': 0,
    'male': 1, 'm': 1, 'man': 1, '1': 1,
}


def canonical_feature_name(name):
    return re.sub(r'[^a-z0-9]+', ' ', str(name).strip().lower()).strip()


def default_clinical_columns(df):
    preferred = [
        'Age', 'Sex', 'Congestive heart failure', 'Hypertension',
        'Diabetes mellitus', 'Stroke', 'Vascular disease', 'Body mass index',
        'CHA2DS2-VASc', 'Obstructive sleep apnea syndrom', 'Smoking', 'Dyslipidemia',
    ]
    stripped = {str(c).strip(): c for c in df.columns}
    cols = [stripped[c] for c in preferred if c in stripped]
    if cols:
        return cols
    exclude = {
        'patient_id', 'cohort', 'cohort_folder', 'label', 'eye',
        'sup_path', 'deep_path', 'cc_path', 'fold', 'has_all_layers'
    }
    return [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]


def validate_patient_folds(df, n_splits=None):
    per_patient = df.groupby('patient_id')['fold'].nunique()
    bad = per_patient[per_patient > 1]
    if len(bad):
        raise AssertionError(f'Patient leakage across folds: {bad.index.tolist()[:10]}')
    if n_splits is not None:
        got = set(df['fold'].dropna().astype(int).unique())
        expected = set(range(n_splits))
        if not got.issubset(expected):
            raise AssertionError(f'Unexpected fold ids: {got - expected}')


def validate_no_patient_overlap_between_splits(train_df, val_df):
    overlap = sorted(set(train_df['patient_id'].astype(str)) & set(val_df['patient_id'].astype(str)))
    if overlap:
        raise AssertionError(f'Patient overlap between train/val: {overlap[:10]}')
    return True


def read_grayscale_array(path):
    with Image.open(path) as img:
        return np.asarray(img.convert('L'), dtype=np.float32)


def percentile_normalise(arr, lower=2, upper=98):
    lo, hi = np.percentile(arr, [lower, upper])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(arr, dtype=np.float32)
    return np.clip((arr - lo) / (hi - lo), 0, 1).astype(np.float32)


def apply_clahe(arr01, clip_limit=2.0, tile_grid_size=(8, 8)):
    arr01 = np.clip(arr01, 0, 1).astype(np.float32)
    if cv2 is not None:
        u8 = (arr01 * 255).astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
        return clahe.apply(u8).astype(np.float32) / 255.0
    if exposure is not None:
        return exposure.equalize_adapthist(arr01, clip_limit=clip_limit / 40.0, nbins=256).astype(np.float32)
    return arr01


def resize_array(arr01, image_size=IMAGE_SIZE, resample=Image.Resampling.BICUBIC):
    img = Image.fromarray((np.clip(arr01, 0, 1) * 255).astype(np.uint8), mode='L')
    img = img.resize((image_size, image_size), resample=resample)
    return np.asarray(img, dtype=np.float32) / 255.0


def preprocess_octa_image(path, image_size=IMAGE_SIZE, use_clahe=True):
    arr = read_grayscale_array(path)
    arr = percentile_normalise(arr, lower=2, upper=98)
    if use_clahe:
        arr = apply_clahe(arr, clip_limit=2.0, tile_grid_size=(8, 8))
    return resize_array(arr, image_size=image_size, resample=Image.Resampling.BICUBIC).astype(np.float32)


def preprocess_mask_image(path, image_size=IMAGE_SIZE):
    if path is None or (isinstance(path, float) and np.isnan(path)) or not Path(path).exists():
        return np.zeros((image_size, image_size), dtype=np.float32)
    arr = read_grayscale_array(path)
    arr = (arr > 0).astype(np.float32)
    return resize_array(arr, image_size=image_size, resample=Image.Resampling.NEAREST)


def compute_layer_stats(df, image_size=IMAGE_SIZE, show_progress=False, desc='Layer statistics'):
    stats = {}
    for layer, col in LAYER_PATH_COLS.items():
        n_pixels = 0
        total = 0.0
        total_sq = 0.0
        paths = df[col].dropna().tolist()
        iterator = paths
        if show_progress and 'phase3_progress' in globals():
            iterator = phase3_progress(paths, desc=f'{desc}: {layer}', unit='image', leave=False)
        for p in iterator:
            arr = preprocess_octa_image(p, image_size=image_size, use_clahe=True)
            total += float(arr.sum())
            total_sq += float((arr ** 2).sum())
            n_pixels += int(arr.size)
        if n_pixels == 0:
            mean, std = 0.0, 1.0
        else:
            mean = total / n_pixels
            var = max(total_sq / n_pixels - mean ** 2, 1e-8)
            std = float(np.sqrt(var))
        stats[layer] = {'mean': float(mean), 'std': float(std), 'n_pixels': int(n_pixels)}
    return stats


def _array_to_pil(arr01):
    return Image.fromarray((np.clip(arr01, 0, 1) * 255).astype(np.uint8), mode='L')


def _pil_to_array(img):
    return np.asarray(img.convert('L'), dtype=np.float32) / 255.0


def sample_octa_augmentation_params(rng):
    return {
        'hflip': bool(rng.random() < 0.5),
        'vflip': bool(rng.random() < 0.3),
        'angle': float(rng.uniform(-15, 15)),
        'translate': (int(rng.integers(-11, 12)), int(rng.integers(-11, 12))),
        'scale': float(rng.uniform(0.95, 1.05)),
        'shear': float(rng.uniform(-5, 5)),
        'brightness': float(rng.uniform(0.9, 1.1)),
        'contrast': float(rng.uniform(0.9, 1.1)),
        'noise': bool(rng.random() < 0.2),
        'noise_sigma': 0.01,
    }


def apply_octa_group_augmentation(arrays, params):
    out = []
    for arr in arrays:
        img = _array_to_pil(arr)
        if params['hflip']:
            img = ImageOps.mirror(img)
        if params['vflip']:
            img = ImageOps.flip(img)
        if TVF is not None:
            img = TVF.affine(
                img, angle=params['angle'], translate=params['translate'], scale=params['scale'],
                shear=[params['shear'], 0.0], interpolation=InterpolationMode.BICUBIC, fill=0
            )
        else:
            img = img.rotate(params['angle'], resample=Image.Resampling.BICUBIC, fillcolor=0)
            tx, ty = params['translate']
            img = img.transform(img.size, Image.Transform.AFFINE, (1, 0, -tx, 0, 1, -ty),
                                resample=Image.Resampling.BICUBIC, fillcolor=0)
        img = ImageEnhance.Brightness(img).enhance(params['brightness'])
        img = ImageEnhance.Contrast(img).enhance(params['contrast'])
        aug = _pil_to_array(img)
        if params['noise']:
            aug = np.clip(aug + np.random.normal(0, params['noise_sigma'], size=aug.shape), 0, 1)
        out.append(aug.astype(np.float32))
    return out


class ClinicalPreprocessor:
    def __init__(self, clinical_cols, specs=None):
        self.clinical_cols = list(clinical_cols)
        self.specs = dict(specs) if specs is not None else None
        self.continuous_stats = {}
        self.category_maps = {}
        self.ordinal_cardinalities = {}

    @staticmethod
    def _encode_ordinal_value(value):
        if pd.isna(value):
            return np.nan
        if isinstance(value, str):
            return SMOKING_MAP.get(canonical_feature_name(value), np.nan)
        return int(value)

    def transform_row(self, row):
        values, mask = [], []
        for col in self.clinical_cols:
            value = row[col] if col in row else np.nan
            if pd.isna(value):
                values.append(0.0)
                mask.append(0.0)
                continue
            kind = self.specs[col]
            if kind == 'continuous':
                value = float(pd.to_numeric(value, errors='coerce'))
                stats = self.continuous_stats[col]
                encoded = (value - stats['mean']) / stats['std'] if np.isfinite(value) else 0.0
            elif kind == 'ordinal':
                encoded = self._encode_ordinal_value(value)
                encoded = 0.0 if pd.isna(encoded) else float(encoded)
            else:
                mapping = self.category_maps.get(col, {})
                encoded = mapping.get(value, mapping.get(str(value), value))
                if isinstance(encoded, str):
                    encoded = SEX_MAP.get(canonical_feature_name(encoded), 0)
                encoded = float(encoded)
            values.append(float(encoded))
            mask.append(1.0)
        return np.asarray(values, dtype=np.float32), np.asarray(mask, dtype=np.float32)

    def transform_dataframe(self, df):
        vals, masks = [], []
        for _, row in df.iterrows():
            v, m = self.transform_row(row)
            vals.append(v)
            masks.append(m)
        return np.vstack(vals).astype(np.float32), np.vstack(masks).astype(np.float32)

    @property
    def ordinal_indices(self):
        return [i for i, c in enumerate(self.clinical_cols) if self.specs.get(c) == 'ordinal']

    @classmethod
    def from_dict(cls, state):
        obj = cls(state['clinical_cols'], specs=state['specs'])
        obj.continuous_stats = state.get('continuous_stats', {})
        obj.category_maps = state.get('category_maps', {})
        obj.ordinal_cardinalities = state.get('ordinal_cardinalities', {})
        return obj


if nn is not None:
    class ClinicalMaskTokenImputer(nn.Module):
        def __init__(self, num_features, ordinal_indices=None, ordinal_cardinalities=None):
            super().__init__()
            self.num_features = int(num_features)
            self.missing_tokens = nn.Parameter(torch.zeros(self.num_features))
            self.ordinal_indices = [int(i) for i in (ordinal_indices or [])]
            self.ordinal_embeddings = nn.ModuleDict()
            ordinal_cardinalities = ordinal_cardinalities or {}
            for idx in self.ordinal_indices:
                card = int(ordinal_cardinalities.get(idx, 3))
                self.ordinal_embeddings[str(idx)] = nn.Embedding(card, 1)

        def forward(self, values, mask):
            x = values.float().clone()
            mask = mask.float()
            for idx in self.ordinal_indices:
                if str(idx) in self.ordinal_embeddings:
                    codes = values[:, idx].round().long().clamp(min=0, max=self.ordinal_embeddings[str(idx)].num_embeddings - 1)
                    x[:, idx] = self.ordinal_embeddings[str(idx)](codes).squeeze(-1)
            tokens = self.missing_tokens.view(1, -1).expand_as(x)
            return torch.where(mask > 0.5, x, tokens)
else:
    ClinicalMaskTokenImputer = None


def make_clinical_mask_token_imputer(clinical_preprocessor):
    if ClinicalMaskTokenImputer is None:
        raise ImportError('PyTorch is required for ClinicalMaskTokenImputer')
    ordinal_cardinalities = {
        i: clinical_preprocessor.ordinal_cardinalities.get(col, 3)
        for i, col in enumerate(clinical_preprocessor.clinical_cols)
        if clinical_preprocessor.specs.get(col) == 'ordinal'
    }
    return ClinicalMaskTokenImputer(
        num_features=len(clinical_preprocessor.clinical_cols),
        ordinal_indices=list(ordinal_cardinalities.keys()),
        ordinal_cardinalities=ordinal_cardinalities,
    )


class RastaPhase2Dataset(Dataset):
    def __init__(self, df, clinical_preprocessor, layer_stats, train=False, image_size=IMAGE_SIZE,
                 clinical_mask_dropout=0.10, mask_path_cols=None, seed=PHASE2_SEED):
        if torch is None:
            raise ImportError('PyTorch is required to instantiate RastaPhase2Dataset')
        self.df = df.reset_index(drop=True).copy()
        self.clinical_preprocessor = clinical_preprocessor
        self.layer_stats = layer_stats
        self.train = bool(train)
        self.image_size = int(image_size)
        self.clinical_mask_dropout = float(clinical_mask_dropout)
        self.mask_path_cols = mask_path_cols or {'vessel_mask': None, 'faz_mask': None, 'flow_mask': None}
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.df)

    def _load_layer(self, row, layer):
        return preprocess_octa_image(row[LAYER_PATH_COLS[layer]], image_size=self.image_size, use_clahe=True)

    def _standardize_layer(self, arr, layer):
        mean = float(self.layer_stats[layer]['mean'])
        std = float(self.layer_stats[layer]['std'])
        return ((arr - mean) / max(std, 1e-6)).astype(np.float32)

    def _load_optional_mask(self, row, key):
        col = self.mask_path_cols.get(key)
        path = row[col] if col and col in row and pd.notna(row[col]) else None
        arr = preprocess_mask_image(path, image_size=self.image_size)
        return torch.from_numpy(arr[None, :, :].astype(np.float32))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        arrays = [self._load_layer(row, layer) for layer in ['sup', 'deep', 'cc']]
        if self.train:
            arrays = apply_octa_group_augmentation(arrays, sample_octa_augmentation_params(self.rng))
        sup, deep, cc = [self._standardize_layer(arr, layer) for arr, layer in zip(arrays, ['sup', 'deep', 'cc'])]
        clinical_values, clinical_mask = self.clinical_preprocessor.transform_row(row)
        if self.train and self.clinical_mask_dropout > 0:
            observed = np.where(clinical_mask > 0.5)[0]
            if len(observed):
                drop = observed[self.rng.random(len(observed)) < self.clinical_mask_dropout]
                clinical_values[drop] = 0.0
                clinical_mask[drop] = 0.0
        return {
            'sup_image': torch.from_numpy(sup[None, :, :]),
            'deep_image': torch.from_numpy(deep[None, :, :]),
            'cc_image': torch.from_numpy(cc[None, :, :]),
            'clinical_features': torch.from_numpy(clinical_values.astype(np.float32)),
            'clinical_mask': torch.from_numpy(clinical_mask.astype(np.float32)),
            'vessel_mask': self._load_optional_mask(row, 'vessel_mask'),
            'faz_mask': self._load_optional_mask(row, 'faz_mask'),
            'flow_mask': self._load_optional_mask(row, 'flow_mask'),
            'label': torch.tensor(int(row['label']), dtype=torch.long),
            'patient_id': str(row['patient_id']),
            'eye': str(row['eye']),
            'fold': int(row['fold']) if 'fold' in row else -1,
        }


def make_phase2_datasets_for_fold(fold_df, fold, clinical_preprocessor, layer_stats, **dataset_kwargs):
    train_df = fold_df[fold_df['fold'] != fold].copy()
    val_df = fold_df[fold_df['fold'] == fold].copy()
    validate_no_patient_overlap_between_splits(train_df, val_df)
    train_ds = RastaPhase2Dataset(train_df, clinical_preprocessor, layer_stats, train=True, **dataset_kwargs)
    val_ds = RastaPhase2Dataset(val_df, clinical_preprocessor, layer_stats, train=False, **dataset_kwargs)
    return train_ds, val_ds


def _load_json(path):
    return json.loads(Path(path).read_text())


def _require_phase_artifacts(paths_by_name):
    missing = {name: str(path) for name, path in paths_by_name.items() if not Path(path).exists()}
    if missing:
        msg = [
            'Cannot fast-load Phase 3 prerequisites because required Phase 2 files are missing:',
            *[f'  - {name}: {path}' for name, path in missing.items()],
            '',
            'Restore these existing Phase 2 artifacts, or run the Phase 2 build-artifacts cell once, then retry.',
        ]
        raise RuntimeError('\n'.join(msg))


phase1_manifest_path = OUTPUT_DIR / 'manifest_model_ready_with_clinical.csv'
phase1_folds_path = OUTPUT_DIR / 'preliminary_patient_level_folds.csv'
phase2_manifest_path = PHASE2_OUTPUT_DIR / 'phase2_manifest_qc_pass.csv'
phase2_folds_path = PHASE2_OUTPUT_DIR / 'phase2_patient_stratified_folds.csv'
phase2_cols_path = PHASE2_OUTPUT_DIR / 'phase2_clinical_columns.json'
phase2_stats_all_path = PHASE2_OUTPUT_DIR / 'octa_train_stats_all_folds.json'
phase2_qc_log_path = PHASE2_OUTPUT_DIR / 'phase2_image_qc_full_log.csv'
phase2_exclusions_path = PHASE2_OUTPUT_DIR / 'phase2_image_qc_exclusions.csv'

# Phase 3 consumes only the QC-passed fold manifest and per-fold OCTA statistics.
# Phase 1 and clinical artifacts are useful summary state, but must not block Phase 3.
required = {
    'Phase 2 patient folds': phase2_folds_path,
}
for fold in range(PHASE2_N_SPLITS):
    required[f'Phase 2 OCTA stats fold {fold}'] = PHASE2_OUTPUT_DIR / f'octa_train_stats_fold{fold}.json'
_require_phase_artifacts(required)

manifest_clin = pd.read_csv(phase1_manifest_path) if phase1_manifest_path.exists() else pd.DataFrame()
fold_df = pd.read_csv(phase1_folds_path) if phase1_folds_path.exists() else None
phase2_manifest = pd.read_csv(phase2_manifest_path) if phase2_manifest_path.exists() else pd.DataFrame()
phase2_folds = pd.read_csv(phase2_folds_path)
phase2_qc_log = pd.read_csv(phase2_qc_log_path) if phase2_qc_log_path.exists() else pd.DataFrame()
phase2_exclusions = pd.read_csv(phase2_exclusions_path) if phase2_exclusions_path.exists() else pd.DataFrame()

clinical_cols_phase2 = _load_json(phase2_cols_path).get('clinical_cols', []) if phase2_cols_path.exists() else []
clinical_cols = default_clinical_columns(manifest_clin) if not manifest_clin.empty else []
clinical_preprocessors = {
    fold: ClinicalPreprocessor.from_dict(_load_json(PHASE2_OUTPUT_DIR / f'clinical_preprocessor_fold{fold}.json'))
    for fold in range(PHASE2_N_SPLITS)
    if (PHASE2_OUTPUT_DIR / f'clinical_preprocessor_fold{fold}.json').exists()
}
layer_stats_by_fold = {
    fold: _load_json(PHASE2_OUTPUT_DIR / f'octa_train_stats_fold{fold}.json')
    for fold in range(PHASE2_N_SPLITS)
}
# Also parse aggregate stats in case later cells inspect it directly.
octa_train_stats_all_folds = (
    {int(fold): stats for fold, stats in _load_json(phase2_stats_all_path).items()}
    if phase2_stats_all_path.exists() else dict(layer_stats_by_fold)
)

validate_patient_folds(phase2_folds, n_splits=PHASE2_N_SPLITS)
seed_everything(PHASE2_SEED)

print('Fast reload complete. Existing Phase 1/2 outputs were read only; nothing was regenerated.')
print('manifest_clin:', manifest_clin.shape, '(optional summary)')
print('phase2_manifest:', phase2_manifest.shape, '(optional summary)')
print('phase2_folds:', phase2_folds.shape)
print('clinical preprocessors:', sorted(clinical_preprocessors.keys()))
print('layer stats folds:', sorted(layer_stats_by_fold.keys()))
print('Next: run Phase 3 configuration cell and continue.')

In [ ]:
# ============================================================
# Phase 3 configuration: OCTA-SimCLR pretraining
# ============================================================

def resolve_phase3_output_dir():
    """Reuse an existing expanded-run directory; default new runs to the 150-epoch name."""
    candidates = [
        Path('outputs/phase3_octa_simclr_8class_150epochs'),
        Path('outputs/phase3_octa_simclr_8class_120epochs'),
    ]
    existing = [path for path in candidates if path.is_dir()]
    if not existing:
        return candidates[0]
    if len(existing) == 1:
        return existing[0]

    # If both names exist, use the directory with the strongest resume evidence.
    def resume_score(path):
        last_checkpoints = list(path.glob('simclr_*_last.pt'))
        latest_checkpoint = max((p.stat().st_mtime for p in last_checkpoints), default=0.0)
        prefer_150_on_tie = int(path.name.endswith('_150epochs'))
        return len(last_checkpoints), latest_checkpoint, prefer_150_on_tie

    return max(existing, key=resume_score)


PHASE3_OUTPUT_DIR = ensure_dir(resolve_phase3_output_dir())
PHASE3_SEED = 42
PHASE3_IMAGE_SIZE = IMAGE_SIZE
PHASE3_LAYERS = ['sup', 'deep', 'cc']

# Publication-conservative default: pretrain within each downstream training fold only.
# Optional transductive/self-supervised mode: set to 'all_qc_pass' to use all QC-passed OCTA images label-free.
PHASE3_PRETRAIN_SCOPE = 'fold_train'  # {'fold_train', 'all_qc_pass'}

# Expensive CUDA job. True launches Phase 3 on the CUDA GPU machine.
RUN_PHASE3_PRETRAINING = True
PHASE3_REQUIRE_CUDA = True
PHASE3_FOLDS_TO_RUN = list(range(PHASE2_N_SPLITS))  # used when PHASE3_PRETRAIN_SCOPE == 'fold_train'
PHASE3_LAYERS_TO_RUN = PHASE3_LAYERS

# SimCLR hyperparameters for the expanded eight-class run. This uses a fresh output directory.
PHASE3_EPOCHS = 150
PHASE3_BATCH_SIZE = 128
PHASE3_GRAD_ACCUM_STEPS = 1          # increase if GPU memory requires smaller physical batches
PHASE3_NUM_WORKERS = 0
PHASE3_TEMPERATURE = 0.07
PHASE3_LR = 0.03  # lowered after AMP overflow/no-update preflight; was 0.1
PHASE3_MOMENTUM = 0.9
PHASE3_WEIGHT_DECAY = 1e-4
PHASE3_USE_AMP = False  # disabled after AMP produced inf gradients and skipped optimizer steps
PHASE3_IMAGENET_INIT = True
PHASE3_CHECKPOINT_EVERY = 10
PHASE3_RESUME_PRETRAINING = True       # auto-load *_last.pt and continue from next epoch
PHASE3_SAVE_LAST_EVERY_EPOCH = True    # minimize lost work if notebook/runtime stops
PHASE3_SHOW_PROGRESS = False             # keep tqdm/widget progress disabled unless explicitly wanted
PHASE3_SHOW_BATCH_PROGRESS = False       # keep notebook output compact; rely on console epoch heartbeat by default
PHASE3_PROGRESS_UPDATE_EVERY = 10        # used only if batch progress is explicitly re-enabled
PHASE3_LOG_EVERY_N_EPOCHS = 1            # console heartbeat: print every completed epoch
PHASE3_LOG_FIRST_N_EPOCHS = 3            # retained for compatibility; with LOG_EVERY_N_EPOCHS=1 all epochs print

# Safety diagnostics for the flat-loss failure mode: ln(2N-1) means random SimCLR behavior.
RUN_PHASE3_PREFLIGHT_DIAGNOSTICS = True
PHASE3_ABORT_ON_PREFLIGHT_FAILURE = True
PHASE3_ABORT_FLAT_LOSS_AFTER_EPOCH = 10  # abort a layer-run if still at random baseline this late; set None to disable
PHASE3_FLAT_LOSS_TOLERANCE = 0.02
PHASE3_MAX_GRAD_NORM = 1.0  # finite-gradient clipping; set None to disable

if torch is None:
    raise ImportError('PyTorch is required for Phase 3 OCTA-SimCLR pretraining.')

import math
import time
from PIL import ImageFilter
from torch.utils.data import DataLoader

try:
    from tqdm.auto import tqdm
except Exception:
    class _NoOpProgress:
        """Minimal tqdm-compatible fallback if tqdm is unavailable."""
        def __init__(self, iterable=None, total=None, desc=None, unit=None, **kwargs):
            self.iterable = iterable
            self.total = total
            self.desc = desc or 'progress'
            self.unit = unit or 'it'
            self.n = 0
            self.disable = kwargs.get('disable', False)
            if iterable is None and not self.disable:
                print(f'{self.desc}: started' + (f' (total={total} {self.unit})' if total is not None else ''))

        def __iter__(self):
            if self.iterable is None:
                return iter(())
            if not self.disable:
                print(f'{self.desc}: started')
            for item in self.iterable:
                yield item
            if not self.disable:
                print(f'{self.desc}: done')

        def update(self, n=1):
            self.n += n
            if not self.disable and self.total is not None:
                print(f'{self.desc}: {self.n}/{self.total} {self.unit}')

        def set_postfix(self, *args, **kwargs):
            return None

        def set_description_str(self, desc):
            self.desc = desc

        def close(self):
            if not self.disable:
                print(f'{self.desc}: done')

    def tqdm(iterable=None, *args, **kwargs):
        return _NoOpProgress(iterable=iterable, **kwargs)


def format_duration(seconds):
    seconds = max(0.0, float(seconds))
    if seconds < 60:
        return f'{seconds:.1f}s'
    minutes, sec = divmod(seconds, 60)
    if minutes < 60:
        return f'{int(minutes)}m {sec:04.1f}s'
    hours, minutes = divmod(minutes, 60)
    return f'{int(hours)}h {int(minutes):02d}m {sec:04.1f}s'


def phase3_progress(iterable=None, **kwargs):
    """tqdm wrapper controlled by PHASE3_SHOW_PROGRESS."""
    kwargs.setdefault('dynamic_ncols', True)
    kwargs['disable'] = bool(kwargs.get('disable', False) or not PHASE3_SHOW_PROGRESS)
    return tqdm(iterable, **kwargs)


def should_print_phase3_epoch(epoch, total_epochs):
    """Console heartbeat policy for end-of-epoch logs."""
    epoch = int(epoch)
    total_epochs = int(total_epochs)
    if epoch <= int(PHASE3_LOG_FIRST_N_EPOCHS):
        return True
    if epoch == total_epochs:
        return True
    return (epoch % max(1, int(PHASE3_LOG_EVERY_N_EPOCHS))) == 0


def phase3_mean_epoch_seconds(history):
    vals = []
    for rec in history or []:
        try:
            sec = float(rec.get('seconds'))
        except Exception:
            continue
        if sec > 0 and sec == sec and sec != float('inf'):
            vals.append(sec)
    return (sum(vals) / len(vals)) if vals else None

try:
    from torchvision import models
except Exception as e:
    models = None
    print('torchvision.models unavailable; ConvNeXt-Tiny cannot be constructed until this is fixed:', repr(e))

seed_everything(PHASE3_SEED)
print('PHASE3_OUTPUT_DIR =', PHASE3_OUTPUT_DIR.resolve())
print('RUN_PHASE3_PRETRAINING =', RUN_PHASE3_PRETRAINING)
print('PHASE3_REQUIRE_CUDA =', PHASE3_REQUIRE_CUDA)
print('PHASE3_PRETRAIN_SCOPE =', PHASE3_PRETRAIN_SCOPE)
print('PHASE3_RESUME_PRETRAINING =', PHASE3_RESUME_PRETRAINING)
print('PHASE3_SAVE_LAST_EVERY_EPOCH =', PHASE3_SAVE_LAST_EVERY_EPOCH)

In [ ]:
# ============================================================
# Phase 3 positive-pair dataset and conservative OCTA augmentations
# ============================================================


def sample_simclr_augmentation_params(rng=np.random):
    """Conservative SimCLR augmentation that preserves OCTA vascular morphology."""
    return {
        'hflip': bool(rng.random() < 0.5),
        'rotate': bool(rng.random() < 0.4),
        'angle': float(rng.uniform(-15.0, 15.0)),
        'crop_scale': float(rng.uniform(0.70, 1.00)),
        'blur': bool(rng.random() < 0.3),
        'blur_sigma': float(rng.uniform(0.1, 2.0)),
        'brightness': float(rng.uniform(0.90, 1.10)),
        'contrast': float(rng.uniform(0.90, 1.10)),
    }


def apply_simclr_augmentation(arr01, params, image_size=PHASE3_IMAGE_SIZE):
    """Apply one independently sampled SimCLR view transform to a preprocessed [0,1] OCTA array."""
    img = _array_to_pil(arr01)

    # Random resized crop, implemented before other transforms.
    w, h = img.size
    crop_size = max(1, int(round(min(w, h) * params['crop_scale'])))
    if crop_size < min(w, h):
        left = int(np.random.randint(0, w - crop_size + 1))
        top = int(np.random.randint(0, h - crop_size + 1))
        img = img.crop((left, top, left + crop_size, top + crop_size))
    img = img.resize((image_size, image_size), resample=Image.Resampling.BICUBIC)

    if params['hflip']:
        img = ImageOps.mirror(img)
    if params['rotate']:
        img = img.rotate(params['angle'], resample=Image.Resampling.BICUBIC, fillcolor=0)
    if params['blur']:
        img = img.filter(ImageFilter.GaussianBlur(radius=params['blur_sigma']))

    img = ImageEnhance.Brightness(img).enhance(params['brightness'])
    img = ImageEnhance.Contrast(img).enhance(params['contrast'])
    return _pil_to_array(img).astype(np.float32)


def standardize_octa_array(arr01, layer, layer_stats):
    mean = float(layer_stats[layer]['mean']) if layer_stats and layer in layer_stats else 0.0
    std = float(layer_stats[layer]['std']) if layer_stats and layer in layer_stats else 1.0
    return ((arr01 - mean) / max(std, 1e-6)).astype(np.float32)


def simclr_tensor_from_path(path, layer, layer_stats, augment=True, image_size=PHASE3_IMAGE_SIZE):
    arr = preprocess_octa_image(path, image_size=image_size, use_clahe=True)
    if augment:
        arr = apply_simclr_augmentation(arr, sample_simclr_augmentation_params(), image_size=image_size)
    arr = standardize_octa_array(arr, layer, layer_stats)
    return torch.from_numpy(arr[None, :, :].astype(np.float32))


class RastaSimCLRLayerDataset(Dataset):
    """Layer-specific SimCLR Dataset using OD/OS as natural positives when available.

    For patients with both eyes: view1 and view2 are the same OCTA layer from opposite eyes.
    For unilateral patients: view1 and view2 are two independent augmentations of the same image.
    """
    def __init__(self, df, layer, layer_stats, image_size=PHASE3_IMAGE_SIZE):
        if layer not in LAYER_PATH_COLS:
            raise ValueError(f'Unknown OCTA layer: {layer}')
        self.df = df.reset_index(drop=True).copy()
        self.layer = layer
        self.path_col = LAYER_PATH_COLS[layer]
        self.layer_stats = layer_stats
        self.image_size = int(image_size)
        self.eye_path_by_patient = {}
        for patient_id, g in self.df.groupby('patient_id'):
            eye_map = {}
            for _, row in g.iterrows():
                if pd.notna(row.get(self.path_col)):
                    eye_map[str(row['eye'])] = str(row[self.path_col])
            self.eye_path_by_patient[str(patient_id)] = eye_map

    def __len__(self):
        return len(self.df)

    def _positive_path(self, row):
        patient_id = str(row['patient_id'])
        eye = str(row['eye'])
        eye_map = self.eye_path_by_patient.get(patient_id, {})
        opposite_eye = 'OS' if eye == 'OD' else 'OD'
        if opposite_eye in eye_map:
            return eye_map[opposite_eye], opposite_eye, 'bilateral_od_os'
        return str(row[self.path_col]), eye, 'self_augmented_unilateral'

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path1 = str(row[self.path_col])
        path2, eye2, positive_type = self._positive_path(row)
        x1 = simclr_tensor_from_path(path1, self.layer, self.layer_stats, augment=True, image_size=self.image_size)
        x2 = simclr_tensor_from_path(path2, self.layer, self.layer_stats, augment=True, image_size=self.image_size)
        return {
            'view1': x1,
            'view2': x2,
            'patient_id': str(row['patient_id']),
            'eye1': str(row['eye']),
            'eye2': str(eye2),
            'layer': self.layer,
            'positive_type': positive_type,
        }


def summarize_simclr_pairs(df):
    rows = []
    for layer, path_col in LAYER_PATH_COLS.items():
        n_rows = int(df[path_col].notna().sum())
        bilateral_eye_rows = 0
        unilateral_eye_rows = 0
        bilateral_patients = 0
        unilateral_patients = 0
        for _, g in df.groupby('patient_id'):
            eyes = set(g.loc[g[path_col].notna(), 'eye'].astype(str))
            if {'OD', 'OS'}.issubset(eyes):
                bilateral_patients += 1
                bilateral_eye_rows += int(g[path_col].notna().sum())
            elif len(eyes) == 1:
                unilateral_patients += 1
                unilateral_eye_rows += int(g[path_col].notna().sum())
        rows.append({
            'layer': layer,
            'eye_rows': n_rows,
            'bilateral_positive_eye_rows': bilateral_eye_rows,
            'self_augmented_eye_rows': unilateral_eye_rows,
            'bilateral_patients': bilateral_patients,
            'unilateral_patients': unilateral_patients,
        })
    return pd.DataFrame(rows)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [ ]:
# ============================================================
# Phase 3 ConvNeXt-Tiny SimCLR model and NT-Xent loss
# ============================================================


class ConvNeXtTinyBackbone(nn.Module):
    """ConvNeXt-Tiny feature extractor adapted to 1-channel OCTA input."""
    output_dim = 768

    def __init__(self, in_chans=1, imagenet_init=True):
        super().__init__()
        if models is None:
            raise ImportError('torchvision.models is required to build ConvNeXt-Tiny.')
        weights = None
        if imagenet_init:
            try:
                weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
            except Exception:
                weights = 'IMAGENET1K_V1'
        try:
            backbone = models.convnext_tiny(weights=weights)
        except Exception as e:
            print('Warning: failed to load ImageNet ConvNeXt-Tiny weights; falling back to random init:', repr(e))
            backbone = models.convnext_tiny(weights=None)
        self.features = backbone.features
        self._adapt_first_conv(in_chans=in_chans)

    def _adapt_first_conv(self, in_chans=1):
        stem_conv = self.features[0][0]
        if not isinstance(stem_conv, nn.Conv2d) or stem_conv.in_channels == in_chans:
            return
        new_conv = nn.Conv2d(
            in_chans,
            stem_conv.out_channels,
            kernel_size=stem_conv.kernel_size,
            stride=stem_conv.stride,
            padding=stem_conv.padding,
            dilation=stem_conv.dilation,
            groups=1,
            bias=stem_conv.bias is not None,
            padding_mode=stem_conv.padding_mode,
        )
        with torch.no_grad():
            if in_chans == 1 and stem_conv.weight.shape[1] == 3:
                new_conv.weight.copy_(stem_conv.weight.mean(dim=1, keepdim=True))
            else:
                nn.init.kaiming_normal_(new_conv.weight, mode='fan_out', nonlinearity='relu')
            if stem_conv.bias is not None:
                new_conv.bias.copy_(stem_conv.bias)
        self.features[0][0] = new_conv

    def forward_features(self, x):
        return self.features(x)

    def forward(self, x):
        fmap = self.forward_features(x)
        return fmap.mean(dim=(-2, -1))


class OCTASimCLR(nn.Module):
    """ConvNeXt-Tiny encoder plus 2-layer SimCLR projection head."""
    def __init__(self, projection_hidden=256, projection_dim=128, imagenet_init=True):
        super().__init__()
        self.encoder = ConvNeXtTinyBackbone(in_chans=1, imagenet_init=imagenet_init)
        self.projector = nn.Sequential(
            nn.Linear(self.encoder.output_dim, projection_hidden),
            nn.GELU(),
            nn.Linear(projection_hidden, projection_dim),
        )

    def forward(self, x, return_features=False):
        h = self.encoder(x)
        z = nn.functional.normalize(self.projector(h), dim=1)
        if return_features:
            return h, z
        return z


def nt_xent_loss(z1, z2, temperature=PHASE3_TEMPERATURE):
    """Normalized temperature-scaled cross entropy for paired batches."""
    if z1.shape != z2.shape:
        raise ValueError(f'z1 and z2 must have identical shapes, got {z1.shape} and {z2.shape}')
    batch_size = z1.size(0)
    if batch_size < 2:
        raise ValueError('NT-Xent requires batch_size >= 2')
    z = torch.cat([z1, z2], dim=0)
    sim = torch.matmul(z, z.T) / float(temperature)
    self_mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z.device)
    mask_value = torch.finfo(sim.dtype).min if sim.is_floating_point() else -1e9
    sim = sim.masked_fill(self_mask, mask_value)
    targets = torch.arange(2 * batch_size, device=z.device)
    targets = (targets + batch_size) % (2 * batch_size)
    return nn.functional.cross_entropy(sim, targets)


def make_grad_scaler(enabled):
    try:
        return torch.amp.GradScaler('cuda', enabled=enabled)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=enabled)


def autocast_context(device, enabled):
    try:
        return torch.amp.autocast(device_type=device.type, enabled=enabled)
    except Exception:
        return torch.cuda.amp.autocast(enabled=enabled)

In [ ]:
# ============================================================
# Phase 3 data selection, statistics, checkpoint paths, and training loop
# ============================================================


def get_phase3_pretraining_df(fold=None, scope=PHASE3_PRETRAIN_SCOPE):
    """Return unlabeled eye rows for SimCLR pretraining.

    scope='fold_train': only rows outside the held-out fold, avoiding validation/test image exposure.
    scope='all_qc_pass': all QC-passed rows, label-free transductive self-supervised pretraining.
    """
    if 'phase2_folds' not in globals():
        raise RuntimeError('Run Phase 2 artifact cell first so phase2_folds is available.')
    if scope == 'fold_train':
        if fold is None:
            raise ValueError('fold must be provided when scope="fold_train"')
        df = phase2_folds[phase2_folds['fold'] != fold].copy()
    elif scope == 'all_qc_pass':
        df = phase2_folds.copy()
    else:
        raise ValueError(f'Unknown PHASE3_PRETRAIN_SCOPE: {scope}')
    missing = [col for col in LAYER_PATH_COLS.values() if col not in df.columns]
    if missing:
        raise ValueError(f'Missing layer path columns for Phase 3: {missing}')
    return df.reset_index(drop=True)


def phase3_stats_cache_path(fold=None, scope=PHASE3_PRETRAIN_SCOPE):
    tag = f'{scope}_fold{fold}' if scope == 'fold_train' else scope
    return PHASE3_OUTPUT_DIR / f'octa_simclr_stats_{tag}.json'


def get_phase3_layer_stats(pretrain_df, fold=None, scope=PHASE3_PRETRAIN_SCOPE):
    if scope == 'fold_train' and 'layer_stats_by_fold' in globals() and fold in layer_stats_by_fold:
        return layer_stats_by_fold[fold]
    path = phase3_stats_cache_path(fold=fold, scope=scope)
    if path.exists():
        return json.loads(path.read_text())
    tag = f'{scope}_fold{fold}' if scope == 'fold_train' else scope
    try:
        stats = compute_layer_stats(
            pretrain_df,
            image_size=PHASE3_IMAGE_SIZE,
            show_progress=PHASE3_SHOW_PROGRESS,
            desc=f'Phase 3 stats {tag}',
        )
    except TypeError:
        # Backward-compatible if an older in-memory compute_layer_stats was not rerun.
        stats = compute_layer_stats(pretrain_df, image_size=PHASE3_IMAGE_SIZE)
    save_json(stats, path)
    return stats


def phase3_checkpoint_prefix(layer, fold=None, scope=PHASE3_PRETRAIN_SCOPE):
    tag = f'{scope}_fold{fold}' if scope == 'fold_train' else scope
    return PHASE3_OUTPUT_DIR / f'simclr_{tag}_{layer}'


def phase3_checkpoint_path(layer, fold=None, scope=PHASE3_PRETRAIN_SCOPE, kind='last'):
    if kind not in {'last', 'best'}:
        raise ValueError(f'Unknown checkpoint kind: {kind}')
    suffix = '_best.pt' if kind == 'best' else '_last.pt'
    return Path(str(phase3_checkpoint_prefix(layer, fold=fold, scope=scope)) + suffix)


def phase3_backbone_path(layer, fold=None, scope=PHASE3_PRETRAIN_SCOPE):
    return Path(str(phase3_checkpoint_prefix(layer, fold=fold, scope=scope)) + '_backbone.pt')


def phase3_torch_load(path, map_location=None):
    """Load full training checkpoints across PyTorch versions."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def phase3_find_resume_checkpoint(layer, fold=None, scope=PHASE3_PRETRAIN_SCOPE):
    path = phase3_checkpoint_path(layer, fold=fold, scope=scope, kind='last')
    return path if path.exists() else None


def phase3_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['cuda'] = torch.cuda.get_rng_state_all()
    return state


def phase3_restore_rng_state(state):
    if not state:
        return
    try:
        if 'python' in state:
            random.setstate(state['python'])
        if 'numpy' in state:
            np.random.set_state(state['numpy'])
        if 'torch' in state:
            torch.set_rng_state(state['torch'].cpu() if torch.is_tensor(state['torch']) else state['torch'])
        if torch.cuda.is_available() and 'cuda' in state:
            cuda_states = [x.cpu() if torch.is_tensor(x) else x for x in state['cuda']]
            torch.cuda.set_rng_state_all(cuda_states)
    except Exception as e:
        print('Warning: could not fully restore Phase 3 RNG state:', repr(e))


def phase3_nt_xent_random_baseline(batch_size):
    """Expected NT-Xent loss when positives are no more similar than negatives."""
    return float(math.log(max(2, 2 * int(batch_size) - 1)))


def phase3_grads_are_finite(model):
    """Return False if any trainable parameter has NaN/Inf gradients."""
    for p in model.parameters():
        if p.requires_grad and p.grad is not None and not torch.isfinite(p.grad).all():
            return False
    return True


def phase3_clip_gradients_if_needed(model):
    if PHASE3_MAX_GRAD_NORM is None:
        return None
    return float(torch.nn.utils.clip_grad_norm_(model.parameters(), float(PHASE3_MAX_GRAD_NORM)).detach().cpu())


def save_simclr_checkpoint(model, optimizer, scheduler, epoch, history, layer, fold, scope, layer_stats, is_best=False):
    checkpoint_path = phase3_checkpoint_path(layer, fold=fold, scope=scope, kind=('best' if is_best else 'last'))
    payload = {
        'epoch': int(epoch),
        'layer': layer,
        'fold': None if fold is None else int(fold),
        'scope': scope,
        'model_state_dict': model.state_dict(),
        'encoder_state_dict': model.encoder.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if optimizer is not None else None,
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'history': history,
        'layer_stats': layer_stats,
        'config': {
            'epochs': PHASE3_EPOCHS,
            'batch_size': PHASE3_BATCH_SIZE,
            'grad_accum_steps': PHASE3_GRAD_ACCUM_STEPS,
            'temperature': PHASE3_TEMPERATURE,
            'lr': PHASE3_LR,
            'momentum': PHASE3_MOMENTUM,
            'weight_decay': PHASE3_WEIGHT_DECAY,
            'imagenet_init': PHASE3_IMAGENET_INIT,
            'require_cuda': PHASE3_REQUIRE_CUDA,
            'max_grad_norm': PHASE3_MAX_GRAD_NORM,
            'resume_pretraining': PHASE3_RESUME_PRETRAINING,
            'save_last_every_epoch': PHASE3_SAVE_LAST_EVERY_EPOCH,
        },
        'rng_state': phase3_rng_state(),
    }
    torch.save(payload, checkpoint_path)
    return checkpoint_path


def save_simclr_backbone(model, layer, fold, scope, layer_stats, history):
    backbone_path = phase3_backbone_path(layer, fold=fold, scope=scope)
    payload = {
        'layer': layer,
        'fold': None if fold is None else int(fold),
        'scope': scope,
        'encoder_state_dict': model.encoder.state_dict(),
        'layer_stats': layer_stats,
        'history': history,
        'output_dim': model.encoder.output_dim,
        'note': 'Projection head discarded; load encoder_state_dict into ConvNeXtTinyBackbone for Phase 4.',
    }
    torch.save(payload, backbone_path)
    return backbone_path


def train_simclr_one_layer(layer, pretrain_df, layer_stats, fold=None, scope=PHASE3_PRETRAIN_SCOPE, resume_path=None):
    if len(pretrain_df) < 2:
        raise ValueError(f'Need at least 2 eye rows for SimCLR pretraining, got {len(pretrain_df)}')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if PHASE3_REQUIRE_CUDA and device.type != 'cuda':
        raise RuntimeError('Phase 3 is configured for the CUDA-enabled training machine, but CUDA is not available in this runtime.')
    amp_enabled = bool(PHASE3_USE_AMP and device.type == 'cuda')
    dataset = RastaSimCLRLayerDataset(pretrain_df, layer=layer, layer_stats=layer_stats, image_size=PHASE3_IMAGE_SIZE)
    generator = torch.Generator()
    generator.manual_seed(PHASE3_SEED)
    loader = DataLoader(
        dataset,
        batch_size=PHASE3_BATCH_SIZE,
        shuffle=True,
        num_workers=PHASE3_NUM_WORKERS,
        pin_memory=(device.type == 'cuda'),
        drop_last=(len(dataset) >= PHASE3_BATCH_SIZE),
        worker_init_fn=seed_worker,
        generator=generator,
    )
    if len(loader) == 0:
        raise ValueError('DataLoader has zero batches; reduce PHASE3_BATCH_SIZE or add more data.')

    model = OCTASimCLR(imagenet_init=PHASE3_IMAGENET_INIT).to(device)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=PHASE3_LR, momentum=PHASE3_MOMENTUM, weight_decay=PHASE3_WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE3_EPOCHS, eta_min=0.0)
    scaler = make_grad_scaler(amp_enabled)
    start_epoch = 1
    history = []
    best_loss = float('inf')

    if resume_path is None and PHASE3_RESUME_PRETRAINING:
        resume_path = phase3_find_resume_checkpoint(layer, fold=fold, scope=scope)
    if resume_path is not None and Path(resume_path).exists():
        ckpt = phase3_torch_load(resume_path, map_location=device)
        if ckpt.get('layer') != layer or ckpt.get('scope') != scope or ckpt.get('fold') != (None if fold is None else int(fold)):
            raise ValueError(
                f'Resume checkpoint metadata mismatch for layer={layer} fold={fold} scope={scope}: {resume_path}'
            )
        model.load_state_dict(ckpt['model_state_dict'])
        if ckpt.get('optimizer_state_dict') is not None:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if ckpt.get('scheduler_state_dict') is not None:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        phase3_restore_rng_state(ckpt.get('rng_state'))
        history = list(ckpt.get('history', []))
        start_epoch = int(ckpt.get('epoch', 0)) + 1
        if history:
            best_loss = min(float(h['loss']) for h in history)
        if start_epoch > PHASE3_EPOCHS:
            print(f'Checkpoint already reached epoch {start_epoch - 1}/{PHASE3_EPOCHS}: {resume_path}')
        else:
            print(f'Resumed {layer} SimCLR from {resume_path} at epoch {start_epoch}/{PHASE3_EPOCHS}')

    print(f'Phase 3 SimCLR | layer={layer} fold={fold} scope={scope} device={device} n={len(dataset)} batches={len(loader)}')
    history_path = Path(str(phase3_checkpoint_prefix(layer, fold=fold, scope=scope)) + '_history.csv')
    run_t0 = time.time()
    remaining_epochs = max(0, PHASE3_EPOCHS - start_epoch + 1)
    epoch_bar = phase3_progress(
        range(start_epoch, PHASE3_EPOCHS + 1),
        total=remaining_epochs,
        desc=f'Phase 3 {scope} fold={fold} layer={layer}',
        unit='epoch',
        leave=True,
    )
    for epoch in epoch_bar:
        model.train()
        t0 = time.time()
        running_loss = 0.0
        optimizer.zero_grad(set_to_none=True)
        valid_steps = 0
        effective_batch_size = None

        batch_bar = phase3_progress(
            enumerate(loader, start=1),
            total=len(loader),
            desc=f'{layer} epoch {epoch:03d}/{PHASE3_EPOCHS}',
            unit='batch',
            leave=False,
            disable=(not PHASE3_SHOW_BATCH_PROGRESS),
        )
        for step, batch in batch_bar:
            x1 = batch['view1'].to(device, non_blocking=True)
            x2 = batch['view2'].to(device, non_blocking=True)
            if x1.size(0) < 2:
                continue
            if effective_batch_size is None:
                effective_batch_size = int(x1.size(0))
            with autocast_context(device, amp_enabled):
                z1 = model(x1)
                z2 = model(x2)
                loss = nt_xent_loss(z1, z2, temperature=PHASE3_TEMPERATURE)
                scaled_loss = loss / max(1, PHASE3_GRAD_ACCUM_STEPS)
            scaler.scale(scaled_loss).backward()
            running_loss += float(loss.detach().cpu())
            valid_steps += 1
            if PHASE3_SHOW_BATCH_PROGRESS and (step == 1 or step % PHASE3_PROGRESS_UPDATE_EVERY == 0 or step == len(loader)):
                batch_bar.set_postfix(loss=f'{running_loss / max(1, valid_steps):.4f}', refresh=False)
            if step % PHASE3_GRAD_ACCUM_STEPS == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                if not phase3_grads_are_finite(model):
                    raise RuntimeError(
                        f'Phase 3 abort: non-finite gradients at layer={layer} fold={fold} '
                        f'epoch={epoch} step={step}. Try lower LR/batch size or keep PHASE3_USE_AMP=False.'
                    )
                phase3_clip_gradients_if_needed(model)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        batch_bar.close()

        scheduler.step()
        mean_loss = running_loss / max(1, valid_steps)
        random_baseline_loss = phase3_nt_xent_random_baseline(effective_batch_size or PHASE3_BATCH_SIZE)
        epoch_seconds = time.time() - t0
        total_seconds = time.time() - run_t0
        rec = {
            'epoch': int(epoch),
            'loss': float(mean_loss),
            'lr': float(optimizer.param_groups[0]['lr']),
            'random_baseline_loss': float(random_baseline_loss),
            'loss_minus_random_baseline': float(mean_loss - random_baseline_loss),
            'seconds': float(epoch_seconds),
            'total_seconds': float(total_seconds),
        }
        history.append(rec)
        pd.DataFrame(history).to_csv(history_path, index=False)  # exact per-epoch log, updated every epoch without spamming notebook output
        is_best = mean_loss < best_loss
        if is_best:
            best_loss = mean_loss
            save_simclr_checkpoint(model, optimizer, scheduler, epoch, history, layer, fold, scope, layer_stats, is_best=True)
        if PHASE3_SAVE_LAST_EVERY_EPOCH or epoch == PHASE3_EPOCHS or epoch % PHASE3_CHECKPOINT_EVERY == 0:
            save_simclr_checkpoint(model, optimizer, scheduler, epoch, history, layer, fold, scope, layer_stats, is_best=False)
        if (PHASE3_ABORT_FLAT_LOSS_AFTER_EPOCH is not None
                and epoch >= int(PHASE3_ABORT_FLAT_LOSS_AFTER_EPOCH)
                and abs(mean_loss - random_baseline_loss) <= PHASE3_FLAT_LOSS_TOLERANCE):
            save_simclr_checkpoint(model, optimizer, scheduler, epoch, history, layer, fold, scope, layer_stats, is_best=False)
            raise RuntimeError(
                f'Phase 3 abort: {layer} fold={fold} loss {mean_loss:.4f} remains near random baseline '
                f'{random_baseline_loss:.4f} at epoch {epoch}. Check preflight diagnostics before continuing.'
            )
        epoch_bar.set_postfix(
            loss=f'{mean_loss:.4f}',
            lr=f"{rec['lr']:.5f}",
            epoch_time=format_duration(epoch_seconds),
            total_time=format_duration(total_seconds),
            refresh=True,
        )
        if should_print_phase3_epoch(epoch, PHASE3_EPOCHS):
            print(
                f"[{layer}] epoch {epoch:03d}/{PHASE3_EPOCHS} "
                f"loss={mean_loss:.4f} baseline={random_baseline_loss:.4f} lr={rec['lr']:.5f} "
                f"epoch_time={format_duration(epoch_seconds)} total_time={format_duration(total_seconds)}"
            )
    epoch_bar.close()

    backbone_path = save_simclr_backbone(model, layer, fold, scope, layer_stats, history)
    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False)
    return {'layer': layer, 'fold': fold, 'scope': scope, 'best_loss': best_loss, 'backbone_path': str(backbone_path), 'history_path': str(history_path)}

In [ ]:
import gc, torch 
gc.collect() 
torch.cuda.empty_cache() 
torch.cuda.reset_peak_memory_stats()

In [ ]:
# ============================================================
# Phase 3 data selection, statistics, checkpoint paths, and training loop
# ============================================================


def get_phase3_pretraining_df(fold=None, scope=PHASE3_PRETRAIN_SCOPE):
    """Return unlabeled eye rows for SimCLR pretraining.

    scope='fold_train': only rows outside the held-out fold, avoiding validation/test image exposure.
    scope='all_qc_pass': all QC-passed rows, label-free transductive self-supervised pretraining.
    """
    if 'phase2_folds' not in globals():
        raise RuntimeError('Run Phase 2 artifact cell first so phase2_folds is available.')
    if scope == 'fold_train':
        if fold is None:
            raise ValueError('fold must be provided when scope="fold_train"')
        df = phase2_folds[phase2_folds['fold'] != fold].copy()
    elif scope == 'all_qc_pass':
        df = phase2_folds.copy()
    else:
        raise ValueError(f'Unknown PHASE3_PRETRAIN_SCOPE: {scope}')
    missing = [col for col in LAYER_PATH_COLS.values() if col not in df.columns]
    if missing:
        raise ValueError(f'Missing layer path columns for Phase 3: {missing}')
    return df.reset_index(drop=True)


def phase3_stats_cache_path(fold=None, scope=PHASE3_PRETRAIN_SCOPE):
    tag = f'{scope}_fold{fold}' if scope == 'fold_train' else scope
    return PHASE3_OUTPUT_DIR / f'octa_simclr_stats_{tag}.json'


def get_phase3_layer_stats(pretrain_df, fold=None, scope=PHASE3_PRETRAIN_SCOPE):
    if scope == 'fold_train' and 'layer_stats_by_fold' in globals() and fold in layer_stats_by_fold:
        return layer_stats_by_fold[fold]
    path = phase3_stats_cache_path(fold=fold, scope=scope)
    if path.exists():
        return json.loads(path.read_text())
    tag = f'{scope}_fold{fold}' if scope == 'fold_train' else scope
    try:
        stats = compute_layer_stats(
            pretrain_df,
            image_size=PHASE3_IMAGE_SIZE,
            show_progress=PHASE3_SHOW_PROGRESS,
            desc=f'Phase 3 stats {tag}',
        )
    except TypeError:
        # Backward-compatible if an older in-memory compute_layer_stats was not rerun.
        stats = compute_layer_stats(pretrain_df, image_size=PHASE3_IMAGE_SIZE)
    save_json(stats, path)
    return stats


def phase3_checkpoint_prefix(layer, fold=None, scope=PHASE3_PRETRAIN_SCOPE):
    tag = f'{scope}_fold{fold}' if scope == 'fold_train' else scope
    return PHASE3_OUTPUT_DIR / f'simclr_{tag}_{layer}'


def phase3_checkpoint_path(layer, fold=None, scope=PHASE3_PRETRAIN_SCOPE, kind='last'):
    if kind not in {'last', 'best'}:
        raise ValueError(f'Unknown checkpoint kind: {kind}')
    suffix = '_best.pt' if kind == 'best' else '_last.pt'
    return Path(str(phase3_checkpoint_prefix(layer, fold=fold, scope=scope)) + suffix)


def phase3_backbone_path(layer, fold=None, scope=PHASE3_PRETRAIN_SCOPE):
    return Path(str(phase3_checkpoint_prefix(layer, fold=fold, scope=scope)) + '_backbone.pt')


def phase3_torch_load(path, map_location=None):
    """Load full training checkpoints across PyTorch versions."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def phase3_find_resume_checkpoint(layer, fold=None, scope=PHASE3_PRETRAIN_SCOPE):
    path = phase3_checkpoint_path(layer, fold=fold, scope=scope, kind='last')
    return path if path.exists() else None


def phase3_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['cuda'] = torch.cuda.get_rng_state_all()
    return state


def phase3_restore_rng_state(state):
    if not state:
        return
    try:
        if 'python' in state:
            random.setstate(state['python'])
        if 'numpy' in state:
            np.random.set_state(state['numpy'])
        if 'torch' in state:
            torch.set_rng_state(state['torch'].cpu() if torch.is_tensor(state['torch']) else state['torch'])
        if torch.cuda.is_available() and 'cuda' in state:
            cuda_states = [x.cpu() if torch.is_tensor(x) else x for x in state['cuda']]
            torch.cuda.set_rng_state_all(cuda_states)
    except Exception as e:
        print('Warning: could not fully restore Phase 3 RNG state:', repr(e))


def phase3_nt_xent_random_baseline(batch_size):
    """Expected NT-Xent loss when positives are no more similar than negatives."""
    return float(math.log(max(2, 2 * int(batch_size) - 1)))


def phase3_grads_are_finite(model):
    """Return False if any trainable parameter has NaN/Inf gradients."""
    for p in model.parameters():
        if p.requires_grad and p.grad is not None and not torch.isfinite(p.grad).all():
            return False
    return True


def phase3_clip_gradients_if_needed(model):
    if PHASE3_MAX_GRAD_NORM is None:
        return None
    return float(torch.nn.utils.clip_grad_norm_(model.parameters(), float(PHASE3_MAX_GRAD_NORM)).detach().cpu())


def save_simclr_checkpoint(model, optimizer, scheduler, epoch, history, layer, fold, scope, layer_stats, is_best=False):
    checkpoint_path = phase3_checkpoint_path(layer, fold=fold, scope=scope, kind=('best' if is_best else 'last'))
    payload = {
        'epoch': int(epoch),
        'layer': layer,
        'fold': None if fold is None else int(fold),
        'scope': scope,
        'model_state_dict': model.state_dict(),
        'encoder_state_dict': model.encoder.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if optimizer is not None else None,
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'history': history,
        'layer_stats': layer_stats,
        'config': {
            'epochs': PHASE3_EPOCHS,
            'batch_size': PHASE3_BATCH_SIZE,
            'grad_accum_steps': PHASE3_GRAD_ACCUM_STEPS,
            'temperature': PHASE3_TEMPERATURE,
            'lr': PHASE3_LR,
            'momentum': PHASE3_MOMENTUM,
            'weight_decay': PHASE3_WEIGHT_DECAY,
            'imagenet_init': PHASE3_IMAGENET_INIT,
            'require_cuda': PHASE3_REQUIRE_CUDA,
            'max_grad_norm': PHASE3_MAX_GRAD_NORM,
            'resume_pretraining': PHASE3_RESUME_PRETRAINING,
            'save_last_every_epoch': PHASE3_SAVE_LAST_EVERY_EPOCH,
        },
        'rng_state': phase3_rng_state(),
    }
    torch.save(payload, checkpoint_path)
    return checkpoint_path


def save_simclr_backbone(model, layer, fold, scope, layer_stats, history):
    backbone_path = phase3_backbone_path(layer, fold=fold, scope=scope)
    payload = {
        'layer': layer,
        'fold': None if fold is None else int(fold),
        'scope': scope,
        'encoder_state_dict': model.encoder.state_dict(),
        'layer_stats': layer_stats,
        'history': history,
        'output_dim': model.encoder.output_dim,
        'note': 'Projection head discarded; load encoder_state_dict into ConvNeXtTinyBackbone for Phase 4.',
    }
    torch.save(payload, backbone_path)
    return backbone_path


def train_simclr_one_layer(layer, pretrain_df, layer_stats, fold=None, scope=PHASE3_PRETRAIN_SCOPE, resume_path=None):
    if len(pretrain_df) < 2:
        raise ValueError(f'Need at least 2 eye rows for SimCLR pretraining, got {len(pretrain_df)}')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if PHASE3_REQUIRE_CUDA and device.type != 'cuda':
        raise RuntimeError('Phase 3 is configured for the CUDA-enabled training machine, but CUDA is not available in this runtime.')
    amp_enabled = bool(PHASE3_USE_AMP and device.type == 'cuda')
    dataset = RastaSimCLRLayerDataset(pretrain_df, layer=layer, layer_stats=layer_stats, image_size=PHASE3_IMAGE_SIZE)
    generator = torch.Generator()
    generator.manual_seed(PHASE3_SEED)
    loader = DataLoader(
        dataset,
        batch_size=PHASE3_BATCH_SIZE,
        shuffle=True,
        num_workers=PHASE3_NUM_WORKERS,
        pin_memory=(device.type == 'cuda'),
        drop_last=(len(dataset) >= PHASE3_BATCH_SIZE),
        worker_init_fn=seed_worker,
        generator=generator,
    )
    if len(loader) == 0:
        raise ValueError('DataLoader has zero batches; reduce PHASE3_BATCH_SIZE or add more data.')

    model = OCTASimCLR(imagenet_init=PHASE3_IMAGENET_INIT).to(device)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=PHASE3_LR, momentum=PHASE3_MOMENTUM, weight_decay=PHASE3_WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE3_EPOCHS, eta_min=0.0)
    scaler = make_grad_scaler(amp_enabled)
    start_epoch = 1
    history = []
    best_loss = float('inf')

    if resume_path is None and PHASE3_RESUME_PRETRAINING:
        resume_path = phase3_find_resume_checkpoint(layer, fold=fold, scope=scope)
    if resume_path is not None and Path(resume_path).exists():
        ckpt = phase3_torch_load(resume_path, map_location=device)
        if ckpt.get('layer') != layer or ckpt.get('scope') != scope or ckpt.get('fold') != (None if fold is None else int(fold)):
            raise ValueError(
                f'Resume checkpoint metadata mismatch for layer={layer} fold={fold} scope={scope}: {resume_path}'
            )
        model.load_state_dict(ckpt['model_state_dict'])
        if ckpt.get('optimizer_state_dict') is not None:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if ckpt.get('scheduler_state_dict') is not None:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        phase3_restore_rng_state(ckpt.get('rng_state'))
        history = list(ckpt.get('history', []))
        start_epoch = int(ckpt.get('epoch', 0)) + 1
        if history:
            best_loss = min(float(h['loss']) for h in history)
        if start_epoch > PHASE3_EPOCHS:
            print(f'Checkpoint already reached epoch {start_epoch - 1}/{PHASE3_EPOCHS}: {resume_path}')
        else:
            print(f'Resumed {layer} SimCLR from {resume_path} at epoch {start_epoch}/{PHASE3_EPOCHS}')

    print(f'Phase 3 SimCLR | layer={layer} fold={fold} scope={scope} device={device} n={len(dataset)} batches={len(loader)}')
    history_path = Path(str(phase3_checkpoint_prefix(layer, fold=fold, scope=scope)) + '_history.csv')
    run_t0 = time.time()
    remaining_epochs = max(0, PHASE3_EPOCHS - start_epoch + 1)
    avg_epoch_seconds = phase3_mean_epoch_seconds(history)
    if remaining_epochs > 0:
        if avg_epoch_seconds is None:
            print(
                f'Phase 3 SimCLR START | layer={layer} fold={fold} scope={scope} '
                f'start_epoch={start_epoch}/{PHASE3_EPOCHS} remaining_epochs={remaining_epochs} '
                f'batches_per_epoch={len(loader)} eta=pending_first_epoch'
            )
        else:
            print(
                f'Phase 3 SimCLR START | layer={layer} fold={fold} scope={scope} '
                f'start_epoch={start_epoch}/{PHASE3_EPOCHS} remaining_epochs={remaining_epochs} '
                f'batches_per_epoch={len(loader)} avg_epoch={format_duration(avg_epoch_seconds)} '
                f'est_remaining={format_duration(avg_epoch_seconds * remaining_epochs)} '
                f'est_total_layer={format_duration(avg_epoch_seconds * PHASE3_EPOCHS)}'
            )
    epoch_bar = phase3_progress(
        range(start_epoch, PHASE3_EPOCHS + 1),
        total=remaining_epochs,
        desc=f'Phase 3 {scope} fold={fold} layer={layer}',
        unit='epoch',
        leave=True,
    )
    for epoch in epoch_bar:
        avg_epoch_seconds_before = phase3_mean_epoch_seconds(history)
        remaining_including_current = max(0, PHASE3_EPOCHS - epoch + 1)
        if avg_epoch_seconds_before is None:
            print(
                f'[{layer}] START epoch {epoch:03d}/{PHASE3_EPOCHS} '
                f'batches={len(loader)} remaining_including_this={remaining_including_current} '
                f'eta=pending_first_epoch'
            )
        else:
            print(
                f'[{layer}] START epoch {epoch:03d}/{PHASE3_EPOCHS} '
                f'batches={len(loader)} remaining_including_this={remaining_including_current} '
                f'avg_epoch={format_duration(avg_epoch_seconds_before)} '
                f'est_remaining={format_duration(avg_epoch_seconds_before * remaining_including_current)} '
                f'est_total_layer={format_duration(avg_epoch_seconds_before * PHASE3_EPOCHS)}'
            )
        model.train()
        t0 = time.time()
        running_loss = 0.0
        optimizer.zero_grad(set_to_none=True)
        valid_steps = 0
        effective_batch_size = None

        batch_bar = phase3_progress(
            enumerate(loader, start=1),
            total=len(loader),
            desc=f'{layer} epoch {epoch:03d}/{PHASE3_EPOCHS}',
            unit='batch',
            leave=False,
            disable=(not PHASE3_SHOW_BATCH_PROGRESS),
        )
        for step, batch in batch_bar:
            x1 = batch['view1'].to(device, non_blocking=True)
            x2 = batch['view2'].to(device, non_blocking=True)
            if x1.size(0) < 2:
                continue
            if effective_batch_size is None:
                effective_batch_size = int(x1.size(0))
            with autocast_context(device, amp_enabled):
                z1 = model(x1)
                z2 = model(x2)
                loss = nt_xent_loss(z1, z2, temperature=PHASE3_TEMPERATURE)
                scaled_loss = loss / max(1, PHASE3_GRAD_ACCUM_STEPS)
            scaler.scale(scaled_loss).backward()
            running_loss += float(loss.detach().cpu())
            valid_steps += 1
            if PHASE3_SHOW_BATCH_PROGRESS and (step == 1 or step % PHASE3_PROGRESS_UPDATE_EVERY == 0 or step == len(loader)):
                batch_bar.set_postfix(loss=f'{running_loss / max(1, valid_steps):.4f}', refresh=False)
            if step % PHASE3_GRAD_ACCUM_STEPS == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                if not phase3_grads_are_finite(model):
                    raise RuntimeError(
                        f'Phase 3 abort: non-finite gradients at layer={layer} fold={fold} '
                        f'epoch={epoch} step={step}. Try lower LR/batch size or keep PHASE3_USE_AMP=False.'
                    )
                phase3_clip_gradients_if_needed(model)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        batch_bar.close()

        scheduler.step()
        mean_loss = running_loss / max(1, valid_steps)
        random_baseline_loss = phase3_nt_xent_random_baseline(effective_batch_size or PHASE3_BATCH_SIZE)
        epoch_seconds = time.time() - t0
        total_seconds = time.time() - run_t0
        rec = {
            'epoch': int(epoch),
            'loss': float(mean_loss),
            'lr': float(optimizer.param_groups[0]['lr']),
            'random_baseline_loss': float(random_baseline_loss),
            'loss_minus_random_baseline': float(mean_loss - random_baseline_loss),
            'seconds': float(epoch_seconds),
            'total_seconds': float(total_seconds),
        }
        history.append(rec)
        pd.DataFrame(history).to_csv(history_path, index=False)  # exact per-epoch log, updated every epoch without spamming notebook output
        is_best = mean_loss < best_loss
        if is_best:
            best_loss = mean_loss
            save_simclr_checkpoint(model, optimizer, scheduler, epoch, history, layer, fold, scope, layer_stats, is_best=True)
        if PHASE3_SAVE_LAST_EVERY_EPOCH or epoch == PHASE3_EPOCHS or epoch % PHASE3_CHECKPOINT_EVERY == 0:
            save_simclr_checkpoint(model, optimizer, scheduler, epoch, history, layer, fold, scope, layer_stats, is_best=False)
        if (PHASE3_ABORT_FLAT_LOSS_AFTER_EPOCH is not None
                and epoch >= int(PHASE3_ABORT_FLAT_LOSS_AFTER_EPOCH)
                and abs(mean_loss - random_baseline_loss) <= PHASE3_FLAT_LOSS_TOLERANCE):
            save_simclr_checkpoint(model, optimizer, scheduler, epoch, history, layer, fold, scope, layer_stats, is_best=False)
            raise RuntimeError(
                f'Phase 3 abort: {layer} fold={fold} loss {mean_loss:.4f} remains near random baseline '
                f'{random_baseline_loss:.4f} at epoch {epoch}. Check preflight diagnostics before continuing.'
            )
        avg_epoch_seconds = phase3_mean_epoch_seconds(history)
        remaining_after_epoch = max(0, PHASE3_EPOCHS - epoch)
        epoch_bar.set_postfix(
            loss=f'{mean_loss:.4f}',
            lr=f"{rec['lr']:.5f}",
            epoch_time=format_duration(epoch_seconds),
            total_time=format_duration(total_seconds),
            refresh=True,
        )
        if should_print_phase3_epoch(epoch, PHASE3_EPOCHS):
            avg_epoch_text = 'n/a' if avg_epoch_seconds is None else format_duration(avg_epoch_seconds)
            est_remaining_text = 'n/a' if avg_epoch_seconds is None else format_duration(avg_epoch_seconds * remaining_after_epoch)
            est_total_text = 'n/a' if avg_epoch_seconds is None else format_duration(avg_epoch_seconds * PHASE3_EPOCHS)
            print(
                f"[{layer}] END   epoch {epoch:03d}/{PHASE3_EPOCHS} "
                f"loss={mean_loss:.4f} baseline={random_baseline_loss:.4f} lr={rec['lr']:.5f} "
                f"epoch_time={format_duration(epoch_seconds)} avg_epoch={avg_epoch_text} "
                f"elapsed_this_run={format_duration(total_seconds)} est_remaining={est_remaining_text} "
                f"est_total_layer={est_total_text}"
            )
    epoch_bar.close()

    backbone_path = save_simclr_backbone(model, layer, fold, scope, layer_stats, history)
    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False)
    return {'layer': layer, 'fold': fold, 'scope': scope, 'best_loss': best_loss, 'backbone_path': str(backbone_path), 'history_path': str(history_path)}

In [ ]:
# ============================================================
# Phase 3 SimCLR preflight diagnostics: gradients, parameter update, embedding geometry
# ============================================================


def _phase3_float(x):
    return float(x.detach().cpu()) if torch.is_tensor(x) else float(x)


def phase3_projection_geometry(z1, z2):
    """Cosine stats for normalized SimCLR projections."""
    with torch.no_grad():
        n = int(z1.size(0))
        z = torch.cat([z1, z2], dim=0).float()
        sim = torch.matmul(z, z.T)
        idx = torch.arange(2 * n, device=z.device)
        pos_idx = (idx + n) % (2 * n)
        pos = sim[idx, pos_idx]
        self_mask = torch.eye(2 * n, dtype=torch.bool, device=z.device)
        pos_mask = torch.zeros_like(self_mask)
        pos_mask[idx, pos_idx] = True
        neg = sim[~self_mask & ~pos_mask]
        return {
            'positive_cosine_mean': _phase3_float(pos.mean()),
            'positive_cosine_std': _phase3_float(pos.std(unbiased=False)),
            'negative_cosine_mean': _phase3_float(neg.mean()),
            'negative_cosine_std': _phase3_float(neg.std(unbiased=False)),
            'pos_minus_neg_cosine': _phase3_float(pos.mean() - neg.mean()),
        }


def phase3_embedding_stats(h1, h2, z1, z2):
    with torch.no_grad():
        h = torch.cat([h1, h2], dim=0).float()
        z = torch.cat([z1, z2], dim=0).float()
        return {
            'feature_mean': _phase3_float(h.mean()),
            'feature_std': _phase3_float(h.std(unbiased=False)),
            'projection_mean': _phase3_float(z.mean()),
            'projection_std': _phase3_float(z.std(unbiased=False)),
            'projection_norm_mean': _phase3_float(z.norm(dim=1).mean()),
            'projection_norm_std': _phase3_float(z.norm(dim=1).std(unbiased=False)),
        }


def phase3_grad_stats(model):
    trainable_tensors = 0
    grad_tensors = 0
    none_grad_tensors = 0
    total_params = 0
    trainable_params = 0
    grad_sq = 0.0
    max_abs_grad = 0.0
    for p in model.parameters():
        total_params += p.numel()
        if not p.requires_grad:
            continue
        trainable_tensors += 1
        trainable_params += p.numel()
        if p.grad is None:
            none_grad_tensors += 1
            continue
        grad_tensors += 1
        g = p.grad.detach().float()
        grad_sq += float(torch.sum(g * g).cpu())
        max_abs_grad = max(max_abs_grad, float(g.abs().max().cpu()))
    return {
        'total_params': int(total_params),
        'trainable_params': int(trainable_params),
        'trainable_tensors': int(trainable_tensors),
        'grad_tensors': int(grad_tensors),
        'none_grad_tensors': int(none_grad_tensors),
        'grad_norm_l2': float(math.sqrt(grad_sq)),
        'max_abs_grad': float(max_abs_grad),
    }


def phase3_eval_batch_metrics(model, x1, x2, device, amp_enabled):
    was_training = model.training
    model.eval()
    with torch.no_grad():
        with autocast_context(device, amp_enabled):
            h1, z1 = model(x1, return_features=True)
            h2, z2 = model(x2, return_features=True)
            loss = nt_xent_loss(z1, z2, temperature=PHASE3_TEMPERATURE)
    if was_training:
        model.train()
    out = {
        'loss_eval': _phase3_float(loss),
        'random_baseline_loss': phase3_nt_xent_random_baseline(int(x1.size(0))),
    }
    out['loss_minus_random_baseline'] = out['loss_eval'] - out['random_baseline_loss']
    out.update(phase3_embedding_stats(h1, h2, z1, z2))
    out.update(phase3_projection_geometry(z1, z2))
    return out


def phase3_simclr_preflight_diagnostics(layer, pretrain_df, layer_stats, fold=None, scope=PHASE3_PRETRAIN_SCOPE):
    """Run one throwaway optimizer step and report whether SimCLR can learn/update at all."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if PHASE3_REQUIRE_CUDA and device.type != 'cuda':
        raise RuntimeError('Phase 3 preflight requires CUDA, but CUDA is not available.')
    amp_enabled = bool(PHASE3_USE_AMP and device.type == 'cuda')

    dataset = RastaSimCLRLayerDataset(pretrain_df, layer=layer, layer_stats=layer_stats, image_size=PHASE3_IMAGE_SIZE)
    generator = torch.Generator()
    generator.manual_seed(PHASE3_SEED)
    loader = DataLoader(
        dataset,
        batch_size=PHASE3_BATCH_SIZE,
        shuffle=True,
        num_workers=PHASE3_NUM_WORKERS,
        pin_memory=(device.type == 'cuda'),
        drop_last=(len(dataset) >= PHASE3_BATCH_SIZE),
        worker_init_fn=seed_worker,
        generator=generator,
    )
    batch = next(iter(loader))
    x1 = batch['view1'].to(device, non_blocking=True)
    x2 = batch['view2'].to(device, non_blocking=True)
    if x1.size(0) < 2:
        raise ValueError('Phase 3 preflight needs batch size >= 2.')

    model = OCTASimCLR(imagenet_init=PHASE3_IMAGENET_INIT).to(device)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=PHASE3_LR, momentum=PHASE3_MOMENTUM, weight_decay=PHASE3_WEIGHT_DECAY
    )
    scaler = make_grad_scaler(amp_enabled)

    before = phase3_eval_batch_metrics(model, x1, x2, device, amp_enabled)
    tracked_name = None
    tracked_before = None

    model.train()
    optimizer.zero_grad(set_to_none=True)
    with autocast_context(device, amp_enabled):
        z1 = model(x1)
        z2 = model(x2)
        loss_train_before = nt_xent_loss(z1, z2, temperature=PHASE3_TEMPERATURE)
    scaler.scale(loss_train_before).backward()
    scaler.unscale_(optimizer)
    grads = phase3_grad_stats(model)

    for name, p in model.named_parameters():
        if p.requires_grad and p.grad is not None:
            tracked_name = name
            tracked_before = p.detach().float().clone()
            break

    nonfinite_grads = (not np.isfinite(grads['grad_norm_l2'])) or (not np.isfinite(grads['max_abs_grad']))
    if not nonfinite_grads:
        phase3_clip_gradients_if_needed(model)
        scaler.step(optimizer)
        scaler.update()
    else:
        print('Phase 3 preflight: non-finite gradients detected; skipped optimizer step to avoid corrupting weights.')
    optimizer.zero_grad(set_to_none=True)
    after = phase3_eval_batch_metrics(model, x1, x2, device, amp_enabled)

    param_delta_l2 = np.nan
    param_delta_max_abs = np.nan
    if tracked_name is not None:
        tracked_after = dict(model.named_parameters())[tracked_name].detach().float()
        delta = tracked_after - tracked_before
        param_delta_l2 = float(delta.norm().cpu())
        param_delta_max_abs = float(delta.abs().max().cpu())

    rec = {
        'layer': layer,
        'fold': None if fold is None else int(fold),
        'scope': scope,
        'device': str(device),
        'batch_size': int(x1.size(0)),
        'num_batches': int(len(loader)),
        'temperature': float(PHASE3_TEMPERATURE),
        'lr': float(PHASE3_LR),
        'amp_enabled': bool(amp_enabled),
        'max_grad_norm': PHASE3_MAX_GRAD_NORM,
        'loss_train_before_step': _phase3_float(loss_train_before),
        'loss_eval_before_step': before['loss_eval'],
        'loss_eval_after_one_step': after['loss_eval'],
        'loss_eval_delta_after_one_step': after['loss_eval'] - before['loss_eval'],
        'random_baseline_loss': before['random_baseline_loss'],
        'baseline_close_initial': bool(abs(before['loss_eval'] - before['random_baseline_loss']) <= PHASE3_FLAT_LOSS_TOLERANCE),
        'tracked_param': tracked_name,
        'tracked_param_delta_l2': param_delta_l2,
        'tracked_param_delta_max_abs': param_delta_max_abs,
        'view_absdiff_mean': _phase3_float((x1 - x2).abs().mean()),
        'view1_mean': _phase3_float(x1.float().mean()),
        'view1_std': _phase3_float(x1.float().std(unbiased=False)),
        'view2_mean': _phase3_float(x2.float().mean()),
        'view2_std': _phase3_float(x2.float().std(unbiased=False)),
    }
    for k, v in grads.items():
        rec[f'grad_{k}'] = v
    for k, v in before.items():
        rec[f'before_{k}'] = v
    for k, v in after.items():
        rec[f'after_{k}'] = v

    failures = []
    warnings = []
    if rec['grad_trainable_params'] <= 0:
        failures.append('no_trainable_params')
    if rec['grad_grad_tensors'] <= 0 or rec['grad_grad_norm_l2'] <= 0:
        failures.append('no_gradient_flow')
    if not np.isfinite(rec['grad_grad_norm_l2']) or not np.isfinite(rec['grad_max_abs_grad']):
        failures.append('nonfinite_gradients')
    if not np.isfinite(rec['tracked_param_delta_l2']) or rec['tracked_param_delta_l2'] <= 0:
        failures.append('optimizer_did_not_change_tracked_param')
    if not np.isfinite(rec['before_projection_std']) or rec['before_projection_std'] < 1e-4:
        failures.append('projection_collapse_or_constant')
    if not np.isfinite(rec['before_feature_std']) or rec['before_feature_std'] < 1e-4:
        failures.append('feature_collapse_or_constant')
    if abs(rec['before_pos_minus_neg_cosine']) < 0.01:
        warnings.append('initial_positive_similarity_matches_negatives_expected_at_random_init')
    if abs(rec['loss_eval_after_one_step'] - rec['random_baseline_loss']) <= PHASE3_FLAT_LOSS_TOLERANCE:
        warnings.append('one_step_loss_still_near_random_baseline_watch_first_epochs')

    rec['failures'] = ';'.join(failures)
    rec['warnings'] = ';'.join(warnings)
    rec['passed'] = bool(not failures)
    return rec


if RUN_PHASE3_PREFLIGHT_DIAGNOSTICS:
    if PHASE3_PRETRAIN_SCOPE == 'fold_train':
        phase3_diag_fold = PHASE3_FOLDS_TO_RUN[0] if PHASE3_FOLDS_TO_RUN else 0
        phase3_diag_df = get_phase3_pretraining_df(fold=phase3_diag_fold, scope=PHASE3_PRETRAIN_SCOPE)
    else:
        phase3_diag_fold = None
        phase3_diag_df = get_phase3_pretraining_df(fold=None, scope=PHASE3_PRETRAIN_SCOPE)
    phase3_diag_layer = PHASE3_LAYERS_TO_RUN[0] if PHASE3_LAYERS_TO_RUN else PHASE3_LAYERS[0]
    print(f'Running Phase 3 preflight diagnostics: fold={phase3_diag_fold} layer={phase3_diag_layer}')
    phase3_diag_stats = get_phase3_layer_stats(phase3_diag_df, fold=phase3_diag_fold, scope=PHASE3_PRETRAIN_SCOPE)
    phase3_preflight = phase3_simclr_preflight_diagnostics(
        phase3_diag_layer, phase3_diag_df, phase3_diag_stats, fold=phase3_diag_fold, scope=PHASE3_PRETRAIN_SCOPE
    )
    phase3_preflight_path = PHASE3_OUTPUT_DIR / 'phase3_preflight_diagnostics.csv'
    pd.DataFrame([phase3_preflight]).to_csv(phase3_preflight_path, index=False)
    display(pd.DataFrame([phase3_preflight]).T.rename(columns={0: 'value'}))
    print('Saved Phase 3 preflight diagnostics:', phase3_preflight_path.resolve())
    if PHASE3_ABORT_ON_PREFLIGHT_FAILURE and not phase3_preflight['passed']:
        raise RuntimeError(f"Phase 3 preflight failed: {phase3_preflight['failures']}")
else:
    print('Phase 3 preflight diagnostics skipped.')

In [ ]:
# ============================================================
# Phase 3 launch / audit cell
# ============================================================

if 'phase2_folds' not in globals():
    raise RuntimeError('Run Phase 2 cells before Phase 3 so QC-passed folds are available.')

phase3_registry = []
if PHASE3_PRETRAIN_SCOPE == 'fold_train':
    audit_fold = PHASE3_FOLDS_TO_RUN[0] if PHASE3_FOLDS_TO_RUN else 0
    phase3_audit_df = get_phase3_pretraining_df(fold=audit_fold, scope=PHASE3_PRETRAIN_SCOPE)
else:
    phase3_audit_df = get_phase3_pretraining_df(fold=None, scope=PHASE3_PRETRAIN_SCOPE)

pair_summary = summarize_simclr_pairs(phase3_audit_df)
pair_summary.to_csv(PHASE3_OUTPUT_DIR / 'simclr_positive_pair_summary_preview.csv', index=False)
display(pair_summary)

if RUN_PHASE3_PRETRAINING:
    if PHASE3_PRETRAIN_SCOPE == 'fold_train':
        fold_plan = PHASE3_FOLDS_TO_RUN
    else:
        fold_plan = [None]

    total_phase3_runs = len(fold_plan) * len(PHASE3_LAYERS_TO_RUN)
    phase3_total_t0 = time.time()
    print(f'Launching Phase 3: {total_phase3_runs} layer-training run(s) across folds={fold_plan} layers={PHASE3_LAYERS_TO_RUN}')
    print(f'Resume enabled={PHASE3_RESUME_PRETRAINING}; last-checkpoint every epoch={PHASE3_SAVE_LAST_EVERY_EPOCH}')
    phase3_total_bar = phase3_progress(
        total=total_phase3_runs,
        desc='Phase 3 total training progress',
        unit='layer-run',
        leave=True,
    )

    try:
        for fold in fold_plan:
            pretrain_df = get_phase3_pretraining_df(fold=fold, scope=PHASE3_PRETRAIN_SCOPE)
            tag = f'{PHASE3_PRETRAIN_SCOPE}_fold{fold}' if PHASE3_PRETRAIN_SCOPE == 'fold_train' else PHASE3_PRETRAIN_SCOPE
            print(f'Preparing Phase 3 data/statistics for {tag}: {len(pretrain_df)} QC-passed eye rows')
            layer_stats = get_phase3_layer_stats(pretrain_df, fold=fold, scope=PHASE3_PRETRAIN_SCOPE)
            summarize_simclr_pairs(pretrain_df).to_csv(PHASE3_OUTPUT_DIR / f'simclr_positive_pair_summary_{tag}.csv', index=False)
            save_json(layer_stats, PHASE3_OUTPUT_DIR / f'simclr_layer_stats_{tag}.json')

            for layer in PHASE3_LAYERS_TO_RUN:
                phase3_total_bar.set_description_str(f'Phase 3 total fold={fold} layer={layer}')
                result = train_simclr_one_layer(layer, pretrain_df, layer_stats, fold=fold, scope=PHASE3_PRETRAIN_SCOPE)
                phase3_registry.append(result)
                pd.DataFrame(phase3_registry).to_csv(PHASE3_OUTPUT_DIR / 'phase3_pretrained_backbone_registry.csv', index=False)
                save_json(phase3_registry, PHASE3_OUTPUT_DIR / 'phase3_pretrained_backbone_registry.json')

                completed = len(phase3_registry)
                elapsed = time.time() - phase3_total_t0
                avg_per_run = elapsed / max(1, completed)
                remaining = max(0, total_phase3_runs - completed) * avg_per_run
                phase3_total_bar.update(1)
                phase3_total_bar.set_postfix(
                    done=f'{completed}/{total_phase3_runs}',
                    total_time=format_duration(elapsed),
                    avg_run=format_duration(avg_per_run),
                    eta=format_duration(remaining),
                    refresh=True,
                )
                print(
                    f"Completed layer-run {completed}/{total_phase3_runs}: fold={fold} layer={layer} "
                    f"elapsed={format_duration(elapsed)} avg_run={format_duration(avg_per_run)} eta={format_duration(remaining)}"
                )
    finally:
        phase3_total_bar.close()

    total_elapsed = time.time() - phase3_total_t0
    print(f'Phase 3 OCTA-SimCLR completed in {format_duration(total_elapsed)}. Backbone registry:')
    display(pd.DataFrame(phase3_registry))
else:
    print('Phase 3 code is ready but pretraining was not launched.')
    print('Set RUN_PHASE3_PRETRAINING = True on the GPU machine to train independent Sup/Deep/CC SimCLR encoders.')
    print('Recommended outputs after training: *_backbone.pt files in', PHASE3_OUTPUT_DIR.resolve())

In [ ]:
# ============================================================
# SUMMARIZATION CELL — Status through Phase 3 only
# ============================================================
# Purpose:
#   Run this cell after opening the notebook to quickly verify completed Phase 1–3 artifacts.
#   It does NOT rebuild manifests, recompute preprocessing stats, or train SimCLR.
#   Safe to run on a CPU/local copy, including when large Phase 3 .pt checkpoint files are absent.
# ============================================================

from pathlib import Path as _SummaryPath
import json as _summary_json
import math as _summary_math
import pandas as _summary_pd

_SUMMARY_PHASE1_DIR = _SummaryPath('outputs/phase1_dataset_analysis_8class')
_SUMMARY_PHASE2_DIR = _SummaryPath('outputs/phase2_preprocessing_8class')
if 'PHASE3_OUTPUT_DIR' in globals():
    _SUMMARY_PHASE3_DIR = _SummaryPath(PHASE3_OUTPUT_DIR)
else:
    _summary_phase3_candidates = [
        _SummaryPath('outputs/phase3_octa_simclr_8class_150epochs'),
        _SummaryPath('outputs/phase3_octa_simclr_8class_120epochs'),
    ]
    _summary_phase3_existing = [path for path in _summary_phase3_candidates if path.is_dir()]
    if not _summary_phase3_existing:
        _SUMMARY_PHASE3_DIR = _summary_phase3_candidates[0]
    elif len(_summary_phase3_existing) == 1:
        _SUMMARY_PHASE3_DIR = _summary_phase3_existing[0]
    else:
        def _summary_phase3_resume_score(path):
            checkpoints = list(path.glob('simclr_*_last.pt'))
            latest = max((p.stat().st_mtime for p in checkpoints), default=0.0)
            return len(checkpoints), latest, int(path.name.endswith('_150epochs'))
        _SUMMARY_PHASE3_DIR = max(_summary_phase3_existing, key=_summary_phase3_resume_score)


def _summary_read_json(path):
    path = _SummaryPath(path)
    if not path.exists():
        return None
    return _summary_json.loads(path.read_text())


def _summary_read_csv(path):
    path = _SummaryPath(path)
    if not path.exists():
        return None
    return _summary_pd.read_csv(path)


def _summary_exists(path):
    return _SummaryPath(path).exists()


# ------------------------------------------------------------
# Phase 1 summary — dataset analysis artifacts
# ------------------------------------------------------------
phase1_audit = _summary_read_json(_SUMMARY_PHASE1_DIR / 'dataset_audit.json')
if phase1_audit is None:
    print('PHASE 1: MISSING dataset_audit.json')
else:
    print('PHASE 1: OK')
    print('  patients:', phase1_audit.get('n_patients'))
    print('  eye samples:', phase1_audit.get('n_eye_samples'))
    print('  patient class counts:', phase1_audit.get('class_counts_patient'))
    print('  eye class counts:', phase1_audit.get('class_counts_eye'))
    print('  imbalance ratio patient:', phase1_audit.get('class_imbalance_ratio_patient'))
    print('  bilateral availability:', phase1_audit.get('bilateral_availability'))
    print('  layer missing counts:', phase1_audit.get('layer_missing_counts'))
    print('  patients in multiple cohorts:', phase1_audit.get('patients_in_multiple_cohorts'))


# ------------------------------------------------------------
# Phase 2 summary — preprocessing/folds/QC artifacts
# ------------------------------------------------------------
phase2_folds = _summary_read_csv(_SUMMARY_PHASE2_DIR / 'phase2_patient_stratified_folds.csv')
phase2_qc_exclusions = _summary_read_csv(_SUMMARY_PHASE2_DIR / 'phase2_image_qc_exclusions.csv')
print('\nPHASE 2:', 'OK' if phase2_folds is not None else 'MISSING phase2_patient_stratified_folds.csv')
if phase2_folds is not None:
    print('  rows:', len(phase2_folds))
    print('  folds:', sorted(phase2_folds['fold'].dropna().astype(int).unique().tolist()) if 'fold' in phase2_folds else 'missing fold column')
    print('  fold eye counts:', phase2_folds['fold'].value_counts().sort_index().to_dict() if 'fold' in phase2_folds else None)
    patient_fold_counts = phase2_folds.groupby('patient_id')['fold'].nunique() if {'patient_id', 'fold'}.issubset(phase2_folds.columns) else None
    leakage_count = int((patient_fold_counts > 1).sum()) if patient_fold_counts is not None else None
    print('  patient leakage count:', leakage_count)
    print('  class × fold:')
    if {'fold', 'cohort'}.issubset(phase2_folds.columns):
        display(_summary_pd.crosstab(phase2_folds['fold'], phase2_folds['cohort']))
if phase2_qc_exclusions is not None:
    print('  QC exclusions:', len(phase2_qc_exclusions))
else:
    print('  QC exclusions: missing file')

phase2_expected = []
for fold in range(5):
    phase2_expected.extend([
        _SUMMARY_PHASE2_DIR / f'clinical_preprocessor_fold{fold}.json',
        _SUMMARY_PHASE2_DIR / f'octa_train_stats_fold{fold}.json',
        _SUMMARY_PHASE2_DIR / f'clinical_mutual_information_fold{fold}.csv',
    ])
phase2_missing = [str(p) for p in phase2_expected if not p.exists()]
print('  missing fold artifacts:', phase2_missing if phase2_missing else 'none')


# ------------------------------------------------------------
# Phase 3 summary — OCTA-SimCLR artifacts, no checkpoint loading
# ------------------------------------------------------------
phase3_registry = _summary_read_csv(_SUMMARY_PHASE3_DIR / 'phase3_pretrained_backbone_registry.csv')
phase3_histories = sorted(_SUMMARY_PHASE3_DIR.glob('simclr_fold_train_fold*_history.csv'))
phase3_backbones = sorted(_SUMMARY_PHASE3_DIR.glob('*.pt'))
print('\nPHASE 3:', 'OK' if phase3_registry is not None and len(phase3_histories) else 'MISSING registry or histories')
print('  registry rows:', 0 if phase3_registry is None else len(phase3_registry))
print('  history CSVs:', len(phase3_histories))
print('  .pt files present locally:', len(phase3_backbones), '(OK if intentionally omitted from copied outputs)')

if phase3_registry is not None and len(phase3_registry):
    print('  registry best losses:')
    display(phase3_registry.pivot(index='fold', columns='layer', values='best_loss') if {'fold','layer','best_loss'}.issubset(phase3_registry.columns) else phase3_registry)

phase3_history_rows = []
for hist_path in phase3_histories:
    hist = _summary_pd.read_csv(hist_path)
    if not len(hist):
        phase3_history_rows.append({'file': hist_path.name, 'epochs': 0, 'status': 'empty'})
        continue
    final_loss = float(hist['loss'].iloc[-1]) if 'loss' in hist else float('nan')
    min_loss = float(hist['loss'].min()) if 'loss' in hist else float('nan')
    baseline = float(hist['random_baseline_loss'].iloc[-1]) if 'random_baseline_loss' in hist else _summary_math.nan
    phase3_history_rows.append({
        'file': hist_path.name,
        'epochs': int(len(hist)),
        'start_epoch': int(hist['epoch'].iloc[0]) if 'epoch' in hist else None,
        'end_epoch': int(hist['epoch'].iloc[-1]) if 'epoch' in hist else None,
        'final_loss': final_loss,
        'min_loss': min_loss,
        'random_baseline': baseline,
        'final_minus_baseline': final_loss - baseline if baseline == baseline else None,
        'finite_loss': bool(hist['loss'].map(_summary_math.isfinite).all()) if 'loss' in hist else None,
    })
phase3_history_summary = _summary_pd.DataFrame(phase3_history_rows)
phase3_complete_150 = False
if len(phase3_history_summary):
    print('  history sanity:')
    display(phase3_history_summary)
    phase3_complete_150 = bool((phase3_history_summary['epochs'] == 150).all()) if 'epochs' in phase3_history_summary else False
    finite_all = bool(phase3_history_summary['finite_loss'].fillna(False).all()) if 'finite_loss' in phase3_history_summary else False
    below_baseline = bool((phase3_history_summary['final_minus_baseline'] < -0.5).all()) if 'final_minus_baseline' in phase3_history_summary else False
    print('  complete 150 epochs all runs:', phase3_complete_150)
    print('  finite losses all runs:', finite_all)
    print('  final losses clearly below random baseline:', below_baseline)

pair_summaries = sorted(_SUMMARY_PHASE3_DIR.glob('simclr_positive_pair_summary_fold_train_fold*.csv'))
print('  positive-pair summary files:', len(pair_summaries))
if pair_summaries:
    pair_df = _summary_pd.concat([_summary_pd.read_csv(p).assign(file=p.name) for p in pair_summaries], ignore_index=True)
    display(pair_df[['file', 'layer', 'eye_rows', 'bilateral_positive_eye_rows', 'self_augmented_eye_rows', 'bilateral_patients', 'unilateral_patients']])

print('\nSUMMARY VERDICT:')
phase1_has_eight_classes = bool(phase1_audit and len(phase1_audit.get('class_counts_patient', {})) == 8)
if phase1_has_eight_classes and phase2_folds is not None and phase3_registry is not None and len(phase3_histories) == 15 and phase3_complete_150:
    print('  Expanded eight-class Phase 1–3 artifact set looks complete enough for later integration, except large .pt backbones if this is a copied local directory.')
else:
    print('  Some expected Phase 1–3 summary artifacts are missing; inspect messages above.')

In [ ]:
# ============================================================
# Eight-class segmentation architecture comparison configuration
# ============================================================
# Phases 1-3 above are exact source copies from main.ipynb.
# All Phase 4-14 comparison artifacts are isolated below.

SEGCOMP_ROOT = ensure_dir('outputs/segmentation_comparison_8class')
SEGCOMP_SHARED_DIR = ensure_dir(SEGCOMP_ROOT / 'shared')
SEGCOMP_MODEL_IDS = [
    'unetpp',
    'unet',
    'attention_unet',
    'deeplabv3plus',
    'segformer_b0',
    'imn',
]
SEGCOMP_MASK_SOURCES = {model_id: f'{model_id}_student' for model_id in SEGCOMP_MODEL_IDS}
SEGCOMP_ACTIVE_MODEL_ID = 'unetpp'
SEGCOMP_ACTIVE_MASK_SOURCE = SEGCOMP_MASK_SOURCES[SEGCOMP_ACTIVE_MODEL_ID]
SEGCOMP_RUN_SEGMENTATION = True
SEGCOMP_RUN_MASK_MATERIALIZATION = True
SEGCOMP_RUN_CLASSIFIER_TRAINING = True
SEGCOMP_RUN_EXPLAINABILITY = False
SEGCOMP_RUN_EVALUATION = True

if 'COHORT_TO_LABEL' in globals():
    expected_segcomp_labels = list(range(8))
    actual_segcomp_labels = sorted(int(v) for v in COHORT_TO_LABEL.values())
    if actual_segcomp_labels != expected_segcomp_labels:
        raise ValueError(f'Eight-class comparison requires labels 0-7, got {actual_segcomp_labels}')


def segcomp_model_dir(model_id):
    if model_id not in SEGCOMP_MODEL_IDS:
        raise ValueError(f'Unknown segmentation model {model_id!r}; expected one of {SEGCOMP_MODEL_IDS}')
    return ensure_dir(SEGCOMP_ROOT / model_id)


def segcomp_activate_model(model_id):
    """Set architecture-specific paths used by the shared Phase 8-14 code."""
    global SEGCOMP_ACTIVE_MODEL_ID, SEGCOMP_ACTIVE_MASK_SOURCE
    global PHASE8_STUDENT_OUTPUT_DIR, PHASE8_STUDENT_MASK_SOURCE
    global PHASE9_OUTPUT_DIR, PHASE9_UNETPP_MASK_DIR, PHASE9_SELECTED_MASK_SOURCE
    global PHASE11_OUTPUT_DIR, PHASE12_OUTPUT_DIR, PHASE13_OUTPUT_DIR
    global PHASE14_OUTPUT_DIR, PHASE14_OOF_PATH, PHASE14_MC_DROPOUT_PATH

    model_dir = segcomp_model_dir(model_id)
    SEGCOMP_ACTIVE_MODEL_ID = model_id
    SEGCOMP_ACTIVE_MASK_SOURCE = SEGCOMP_MASK_SOURCES[model_id]
    PHASE8_STUDENT_OUTPUT_DIR = ensure_dir(model_dir / 'phase8_segmentation')
    PHASE8_STUDENT_MASK_SOURCE = SEGCOMP_ACTIVE_MASK_SOURCE
    PHASE9_OUTPUT_DIR = ensure_dir(model_dir / 'phase9_masks_biomarkers')
    PHASE9_UNETPP_MASK_DIR = ensure_dir(PHASE9_OUTPUT_DIR / 'student_masks')
    PHASE9_SELECTED_MASK_SOURCE = SEGCOMP_ACTIVE_MASK_SOURCE
    PHASE11_OUTPUT_DIR = ensure_dir(model_dir / 'phase11_augmentation')
    PHASE12_OUTPUT_DIR = ensure_dir(model_dir / 'phase12_training')
    PHASE13_OUTPUT_DIR = ensure_dir(model_dir / 'phase13_explainability')
    PHASE14_OUTPUT_DIR = ensure_dir(model_dir / 'phase14_evaluation')
    PHASE14_OOF_PATH = PHASE12_OUTPUT_DIR / 'phase12_all_outer_test_predictions.csv'
    PHASE14_MC_DROPOUT_PATH = PHASE14_OUTPUT_DIR / 'phase14_mc_dropout_oof_predictions.csv'
    return {
        'model_id': model_id,
        'mask_source': SEGCOMP_ACTIVE_MASK_SOURCE,
        'model_dir': str(model_dir),
    }


print('Eight-class segmentation comparison root:', SEGCOMP_ROOT)
print('Models:', SEGCOMP_MODEL_IDS)
print('Segmentation training, mask materialization, classifier training, and Phase 14 evaluation are enabled.')


In [ ]:
# ============================================================
# Phase 4 — Image Encoders with Projection Heads
# ============================================================

PHASE4_OUTPUT_DIR = ensure_dir('outputs/segmentation_comparison_8class/shared/phase4_image_encoders')
PHASE4_LAYERS = ['sup', 'deep', 'cc']
PHASE4_IMAGE_SIZE = IMAGE_SIZE
PHASE4_TOKEN_DIM = 256
PHASE4_PROJECTION_HIDDEN_DIM = 512
PHASE4_BACKBONE_DIM = 768
PHASE4_REQUIRE_SIMCLR_BACKBONES = True # local copy may omit large .pt files
PHASE4_RUN_SMOKE_TEST = True            # set True only on a machine with torch/torchvision ready
PHASE4_DEFAULT_FREEZE_BACKBONES = False

if torch is None:
    raise ImportError('PyTorch is required for Phase 4 image encoders.')
if models is None:
    raise ImportError('torchvision.models is required for ConvNeXt-Tiny Phase 4 encoders.')


class OCTAProjectionHead(nn.Module):
    """PLAN Phase 4 projection: 768 -> 512 -> 256 with GELU/LayerNorm."""
    def __init__(self, input_dim=PHASE4_BACKBONE_DIM, hidden_dim=PHASE4_PROJECTION_HIDDEN_DIM, output_dim=PHASE4_TOKEN_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, output_dim),
            nn.LayerNorm(output_dim),
        )

    def forward(self, x):
        return self.net(x)


class Phase4LayerImageEncoder(nn.Module):
    """One OCTA layer encoder: ConvNeXt-Tiny backbone + projection head."""
    def __init__(self, layer, imagenet_init=True):
        super().__init__()
        if layer not in PHASE4_LAYERS:
            raise ValueError(f'Unknown layer {layer}; expected one of {PHASE4_LAYERS}')
        self.layer = layer
        self.backbone = ConvNeXtTinyBackbone(in_chans=1, imagenet_init=imagenet_init)
        self.projection = OCTAProjectionHead(input_dim=self.backbone.output_dim)

    def forward(self, x, return_spatial=True):
        spatial = self.backbone.forward_features(x)  # expected [B, 768, 7, 7] for 224x224 input
        pooled = spatial.mean(dim=(-2, -1))
        token = self.projection(pooled)
        out = {'token': token, 'pooled': pooled}
        if return_spatial:
            out['spatial'] = spatial
        return out


class Phase4OCTAImageEncoders(nn.Module):
    """Independent Sup/Deep/CC encoders. Weights are intentionally not shared."""
    def __init__(self, layers=PHASE4_LAYERS, imagenet_init=True):
        super().__init__()
        self.layers = list(layers)
        self.encoders = nn.ModuleDict({
            layer: Phase4LayerImageEncoder(layer=layer, imagenet_init=imagenet_init)
            for layer in self.layers
        })
        self.assert_no_shared_backbone_parameters()

    def assert_no_shared_backbone_parameters(self):
        seen = {}
        for layer, encoder in self.encoders.items():
            for name, param in encoder.backbone.named_parameters():
                ptr = param.data_ptr()
                if ptr in seen:
                    raise AssertionError(f'Backbone parameter shared between {seen[ptr]} and {layer}.{name}')
                seen[ptr] = f'{layer}.{name}'
        return True

    def forward(self, batch, return_spatial=True):
        """Accepts a Phase 2 sample/batch dict with sup_image/deep_image/cc_image tensors."""
        tokens, pooled_features, spatial_maps = {}, {}, {}
        ordered_tokens = []
        for layer in self.layers:
            key = f'{layer}_image'
            if key not in batch:
                raise KeyError(f'Missing {key} in batch for Phase 4 encoders')
            layer_out = self.encoders[layer](batch[key], return_spatial=return_spatial)
            tokens[layer] = layer_out['token']
            pooled_features[layer] = layer_out['pooled']
            if return_spatial:
                spatial_maps[layer] = layer_out['spatial']
            ordered_tokens.append(layer_out['token'])
        return {
            'layer_tokens': torch.stack(ordered_tokens, dim=1),  # [B, 3, 256]
            'tokens': tokens,
            'pooled_features': pooled_features,
            'spatial_maps': spatial_maps,
        }


def _phase4_normalize_path(path_like):
    if path_like is None or (isinstance(path_like, float) and np.isnan(path_like)):
        return None
    return Path(str(path_like).replace('\\\\', '/').replace('\\', '/'))


def _phase4_load_torch_payload(path, map_location='cpu'):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def _phase4_extract_backbone_state(payload):
    """Support backbone-only exports and full SimCLR checkpoints."""
    if isinstance(payload, dict):
        for key in ['backbone_state_dict', 'encoder_state_dict', 'state_dict', 'model_state_dict']:
            state = payload.get(key)
            if isinstance(state, dict):
                if key == 'model_state_dict':
                    # Full OCTASimCLR: strip encoder. prefix if present.
                    stripped = {}
                    for k, v in state.items():
                        if k.startswith('encoder.'):
                            stripped[k[len('encoder.'):]] = v
                    return stripped or state
                return state
    if isinstance(payload, dict) and all(torch.is_tensor(v) for v in payload.values()):
        return payload
    raise ValueError('Could not find a ConvNeXt backbone state dict in Phase 3 payload')


def phase4_registry_rows():
    registry_path = PHASE3_OUTPUT_DIR / 'phase3_pretrained_backbone_registry.csv'
    if registry_path.exists():
        return pd.read_csv(registry_path)
    return pd.DataFrame(columns=['layer', 'fold', 'scope', 'best_loss', 'backbone_path', 'history_path'])


def phase4_find_backbone_path(layer, fold=None, scope=PHASE3_PRETRAIN_SCOPE):
    """Prefer actual expected path; fall back to registry path for copied outputs."""
    expected = phase3_backbone_path(layer, fold=fold, scope=scope)
    if expected.exists():
        return expected
    reg = phase4_registry_rows()
    if len(reg):
        mask = (reg['layer'].astype(str) == str(layer)) & (reg['scope'].astype(str) == str(scope))
        if fold is None:
            mask = mask & reg['fold'].isna()
        else:
            mask = mask & (reg['fold'].fillna(-999).astype(int) == int(fold))
        if mask.any():
            candidate = _phase4_normalize_path(reg.loc[mask, 'backbone_path'].iloc[0])
            if candidate is not None and candidate.exists():
                return candidate
    return expected


def phase4_load_simclr_backbone(layer_encoder, layer, fold=None, scope=PHASE3_PRETRAIN_SCOPE, strict=True, map_location='cpu'):
    path = phase4_find_backbone_path(layer, fold=fold, scope=scope)
    if not path.exists():
        msg = f'Missing Phase 3 backbone for layer={layer} fold={fold} scope={scope}: {path}'
        if strict:
            raise FileNotFoundError(msg)
        return {'loaded': False, 'path': str(path), 'message': msg}

    payload = _phase4_load_torch_payload(path, map_location=map_location)
    if isinstance(payload, dict):
        meta_layer = payload.get('layer')
        meta_fold = payload.get('fold')
        meta_scope = payload.get('scope')
        if meta_layer is not None and str(meta_layer) != str(layer):
            raise ValueError(f'Backbone layer metadata mismatch: expected {layer}, got {meta_layer}')
        if meta_scope is not None and str(meta_scope) != str(scope):
            raise ValueError(f'Backbone scope metadata mismatch: expected {scope}, got {meta_scope}')
        if meta_fold is not None and fold is not None and int(meta_fold) != int(fold):
            raise ValueError(f'Backbone fold metadata mismatch: expected {fold}, got {meta_fold}')
    state = _phase4_extract_backbone_state(payload)
    missing, unexpected = layer_encoder.backbone.load_state_dict(state, strict=False)
    if strict and (missing or unexpected):
        raise RuntimeError(f'Backbone load not clean for {layer}/fold={fold}: missing={missing[:5]}, unexpected={unexpected[:5]}')
    return {
        'loaded': True,
        'path': str(path),
        'missing_keys': list(missing),
        'unexpected_keys': list(unexpected),
    }


def build_phase4_image_encoders(fold=0, scope=PHASE3_PRETRAIN_SCOPE, load_simclr=True, freeze_backbones=PHASE4_DEFAULT_FREEZE_BACKBONES, require_backbones=PHASE4_REQUIRE_SIMCLR_BACKBONES, device=None):
    """Build independent Sup/Deep/CC encoders and optionally load fold-specific SimCLR backbones."""
    model = Phase4OCTAImageEncoders(layers=PHASE4_LAYERS, imagenet_init=True)
    load_report = []
    if load_simclr:
        for layer in PHASE4_LAYERS:
            report = phase4_load_simclr_backbone(
                model.encoders[layer], layer=layer, fold=fold, scope=scope,
                strict=require_backbones, map_location='cpu'
            )
            load_report.append({'layer': layer, **report})
    if freeze_backbones:
        for encoder in model.encoders.values():
            for p in encoder.backbone.parameters():
                p.requires_grad = False
    if device is not None:
        model = model.to(device)
    model.phase4_load_report = load_report
    return model


def phase4_backbone_availability_audit(scope=PHASE3_PRETRAIN_SCOPE, folds=None):
    if folds is None:
        folds = PHASE3_FOLDS_TO_RUN if scope == 'fold_train' else [None]
    reg = phase4_registry_rows()
    rows = []
    for fold in folds:
        for layer in PHASE4_LAYERS:
            expected = phase3_backbone_path(layer, fold=fold, scope=scope)
            reg_row = None
            if len(reg):
                mask = (reg['layer'].astype(str) == str(layer)) & (reg['scope'].astype(str) == str(scope))
                if fold is None:
                    mask = mask & reg['fold'].isna()
                else:
                    mask = mask & (reg['fold'].fillna(-999).astype(int) == int(fold))
                if mask.any():
                    reg_row = reg.loc[mask].iloc[0]
            registry_path = _phase4_normalize_path(reg_row['backbone_path']) if reg_row is not None and 'backbone_path' in reg_row else None
            history_path = _phase4_normalize_path(reg_row['history_path']) if reg_row is not None and 'history_path' in reg_row else None
            resolved = phase4_find_backbone_path(layer, fold=fold, scope=scope)
            rows.append({
                'fold': fold,
                'layer': layer,
                'scope': scope,
                'expected_backbone_path': str(expected),
                'registry_backbone_path': None if registry_path is None else str(registry_path),
                'resolved_backbone_path': str(resolved),
                'backbone_exists_here': bool(resolved.exists()),
                'history_path': None if history_path is None else str(history_path),
                'history_exists_here': bool(history_path.exists()) if history_path is not None else False,
                'best_loss': None if reg_row is None or 'best_loss' not in reg_row else reg_row['best_loss'],
            })
    audit = pd.DataFrame(rows)
    audit.to_csv(PHASE4_OUTPUT_DIR / 'phase4_backbone_availability.csv', index=False)
    return audit


phase4_audit = phase4_backbone_availability_audit()
print('Phase 4 backbone availability audit saved:', PHASE4_OUTPUT_DIR / 'phase4_backbone_availability.csv')
display(phase4_audit)

if PHASE4_RUN_SMOKE_TEST:
    phase4_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    phase4_model = build_phase4_image_encoders(
        fold=0,
        load_simclr=True,
        freeze_backbones=False,
        require_backbones=PHASE4_REQUIRE_SIMCLR_BACKBONES,
        device=phase4_device,
    )
    dummy = {f'{layer}_image': torch.randn(2, 1, PHASE4_IMAGE_SIZE, PHASE4_IMAGE_SIZE, device=phase4_device) for layer in PHASE4_LAYERS}
    with torch.no_grad():
        phase4_out = phase4_model(dummy)
    print('layer_tokens:', tuple(phase4_out['layer_tokens'].shape))
    print('spatial maps:', {k: tuple(v.shape) for k, v in phase4_out['spatial_maps'].items()})
else:
    print('Phase 4 code ready. Set PHASE4_RUN_SMOKE_TEST=True to instantiate/test encoders.')

In [ ]:
# ============================================================
# Phase 5 — Tabular metadata encoder
# ============================================================

PHASE5_OUTPUT_DIR = ensure_dir('outputs/segmentation_comparison_8class/shared/phase5_tabular_encoder')
PHASE5_CLINICAL_MASK_DROPOUT = 0.10
PHASE5_TABULAR_DROPOUT1 = 0.30
PHASE5_TABULAR_DROPOUT2 = 0.20
PHASE5_OUTPUT_DIM = 256

# Includes future Phase 8 generated names plus current RASTA/table biomarker names.
DEFAULT_BIOMARKER_PATTERNS = (
    r'^(FAZ|Faz|DENS_|PERF_)',
    r'faz', r'vessel.*density', r'perfusion', r'flow.*deficit', r'cc.*deficit',
    r'skeleton.*density', r'vessel.*length', r'circularity', r'centroid',
)
PHASE5_EYE_SUFFIX_RE = re.compile(r'^(?P<base>.+)_(?P<eye>OD|OS)$', re.IGNORECASE)


def _phase5_is_numericish(series):
    return pd.api.types.is_numeric_dtype(series) or pd.to_numeric(series, errors='coerce').notna().any()


def infer_phase5_biomarker_specs(df, exclude_cols=None):
    """Infer biomarker specs.

    Returns generic specs. For side-specific table columns like `FAZRawSize_OD` and
    `FAZRawSize_OS`, one feature is emitted and resolved by each sample's `eye`.
    This avoids feeding the contralateral eye as a separate tabular feature.
    """
    exclude = set(exclude_cols or [])
    patterns = [re.compile(p, re.IGNORECASE) for p in DEFAULT_BIOMARKER_PATTERNS]
    candidates = []
    for col in df.columns:
        if col in exclude:
            continue
        name = str(col)
        if any(p.search(name) for p in patterns) and _phase5_is_numericish(df[col]):
            candidates.append(col)

    by_base, generic = {}, []
    for col in candidates:
        m = PHASE5_EYE_SUFFIX_RE.match(str(col))
        if m:
            base = m.group('base')
            eye = m.group('eye').upper()
            by_base.setdefault(base, {})[eye] = col
        else:
            generic.append({'name': str(col), 'columns': {'ANY': col}})

    specs = []
    for base, cols in sorted(by_base.items()):
        specs.append({'name': base, 'columns': cols})
    specs.extend(generic)
    return specs


def phase5_biomarker_names(biomarker_specs):
    return [spec['name'] if isinstance(spec, dict) else str(spec) for spec in (biomarker_specs or [])]


def _phase5_resolve_biomarker_value(row, spec):
    if not isinstance(spec, dict):
        return row[spec] if spec in row else np.nan
    cols = spec.get('columns', {})
    if 'ANY' in cols:
        col = cols['ANY']
    else:
        eye = str(row.get('eye', '')).upper()
        col = cols.get(eye)
    return row[col] if col and col in row else np.nan


def fit_phase5_biomarker_stats(train_df, biomarker_specs):
    """Fold-fitted z-score stats over eye-aligned biomarker values."""
    stats = {}
    for spec in biomarker_specs:
        name = spec['name'] if isinstance(spec, dict) else str(spec)
        vals = []
        for _, row in train_df.iterrows():
            raw = _phase5_resolve_biomarker_value(row, spec)
            vals.append(pd.to_numeric(raw, errors='coerce'))
        vals = pd.Series(vals, dtype='float64')
        mean = float(vals.mean()) if vals.notna().any() else 0.0
        std = float(vals.std(ddof=0)) if vals.notna().any() else 1.0
        if not np.isfinite(std) or std < 1e-6:
            std = 1.0
        stats[name] = {'mean': mean, 'std': std, 'observed_fraction': float(vals.notna().mean()) if len(vals) else 0.0}
    return stats


def transform_phase5_biomarkers(row, biomarker_specs, biomarker_stats):
    values, mask = [], []
    for spec in biomarker_specs:
        name = spec['name'] if isinstance(spec, dict) else str(spec)
        raw = _phase5_resolve_biomarker_value(row, spec)
        val = pd.to_numeric(raw, errors='coerce') if pd.notna(raw) else np.nan
        if pd.notna(val) and np.isfinite(float(val)):
            st = biomarker_stats[name]
            values.append((float(val) - st['mean']) / max(st['std'], 1e-6))
            mask.append(1.0)
        else:
            values.append(0.0)
            mask.append(0.0)
    return np.asarray(values, dtype=np.float32), np.asarray(mask, dtype=np.float32)


if nn is not None:
    class Phase5TabularMissingTokenLayer(nn.Module):
        """Learned imputation for clinical features and biomarkers."""
        def __init__(self, clinical_dim, ordinal_indices=None, ordinal_cardinalities=None, biomarker_dim=0):
            super().__init__()
            self.clinical_dim = int(clinical_dim)
            self.biomarker_dim = int(biomarker_dim)
            self.clinical_missing_tokens = nn.Parameter(torch.zeros(self.clinical_dim))
            self.biomarker_missing_tokens = nn.Parameter(torch.zeros(self.biomarker_dim)) if self.biomarker_dim else None
            self.ordinal_indices = [int(i) for i in (ordinal_indices or [])]
            self.ordinal_embeddings = nn.ModuleDict()
            ordinal_cardinalities = ordinal_cardinalities or {}
            for idx in self.ordinal_indices:
                self.ordinal_embeddings[str(idx)] = nn.Embedding(max(int(ordinal_cardinalities.get(idx, 3)), 1), 1)

        def forward(self, clinical_values, clinical_mask, biomarkers=None, biomarker_mask=None):
            x_clin = clinical_values.float().clone()
            clinical_mask = clinical_mask.float()
            if x_clin.ndim != 2 or x_clin.shape[1] != self.clinical_dim:
                raise ValueError(f'clinical_values must be [B,{self.clinical_dim}], got {tuple(x_clin.shape)}')
            for idx in self.ordinal_indices:
                key = str(idx)
                if key in self.ordinal_embeddings:
                    emb = self.ordinal_embeddings[key]
                    codes = clinical_values[:, idx].round().long().clamp(min=0, max=emb.num_embeddings - 1)
                    x_clin[:, idx] = emb(codes).squeeze(-1)
            x_clin = torch.where(clinical_mask > 0.5, x_clin, self.clinical_missing_tokens.view(1, -1).expand_as(x_clin))

            if self.biomarker_dim == 0:
                return x_clin
            b = x_clin.shape[0]
            if biomarkers is None:
                biomarkers = torch.zeros(b, self.biomarker_dim, device=x_clin.device, dtype=x_clin.dtype)
            else:
                biomarkers = biomarkers.float().to(x_clin.device)
            if biomarker_mask is None:
                biomarker_mask = torch.zeros(b, self.biomarker_dim, device=x_clin.device, dtype=x_clin.dtype)
            else:
                biomarker_mask = biomarker_mask.float().to(x_clin.device)
            if biomarkers.shape[1] != self.biomarker_dim:
                raise ValueError(f'biomarkers must be [B,{self.biomarker_dim}], got {tuple(biomarkers.shape)}')
            x_bio = torch.where(biomarker_mask > 0.5, biomarkers, self.biomarker_missing_tokens.view(1, -1).expand_as(biomarkers))
            return torch.cat([x_clin, x_bio], dim=1)


    class Phase5TabularEncoder(nn.Module):
        """PLAN Phase 5: LayerNorm -> FC 128 -> GELU/Dropout -> FC 256 -> GELU/LayerNorm/Dropout."""
        def __init__(self, clinical_dim, ordinal_indices=None, ordinal_cardinalities=None,
                     biomarker_dim=0, clinical_mask_dropout=PHASE5_CLINICAL_MASK_DROPOUT):
            super().__init__()
            self.clinical_dim = int(clinical_dim)
            self.biomarker_dim = int(biomarker_dim)
            self.input_dim = self.clinical_dim + self.biomarker_dim
            self.clinical_mask_dropout = float(clinical_mask_dropout)
            self.missing_layer = Phase5TabularMissingTokenLayer(
                self.clinical_dim, ordinal_indices=ordinal_indices,
                ordinal_cardinalities=ordinal_cardinalities, biomarker_dim=self.biomarker_dim,
            )
            self.net = nn.Sequential(
                nn.LayerNorm(self.input_dim),
                nn.Linear(self.input_dim, 128),
                nn.GELU(),
                nn.Dropout(PHASE5_TABULAR_DROPOUT1),
                nn.Linear(128, PHASE5_OUTPUT_DIM),
                nn.GELU(),
                nn.LayerNorm(PHASE5_OUTPUT_DIM),
                nn.Dropout(PHASE5_TABULAR_DROPOUT2),
            )

        def _drop_observed_clinical(self, clinical_mask):
            if not self.training or self.clinical_mask_dropout <= 0:
                return clinical_mask.float()
            drop = (torch.rand_like(clinical_mask.float()) < self.clinical_mask_dropout) & (clinical_mask > 0.5)
            return clinical_mask.float().masked_fill(drop, 0.0)

        def forward(self, clinical_features, clinical_mask, biomarkers=None, biomarker_mask=None, return_tabular_input=False):
            clinical_mask = self._drop_observed_clinical(clinical_mask)
            x = self.missing_layer(clinical_features, clinical_mask, biomarkers=biomarkers, biomarker_mask=biomarker_mask)
            out = self.net(x)
            if return_tabular_input:
                return {'tabular_features': out, 'tabular_input': x, 'clinical_mask_after_dropout': clinical_mask}
            return out
else:
    Phase5TabularMissingTokenLayer = None
    Phase5TabularEncoder = None


def phase5_ordinal_metadata(clinical_preprocessor):
    ordinal_cardinalities = {
        i: clinical_preprocessor.ordinal_cardinalities.get(col, 3)
        for i, col in enumerate(clinical_preprocessor.clinical_cols)
        if clinical_preprocessor.specs.get(col) == 'ordinal'
    }
    return list(ordinal_cardinalities.keys()), ordinal_cardinalities


def make_phase5_tabular_encoder(clinical_preprocessor, biomarker_specs=None, clinical_mask_dropout=PHASE5_CLINICAL_MASK_DROPOUT):
    if Phase5TabularEncoder is None:
        raise ImportError('PyTorch is required for Phase5TabularEncoder')
    ordinal_indices, ordinal_cardinalities = phase5_ordinal_metadata(clinical_preprocessor)
    return Phase5TabularEncoder(
        clinical_dim=len(clinical_preprocessor.clinical_cols),
        ordinal_indices=ordinal_indices,
        ordinal_cardinalities=ordinal_cardinalities,
        biomarker_dim=len(biomarker_specs or []),
        clinical_mask_dropout=clinical_mask_dropout,
    )


def build_phase5_tabular_artifacts(fold_df, clinical_preprocessors, output_dir=PHASE5_OUTPUT_DIR):
    """Save fold-wise biomarker stats and tabular schema metadata."""
    ensure_dir(output_dir)
    clinical_cols = list(next(iter(clinical_preprocessors.values())).clinical_cols) if clinical_preprocessors else []
    exclude = set(clinical_cols) | {'label', 'fold'} | set(LAYER_PATH_COLS.values()) | {'patient_id', 'cohort', 'cohort_folder', 'eye', 'has_all_layers'}
    biomarker_specs = infer_phase5_biomarker_specs(fold_df, exclude_cols=exclude)
    biomarker_names = phase5_biomarker_names(biomarker_specs)
    summary_rows = []
    biomarker_stats_by_fold = {}
    for fold in sorted(clinical_preprocessors):
        train_df = fold_df[fold_df['fold'] != fold].copy() if 'fold' in fold_df else fold_df.copy()
        stats = fit_phase5_biomarker_stats(train_df, biomarker_specs)
        biomarker_stats_by_fold[fold] = stats
        save_json(stats, Path(output_dir) / f'phase5_biomarker_stats_fold{fold}.json')
        pre = clinical_preprocessors[fold]
        ordinal_indices, ordinal_cardinalities = phase5_ordinal_metadata(pre)
        schema = {
            'fold': fold,
            'clinical_cols': pre.clinical_cols,
            'clinical_dim': len(pre.clinical_cols),
            'biomarker_names': biomarker_names,
            'biomarker_specs': biomarker_specs,
            'biomarker_dim': len(biomarker_specs),
            'd_tab': len(pre.clinical_cols) + len(biomarker_specs),
            'ordinal_indices': ordinal_indices,
            'ordinal_cardinalities': ordinal_cardinalities,
            'output_dim': PHASE5_OUTPUT_DIM,
            'architecture': 'LayerNorm -> Linear(D_tab,128) -> GELU -> Dropout(0.3) -> Linear(128,256) -> GELU -> LayerNorm -> Dropout(0.2)',
            'missingness': 'learned per-feature scalar missing tokens; biomarker missing tokens represent absent/QC-failed biomarkers; 10% observed clinical features masked during training',
            'eye_specific_biomarkers': 'columns ending _OD/_OS are resolved by each row eye and exported as one feature name',
        }
        save_json(schema, Path(output_dir) / f'phase5_tabular_schema_fold{fold}.json')
        summary_rows.append({k: schema[k] for k in ['fold', 'clinical_dim', 'biomarker_dim', 'd_tab', 'output_dim']})
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(Path(output_dir) / 'phase5_tabular_encoder_summary.csv', index=False)
    return biomarker_specs, biomarker_stats_by_fold, summary


if 'phase2_folds' in globals() and 'clinical_preprocessors' in globals():
    phase5_biomarker_specs, phase5_biomarker_stats_by_fold, phase5_summary = build_phase5_tabular_artifacts(
        phase2_folds, clinical_preprocessors, output_dir=PHASE5_OUTPUT_DIR
    )
    display(phase5_summary)
    print('Phase 5 biomarker features detected:', phase5_biomarker_names(phase5_biomarker_specs))
    print('Phase 5 artifacts saved to:', PHASE5_OUTPUT_DIR.resolve())

    if torch is not None and clinical_preprocessors:
        fold0 = sorted(clinical_preprocessors)[0]
        encoder0 = make_phase5_tabular_encoder(clinical_preprocessors[fold0], biomarker_specs=phase5_biomarker_specs)
        b = 4
        clinical_dim = len(clinical_preprocessors[fold0].clinical_cols)
        biomarker_dim = len(phase5_biomarker_specs)
        dummy_clin = torch.zeros(b, clinical_dim)
        dummy_mask = torch.ones(b, clinical_dim)
        dummy_bio = torch.zeros(b, biomarker_dim) if biomarker_dim else None
        dummy_bio_mask = torch.zeros(b, biomarker_dim) if biomarker_dim else None
        with torch.no_grad():
            y = encoder0(dummy_clin, dummy_mask, biomarkers=dummy_bio, biomarker_mask=dummy_bio_mask)
        print('Phase 5 encoder smoke-test output shape:', tuple(y.shape))
else:
    print('Phase 5 classes defined. Run Phase 2/fast-reload first to build fold-wise tabular artifacts.')

In [ ]:
# ============================================================
# Phase 6 — Cross-Layer Attention Module
# ============================================================

PHASE6_OUTPUT_DIR = ensure_dir('outputs/segmentation_comparison_8class/shared/phase6_cross_layer_attention')
PHASE6_LAYERS = ['sup', 'deep', 'cc']
PHASE6_TOKEN_DIM = 256
PHASE6_NUM_HEADS = 8
PHASE6_HEAD_DIM = PHASE6_TOKEN_DIM // PHASE6_NUM_HEADS
PHASE6_FFN_DIM = 1024
PHASE6_DROPOUT = 0.1
PHASE6_NUM_BLOCKS = 1
PHASE6_RUN_SMOKE_TEST = True

if PHASE6_TOKEN_DIM % PHASE6_NUM_HEADS != 0:
    raise ValueError('PHASE6_TOKEN_DIM must be divisible by PHASE6_NUM_HEADS')
if PHASE6_HEAD_DIM != 32:
    raise ValueError(f'PLAN expects head_dim=32, got {PHASE6_HEAD_DIM}')


class Phase6TransformerEncoderBlock(nn.Module):
    """Pre-norm Transformer encoder block for 3 OCTA layer tokens."""
    def __init__(self, token_dim=PHASE6_TOKEN_DIM, num_heads=PHASE6_NUM_HEADS, ffn_dim=PHASE6_FFN_DIM, dropout=PHASE6_DROPOUT):
        super().__init__()
        self.norm_attn = nn.LayerNorm(token_dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=token_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attn_dropout = nn.Dropout(dropout)
        self.norm_ffn = nn.LayerNorm(token_dim)
        self.ffn = nn.Sequential(
            nn.Linear(token_dim, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, token_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, return_attention=True):
        qkv = self.norm_attn(x)
        if return_attention:
            try:
                attn_out, attn_weights = self.attn(
                    qkv, qkv, qkv,
                    need_weights=True,
                    average_attn_weights=False,
                )
            except TypeError:
                # Older PyTorch returns attention averaged over heads: [B, 3, 3].
                attn_out, attn_weights = self.attn(qkv, qkv, qkv, need_weights=True)
                attn_weights = attn_weights.unsqueeze(1)
        else:
            attn_out, attn_weights = self.attn(qkv, qkv, qkv, need_weights=False)
            attn_weights = None
        x = x + self.attn_dropout(attn_out)
        x = x + self.ffn(self.norm_ffn(x))
        return x, attn_weights


class Phase6CrossLayerAttention(nn.Module):
    """Layer-aware self-attention over ordered [Sup, Deep, CC] image tokens."""
    def __init__(
        self,
        token_dim=PHASE6_TOKEN_DIM,
        num_heads=PHASE6_NUM_HEADS,
        ffn_dim=PHASE6_FFN_DIM,
        dropout=PHASE6_DROPOUT,
        num_blocks=PHASE6_NUM_BLOCKS,
        layers=PHASE6_LAYERS,
    ):
        super().__init__()
        self.token_dim = int(token_dim)
        self.layers = list(layers)
        if len(self.layers) != 3:
            raise ValueError('Phase 6 expects exactly three layer tokens: sup, deep, cc')
        self.layer_identity = nn.Parameter(torch.empty(1, len(self.layers), self.token_dim))
        nn.init.normal_(self.layer_identity, mean=0.0, std=0.02)
        self.blocks = nn.ModuleList([
            Phase6TransformerEncoderBlock(token_dim=token_dim, num_heads=num_heads, ffn_dim=ffn_dim, dropout=dropout)
            for _ in range(num_blocks)
        ])
        self.output_norm = nn.LayerNorm(token_dim)
        self.last_attention_weights = None

    def forward(self, layer_tokens, return_attention=True):
        """
        Args:
            layer_tokens: Tensor [B, 3, 256], ordered as [sup, deep, cc].

        Returns:
            dict with F_octa [B, 256], attended tokens [B, 3, 256], and attention [B, H, 3, 3].
        """
        if layer_tokens.ndim != 3:
            raise ValueError(f'layer_tokens must be [B, 3, D], got {tuple(layer_tokens.shape)}')
        if layer_tokens.shape[1] != len(self.layers):
            raise ValueError(f'Expected {len(self.layers)} layer tokens, got {layer_tokens.shape[1]}')
        if layer_tokens.shape[2] != self.token_dim:
            raise ValueError(f'Expected token_dim={self.token_dim}, got {layer_tokens.shape[2]}')

        x = layer_tokens + self.layer_identity.to(dtype=layer_tokens.dtype, device=layer_tokens.device)
        attention_by_block = []
        for block in self.blocks:
            x, attn = block(x, return_attention=return_attention)
            if attn is not None:
                attention_by_block.append(attn)
        x = self.output_norm(x)
        f_octa = x.mean(dim=1)

        attention_weights = attention_by_block[-1] if attention_by_block else None
        self.last_attention_weights = attention_weights
        return {
            'F_octa': f_octa,
            'attended_layer_tokens': x,
            'attention_weights': attention_weights,
            'attention_weights_by_block': attention_by_block,
            'layer_identity_embeddings': self.layer_identity,
            'layer_order': list(self.layers),
        }


class Phase6LayerAwareOCTAEncoder(nn.Module):
    """Convenience wrapper: Phase 4 image encoders -> Phase 6 cross-layer attention."""
    def __init__(self, image_encoders, cross_layer_attention=None):
        super().__init__()
        self.image_encoders = image_encoders
        self.cross_layer_attention = cross_layer_attention or Phase6CrossLayerAttention()

    def forward(self, batch, return_spatial=True, return_attention=True):
        image_out = self.image_encoders(batch, return_spatial=return_spatial)
        attention_out = self.cross_layer_attention(image_out['layer_tokens'], return_attention=return_attention)
        return {**image_out, **attention_out}


def phase6_layer_dominance(attention_weights, layer_names=PHASE6_LAYERS):
    """Compute dominance_k = mean attention received by layer k over queries and heads."""
    if attention_weights is None:
        raise ValueError('attention_weights is None')
    attn = attention_weights
    if attn.ndim == 3:  # [B, 3, 3]
        attn = attn.unsqueeze(1)
    if attn.ndim != 4:
        raise ValueError(f'Expected attention [B,H,3,3], got {tuple(attn.shape)}')
    dominance = attn.mean(dim=(1, 2))  # average over heads and query tokens -> [B, 3]
    return pd.DataFrame(
        dominance.detach().cpu().numpy(),
        columns=[f'{name}_dominance' for name in layer_names],
    )


def phase6_attention_audit(attention_weights):
    """Basic sanity checks for self-attention probabilities."""
    if attention_weights is None:
        return {'has_attention': False}
    attn = attention_weights.detach().float().cpu()
    if attn.ndim == 3:
        attn = attn.unsqueeze(1)
    row_sums = attn.sum(dim=-1)
    return {
        'has_attention': True,
        'shape': tuple(attn.shape),
        'row_sum_min': float(row_sums.min()),
        'row_sum_max': float(row_sums.max()),
        'row_sum_mean': float(row_sums.mean()),
        'attention_min': float(attn.min()),
        'attention_max': float(attn.max()),
    }


def build_phase6_cross_layer_attention(device=None):
    model = Phase6CrossLayerAttention()
    if device is not None:
        model = model.to(device)
    return model


phase6_summary = {
    'token_dim': PHASE6_TOKEN_DIM,
    'num_heads': PHASE6_NUM_HEADS,
    'head_dim': PHASE6_HEAD_DIM,
    'ffn_dim': PHASE6_FFN_DIM,
    'dropout': PHASE6_DROPOUT,
    'num_blocks': PHASE6_NUM_BLOCKS,
    'layers': PHASE6_LAYERS,
    'aggregation': 'mean_pool_attended_tokens',
    'output': 'F_octa [B, 256]',
}
save_json(phase6_summary, PHASE6_OUTPUT_DIR / 'phase6_cross_layer_attention_summary.json')

if PHASE6_RUN_SMOKE_TEST:
    phase6_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    phase6_model = build_phase6_cross_layer_attention(device=phase6_device)
    phase6_model.eval()
    dummy_tokens = torch.randn(4, 3, PHASE6_TOKEN_DIM, device=phase6_device)
    with torch.no_grad():
        phase6_out = phase6_model(dummy_tokens, return_attention=True)
    audit = phase6_attention_audit(phase6_out['attention_weights'])
    dominance_df = phase6_layer_dominance(phase6_out['attention_weights'])
    pd.DataFrame([audit]).to_csv(PHASE6_OUTPUT_DIR / 'phase6_smoke_attention_audit.csv', index=False)
    dominance_df.to_csv(PHASE6_OUTPUT_DIR / 'phase6_smoke_layer_dominance.csv', index=False)
    print('Phase 6 smoke test OK')
    print('F_octa:', tuple(phase6_out['F_octa'].shape))
    print('attended_layer_tokens:', tuple(phase6_out['attended_layer_tokens'].shape))
    print('attention audit:', audit)
    display(dominance_df.head())
else:
    print('Phase 6 code ready. Set PHASE6_RUN_SMOKE_TEST=True to test cross-layer attention.')

In [ ]:
# ============================================================
# Phase 7 — Multimodal Cross-Attention Fusion
# ============================================================

PHASE7_OUTPUT_DIR = ensure_dir('outputs/segmentation_comparison_8class/shared/phase7_multimodal_fusion')
PHASE7_TOKEN_DIM = 256
PHASE7_MASK_CHANNELS = 5
PHASE7_MASK_FEATURE_DIM = 128
PHASE7_FUSED_DIM = 512
PHASE7_CROSS_ATTENTION_HEADS = 4
PHASE7_DROPOUT = 0.10
PHASE7_RUN_SMOKE_TEST = True

if torch is None:
    raise ImportError('PyTorch is required for Phase 7 multimodal fusion.')


class Phase7CrossAttentionFusion(nn.Module):
    """OCTA query attends to tabular/biomarker context, then fuses mask branch.

    Inputs:
      f_octa:    [B, 256] from Phase 6 cross-layer attention
      f_tabular: [B, 256] from Phase 5 tabular+biomarker encoder
      f_mask:    [B, 128] from generated-mask encoder

    Output:
      h: [B, 512] unified multimodal representation for Phase 9 classifier.
    """
    def __init__(self, token_dim=PHASE7_TOKEN_DIM, mask_dim=PHASE7_MASK_FEATURE_DIM,
                 fused_dim=PHASE7_FUSED_DIM, num_heads=PHASE7_CROSS_ATTENTION_HEADS,
                 dropout=PHASE7_DROPOUT):
        super().__init__()
        self.token_dim = int(token_dim)
        self.mask_dim = int(mask_dim)
        self.fused_dim = int(fused_dim)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=self.token_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.octa_norm = nn.LayerNorm(self.token_dim)
        self.tabular_norm = nn.LayerNorm(self.token_dim)
        self.post_attn_norm = nn.LayerNorm(self.token_dim)
        self.mask_projection = nn.Sequential(
            nn.LayerNorm(self.mask_dim),
            nn.Linear(self.mask_dim, self.mask_dim),
            nn.GELU(),
            nn.LayerNorm(self.mask_dim),
        )
        # PLAN Phase 7: concat attended OCTA + mask, then include tabular again -> 512.
        self.output_projection = nn.Sequential(
            nn.Linear(self.token_dim + self.mask_dim + self.token_dim, self.fused_dim),
            nn.GELU(),
            nn.LayerNorm(self.fused_dim),
        )

    def forward(self, f_octa, f_tabular, f_mask, need_weights=True):
        if f_octa.ndim != 2 or f_tabular.ndim != 2 or f_mask.ndim != 2:
            raise ValueError('Expected f_octa [B,256], f_tabular [B,256], f_mask [B,128].')
        q = self.octa_norm(f_octa).unsqueeze(1)       # [B, 1, 256]
        kv = self.tabular_norm(f_tabular).unsqueeze(1) # [B, 1, 256]
        attended, attn_weights = self.cross_attention(q, kv, kv, need_weights=need_weights)
        attended = attended.squeeze(1)
        f_cross = self.post_attn_norm(f_octa + attended)
        f_mask_proj = self.mask_projection(f_mask)
        h = self.output_projection(torch.cat([f_cross, f_mask_proj, f_tabular], dim=1))
        return {
            'h': h,
            'f_fused_octa_tabular': f_cross,
            'f_mask': f_mask_proj,
            'cross_attention_weights': attn_weights,
        }


class Phase7MaskEncoder(nn.Module):
    """Lightweight generated-mask branch: 5 masks -> 128-d feature.

    Expected mask order:
      [sup_vessel, deep_vessel, sup_faz, deep_faz, cc_flow]
    Masks are generated pseudo-masks from Phase 8, not human ground truth.
    """
    def __init__(self, in_channels=PHASE7_MASK_CHANNELS, output_dim=PHASE7_MASK_FEATURE_DIM):
        super().__init__()
        self.in_channels = int(in_channels)
        self.output_dim = int(output_dim)
        self.features = nn.Sequential(
            nn.Conv2d(self.in_channels, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, self.output_dim),
            nn.GELU(),
            nn.LayerNorm(self.output_dim),
        )

    def forward(self, masks):
        if masks.ndim != 4 or masks.size(1) != self.in_channels:
            raise ValueError(f'Expected masks [B,{self.in_channels},H,W], got {tuple(masks.shape)}')
        return self.projection(self.features(masks.float()))


def phase7_stack_generated_masks(batch):
    """Stack Phase 2/8 mask tensors into [B,5,224,224]."""
    keys = ['sup_vessel_mask', 'deep_vessel_mask', 'sup_faz_mask', 'deep_faz_mask', 'cc_flow_mask']
    missing = [k for k in keys if k not in batch]
    if missing:
        raise KeyError(f'Missing generated mask keys for Phase 7: {missing}')
    masks = []
    for key in keys:
        x = batch[key]
        if x.ndim == 3:          # [B,H,W]
            x = x.unsqueeze(1)
        elif x.ndim == 2:        # [H,W]
            x = x.unsqueeze(0).unsqueeze(0)
        elif x.ndim != 4:
            raise ValueError(f'{key} must be [B,1,H,W], [B,H,W], or [H,W]; got {tuple(x.shape)}')
        masks.append(x.float())
    return torch.cat(masks, dim=1)


def _phase7_extract_f_octa(octa_output):
    """Accept likely Phase 6 output key variants."""
    for key in ('f_octa', 'F_octa', 'octa_features', 'layer_aware_octa'):
        if isinstance(octa_output, dict) and key in octa_output:
            return octa_output[key]
    if torch.is_tensor(octa_output):
        return octa_output
    raise KeyError('Could not find F_octa in Phase 6 output.')


class Phase7MultimodalFusionModel(nn.Module):
    """Composable Phase 7 wrapper around Phase 6 OCTA, Phase 5 tabular, and mask branch."""
    def __init__(self, octa_encoder=None, tabular_encoder=None, mask_encoder=None, fusion=None):
        super().__init__()
        self.octa_encoder = octa_encoder
        self.tabular_encoder = tabular_encoder
        self.mask_encoder = mask_encoder or Phase7MaskEncoder()
        self.fusion = fusion or Phase7CrossAttentionFusion()

    def forward(self, batch, f_octa=None, f_tabular=None, masks=None, **tabular_kwargs):
        if f_octa is None:
            if self.octa_encoder is None:
                raise ValueError('Provide f_octa or attach a Phase 6 octa_encoder.')
            f_octa = _phase7_extract_f_octa(self.octa_encoder(batch))
        if f_tabular is None:
            if self.tabular_encoder is None:
                raise ValueError('Provide f_tabular or attach a Phase 5 tabular_encoder.')
            f_tabular = self.tabular_encoder(
                batch['clinical_features'],
                batch['clinical_mask'],
                biomarkers=batch.get('biomarkers'),
                biomarker_mask=batch.get('biomarker_mask'),
                **tabular_kwargs,
            )
            if isinstance(f_tabular, dict):
                f_tabular = f_tabular.get('f_tabular', f_tabular.get('features'))
        if masks is None:
            masks = phase7_stack_generated_masks(batch)
        f_mask = self.mask_encoder(masks)
        out = self.fusion(f_octa, f_tabular, f_mask)
        out.update({'f_octa': f_octa, 'f_tabular': f_tabular, 'mask_source': batch.get('mask_source', 'generated')})
        return out


def build_phase7_multimodal_fusion(octa_encoder=None, tabular_encoder=None, device=None):
    model = Phase7MultimodalFusionModel(octa_encoder=octa_encoder, tabular_encoder=tabular_encoder)
    if device is not None:
        model = model.to(device)
    return model


# Smoke test: no image reads, no Phase 8 masks needed. Uses placeholder generated masks.
phase7_summary = {
    'token_dim': PHASE7_TOKEN_DIM,
    'mask_channels': PHASE7_MASK_CHANNELS,
    'mask_feature_dim': PHASE7_MASK_FEATURE_DIM,
    'output_dim': PHASE7_FUSED_DIM,
    'cross_attention_heads': PHASE7_CROSS_ATTENTION_HEADS,
    'mask_order': ['sup_vessel_mask', 'deep_vessel_mask', 'sup_faz_mask', 'deep_faz_mask', 'cc_flow_mask'],
    'mask_source_assumption': 'generated pseudo-masks; no human ground-truth masks assumed',
}

if PHASE7_RUN_SMOKE_TEST:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    fusion_model = build_phase7_multimodal_fusion(device=device)
    fusion_model.eval()
    with torch.no_grad():
        b = 2
        fake_batch = {
            'sup_vessel_mask': torch.zeros(b, 1, PHASE4_IMAGE_SIZE, PHASE4_IMAGE_SIZE, device=device),
            'deep_vessel_mask': torch.zeros(b, 1, PHASE4_IMAGE_SIZE, PHASE4_IMAGE_SIZE, device=device),
            'sup_faz_mask': torch.zeros(b, 1, PHASE4_IMAGE_SIZE, PHASE4_IMAGE_SIZE, device=device),
            'deep_faz_mask': torch.zeros(b, 1, PHASE4_IMAGE_SIZE, PHASE4_IMAGE_SIZE, device=device),
            'cc_flow_mask': torch.zeros(b, 1, PHASE4_IMAGE_SIZE, PHASE4_IMAGE_SIZE, device=device),
            'mask_source': 'classical_placeholder',
        }
        out = fusion_model(
            fake_batch,
            f_octa=torch.randn(b, PHASE7_TOKEN_DIM, device=device),
            f_tabular=torch.randn(b, PHASE7_TOKEN_DIM, device=device),
        )
    phase7_summary.update({
        'smoke_test_device': str(device),
        'h_shape': list(out['h'].shape),
        'cross_attention_shape': None if out['cross_attention_weights'] is None else list(out['cross_attention_weights'].shape),
        'smoke_test_passed': list(out['h'].shape) == [b, PHASE7_FUSED_DIM],
    })
    print('Phase 7 smoke test:', phase7_summary)

save_json(phase7_summary, Path(PHASE7_OUTPUT_DIR) / 'phase7_multimodal_fusion_summary.json')
pd.DataFrame([phase7_summary]).to_csv(Path(PHASE7_OUTPUT_DIR) / 'phase7_multimodal_fusion_summary.csv', index=False)

In [ ]:
# ============================================================
# Phase 8 — Self-Generated Segmentation Masks and Biomarkers
# ============================================================

PHASE8_OUTPUT_DIR = ensure_dir('outputs/segmentation_comparison_8class/shared/phase8_classical_pseudo_labels')
PHASE8_MASK_DIR = ensure_dir(PHASE8_OUTPUT_DIR / 'masks')
PHASE8_MONTAGE_DIR = ensure_dir(PHASE8_OUTPUT_DIR / 'montages')
PHASE8_IMAGE_SIZE = PHASE4_IMAGE_SIZE if 'PHASE4_IMAGE_SIZE' in globals() else IMAGE_SIZE
PHASE8_MASK_ORDER = ['sup_vessel_mask', 'deep_vessel_mask', 'sup_faz_mask', 'deep_faz_mask', 'cc_flow_mask']
PHASE8_RUN_GENERATION = True  # set True on the data machine after Phase 2/fast reload
PHASE8_MAX_MONTAGE_SAMPLES = 48

import matplotlib.pyplot as plt

try:
    from skimage import filters, morphology, measure, segmentation, util
    from skimage.filters import threshold_sauvola, threshold_local
except Exception as e:
    filters = morphology = measure = segmentation = util = None
    threshold_sauvola = threshold_local = None
    print('Phase 8 will use OpenCV/PIL fallbacks; scikit-image imports failed:', e)


def _phase8_safe_slug(value):
    return re.sub(r'[^a-zA-Z0-9_.-]+', '_', str(value)).strip('_')


def _phase8_uint8(arr01):
    return (np.clip(arr01, 0, 1) * 255).astype(np.uint8)


def _phase8_remove_small_objects_compat(mask, min_size):
    # scikit-image >=0.26 deprecates min_size in favour of max_size.
    # Old behaviour removed components with area < min_size; new max_size removes <= max_size.
    min_size = max(int(min_size), 1)
    try:
        return morphology.remove_small_objects(mask, max_size=max(min_size - 1, 0))
    except TypeError:
        return morphology.remove_small_objects(mask, min_size=min_size)


def _phase8_remove_small_holes_compat(mask, area_threshold):
    # Preserve old area_threshold semantics (< threshold) while avoiding >=0.26 warnings.
    area_threshold = max(int(area_threshold), 1)
    try:
        return morphology.remove_small_holes(mask, max_size=max(area_threshold - 1, 0))
    except TypeError:
        return morphology.remove_small_holes(mask, area_threshold=area_threshold)


def _phase8_binary_cleanup(mask, min_size=24, radius=1, fill_holes=True):
    mask = np.asarray(mask, dtype=bool)
    if morphology is not None:
        if radius > 0:
            selem = morphology.disk(int(radius))
            mask = morphology.opening(mask, footprint=selem)
            mask = morphology.closing(mask, footprint=selem)
        mask = _phase8_remove_small_objects_compat(mask, min_size=int(min_size))
        if fill_holes:
            mask = _phase8_remove_small_holes_compat(mask, area_threshold=max(int(min_size), 16))
    elif cv2 is not None:
        kernel = np.ones((3, 3), np.uint8)
        u8 = mask.astype(np.uint8)
        u8 = cv2.morphologyEx(u8, cv2.MORPH_OPEN, kernel)
        u8 = cv2.morphologyEx(u8, cv2.MORPH_CLOSE, kernel)
        mask = u8.astype(bool)
    return mask.astype(np.float32)


def _phase8_vesselness(arr01):
    arr01 = np.clip(arr01, 0, 1).astype(np.float32)
    if filters is not None:
        try:
            v = filters.frangi(arr01, sigmas=(1, 2, 3), black_ridges=False)
        except Exception:
            v = filters.sato(arr01, sigmas=(1, 2, 3), black_ridges=False)
        v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)
        return percentile_normalise(v, lower=1, upper=99)
    if cv2 is not None:
        blurred = cv2.GaussianBlur(arr01, (0, 0), 1.0)
        return percentile_normalise(arr01 - blurred, lower=1, upper=99)
    return arr01


def _phase8_remove_watermark_corner(mask, corner_h_frac=0.10, corner_w_frac=0.18):
    """Suppress bottom-right vendor watermark region from generated masks."""
    mask = np.asarray(mask, dtype=bool).copy()
    h, w = mask.shape
    mask[int(h * (1 - corner_h_frac)):h, int(w * (1 - corner_w_frac)):w] = False
    return mask


def phase8_generate_vessel_mask(arr01, target_density=(0.045, 0.18)):
    """Improved classical SVC/DVC vessel pseudo-mask, independent of disease labels.

    Combines multiscale vesselness with local bright-vessel contrast. The previous
    rule was too conservative and mainly kept large/edge vessels; this version
    explicitly targets a plausible OCTA vessel-density band and uses lighter
    morphology so capillary texture is not removed.
    """
    arr01 = np.clip(arr01, 0, 1).astype(np.float32)
    if cv2 is not None:
        smooth = cv2.GaussianBlur(arr01, (0, 0), 0.45)
        bg = cv2.GaussianBlur(arr01, (0, 0), 4.0)
        local_contrast = percentile_normalise(arr01 - bg, lower=1, upper=99)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
        top_hat = cv2.morphologyEx(_phase8_uint8(arr01), cv2.MORPH_TOPHAT, kernel).astype(np.float32) / 255.0
        top_hat = percentile_normalise(top_hat, lower=1, upper=99)
    else:
        smooth = arr01
        local_contrast = percentile_normalise(arr01 - np.median(arr01), lower=1, upper=99)
        top_hat = local_contrast

    vesselness = _phase8_vesselness(smooth)
    score = percentile_normalise(0.55 * vesselness + 0.30 * local_contrast + 0.15 * top_hat, lower=1, upper=99)

    masks = []
    if threshold_local is not None:
        for block, offset in [(31, -0.035), (45, -0.025), (65, -0.015)]:
            local = threshold_local(score, block_size=block, offset=offset)
            masks.append(score > local)
    if cv2 is not None:
        u8 = _phase8_uint8(score)
        for block, c in [(31, -4), (45, -3), (65, -2)]:
            masks.append(cv2.adaptiveThreshold(u8, 1, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, block, c).astype(bool))
    for pct in [82, 78, 74, 70]:
        masks.append(score > np.percentile(score, pct))

    mask = np.zeros_like(score, dtype=bool)
    for candidate in masks:
        candidate = _phase8_remove_watermark_corner(candidate)
        candidate = _phase8_binary_cleanup(candidate, min_size=6, radius=0, fill_holes=False) > 0.5
        density = float(candidate.mean())
        if target_density[0] <= density <= target_density[1]:
            mask = candidate
            break
        if density > mask.mean() and density <= 0.25:
            mask = candidate

    if mask.mean() < target_density[0]:
        relaxed = score > np.percentile(score, 68)
        mask = _phase8_binary_cleanup(_phase8_remove_watermark_corner(relaxed), min_size=4, radius=0, fill_holes=False) > 0.5
    elif mask.mean() > target_density[1]:
        stricter = score > np.percentile(score, 100 * (1 - target_density[1]))
        mask = _phase8_binary_cleanup(_phase8_remove_watermark_corner(stricter), min_size=6, radius=0, fill_holes=False) > 0.5

    # Light closing connects broken capillary fragments without erasing fine vessels.
    if morphology is not None:
        mask = morphology.closing(mask, footprint=morphology.disk(1))
        mask = _phase8_remove_small_objects_compat(mask, min_size=4).astype(bool)
    mask = _phase8_remove_watermark_corner(mask)
    skeleton = morphology.skeletonize(mask > 0.5).astype(np.float32) if morphology is not None else mask.astype(np.float32)
    return mask.astype(np.float32), skeleton.astype(np.float32), score.astype(np.float32)


def _phase8_component_props(mask):
    if measure is not None:
        labels = measure.label(mask.astype(bool))
        return measure.regionprops(labels)
    if cv2 is None:
        return []
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    props = []
    for i in range(1, n):
        props.append({'area': stats[i, cv2.CC_STAT_AREA], 'centroid': centroids[i], 'mask': labels == i})
    return props


def phase8_detect_faz_mask(arr01, vessel_mask):
    """Central avascular pseudo-mask for Sup/Deep layers."""
    h, w = vessel_mask.shape
    cy, cx = h / 2.0, w / 2.0
    yy, xx = np.mgrid[:h, :w]
    center_radius = 0.28 * min(h, w)
    central = ((yy - cy) ** 2 + (xx - cx) ** 2) <= center_radius ** 2
    low_flow = (arr01 < np.percentile(arr01[central], 45)) if np.any(central) else (arr01 < np.percentile(arr01, 45))
    candidates = (~(vessel_mask > 0.5)) & central & low_flow
    candidates = _phase8_binary_cleanup(candidates, min_size=32, radius=2, fill_holes=True) > 0.5
    best = np.zeros_like(candidates, dtype=bool)
    best_score = np.inf
    for prop in _phase8_component_props(candidates):
        if isinstance(prop, dict):
            area = float(prop['area'])
            px, py = prop['centroid']
            centroid = (py, px)
            label_mask = prop.get('mask')
        else:
            area = float(prop.area)
            centroid = prop.centroid
            label_mask = prop.image
        area_frac = area / float(h * w)
        dist = float(np.hypot(centroid[0] - cy, centroid[1] - cx) / min(h, w))
        if area_frac < 0.0005 or area_frac > 0.08 or dist > 0.18:
            continue
        score = dist + abs(np.log(max(area_frac, 1e-6) / 0.012)) * 0.03
        if score < best_score:
            best_score = score
            if isinstance(prop, dict) and label_mask is not None:
                best = label_mask.astype(bool)
            elif label_mask is not None:
                best = np.zeros_like(candidates, dtype=bool)
                minr, minc, maxr, maxc = prop.bbox
                best[minr:maxr, minc:maxc] = label_mask
            else:
                best = candidates
    best = _phase8_binary_cleanup(best, min_size=24, radius=2, fill_holes=True)
    return best.astype(np.float32)


def phase8_generate_cc_flow_mask(arr01):
    """Dark choriocapillaris flow-deficit pseudo-mask."""
    arr01 = np.clip(arr01, 0, 1).astype(np.float32)
    if cv2 is not None:
        background = cv2.GaussianBlur(arr01, (0, 0), 8.0)
        corrected = np.clip(arr01 - background + float(np.median(background)), 0, 1)
    else:
        corrected = arr01
    if threshold_sauvola is not None:
        thresh = threshold_sauvola(corrected, window_size=31, k=0.15)
        mask = corrected < thresh
    elif cv2 is not None:
        u8 = _phase8_uint8(corrected)
        mask = cv2.adaptiveThreshold(u8, 1, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 2).astype(bool)
    else:
        mask = corrected < np.percentile(corrected, 35)
    density = float(np.mean(mask))
    if density < 0.01 or density > 0.70:
        mask = corrected < np.percentile(corrected, 30)
    return _phase8_binary_cleanup(mask, min_size=12, radius=1, fill_holes=False).astype(np.float32)


def phase8_mask_qc(mask, kind):
    mask = np.asarray(mask > 0.5, dtype=bool)
    h, w = mask.shape
    area_frac = float(mask.mean())
    qc = {f'{kind}_area_fraction': area_frac, f'{kind}_qc_pass': True, f'{kind}_qc_reason': ''}
    if kind.endswith('vessel') and not (0.035 <= area_frac <= 0.30):
        qc[f'{kind}_qc_pass'] = False
        qc[f'{kind}_qc_reason'] = 'implausible_vessel_density'
    elif kind.endswith('faz'):
        if not (0.0005 <= area_frac <= 0.08):
            qc[f'{kind}_qc_pass'] = False
            qc[f'{kind}_qc_reason'] = 'implausible_faz_area'
        elif measure is not None and mask.any():
            prop = max(measure.regionprops(measure.label(mask)), key=lambda p: p.area)
            dist = float(np.hypot(prop.centroid[0] - h / 2, prop.centroid[1] - w / 2) / min(h, w))
            qc[f'{kind}_centroid_distance'] = dist
            if dist > 0.20:
                qc[f'{kind}_qc_pass'] = False
                qc[f'{kind}_qc_reason'] = 'off_center_faz'
    elif kind == 'cc_flow' and not (0.005 <= area_frac <= 0.70):
        qc[f'{kind}_qc_pass'] = False
        qc[f'{kind}_qc_reason'] = 'implausible_flow_deficit_fraction'
    return qc


def phase8_extract_biomarkers(mask_dict, skeleton_dict=None):
    skeleton_dict = skeleton_dict or {}
    out = {}
    for layer in ['sup', 'deep']:
        vessel = mask_dict[f'{layer}_vessel_mask'] > 0.5
        faz = mask_dict[f'{layer}_faz_mask'] > 0.5
        out[f'phase8_{layer}_vessel_density'] = float(vessel.mean())
        skel = skeleton_dict.get(layer)
        out[f'phase8_{layer}_skeleton_density'] = float((skel > 0.5).mean()) if skel is not None else np.nan
        out[f'phase8_{layer}_faz_area'] = float(faz.mean())
        if measure is not None and faz.any():
            prop = max(measure.regionprops(measure.label(faz)), key=lambda p: p.area)
            perimeter = float(measure.perimeter(faz))
            area_px = float(prop.area)
            circularity = float(4 * np.pi * area_px / max(perimeter ** 2, 1e-6))
            h, w = faz.shape
            centroid_dist = float(np.hypot(prop.centroid[0] - h / 2, prop.centroid[1] - w / 2) / min(h, w))
        else:
            perimeter = circularity = centroid_dist = np.nan
        out[f'phase8_{layer}_faz_perimeter'] = perimeter
        out[f'phase8_{layer}_faz_circularity'] = circularity
        out[f'phase8_{layer}_faz_centroid_distance'] = centroid_dist
    cc = mask_dict['cc_flow_mask'] > 0.5
    out['phase8_cc_flow_deficit_fraction'] = float(cc.mean())
    if measure is not None and cc.any():
        areas = [float(p.area) for p in measure.regionprops(measure.label(cc))]
        out['phase8_cc_flow_deficit_count'] = int(len(areas))
        out['phase8_cc_flow_deficit_mean_area'] = float(np.mean(areas)) / float(cc.size)
        out['phase8_cc_flow_deficit_median_area'] = float(np.median(areas)) / float(cc.size)
    else:
        out['phase8_cc_flow_deficit_count'] = 0
        out['phase8_cc_flow_deficit_mean_area'] = 0.0
        out['phase8_cc_flow_deficit_median_area'] = 0.0
    return out


def phase8_generate_sample_masks(row, image_size=PHASE8_IMAGE_SIZE):
    sup = preprocess_octa_image(row['sup_path'], image_size=image_size, use_clahe=True)
    deep = preprocess_octa_image(row['deep_path'], image_size=image_size, use_clahe=True)
    cc = preprocess_octa_image(row['cc_path'], image_size=image_size, use_clahe=True)
    sup_vessel, sup_skel, _ = phase8_generate_vessel_mask(sup)
    deep_vessel, deep_skel, _ = phase8_generate_vessel_mask(deep)
    masks = {
        'sup_vessel_mask': sup_vessel,
        'deep_vessel_mask': deep_vessel,
        'sup_faz_mask': phase8_detect_faz_mask(sup, sup_vessel),
        'deep_faz_mask': phase8_detect_faz_mask(deep, deep_vessel),
        'cc_flow_mask': phase8_generate_cc_flow_mask(cc),
    }
    skeletons = {'sup': sup_skel, 'deep': deep_skel}
    biomarkers = phase8_extract_biomarkers(masks, skeletons)
    qc = {}
    for key, kind in [('sup_vessel_mask', 'sup_vessel'), ('deep_vessel_mask', 'deep_vessel'), ('sup_faz_mask', 'sup_faz'), ('deep_faz_mask', 'deep_faz'), ('cc_flow_mask', 'cc_flow')]:
        qc.update(phase8_mask_qc(masks[key], kind))
    qc['phase8_mask_qc_pass'] = bool(all(v for k, v in qc.items() if k.endswith('_qc_pass')))
    qc['mask_source'] = 'classical_improved'
    return masks, biomarkers, qc


def _phase8_save_mask(mask, path):
    ensure_dir(Path(path).parent)
    Image.fromarray((np.asarray(mask > 0.5, dtype=np.uint8) * 255), mode='L').save(path)


def phase8_make_montage(row, masks, path):
    arrays = [preprocess_octa_image(row[f'{layer}_path'], image_size=PHASE8_IMAGE_SIZE, use_clahe=True) for layer in ['sup', 'deep', 'cc']]
    titles = ['sup', 'deep', 'cc', 'sup_vessel', 'deep_vessel', 'sup_faz', 'deep_faz', 'cc_flow']
    images = arrays + [masks[k] for k in PHASE8_MASK_ORDER]
    fig, axes = plt.subplots(2, 4, figsize=(10, 5))
    for ax, img, title in zip(axes.ravel(), images, titles):
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        ax.set_title(title, fontsize=9)
        ax.axis('off')
    fig.suptitle(f"{row.get('patient_id')} {row.get('eye')} | label={row.get('label')}", fontsize=10)
    fig.tight_layout()
    ensure_dir(Path(path).parent)
    fig.savefig(path, dpi=150)
    plt.close(fig)


def generate_phase8_artifacts(fold_df, output_dir=PHASE8_OUTPUT_DIR, image_size=PHASE8_IMAGE_SIZE, max_montage_samples=PHASE8_MAX_MONTAGE_SAMPLES):
    """Generate classical pseudo-masks, mask QC, biomarkers, and montage sheets."""
    output_dir = ensure_dir(output_dir)
    mask_dir = ensure_dir(output_dir / 'masks')
    montage_dir = ensure_dir(output_dir / 'montages')
    records = []
    rng = np.random.default_rng(PHASE2_SEED if 'PHASE2_SEED' in globals() else 42)
    montage_indices = set(rng.choice(len(fold_df), size=min(int(max_montage_samples), len(fold_df)), replace=False).tolist()) if len(fold_df) else set()
    for i, (_, row) in enumerate(fold_df.reset_index(drop=True).iterrows()):
        sample_id = f"{_phase8_safe_slug(row.get('patient_id'))}_{_phase8_safe_slug(row.get('eye'))}"
        sample_mask_dir = ensure_dir(mask_dir / sample_id)
        rec = {'patient_id': row.get('patient_id'), 'eye': row.get('eye'), 'fold': int(row.get('fold', -1)), 'label': int(row.get('label', -1)), 'mask_source': 'classical_improved'}
        try:
            masks, biomarkers, qc = phase8_generate_sample_masks(row, image_size=image_size)
            for key, mask in masks.items():
                path = sample_mask_dir / f'{key}.png'
                _phase8_save_mask(mask, path)
                rec[f'{key}_path'] = str(path)
            rec.update(biomarkers)
            rec.update(qc)
            if (i in montage_indices) or (not rec.get('phase8_mask_qc_pass', False)):
                try:
                    phase8_make_montage(row, masks, montage_dir / f'{sample_id}.png')
                except Exception as montage_error:
                    rec['phase8_montage_error'] = repr(montage_error)
        except Exception as e:
            rec['phase8_mask_qc_pass'] = False
            rec['phase8_error'] = repr(e)
        records.append(rec)
        if (i + 1) % 50 == 0:
            print(f'Phase 8 generated {i + 1}/{len(fold_df)} samples')
    artifact_df = pd.DataFrame(records)
    artifact_df.to_csv(output_dir / 'phase8_generated_masks_biomarkers.csv', index=False)
    qc_cols = [c for c in artifact_df.columns if c.endswith('_qc_pass') or c.endswith('_qc_reason') or c.endswith('_area_fraction') or c in ['patient_id', 'eye', 'fold', 'label', 'phase8_error']]
    artifact_df[qc_cols].to_csv(output_dir / 'phase8_mask_qc.csv', index=False)
    summary = {
        'n_samples': int(len(artifact_df)),
        'mask_source': 'classical_improved',
        'mask_order': PHASE8_MASK_ORDER,
        'biomarker_columns': [c for c in artifact_df.columns if c.startswith('phase8_') and not c.endswith('_path') and 'qc' not in c and 'error' not in c],
        'qc_pass_rate': float(artifact_df['phase8_mask_qc_pass'].mean()) if 'phase8_mask_qc_pass' in artifact_df else np.nan,
        'note': 'Deterministic weak pseudo-masks; not human ground truth.',
    }
    save_json(summary, output_dir / 'phase8_generated_masks_summary.json')
    return artifact_df, summary


def phase8_merge_artifacts(fold_df, artifact_df=None, artifact_path=None):
    if artifact_df is None:
        path = Path(artifact_path or PHASE8_OUTPUT_DIR / 'phase8_generated_masks_biomarkers.csv')
        artifact_df = pd.read_csv(path)
    key_cols = ['patient_id', 'eye']
    return fold_df.merge(artifact_df, on=key_cols, how='left', suffixes=('', '_phase8'))


def phase8_load_mask_from_row(row, key, image_size=PHASE8_IMAGE_SIZE):
    path_col = f'{key}_path'
    path = row[path_col] if path_col in row and pd.notna(row[path_col]) else None
    arr = preprocess_mask_image(path, image_size=image_size)
    return torch.from_numpy(arr[None, :, :].astype(np.float32))


class RastaPhase8Dataset(RastaPhase2Dataset):
    """Phase 2 dataset plus generated Phase 8 masks and biomarkers."""
    def __init__(self, df, clinical_preprocessor, layer_stats, biomarker_specs=None, biomarker_stats=None, **kwargs):
        super().__init__(df, clinical_preprocessor, layer_stats, **kwargs)
        self.biomarker_specs = biomarker_specs or []
        self.biomarker_stats = biomarker_stats or {}

    def __getitem__(self, idx):
        sample = super().__getitem__(idx)
        row = self.df.iloc[idx]
        for key in PHASE8_MASK_ORDER:
            sample[key] = phase8_load_mask_from_row(row, key, image_size=self.image_size)
        sample['mask_source'] = str(row.get('mask_source', 'classical_improved'))
        if self.biomarker_specs:
            values, mask = transform_phase5_biomarkers(row, self.biomarker_specs, self.biomarker_stats)
            sample['biomarkers'] = torch.from_numpy(values.astype(np.float32))
            sample['biomarker_mask'] = torch.from_numpy(mask.astype(np.float32))
        return sample


def make_phase8_datasets_for_fold(fold_df_with_phase8, fold, clinical_preprocessor, layer_stats, biomarker_specs=None, biomarker_stats_by_fold=None, **dataset_kwargs):
    train_df = fold_df_with_phase8[fold_df_with_phase8['fold'] != fold].copy()
    val_df = fold_df_with_phase8[fold_df_with_phase8['fold'] == fold].copy()
    validate_no_patient_overlap_between_splits(train_df, val_df)
    biomarker_stats = (biomarker_stats_by_fold or {}).get(fold, {})
    train_ds = RastaPhase8Dataset(train_df, clinical_preprocessor, layer_stats, train=True, biomarker_specs=biomarker_specs, biomarker_stats=biomarker_stats, **dataset_kwargs)
    val_ds = RastaPhase8Dataset(val_df, clinical_preprocessor, layer_stats, train=False, biomarker_specs=biomarker_specs, biomarker_stats=biomarker_stats, **dataset_kwargs)
    return train_ds, val_ds


phase8_summary = {
    'output_dir': str(PHASE8_OUTPUT_DIR),
    'mask_source_default': 'classical_improved',
    'mask_order': PHASE8_MASK_ORDER,
    'run_generation_default': PHASE8_RUN_GENERATION,
    'student_refinement_status': 'not trained by default; add only after classical-mask QC review',
}

if PHASE8_RUN_GENERATION:
    if 'phase2_folds' not in globals():
        raise RuntimeError('Run Phase 2 or fast-reload before Phase 8 generation.')
    phase8_artifacts, phase8_generated_summary = generate_phase8_artifacts(phase2_folds, output_dir=PHASE8_OUTPUT_DIR)
    phase2_folds_with_phase8 = phase8_merge_artifacts(phase2_folds, artifact_df=phase8_artifacts)
    phase2_folds_with_phase8.to_csv(PHASE8_OUTPUT_DIR / 'phase8_manifest_with_masks_biomarkers.csv', index=False)
    phase8_summary.update(phase8_generated_summary)
    print('Phase 8 artifacts saved to:', PHASE8_OUTPUT_DIR.resolve())
else:
    print('Phase 8 classical pseudo-mask pipeline defined. Set PHASE8_RUN_GENERATION=True on the data machine to write masks/biomarkers.')

save_json(phase8_summary, PHASE8_OUTPUT_DIR / 'phase8_pipeline_summary.json')

In [ ]:
# ============================================================
# Phase 8C — Explicit U-Net++ segmentation-student decision gate and definitions
# ============================================================

PHASE8_STUDENT_OUTPUT_DIR = ensure_dir(segcomp_model_dir(SEGCOMP_ACTIVE_MODEL_ID) / 'phase8_segmentation')
import torch.nn.functional as F
import time
PHASE8_STUDENT_DECISION = 'B_train_unetpp_student'  # A_classical_only | B_train_unetpp_student | C_tune_classical_again
PHASE8_STUDENT_DECISION_JUSTIFICATION = (
    'Improved classical masks reached high QC pass rate, but Step C requires explicit evaluation of U-Net++ '
    'unless classical-only is justified after montage/biomarker stability review.'
)
PHASE8_RUN_STUDENT_TRAINING = True  # set True on data/GPU machine after refreshed Phase 8 artifacts exist
PHASE8_STUDENT_EPOCHS = 60
PHASE8_STUDENT_BATCH_SIZE = 16
PHASE8_STUDENT_LR = 3e-4
PHASE8_STUDENT_WEIGHT_DECAY = 1e-4
PHASE8_STUDENT_NUM_WORKERS = 0
PHASE8_STUDENT_AUTO_RESUME = True
PHASE8_STUDENT_LOG_EVERY_N_EPOCHS = 1
PHASE8_STUDENT_MIN_QC_PASS = 0.95
PHASE8_STUDENT_MASK_SOURCE = SEGCOMP_ACTIVE_MASK_SOURCE
PHASE8_STUDENT_THRESHOLD = 0.5
# For a rerun with honest test metrics, use the existing outer Phase 2 fold as test
# and select best epochs on an inner patient-grouped validation split from the remaining folds.
PHASE8_STUDENT_USE_INNER_VAL_FOR_TEST = True
PHASE8_STUDENT_INNER_VAL_FRACTION = 0.20
PHASE8_STUDENT_RANDOM_SEED = PHASE2_SEED if 'PHASE2_SEED' in globals() else 42


def phase8_validate_student_decision(decision=PHASE8_STUDENT_DECISION, justification=PHASE8_STUDENT_DECISION_JUSTIFICATION, artifact_df=None):
    """Enforce PLAN Step C: choose A/B/C explicitly; do not silently skip U-Net++."""
    allowed = {'A_classical_only', 'B_train_unetpp_student', 'C_tune_classical_again'}
    if decision not in allowed:
        raise ValueError(f'PHASE8_STUDENT_DECISION must be one of {sorted(allowed)}, got {decision!r}')
    if not str(justification).strip():
        raise ValueError('Step C requires PHASE8_STUDENT_DECISION_JUSTIFICATION.')
    qc_pass_rate = None
    if artifact_df is not None and 'phase8_mask_qc_pass' in artifact_df:
        qc_pass_rate = float(pd.Series(artifact_df['phase8_mask_qc_pass']).fillna(False).mean())
    if decision == 'A_classical_only':
        if qc_pass_rate is not None and qc_pass_rate < PHASE8_STUDENT_MIN_QC_PASS:
            raise ValueError(f'Decision A invalid: QC pass rate {qc_pass_rate:.3f} < {PHASE8_STUDENT_MIN_QC_PASS:.3f}.')
        if 'unnecessary' not in justification.lower() and 'classical-only' not in justification.lower():
            raise ValueError('Decision A requires explicit reason why student refinement is unnecessary.')
    if decision == 'B_train_unetpp_student' and qc_pass_rate is not None and qc_pass_rate < PHASE8_STUDENT_MIN_QC_PASS:
        raise ValueError(f'Decision B blocked: pseudo-label QC pass rate {qc_pass_rate:.3f} too low; choose C/tune classical first.')
    return {'decision': decision, 'justification': justification, 'qc_pass_rate': qc_pass_rate}


class Phase8StudentPseudoMaskDataset(torch.utils.data.Dataset):
    """QC-passed classical-improved pseudo-labels for frozen upstream U-Net++ student."""
    def __init__(self, df, task='supdeep', image_size=PHASE8_IMAGE_SIZE, train=True):
        self.df = df.copy().reset_index(drop=True)
        if 'phase8_mask_qc_pass' in self.df:
            self.df = self.df[self.df['phase8_mask_qc_pass'].fillna(False)].reset_index(drop=True)
        self.task = task
        self.image_size = int(image_size)
        self.train = bool(train)
        if task not in {'supdeep', 'cc'}:
            raise ValueError("task must be 'supdeep' or 'cc'")
        if task == 'supdeep':
            sup_df = self.df.assign(_seg_layer='sup')
            deep_df = self.df.assign(_seg_layer='deep')
            self.df = pd.concat([sup_df, deep_df], ignore_index=True)

    def __len__(self):
        return len(self.df)

    def _image_tensor(self, path):
        arr = preprocess_octa_image(path, image_size=self.image_size, use_clahe=True).astype(np.float32)
        return torch.from_numpy(arr[None, :, :])

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if self.task == 'supdeep':
            layer = str(row['_seg_layer'])
            x = self._image_tensor(row[f'{layer}_path'])
            vessel = phase8_load_mask_from_row(row, f'{layer}_vessel_mask', image_size=self.image_size)
            faz = phase8_load_mask_from_row(row, f'{layer}_faz_mask', image_size=self.image_size)
            y = torch.cat([vessel, faz], dim=0)
        else:
            x = self._image_tensor(row['cc_path'])
            y = phase8_load_mask_from_row(row, 'cc_flow_mask', image_size=self.image_size)
        return {'image': x.float(), 'mask': y.float(), 'patient_id': str(row.get('patient_id')), 'eye': str(row.get('eye'))}


class _UNetPPConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU(),
        )
    def forward(self, x):
        return self.block(x)


class Phase8UNetPlusPlusStudent(nn.Module):
    """Compact U-Net++ student. 1-channel OCTA -> vessel/FAZ or CC-flow pseudo-mask logits."""
    def __init__(self, in_channels=1, out_channels=2, base=32, deep_supervision=False):
        super().__init__()
        nb = [base, base*2, base*4, base*8, base*16]
        self.deep_supervision = deep_supervision
        self.pool = nn.MaxPool2d(2, 2)
        self.up = lambda x: F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        self.conv0_0 = _UNetPPConvBlock(in_channels, nb[0])
        self.conv1_0 = _UNetPPConvBlock(nb[0], nb[1])
        self.conv2_0 = _UNetPPConvBlock(nb[1], nb[2])
        self.conv3_0 = _UNetPPConvBlock(nb[2], nb[3])
        self.conv4_0 = _UNetPPConvBlock(nb[3], nb[4])
        self.conv0_1 = _UNetPPConvBlock(nb[0] + nb[1], nb[0])
        self.conv1_1 = _UNetPPConvBlock(nb[1] + nb[2], nb[1])
        self.conv2_1 = _UNetPPConvBlock(nb[2] + nb[3], nb[2])
        self.conv3_1 = _UNetPPConvBlock(nb[3] + nb[4], nb[3])
        self.conv0_2 = _UNetPPConvBlock(nb[0]*2 + nb[1], nb[0])
        self.conv1_2 = _UNetPPConvBlock(nb[1]*2 + nb[2], nb[1])
        self.conv2_2 = _UNetPPConvBlock(nb[2]*2 + nb[3], nb[2])
        self.conv0_3 = _UNetPPConvBlock(nb[0]*3 + nb[1], nb[0])
        self.conv1_3 = _UNetPPConvBlock(nb[1]*3 + nb[2], nb[1])
        self.conv0_4 = _UNetPPConvBlock(nb[0]*4 + nb[1], nb[0])
        self.final = nn.Conv2d(nb[0], out_channels, 1)
        self.final_ds = nn.ModuleList([nn.Conv2d(nb[0], out_channels, 1) for _ in range(4)])

    def forward(self, x):
        x0_0 = self.conv0_0(x)
        x1_0 = self.conv1_0(self.pool(x0_0)); x0_1 = self.conv0_1(torch.cat([x0_0, self.up(x1_0)], 1))
        x2_0 = self.conv2_0(self.pool(x1_0)); x1_1 = self.conv1_1(torch.cat([x1_0, self.up(x2_0)], 1)); x0_2 = self.conv0_2(torch.cat([x0_0, x0_1, self.up(x1_1)], 1))
        x3_0 = self.conv3_0(self.pool(x2_0)); x2_1 = self.conv2_1(torch.cat([x2_0, self.up(x3_0)], 1)); x1_2 = self.conv1_2(torch.cat([x1_0, x1_1, self.up(x2_1)], 1)); x0_3 = self.conv0_3(torch.cat([x0_0, x0_1, x0_2, self.up(x1_2)], 1))
        x4_0 = self.conv4_0(self.pool(x3_0)); x3_1 = self.conv3_1(torch.cat([x3_0, self.up(x4_0)], 1)); x2_2 = self.conv2_2(torch.cat([x2_0, x2_1, self.up(x3_1)], 1)); x1_3 = self.conv1_3(torch.cat([x1_0, x1_1, x1_2, self.up(x2_2)], 1)); x0_4 = self.conv0_4(torch.cat([x0_0, x0_1, x0_2, x0_3, self.up(x1_3)], 1))
        if self.deep_supervision:
            return [head(feat) for head, feat in zip(self.final_ds, [x0_1, x0_2, x0_3, x0_4])]
        return self.final(x0_4)


def phase8_soft_dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = (0, 2, 3)
    inter = (probs * targets).sum(dims)
    den = probs.sum(dims) + targets.sum(dims)
    return (1 - (2 * inter + eps) / (den + eps)).mean()


def phase8_tversky_loss(logits, targets, alpha=0.3, beta=0.7, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = (0, 2, 3)
    tp = (probs * targets).sum(dims)
    fp = (probs * (1 - targets)).sum(dims)
    fn = ((1 - probs) * targets).sum(dims)
    return (1 - (tp + eps) / (tp + alpha * fp + beta * fn + eps)).mean()


def phase8_focal_bce_loss(logits, targets, gamma=2.0):
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    pt = torch.exp(-bce)
    return (((1 - pt) ** gamma) * bce).mean()


def phase8_student_loss(logits, targets, alpha=0.5, lambda_dice=0.25):
    if isinstance(logits, (list, tuple)):
        return sum(phase8_student_loss(x, targets, alpha, lambda_dice) for x in logits) / len(logits)
    return alpha * phase8_tversky_loss(logits, targets) + (1 - alpha) * phase8_focal_bce_loss(logits, targets) + lambda_dice * phase8_soft_dice_loss(logits, targets)


@torch.no_grad()
def phase8_student_confusion_counts(logits, targets, threshold=PHASE8_STUDENT_THRESHOLD):
    """Return per-channel TP/TN/FP/FN pixel counts for thresholded segmentation masks."""
    probs = torch.sigmoid(logits[-1] if isinstance(logits, (list, tuple)) else logits)
    threshold_tensor = torch.as_tensor(threshold, device=probs.device, dtype=probs.dtype)
    if threshold_tensor.ndim == 0:
        threshold_tensor = threshold_tensor.repeat(probs.shape[1])
    threshold_tensor = threshold_tensor.view(1, -1, 1, 1)
    pred = (probs >= threshold_tensor).float()
    target = (targets >= 0.5).float()
    dims = (0, 2, 3)
    tp = (pred * target).sum(dims)
    fp = (pred * (1 - target)).sum(dims)
    fn = ((1 - pred) * target).sum(dims)
    tn = ((1 - pred) * (1 - target)).sum(dims)
    return {'tp': tp.detach().cpu(), 'fp': fp.detach().cpu(), 'fn': fn.detach().cpu(), 'tn': tn.detach().cpu()}


def phase8_student_metrics_from_counts(counts, eps=1e-6):
    """Compute macro channel metrics from accumulated pixel confusion counts.

    Note: for binary foreground masks, Dice and foreground F1 use the same formula
    2TP / (2TP + FP + FN). Both names are logged so downstream tables are explicit.
    """
    tp, fp, fn, tn = counts['tp'].float(), counts['fp'].float(), counts['fn'].float(), counts['tn'].float()
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    f1 = dice
    iou = (tp + eps) / (tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    specificity = (tn + eps) / (tn + fp + eps)
    pixel_accuracy = (tp + tn + eps) / (tp + tn + fp + fn + eps)
    balanced_accuracy = 0.5 * (recall + specificity)
    fpr = (fp + eps) / (fp + tn + eps)
    fnr = (fn + eps) / (fn + tp + eps)
    pred_positive_rate = (tp + fp + eps) / (tp + tn + fp + fn + eps)
    target_positive_rate = (tp + fn + eps) / (tp + tn + fp + fn + eps)
    out = {
        'dice': dice.mean().item(),
        'f1': f1.mean().item(),
        'iou': iou.mean().item(),
        'precision': precision.mean().item(),
        'recall': recall.mean().item(),
        'sensitivity': recall.mean().item(),
        'specificity': specificity.mean().item(),
        'pixel_accuracy': pixel_accuracy.mean().item(),
        'balanced_accuracy': balanced_accuracy.mean().item(),
        'fpr': fpr.mean().item(),
        'fnr': fnr.mean().item(),
        'pred_positive_rate': pred_positive_rate.mean().item(),
        'target_positive_rate': target_positive_rate.mean().item(),
    }
    for key, tensor in [('tp', tp), ('fp', fp), ('fn', fn), ('tn', tn)]:
        out[key] = float(tensor.sum().item())
    return out


@torch.no_grad()
def phase8_student_metrics(logits, targets, threshold=PHASE8_STUDENT_THRESHOLD, eps=1e-6):
    return phase8_student_metrics_from_counts(phase8_student_confusion_counts(logits, targets, threshold=threshold), eps=eps)


@torch.no_grad()
def phase8_evaluate_student_model(model, loader, device, threshold=PHASE8_STUDENT_THRESHOLD):
    """Evaluate one split with exact accumulated pixel metrics."""
    model.eval()
    total_loss = 0.0
    n_samples = 0
    total_counts = None
    for batch in loader:
        x, y = batch['image'].to(device), batch['mask'].to(device)
        logits = model(x)
        total_loss += float(phase8_student_loss(logits, y).item()) * x.size(0)
        n_samples += int(x.size(0))
        counts = phase8_student_confusion_counts(logits, y, threshold=threshold)
        if total_counts is None:
            total_counts = {k: v.clone() for k, v in counts.items()}
        else:
            for k, v in counts.items():
                total_counts[k] += v
    metrics = phase8_student_metrics_from_counts(total_counts) if total_counts is not None else {}
    metrics['loss'] = total_loss / max(1, n_samples)
    metrics['n_samples'] = n_samples
    metrics['threshold'] = float(threshold)
    return metrics


def phase8_student_inner_train_val_split(outer_train_df, val_fraction=PHASE8_STUDENT_INNER_VAL_FRACTION, seed=PHASE8_STUDENT_RANDOM_SEED):
    """Patient-grouped inner split so the outer fold can remain a true test split."""
    df = outer_train_df.copy()
    if len(df) < 3:
        return df.copy(), df.iloc[0:0].copy()
    rng = np.random.default_rng(seed)
    group_col = 'patient_id' if 'patient_id' in df.columns else None
    if group_col is not None:
        patients = pd.Series(df[group_col].astype(str).dropna().unique())
        shuffled = patients.sample(frac=1.0, random_state=int(seed)).to_numpy()
        n_val = int(round(len(shuffled) * float(val_fraction)))
        n_val = min(max(n_val, 1), max(len(shuffled) - 1, 1))
        val_patients = set(shuffled[:n_val])
        val_mask = df[group_col].astype(str).isin(val_patients)
    else:
        idx = np.arange(len(df))
        rng.shuffle(idx)
        n_val = int(round(len(idx) * float(val_fraction)))
        n_val = min(max(n_val, 1), max(len(idx) - 1, 1))
        val_idx = set(idx[:n_val].tolist())
        val_mask = pd.Series(np.arange(len(df))).isin(val_idx).to_numpy()
    inner_val_df = df[val_mask].copy()
    inner_train_df = df[~val_mask].copy()
    return inner_train_df, inner_val_df


def train_phase8_unetpp_student(train_df, val_df, task='supdeep', output_dir=PHASE8_STUDENT_OUTPUT_DIR, device=None, run_name=None, test_df=None):
    """Train/resume U-Net++ student; save best/last checkpoints + per-epoch history."""
    if task not in {'supdeep', 'cc'}:
        raise ValueError("task must be 'supdeep' or 'cc'")
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    run_name = run_name or task
    out_dir = ensure_dir(Path(output_dir) / run_name)
    last_path = out_dir / 'last.pt'
    best_path = out_dir / 'best.pt'
    history_path = out_dir / 'history.csv'

    out_channels = 2 if task == 'supdeep' else 1
    model = build_phase8_segmentation_model(SEGCOMP_ACTIVE_MODEL_ID, out_channels=out_channels, deep_supervision=True).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=PHASE8_STUDENT_LR, weight_decay=PHASE8_STUDENT_WEIGHT_DECAY)

    start_epoch = 1
    best_dice = -1.0
    history = []
    if PHASE8_STUDENT_AUTO_RESUME and last_path.exists():
        import warnings
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', message=r'.*weights_only=False.*', category=FutureWarning)
            ckpt = torch.load(last_path, map_location=device)
        ckpt_history = ckpt.get('history', []) or []
        expanded_metrics_present = bool(ckpt_history and ('val_f1' in ckpt_history[-1]) and ('val_pixel_accuracy' in ckpt_history[-1]))
        if PHASE8_STUDENT_USE_INNER_VAL_FOR_TEST and not expanded_metrics_present:
            print(f'RESTART Phase 8 U-Net++ {run_name}: existing checkpoint lacks expanded metrics/inner-val-test protocol.')
        else:
            model.load_state_dict(ckpt['model'])
            if 'optimizer' in ckpt:
                opt.load_state_dict(ckpt['optimizer'])
            history = ckpt_history
            if history:
                best_dice = max(float(r.get('val_dice', -1.0)) for r in history if pd.notna(r.get('val_dice', np.nan)))
            start_epoch = int(ckpt.get('epoch', 0)) + 1
            print(f'RESUME Phase 8 U-Net++ {run_name}: last_epoch={start_epoch-1}, best_dice={best_dice:.4f}, remaining={max(0, PHASE8_STUDENT_EPOCHS-start_epoch+1)}')
    else:
        print(f'START Phase 8 U-Net++ {run_name}: epochs={PHASE8_STUDENT_EPOCHS}, device={device}, batch={PHASE8_STUDENT_BATCH_SIZE}')

    if start_epoch > PHASE8_STUDENT_EPOCHS:
        print(f'COMPLETE Phase 8 U-Net++ {run_name}: checkpoint already at epoch {start_epoch-1}')
        if best_path.exists():
            best_ckpt = torch.load(best_path, map_location=device)
            model.load_state_dict(best_ckpt['model'])
        existing_test_metrics = None
        existing_test_path = out_dir / 'test_metrics.json'
        if existing_test_path.exists():
            existing_record = json.loads(existing_test_path.read_text())
            existing_test_metrics = {k.replace('test_', '', 1): v for k, v in existing_record.items() if k.startswith('test_')}
            print(f"FOUND existing Phase 8 U-Net++ test metrics for {run_name}: "
                  f"test_dice/F1={existing_record.get('test_dice', np.nan):.4f} test_iou={existing_record.get('test_iou', np.nan):.4f} "
                  f"test_acc={existing_record.get('test_pixel_accuracy', np.nan):.4f} test_bal_acc={existing_record.get('test_balanced_accuracy', np.nan):.4f} "
                  f"test_precision={existing_record.get('test_precision', np.nan):.4f} test_recall/sens={existing_record.get('test_recall', np.nan):.4f} "
                  f"test_specificity={existing_record.get('test_specificity', np.nan):.4f} test_fpr={existing_record.get('test_fpr', np.nan):.4f} test_fnr={existing_record.get('test_fnr', np.nan):.4f} "
                  f"test_pred_pos={existing_record.get('test_pred_positive_rate', np.nan):.4f} test_target_pos={existing_record.get('test_target_positive_rate', np.nan):.4f}")
        return model, pd.DataFrame(history), existing_test_metrics

    train_ds = Phase8StudentPseudoMaskDataset(train_df, task=task, train=True)
    val_ds = Phase8StudentPseudoMaskDataset(val_df, task=task, train=False)
    test_ds = Phase8StudentPseudoMaskDataset(test_df, task=task, train=False) if test_df is not None and len(test_df) else None
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=PHASE8_STUDENT_BATCH_SIZE, shuffle=True, num_workers=PHASE8_STUDENT_NUM_WORKERS, pin_memory=(device.type == 'cuda'))
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=PHASE8_STUDENT_BATCH_SIZE, shuffle=False, num_workers=PHASE8_STUDENT_NUM_WORKERS, pin_memory=(device.type == 'cuda'))
    test_loader = None if test_ds is None else torch.utils.data.DataLoader(test_ds, batch_size=PHASE8_STUDENT_BATCH_SIZE, shuffle=False, num_workers=PHASE8_STUDENT_NUM_WORKERS, pin_memory=(device.type == 'cuda'))
    print(f'DATA Phase 8 U-Net++ {run_name}: train={len(train_ds)}, val={len(val_ds)}, test={0 if test_ds is None else len(test_ds)}, task={task}')

    run_start = time.time()
    epoch_times = []
    for epoch in range(start_epoch, PHASE8_STUDENT_EPOCHS + 1):
        t0 = time.time()
        print(f'START epoch {epoch}/{PHASE8_STUDENT_EPOCHS} — Phase 8 U-Net++ {run_name}')
        model.train(); train_loss = 0.0
        for batch in train_loader:
            x, y = batch['image'].to(device), batch['mask'].to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = phase8_student_loss(logits, y)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            train_loss += float(loss.item()) * x.size(0)
        val_metrics = phase8_evaluate_student_model(model, val_loader, device, threshold=PHASE8_STUDENT_THRESHOLD)
        epoch_sec = time.time() - t0
        epoch_times.append(epoch_sec)
        avg_sec = float(np.mean(epoch_times))
        remaining = max(0, PHASE8_STUDENT_EPOCHS - epoch)
        eta_min = remaining * avg_sec / 60.0
        row = {
            'run_name': run_name, 'task': task, 'epoch': epoch,
            'train_loss': train_loss / max(1, len(train_ds)),
            'val_loss': val_metrics.get('loss', np.nan),
            'val_dice': val_metrics.get('dice', np.nan),
            'val_f1': val_metrics.get('f1', np.nan),
            'val_iou': val_metrics.get('iou', np.nan),
            'val_pixel_accuracy': val_metrics.get('pixel_accuracy', np.nan),
            'val_balanced_accuracy': val_metrics.get('balanced_accuracy', np.nan),
            'val_precision': val_metrics.get('precision', np.nan),
            'val_recall': val_metrics.get('recall', np.nan),
            'val_sensitivity': val_metrics.get('sensitivity', np.nan),
            'val_specificity': val_metrics.get('specificity', np.nan),
            'val_fpr': val_metrics.get('fpr', np.nan),
            'val_fnr': val_metrics.get('fnr', np.nan),
            'val_pred_positive_rate': val_metrics.get('pred_positive_rate', np.nan),
            'val_target_positive_rate': val_metrics.get('target_positive_rate', np.nan),
            'epoch_sec': epoch_sec, 'eta_min': eta_min,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)
        payload = {'model': model.state_dict(), 'optimizer': opt.state_dict(), 'epoch': epoch, 'history': history, 'task': task, 'run_name': run_name, 'mask_source': PHASE8_STUDENT_MASK_SOURCE, 'segmentation_model_id': SEGCOMP_ACTIVE_MODEL_ID}
        torch.save(payload, last_path)
        is_best = row['val_dice'] > best_dice
        if is_best:
            best_dice = row['val_dice']; torch.save(payload, best_path)
        if (epoch == start_epoch) or (epoch % PHASE8_STUDENT_LOG_EVERY_N_EPOCHS == 0) or (epoch == PHASE8_STUDENT_EPOCHS):
            print(
                f"END epoch {epoch}/{PHASE8_STUDENT_EPOCHS} — {run_name}: "
                f"train_loss={row['train_loss']:.4f} val_loss={row['val_loss']:.4f} "
                f"val_dice/F1={row['val_dice']:.4f} val_iou={row['val_iou']:.4f} "
                f"val_acc={row['val_pixel_accuracy']:.4f} val_bal_acc={row['val_balanced_accuracy']:.4f} "
                f"val_precision={row['val_precision']:.4f} val_recall/sens={row['val_recall']:.4f} "
                f"val_specificity={row['val_specificity']:.4f} val_fpr={row['val_fpr']:.4f} val_fnr={row['val_fnr']:.4f} "
                f"val_pred_pos={row['val_pred_positive_rate']:.4f} val_target_pos={row['val_target_positive_rate']:.4f} "
                f"time={epoch_sec/60:.1f}m avg={avg_sec/60:.1f}m ETA={eta_min:.1f}m best={best_dice:.4f}"
            )
    test_metrics = None
    if test_loader is not None:
        if best_path.exists():
            import warnings
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore', message=r'.*weights_only=False.*', category=FutureWarning)
                best_ckpt = torch.load(best_path, map_location=device)
            model.load_state_dict(best_ckpt['model'])
        test_metrics = phase8_evaluate_student_model(model, test_loader, device, threshold=PHASE8_STUDENT_THRESHOLD)
        test_record = {'run_name': run_name, 'task': task, **{f'test_{k}': v for k, v in test_metrics.items()}}
        save_json(test_record, out_dir / 'test_metrics.json')
        pd.DataFrame([test_record]).to_csv(out_dir / 'test_metrics.csv', index=False)
        print(
            f"TEST Phase 8 U-Net++ {run_name}: "
            f"test_dice/F1={test_record['test_dice']:.4f} test_iou={test_record['test_iou']:.4f} "
            f"test_acc={test_record['test_pixel_accuracy']:.4f} test_bal_acc={test_record['test_balanced_accuracy']:.4f} "
            f"test_precision={test_record['test_precision']:.4f} test_recall/sens={test_record['test_recall']:.4f} "
            f"test_specificity={test_record['test_specificity']:.4f} test_fpr={test_record['test_fpr']:.4f} test_fnr={test_record['test_fnr']:.4f} "
            f"test_pred_pos={test_record['test_pred_positive_rate']:.4f} test_target_pos={test_record['test_target_positive_rate']:.4f}"
        )
    print(f'DONE Phase 8 U-Net++ {run_name}: elapsed={(time.time()-run_start)/60:.1f}m best_dice={best_dice:.4f}')
    return model, pd.DataFrame(history), test_metrics


phase8_student_decision_record = phase8_validate_student_decision()
phase8_student_summary = {
    **phase8_student_decision_record,
    'student_architecture': 'compact U-Net++',
    'tasks': ['supdeep: vessel+FAZ heads', 'cc: flow-deficit head'],
    'run_training_default': PHASE8_RUN_STUDENT_TRAINING,
    'output_dir': str(PHASE8_STUDENT_OUTPUT_DIR),
    'mask_source_if_used': PHASE8_STUDENT_MASK_SOURCE,
    'note': 'Student is upstream/frozen for classifier; trained only against QC-passed classical_improved pseudo-labels, not disease labels.',
    'metric_note': 'For binary masks, foreground F1 equals Dice at the same threshold; rerun logs validation F1/Dice plus pixel accuracy, balanced accuracy, precision, recall/sensitivity, specificity, and optional outer-fold test metrics.',
    'validation_test_protocol': 'If PHASE8_STUDENT_USE_INNER_VAL_FOR_TEST=True, each existing Phase 2 fold is held out as outer test and best epochs are selected on an inner patient-grouped validation split from the remaining folds.',
}


# Phase 8C records the decision and defines the U-Net++ student/training helpers only.
# It does not launch training; use the Phase 8D cell below for resumable training.
save_json(phase8_student_summary, PHASE8_STUDENT_OUTPUT_DIR / 'phase8_student_decision_summary.json')
pd.DataFrame([phase8_student_summary]).to_csv(PHASE8_STUDENT_OUTPUT_DIR / 'phase8_student_decision_summary.csv', index=False)
print('Phase 8C ready: decision recorded, U-Net++ helpers defined, no training launched in this cell.')
print('Next: run the Phase 8D cell to train/resume the segmentation student.')

# ============================================================
# Phase 8C extension - six interchangeable segmentation students
# ============================================================

class _SegCompDoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
        )
    def forward(self, x):
        return self.block(x)


class Phase8UNetStudent(nn.Module):
    def __init__(self, in_channels=1, out_channels=2, base=32, **_):
        super().__init__()
        widths = [base, base*2, base*4, base*8, base*16]
        self.enc = nn.ModuleList()
        prev = in_channels
        for width in widths:
            self.enc.append(_SegCompDoubleConv(prev, width)); prev = width
        self.pool = nn.MaxPool2d(2)
        self.up = nn.ModuleList([nn.ConvTranspose2d(widths[i], widths[i-1], 2, stride=2) for i in range(4,0,-1)])
        self.dec = nn.ModuleList([_SegCompDoubleConv(widths[i-1]*2, widths[i-1]) for i in range(4,0,-1)])
        self.final = nn.Conv2d(widths[0], out_channels, 1)
    def forward(self, x):
        skips=[]
        for i,block in enumerate(self.enc):
            x=block(x)
            if i < len(self.enc)-1:
                skips.append(x); x=self.pool(x)
        for up,block,skip in zip(self.up,self.dec,reversed(skips)):
            x=up(x)
            if x.shape[-2:] != skip.shape[-2:]:
                x=F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
            x=block(torch.cat([skip,x],1))
        return self.final(x)


class _SegCompAttentionGate(nn.Module):
    def __init__(self, skip_ch, gate_ch, inter_ch):
        super().__init__()
        self.skip = nn.Conv2d(skip_ch, inter_ch, 1, bias=False)
        self.gate = nn.Conv2d(gate_ch, inter_ch, 1, bias=False)
        self.psi = nn.Sequential(nn.GELU(), nn.Conv2d(inter_ch, 1, 1), nn.Sigmoid())
    def forward(self, skip, gate):
        gate = F.interpolate(gate, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return skip * self.psi(self.skip(skip) + self.gate(gate))


class Phase8AttentionUNetStudent(nn.Module):
    def __init__(self, in_channels=1, out_channels=2, base=32, **_):
        super().__init__()
        w=[base,base*2,base*4,base*8,base*16]
        self.enc=nn.ModuleList(); prev=in_channels
        for width in w:
            self.enc.append(_SegCompDoubleConv(prev,width)); prev=width
        self.pool=nn.MaxPool2d(2)
        self.up=nn.ModuleList([nn.ConvTranspose2d(w[i],w[i-1],2,2) for i in range(4,0,-1)])
        self.gates=nn.ModuleList([_SegCompAttentionGate(w[i-1],w[i-1],max(base,w[i-1]//2)) for i in range(4,0,-1)])
        self.dec=nn.ModuleList([_SegCompDoubleConv(w[i-1]*2,w[i-1]) for i in range(4,0,-1)])
        self.final=nn.Conv2d(w[0],out_channels,1)
    def forward(self,x):
        skips=[]
        for i,block in enumerate(self.enc):
            x=block(x)
            if i<4: skips.append(x); x=self.pool(x)
        for up,gate,block,skip in zip(self.up,self.gates,self.dec,reversed(skips)):
            x=up(x); attended=gate(skip,x)
            x=block(torch.cat([attended,x],1))
        return self.final(x)


class _SegCompASPP(nn.Module):
    def __init__(self,in_ch,out_ch=128,rates=(1,6,12,18)):
        super().__init__()
        branches=[]
        for rate in rates:
            k=1 if rate==1 else 3; pad=0 if rate==1 else rate
            branches.append(nn.Sequential(nn.Conv2d(in_ch,out_ch,k,padding=pad,dilation=rate,bias=False),nn.BatchNorm2d(out_ch),nn.GELU()))
        self.branches=nn.ModuleList(branches)
        self.project=nn.Sequential(nn.Conv2d(out_ch*len(rates),out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.GELU(),nn.Dropout2d(0.1))
    def forward(self,x):
        return self.project(torch.cat([branch(x) for branch in self.branches],1))


class Phase8DeepLabV3PlusStudent(nn.Module):
    def __init__(self,in_channels=1,out_channels=2,base=32,**_):
        super().__init__()
        self.stem=_SegCompDoubleConv(in_channels,base)
        self.low=_SegCompDoubleConv(base,base*2)
        self.mid=_SegCompDoubleConv(base*2,base*4)
        self.high=nn.Sequential(
            nn.Conv2d(base*4,base*8,3,padding=2,dilation=2,bias=False),nn.BatchNorm2d(base*8),nn.GELU(),
            nn.Conv2d(base*8,base*8,3,padding=4,dilation=4,bias=False),nn.BatchNorm2d(base*8),nn.GELU())
        self.pool=nn.MaxPool2d(2)
        self.aspp=_SegCompASPP(base*8,base*4)
        self.low_proj=nn.Sequential(nn.Conv2d(base*2,base,1,bias=False),nn.BatchNorm2d(base),nn.GELU())
        self.decoder=_SegCompDoubleConv(base*5,base*2)
        self.final=nn.Conv2d(base*2,out_channels,1)
    def forward(self,x):
        size=x.shape[-2:]
        x=self.stem(x); low=self.low(self.pool(x)); x=self.mid(self.pool(low)); x=self.high(self.pool(x)); x=self.aspp(x)
        x=F.interpolate(x,size=low.shape[-2:],mode='bilinear',align_corners=False)
        x=self.decoder(torch.cat([x,self.low_proj(low)],1))
        return F.interpolate(self.final(x),size=size,mode='bilinear',align_corners=False)


class _SegCompOverlapPatch(nn.Module):
    def __init__(self,in_ch,out_ch,kernel,stride,padding):
        super().__init__(); self.proj=nn.Conv2d(in_ch,out_ch,kernel,stride,padding); self.norm=nn.LayerNorm(out_ch)
    def forward(self,x):
        x=self.proj(x); h,w=x.shape[-2:]; x=self.norm(x.flatten(2).transpose(1,2)); return x,h,w


class _SegCompEfficientAttention(nn.Module):
    def __init__(self,dim,heads,sr_ratio):
        super().__init__(); self.heads=heads; self.scale=(dim//heads)**-0.5
        self.q=nn.Linear(dim,dim); self.kv=nn.Linear(dim,dim*2); self.proj=nn.Linear(dim,dim)
        self.sr=nn.Conv2d(dim,dim,sr_ratio,sr_ratio) if sr_ratio>1 else None; self.norm=nn.LayerNorm(dim) if sr_ratio>1 else nn.Identity()
    def forward(self,x,h,w):
        b,n,c=x.shape; q=self.q(x).reshape(b,n,self.heads,c//self.heads).transpose(1,2)
        source=x
        if self.sr is not None:
            source=self.sr(x.transpose(1,2).reshape(b,c,h,w)).flatten(2).transpose(1,2); source=self.norm(source)
        kv=self.kv(source).reshape(b,-1,2,self.heads,c//self.heads).permute(2,0,3,1,4); k,v=kv[0],kv[1]
        attn=(q@k.transpose(-2,-1))*self.scale; x=(attn.softmax(-1)@v).transpose(1,2).reshape(b,n,c)
        return self.proj(x)


class _SegCompMixFFN(nn.Module):
    def __init__(self,dim,ratio=4):
        super().__init__(); hidden=dim*ratio; self.fc1=nn.Linear(dim,hidden); self.dw=nn.Conv2d(hidden,hidden,3,padding=1,groups=hidden); self.fc2=nn.Linear(hidden,dim)
    def forward(self,x,h,w):
        b,n,_=x.shape; x=self.fc1(x); x=self.dw(x.transpose(1,2).reshape(b,-1,h,w)).flatten(2).transpose(1,2); return self.fc2(F.gelu(x))


class _SegCompMiTBlock(nn.Module):
    def __init__(self,dim,heads,sr):
        super().__init__(); self.n1=nn.LayerNorm(dim); self.attn=_SegCompEfficientAttention(dim,heads,sr); self.n2=nn.LayerNorm(dim); self.ffn=_SegCompMixFFN(dim)
    def forward(self,x,h,w):
        x=x+self.attn(self.n1(x),h,w); return x+self.ffn(self.n2(x),h,w)


class Phase8SegFormerB0Student(nn.Module):
    def __init__(self,in_channels=1,out_channels=2,**_):
        super().__init__(); dims=[32,64,160,256]; heads=[1,2,5,8]; srs=[8,4,2,1]
        self.patches=nn.ModuleList([_SegCompOverlapPatch(in_channels,32,7,4,3),_SegCompOverlapPatch(32,64,3,2,1),_SegCompOverlapPatch(64,160,3,2,1),_SegCompOverlapPatch(160,256,3,2,1)])
        self.blocks=nn.ModuleList([nn.ModuleList([_SegCompMiTBlock(dims[i],heads[i],srs[i]) for _ in range(2)]) for i in range(4)])
        self.proj=nn.ModuleList([nn.Conv2d(d,128,1) for d in dims]); self.fuse=nn.Sequential(nn.Conv2d(512,256,1,bias=False),nn.BatchNorm2d(256),nn.GELU(),nn.Dropout2d(0.1)); self.final=nn.Conv2d(256,out_channels,1)
    def forward(self,x):
        size=x.shape[-2:]; feats=[]
        for patch,blocks in zip(self.patches,self.blocks):
            x,h,w=patch(x)
            for block in blocks: x=block(x,h,w)
            x=x.transpose(1,2).reshape(x.shape[0],-1,h,w); feats.append(x)
        target=feats[0].shape[-2:]; fused=torch.cat([F.interpolate(p(f),size=target,mode='bilinear',align_corners=False) for p,f in zip(self.proj,feats)],1)
        return F.interpolate(self.final(self.fuse(fused)),size=size,mode='bilinear',align_corners=False)


class Phase8IMNStudent(nn.Module):
    """OCTA-oriented magnify-then-reduce network for thin structures."""
    def __init__(self,in_channels=1,out_channels=2,base=24,**_):
        super().__init__()
        self.input_block=_SegCompDoubleConv(in_channels,base)
        self.mag1=_SegCompDoubleConv(base,base*2); self.mag2=_SegCompDoubleConv(base*2,base*2)
        self.reduce1=nn.Sequential(nn.Conv2d(base*2,base*4,3,stride=2,padding=1,bias=False),nn.BatchNorm2d(base*4),nn.GELU())
        self.reduce2=nn.Sequential(nn.Conv2d(base*4,base*2,3,stride=2,padding=1,bias=False),nn.BatchNorm2d(base*2),nn.GELU())
        self.refine=_SegCompDoubleConv(base*3,base*2); self.final=nn.Conv2d(base*2,out_channels,1)
    def forward(self,x):
        size=x.shape[-2:]; stem=self.input_block(x)
        x=F.interpolate(stem,scale_factor=2,mode='bilinear',align_corners=False); x=self.mag1(x); x=self.mag2(x)
        x=self.reduce1(x); x=self.reduce2(x)
        if x.shape[-2:] != size: x=F.interpolate(x,size=size,mode='bilinear',align_corners=False)
        return self.final(self.refine(torch.cat([stem,x],1)))


SEGCOMP_SEGMENTER_CLASSES = {
    'unetpp': Phase8UNetPlusPlusStudent,
    'unet': Phase8UNetStudent,
    'attention_unet': Phase8AttentionUNetStudent,
    'deeplabv3plus': Phase8DeepLabV3PlusStudent,
    'segformer_b0': Phase8SegFormerB0Student,
    'imn': Phase8IMNStudent,
}


def build_phase8_segmentation_model(model_id, out_channels, deep_supervision=False):
    if model_id not in SEGCOMP_SEGMENTER_CLASSES:
        raise ValueError(f'Unknown segmentation model: {model_id}')
    cls=SEGCOMP_SEGMENTER_CLASSES[model_id]
    kwargs={'in_channels':1,'out_channels':int(out_channels)}
    if model_id=='unetpp': kwargs['deep_supervision']=bool(deep_supervision)
    return cls(**kwargs)



@torch.no_grad()
def segcomp_select_channel_thresholds(model, loader, device, grid=None):
    """Select one threshold per output channel using inner-validation Dice only."""
    grid = np.asarray(grid if grid is not None else np.linspace(0.10, 0.90, 33), dtype=np.float32)
    totals = None
    model.eval()
    for batch in loader:
        x = batch['image'].to(device)
        target = (batch['mask'].to(device) >= 0.5)
        logits = model(x)
        probs = torch.sigmoid(logits[-1] if isinstance(logits, (list, tuple)) else logits)
        if totals is None:
            totals = {float(t): {'tp': torch.zeros(probs.shape[1]), 'fp': torch.zeros(probs.shape[1]), 'fn': torch.zeros(probs.shape[1])} for t in grid}
        for threshold in grid:
            pred = probs >= float(threshold)
            totals[float(threshold)]['tp'] += (pred & target).sum((0,2,3)).cpu()
            totals[float(threshold)]['fp'] += (pred & ~target).sum((0,2,3)).cpu()
            totals[float(threshold)]['fn'] += (~pred & target).sum((0,2,3)).cpu()
    if totals is None:
        raise ValueError('Cannot select thresholds from an empty validation loader.')
    channels = len(next(iter(totals.values()))['tp'])
    selected=[]; selected_dice=[]
    for channel in range(channels):
        best_t=0.5; best_dice=-1.0
        for threshold,counts in totals.items():
            tp=float(counts['tp'][channel]); fp=float(counts['fp'][channel]); fn=float(counts['fn'][channel])
            dice=(2*tp+1e-6)/(2*tp+fp+fn+1e-6)
            if dice>best_dice:
                best_t=float(threshold); best_dice=float(dice)
        selected.append(best_t); selected_dice.append(best_dice)
    return {'thresholds':selected,'validation_dice_at_threshold':selected_dice,'grid':[float(x) for x in grid]}

def segcomp_parameter_count(model):
    return int(sum(p.numel() for p in model.parameters()))

# Make the existing resumable trainer instantiate the active registry architecture.


In [ ]:
# ============================================================
# Phase 8D - Resumable six-architecture segmentation training
# ============================================================
# Fifty new runs plus ten controlled U-Net++ reference runs:
# six architectures x five outer folds x two tasks.

PHASE8_COMPARISON_FOLDS = [0, 1, 2, 3, 4]


def segcomp_train_segmentation_model(model_id, folds=PHASE8_COMPARISON_FOLDS, fold_df=None, device=None):
    segcomp_activate_model(model_id)
    if fold_df is None:
        fold_df = globals().get('phase2_folds_with_phase8', globals().get('phase2_folds'))
    if fold_df is None:
        raise RuntimeError('Phase 8 comparison requires phase2_folds_with_phase8 or phase2_folds.')
    if len(set(pd.to_numeric(fold_df['label']).dropna().astype(int))) != 8:
        raise ValueError('Phase 8 comparison requires all eight cohort labels.')
    histories=[]; tests=[]
    for fold in folds:
        outer_train=fold_df[fold_df['fold'] != int(fold)].copy()
        outer_test=fold_df[fold_df['fold'] == int(fold)].copy()
        train_df,val_df=phase8_student_inner_train_val_split(outer_train,seed=PHASE8_STUDENT_RANDOM_SEED+int(fold))
        for task in ['supdeep','cc']:
            run_name=f'fold{int(fold)}_{task}'
            model,hist,test_metrics=train_phase8_unetpp_student(
                train_df,val_df,task=task,output_dir=PHASE8_STUDENT_OUTPUT_DIR,
                device=device,run_name=run_name,test_df=outer_test,
            )
            run_dir=Path(PHASE8_STUDENT_OUTPUT_DIR)/run_name
            val_ds=Phase8StudentPseudoMaskDataset(val_df,task=task,train=False)
            val_loader=torch.utils.data.DataLoader(val_ds,batch_size=PHASE8_STUDENT_BATCH_SIZE,shuffle=False,num_workers=PHASE8_STUDENT_NUM_WORKERS)
            threshold_record=segcomp_select_channel_thresholds(model,val_loader,next(model.parameters()).device)
            threshold_record.update({'segmentation_model_id':model_id,'fold':int(fold),'task':task,'source_split':'inner_validation'})
            save_json(threshold_record,run_dir/'thresholds.json')
            if len(outer_test):
                test_ds=Phase8StudentPseudoMaskDataset(outer_test,task=task,train=False)
                test_loader=torch.utils.data.DataLoader(test_ds,batch_size=PHASE8_STUDENT_BATCH_SIZE,shuffle=False,num_workers=PHASE8_STUDENT_NUM_WORKERS)
                test_metrics=phase8_evaluate_student_model(model,test_loader,next(model.parameters()).device,threshold=threshold_record['thresholds'])
                test_record={'segmentation_model_id':model_id,'fold':int(fold),'run_name':run_name,'task':task,**{f'test_{k}':v for k,v in test_metrics.items()}}
                save_json(test_record,run_dir/'test_metrics.json'); pd.DataFrame([test_record]).to_csv(run_dir/'test_metrics.csv',index=False)
            hist=hist.copy(); hist['segmentation_model_id']=model_id; histories.append(hist)
            if test_metrics:
                tests.append({'segmentation_model_id':model_id,'fold':int(fold),'task':task,**test_metrics})
    history_df=pd.concat(histories,ignore_index=True) if histories else pd.DataFrame()
    test_df=pd.DataFrame(tests)
    history_df.to_csv(PHASE8_STUDENT_OUTPUT_DIR/'phase8_all_history.csv',index=False)
    test_df.to_csv(PHASE8_STUDENT_OUTPUT_DIR/'phase8_all_test_metrics.csv',index=False)
    summary={'segmentation_model_id':model_id,'mask_source':SEGCOMP_ACTIVE_MASK_SOURCE,'folds':[int(x) for x in folds],
             'n_history_rows':int(len(history_df)),'n_test_rows':int(len(test_df))}
    save_json(summary,PHASE8_STUDENT_OUTPUT_DIR/'phase8_comparison_summary.json')
    return history_df,test_df


def segcomp_train_all_segmenters(model_ids=SEGCOMP_MODEL_IDS, folds=PHASE8_COMPARISON_FOLDS, fold_df=None, device=None):
    results={}
    for index,model_id in enumerate(model_ids,1):
        print(f'[Segmentation comparison] model {index}/{len(model_ids)}: {model_id}')
        results[model_id]=segcomp_train_segmentation_model(model_id,folds=folds,fold_df=fold_df,device=device)
    return results


if SEGCOMP_RUN_SEGMENTATION:
    segcomp_segmentation_results=segcomp_train_all_segmenters()
else:
    print('Phase 8 comparison ready. Set SEGCOMP_RUN_SEGMENTATION=True or call segcomp_train_all_segmenters().')


In [ ]:
# ============================================================
# Phase 9 - Classification Head with Prototype Augmentation
# ============================================================
# This cell defines Phase 9 only. It does not train the disease classifier.
# It enforces mask_source = unetpp_student. Classical masks are pseudo-label targets only.

PHASE9_OUTPUT_DIR = ensure_dir(segcomp_model_dir(SEGCOMP_ACTIVE_MODEL_ID) / 'phase9_masks_biomarkers')
PHASE9_UNETPP_MASK_DIR = ensure_dir(PHASE9_OUTPUT_DIR / 'student_masks')
PHASE9_SELECTED_MASK_SOURCE = SEGCOMP_ACTIVE_MASK_SOURCE
PHASE9_REQUIRES_UNETPP_CHECKPOINTS = True
PHASE9_MATERIALIZE_BATCH_SIZE = 16
PHASE9_MATERIALIZE_NUM_WORKERS = 0
PHASE9_PROTO_TEMPERATURE = 0.1
PHASE9_PROTO_EMA = 0.95
PHASE9_MC_DROPOUT_PASSES = 30
PHASE9_ENTROPY_EPS = 1e-8
PHASE9_RUN_SMOKE_TEST = False

if torch is None:
    raise ImportError('PyTorch is required for Phase 9.')


def phase9_student_checkpoint_path(fold, task, kind='best'):
    if task not in {'supdeep', 'cc'}:
        raise ValueError("task must be 'supdeep' or 'cc'")
    if kind not in {'best', 'last'}:
        raise ValueError("kind must be 'best' or 'last'")
    return Path(PHASE8_STUDENT_OUTPUT_DIR) / f'fold{int(fold)}_{task}' / f'{kind}.pt'


def phase9_load_trusted_checkpoint(path, map_location='cpu'):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing required U-Net++ checkpoint: {path}')
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        import warnings
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', message=r'.*weights_only=False.*', category=FutureWarning)
            return torch.load(path, map_location=map_location)
    except Exception:
        import warnings
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', message=r'.*weights_only=False.*', category=FutureWarning)
            return torch.load(path, map_location=map_location, weights_only=False)


def phase9_extract_model_state(payload):
    if isinstance(payload, dict):
        for key in ['model', 'model_state_dict', 'state_dict']:
            state = payload.get(key)
            if isinstance(state, dict):
                return state
        if all(torch.is_tensor(v) for v in payload.values()):
            return payload
    raise ValueError('Could not find model state dict in U-Net++ checkpoint payload.')


class Phase9FrozenUNetPPMaskGenerator(nn.Module):
    """Frozen fold-specific Phase 8D U-Net++ generator for downstream masks.

    Input batch must contain unstandardized OCTA tensors in [0, 1]:
      sup_student_image, deep_student_image, cc_student_image
    Output mask order:
      sup_vessel, deep_vessel, sup_faz, deep_faz, cc_flow
    """
    def __init__(self, fold, checkpoint_kind='best', threshold=None, device=None):
        super().__init__()
        self.fold = int(fold)
        self.checkpoint_kind = checkpoint_kind
        self.threshold = threshold
        self.supdeep = build_phase8_segmentation_model(SEGCOMP_ACTIVE_MODEL_ID, out_channels=2, deep_supervision=True)
        self.cc = build_phase8_segmentation_model(SEGCOMP_ACTIVE_MODEL_ID, out_channels=1, deep_supervision=True)
        self.load_report = []
        self.channel_thresholds = {}
        for task, model in [('supdeep', self.supdeep), ('cc', self.cc)]:
            ckpt_path = phase9_student_checkpoint_path(self.fold, task, kind=checkpoint_kind)
            payload = phase9_load_trusted_checkpoint(ckpt_path, map_location='cpu')
            state = phase9_extract_model_state(payload)
            missing, unexpected = model.load_state_dict(state, strict=False)
            if missing or unexpected:
                raise RuntimeError(f'U-Net++ checkpoint load not clean for fold={fold} task={task}: missing={missing[:5]}, unexpected={unexpected[:5]}')
            threshold_path = ckpt_path.parent / 'thresholds.json'
            if not threshold_path.exists():
                raise FileNotFoundError(f'Missing inner-validation threshold file: {threshold_path}')
            threshold_record = json.loads(threshold_path.read_text())
            self.channel_thresholds[task] = [float(x) for x in threshold_record['thresholds']]
            self.load_report.append({'fold': self.fold, 'task': task, 'checkpoint': str(ckpt_path), 'threshold_path': str(threshold_path), 'loaded': True})
        for p in self.parameters():
            p.requires_grad = False
        self.eval()
        if device is not None:
            self.to(device)

    @staticmethod
    def _last_logits(logits):
        return logits[-1] if isinstance(logits, (list, tuple)) else logits

    def _prob_or_binary(self, logits, task):
        probs = torch.sigmoid(self._last_logits(logits))
        thresholds = self.channel_thresholds[task] if self.threshold is None else self.threshold
        threshold_tensor = torch.as_tensor(thresholds, device=probs.device, dtype=probs.dtype)
        if threshold_tensor.ndim == 0:
            threshold_tensor = threshold_tensor.repeat(probs.shape[1])
        return (probs >= threshold_tensor.view(1, -1, 1, 1)).float()

    @torch.no_grad()
    def forward(self, batch):
        required = ['sup_student_image', 'deep_student_image', 'cc_student_image']
        missing = [k for k in required if k not in batch]
        if missing:
            raise KeyError(f'Missing U-Net++ student input tensors: {missing}. Use RastaPhase9Dataset.')
        device = next(self.parameters()).device
        sup = batch['sup_student_image'].to(device).float()
        deep = batch['deep_student_image'].to(device).float()
        cc = batch['cc_student_image'].to(device).float()
        sup_out = self._prob_or_binary(self.supdeep(sup), 'supdeep')
        deep_out = self._prob_or_binary(self.supdeep(deep), 'supdeep')
        cc_out = self._prob_or_binary(self.cc(cc), 'cc')
        masks = torch.cat([
            sup_out[:, 0:1],
            deep_out[:, 0:1],
            sup_out[:, 1:2],
            deep_out[:, 1:2],
            cc_out[:, 0:1],
        ], dim=1)
        return {
            'masks': masks,
            'sup_vessel_mask': masks[:, 0:1],
            'deep_vessel_mask': masks[:, 1:2],
            'sup_faz_mask': masks[:, 2:3],
            'deep_faz_mask': masks[:, 3:4],
            'cc_flow_mask': masks[:, 4:5],
            'mask_source': PHASE9_SELECTED_MASK_SOURCE,
        }


class RastaPhase9Dataset(RastaPhase8Dataset):
    """Phase 8 dataset plus raw [0, 1] OCTA tensors for frozen U-Net++ inference."""
    def __getitem__(self, idx):
        sample = super().__getitem__(idx)
        row = self.df.iloc[idx]
        for layer in ['sup', 'deep', 'cc']:
            arr = preprocess_octa_image(row[f'{layer}_path'], image_size=self.image_size, use_clahe=True).astype(np.float32)
            sample[f'{layer}_student_image'] = torch.from_numpy(arr[None, :, :])
        sample['mask_source'] = PHASE9_SELECTED_MASK_SOURCE
        return sample


def phase9_tensor_masks_to_numpy(mask_tensor):
    arr = mask_tensor.detach().float().cpu().numpy()
    if arr.ndim == 4:
        return arr
    raise ValueError(f'Expected mask tensor [B,5,H,W], got {arr.shape}')


def phase9_biomarkers_from_mask_array(mask5):
    mask_dict = {key: (mask5[i] >= 0.5).astype(np.float32) for i, key in enumerate(PHASE8_MASK_ORDER)}
    skeletons = {}
    if morphology is not None:
        skeletons['sup'] = morphology.skeletonize(mask_dict['sup_vessel_mask'] > 0.5).astype(np.float32)
        skeletons['deep'] = morphology.skeletonize(mask_dict['deep_vessel_mask'] > 0.5).astype(np.float32)
    return phase8_extract_biomarkers(mask_dict, skeletons)


@torch.no_grad()
def phase9_materialize_unetpp_masks_for_fold(fold, fold_df=None, output_dir=None, checkpoint_kind='best', device=None):
    """Run frozen Phase 8D U-Net++ for one classifier fold and write mask PNGs plus biomarkers.

    This is inference from completed Phase 8D checkpoints, not Phase 8 retraining.
    The resulting manifest is the only valid file-based mask manifest for Phase 9+.
    """
    if output_dir is None:
        output_dir = PHASE9_UNETPP_MASK_DIR
    if fold_df is None:
        if 'phase2_folds' not in globals():
            raise RuntimeError('phase2_folds is required. Run Phase 1/2 fast reload first if the kernel was restarted.')
        fold_df = phase2_folds.copy()
    fold = int(fold)
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    generator = Phase9FrozenUNetPPMaskGenerator(fold=fold, checkpoint_kind=checkpoint_kind, threshold=None, device=device)
    out_root = ensure_dir(Path(output_dir) / f'fold_{fold}')
    mask_root = ensure_dir(out_root / 'masks')
    records = []
    ds = RastaPhase9Dataset(
        fold_df,
        clinical_preprocessors[fold],
        layer_stats_by_fold[fold],
        train=False,
        biomarker_specs=[],
        biomarker_stats={},
    )
    loader = torch.utils.data.DataLoader(ds, batch_size=PHASE9_MATERIALIZE_BATCH_SIZE, shuffle=False, num_workers=PHASE9_MATERIALIZE_NUM_WORKERS)
    offset = 0
    for batch in loader:
        out = generator(batch)
        masks_np = phase9_tensor_masks_to_numpy(out['masks'])
        bsz = masks_np.shape[0]
        for j in range(bsz):
            row = fold_df.reset_index(drop=True).iloc[offset + j]
            sample_id = f"{_phase8_safe_slug(row.get('patient_id'))}_{_phase8_safe_slug(row.get('eye'))}"
            sample_dir = ensure_dir(mask_root / sample_id)
            rec = {
                'patient_id': row.get('patient_id'),
                'eye': row.get('eye'),
                'fold': int(row.get('fold', -1)),
                'label': int(row.get('label', -1)),
                'mask_source': PHASE9_SELECTED_MASK_SOURCE,
                'segmentation_model_id': SEGCOMP_ACTIVE_MODEL_ID,
                'pseudo_label_source': globals().get('PHASE8_PSEUDO_LABEL_SOURCE', 'classical_improved'),
                'student_fold': fold,
            }
            for k, key in enumerate(PHASE8_MASK_ORDER):
                path = sample_dir / f'{key}.png'
                _phase8_save_mask(masks_np[j, k], path)
                rec[f'{key}_path'] = str(path)
            biomarkers = phase9_biomarkers_from_mask_array(masks_np[j])
            rec.update(biomarkers)
            qc = {}
            for key, kind in [('sup_vessel_mask', 'sup_vessel'), ('deep_vessel_mask', 'deep_vessel'), ('sup_faz_mask', 'sup_faz'), ('deep_faz_mask', 'deep_faz'), ('cc_flow_mask', 'cc_flow')]:
                qc.update(phase8_mask_qc((masks_np[j, PHASE8_MASK_ORDER.index(key)] >= 0.5).astype(np.float32), kind))
            qc['phase9_unetpp_mask_qc_pass'] = bool(all(v for kk, v in qc.items() if kk.endswith('_qc_pass')))
            rec.update(qc)
            records.append(rec)
        offset += bsz
    artifact_df = pd.DataFrame(records)
    artifact_path = out_root / f'phase9_unetpp_manifest_fold{fold}.csv'
    artifact_df.to_csv(artifact_path, index=False)
    summary = {
        'fold': fold,
        'mask_source': PHASE9_SELECTED_MASK_SOURCE,
        'checkpoint_kind': checkpoint_kind,
        'n_samples': int(len(artifact_df)),
        'qc_pass_rate': float(artifact_df['phase9_unetpp_mask_qc_pass'].mean()) if len(artifact_df) else None,
        'manifest_path': str(artifact_path),
        'load_report': generator.load_report,
    }
    save_json(summary, out_root / f'phase9_unetpp_manifest_fold{fold}_summary.json')
    return artifact_df, summary


def phase9_manifest_path_for_fold(fold):
    return PHASE9_UNETPP_MASK_DIR / f'fold_{int(fold)}' / f'phase9_unetpp_manifest_fold{int(fold)}.csv'


def phase9_load_unetpp_manifest_for_fold(fold, require_exists=True):
    path = phase9_manifest_path_for_fold(fold)
    if not path.exists():
        if require_exists:
            raise FileNotFoundError(
                f'Missing U-Net++ mask manifest for fold {fold}: {path}. '
                f'Run phase9_materialize_unetpp_masks_for_fold({int(fold)}) on the training machine. '
                'Do not use classical_improved as a fallback.'
            )
        return None
    df = pd.read_csv(path)
    if 'mask_source' not in df or set(df['mask_source'].dropna().astype(str)) != {PHASE9_SELECTED_MASK_SOURCE}:
        raise ValueError(f'Invalid Phase 9 manifest mask_source for fold {fold}. Expected only {PHASE9_SELECTED_MASK_SOURCE}.')
    return df


def phase9_merge_unetpp_manifest_for_fold(fold_df, fold):
    manifest = phase9_load_unetpp_manifest_for_fold(fold, require_exists=True)
    return fold_df.merge(manifest, on=['patient_id', 'eye'], how='left', suffixes=('', '_phase9_unetpp'))


class Phase9CompatibilityFusion(nn.Module):
    """Phase 9 forward-only fusion contract: F_cond[256] + F_mask[128] -> H[512]."""
    def __init__(self, token_dim=256, mask_dim=128, fused_dim=512):
        super().__init__()
        self.proj_tab = nn.Linear(token_dim, token_dim)
        self.cond_norm = nn.LayerNorm(token_dim)
        self.mask_encoder = Phase7MaskEncoder(in_channels=5, output_dim=mask_dim)
        self.project = nn.Sequential(
            nn.Linear(token_dim + mask_dim, fused_dim),
            nn.GELU(),
            nn.LayerNorm(fused_dim),
        )

    def forward(self, f_octa, f_tabular, masks):
        f_cond = self.cond_norm(f_octa + self.proj_tab(f_tabular))
        f_mask = self.mask_encoder(masks)
        h = self.project(torch.cat([f_cond, f_mask], dim=1))
        return {'h': h, 'f_cond': f_cond, 'f_mask': f_mask}


class Phase9PrototypeClassifier(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=256, penultimate_dim=128, num_classes=None, dropout=0.3, proto_temperature=PHASE9_PROTO_TEMPERATURE, proto_ema=PHASE9_PROTO_EMA):
        super().__init__()
        if num_classes is None:
            num_classes = int(phase2_folds['label'].nunique()) if 'phase2_folds' in globals() else 8
        self.num_classes = int(num_classes)
        self.proto_temperature = float(proto_temperature)
        self.proto_ema = float(proto_ema)
        self.feature_head = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, penultimate_dim), nn.GELU(), nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(penultimate_dim, self.num_classes)
        self.proto_alpha_raw = nn.Parameter(torch.tensor(-2.0))
        self.register_buffer('prototypes', torch.zeros(self.num_classes, penultimate_dim))
        self.register_buffer('prototype_initialized', torch.zeros(self.num_classes, dtype=torch.bool))

    def standard_logits_from_z(self, z):
        return self.classifier(z)

    def proto_logits_from_z(self, z):
        d2 = torch.cdist(z, self.prototypes.to(z.device), p=2).pow(2)
        return -d2 / max(self.proto_temperature, 1e-6)

    @torch.no_grad()
    def initialize_prototypes(self, z, labels):
        z = z.detach()
        labels = labels.detach().long()
        for cls in range(self.num_classes):
            mask = labels == cls
            if mask.any():
                self.prototypes[cls] = z[mask].mean(dim=0).to(self.prototypes.device)
                self.prototype_initialized[cls] = True

    @torch.no_grad()
    def update_prototypes(self, z, labels):
        z = z.detach()
        labels = labels.detach().long()
        for cls in range(self.num_classes):
            mask = labels == cls
            if not mask.any():
                continue
            batch_proto = z[mask].mean(dim=0).to(self.prototypes.device)
            if not bool(self.prototype_initialized[cls]):
                self.prototypes[cls] = batch_proto
                self.prototype_initialized[cls] = True
            else:
                self.prototypes[cls] = self.proto_ema * self.prototypes[cls] + (1 - self.proto_ema) * batch_proto

    def forward(self, h, use_prototypes=True):
        z = self.feature_head(h)
        standard_logits = self.standard_logits_from_z(z)
        proto_logits = self.proto_logits_from_z(z)
        alpha = F.softplus(self.proto_alpha_raw)
        if use_prototypes and bool(self.prototype_initialized.any()):
            logits = standard_logits + alpha * proto_logits
        else:
            logits = standard_logits
        return {
            'logits': logits,
            'standard_logits': standard_logits,
            'proto_logits': proto_logits,
            'prototype_alpha': alpha,
            'penultimate': z,
        }


class Phase9MultimodalClassifier(nn.Module):
    """Full Phase 9 classifier using frozen U-Net++ masks as the only downstream mask source."""
    def __init__(self, octa_encoder, tabular_encoder, mask_generator, fusion=None, classifier=None, num_classes=None):
        super().__init__()
        if mask_generator is None:
            raise ValueError('Phase 9 requires a frozen U-Net++ mask_generator. No classical fallback is allowed.')
        self.octa_encoder = octa_encoder
        self.tabular_encoder = tabular_encoder
        self.mask_generator = mask_generator
        self.fusion = fusion or Phase9CompatibilityFusion()
        self.classifier = classifier or Phase9PrototypeClassifier(num_classes=num_classes)
        self.mask_source = PHASE9_SELECTED_MASK_SOURCE

    def forward(self, batch, return_attention=True, use_prototypes=True):
        octa_out = self.octa_encoder(batch, return_spatial=True, return_attention=return_attention)
        f_octa = octa_out['F_octa']
        f_tab = self.tabular_encoder(
            batch['clinical_features'],
            batch['clinical_mask'],
            biomarkers=batch.get('biomarkers'),
            biomarker_mask=batch.get('biomarker_mask'),
        )
        mask_out = self.mask_generator(batch)
        masks = mask_out['masks']
        fusion_out = self.fusion(f_octa, f_tab, masks)
        clf_out = self.classifier(fusion_out['h'], use_prototypes=use_prototypes)
        return {**octa_out, **mask_out, **fusion_out, **clf_out, 'f_tabular': f_tab, 'mask_source': self.mask_source}


def phase9_make_datasets_for_fold(fold, fold_df=None, biomarker_specs=None, biomarker_stats_by_fold=None, require_unetpp_manifest=True, **dataset_kwargs):
    if fold_df is None:
        if 'phase2_folds' not in globals():
            raise RuntimeError('phase2_folds is required. Run Phase 1/2 fast reload if needed.')
        fold_df = phase2_folds.copy()
    fold = int(fold)
    merged = phase9_merge_unetpp_manifest_for_fold(fold_df, fold) if require_unetpp_manifest else fold_df.copy()
    if 'mask_source' in merged and set(merged['mask_source'].dropna().astype(str)) != {PHASE9_SELECTED_MASK_SOURCE}:
        raise ValueError('Phase 9 datasets require mask_source=unetpp_student only.')
    train_df = merged[merged['fold'] != fold].copy()
    val_df = merged[merged['fold'] == fold].copy()
    validate_no_patient_overlap_between_splits(train_df, val_df)
    biomarker_specs = biomarker_specs if biomarker_specs is not None else infer_phase5_biomarker_specs(merged)
    train_stats = fit_phase5_biomarker_stats(train_df, biomarker_specs)
    biomarker_stats_by_fold = biomarker_stats_by_fold or {fold: train_stats}
    stats = biomarker_stats_by_fold.get(fold, train_stats)
    train_ds = RastaPhase9Dataset(train_df, clinical_preprocessors[fold], layer_stats_by_fold[fold], train=True, biomarker_specs=biomarker_specs, biomarker_stats=stats, **dataset_kwargs)
    val_ds = RastaPhase9Dataset(val_df, clinical_preprocessors[fold], layer_stats_by_fold[fold], train=False, biomarker_specs=biomarker_specs, biomarker_stats=stats, **dataset_kwargs)
    return train_ds, val_ds, biomarker_specs, stats


def build_phase9_model_for_fold(fold, biomarker_specs=None, device=None, freeze_octa_backbones=True, checkpoint_kind='best'):
    fold = int(fold)
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    image_encoders = build_phase4_image_encoders(fold=fold, freeze_backbones=freeze_octa_backbones, require_backbones=True, device=device)
    octa_encoder = Phase6LayerAwareOCTAEncoder(image_encoders).to(device)
    cp = clinical_preprocessors[fold]
    ord_idx, ord_card = phase5_ordinal_metadata(cp)
    biomarker_dim = len(biomarker_specs or [])
    tabular_encoder = Phase5TabularEncoder(
        clinical_dim=len(cp.clinical_cols),
        ordinal_indices=ord_idx,
        ordinal_cardinalities=ord_card,
        biomarker_dim=biomarker_dim,
    ).to(device)
    mask_generator = Phase9FrozenUNetPPMaskGenerator(fold=fold, checkpoint_kind=checkpoint_kind, threshold=None, device=device)
    num_classes = int(phase2_folds['label'].nunique()) if 'phase2_folds' in globals() else 8
    model = Phase9MultimodalClassifier(octa_encoder, tabular_encoder, mask_generator, num_classes=num_classes).to(device)
    model.phase9_build_report = {
        'fold': fold,
        'mask_source': PHASE9_SELECTED_MASK_SOURCE,
        'student_checkpoints': mask_generator.load_report,
        'biomarker_dim': biomarker_dim,
        'num_classes': num_classes,
    }
    return model


def phase9_fit_temperature(logits, labels, init_temperature=1.0, max_iter=100):
    logits = logits.detach().float()
    labels = labels.detach().long()
    log_temperature = torch.nn.Parameter(torch.tensor(float(np.log(init_temperature)), device=logits.device))
    opt = torch.optim.LBFGS([log_temperature], lr=0.05, max_iter=max_iter)
    ce = nn.CrossEntropyLoss()
    def closure():
        opt.zero_grad(set_to_none=True)
        temp = torch.exp(log_temperature).clamp(0.05, 10.0)
        loss = ce(logits / temp, labels)
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.exp(log_temperature).detach().cpu().clamp(0.05, 10.0))


def phase9_apply_temperature(logits, temperature):
    return logits / max(float(temperature), 1e-6)


@torch.no_grad()
def phase9_mc_dropout_predict(model, batch, passes=PHASE9_MC_DROPOUT_PASSES, temperature=1.0):
    was_training = model.training
    model.train()
    probs = []
    for _ in range(int(passes)):
        out = model(batch)
        probs.append(torch.softmax(phase9_apply_temperature(out['logits'], temperature), dim=1))
    probs = torch.stack(probs, dim=0)
    mean_probs = probs.mean(dim=0)
    epistemic_var = probs.var(dim=0, unbiased=False)
    entropy = -(mean_probs * (mean_probs + PHASE9_ENTROPY_EPS).log()).sum(dim=1)
    model.train(was_training)
    return {'mean_probs': mean_probs, 'epistemic_var': epistemic_var, 'entropy': entropy}


phase9_summary = {
    'mask_source': PHASE9_SELECTED_MASK_SOURCE,
    'requires_unetpp_checkpoints': PHASE9_REQUIRES_UNETPP_CHECKPOINTS,
    'prototype_temperature': PHASE9_PROTO_TEMPERATURE,
    'mc_dropout_passes': PHASE9_MC_DROPOUT_PASSES,
    'note': 'Phase 9 uses frozen Phase 8D U-Net++ masks only. No classical fallback is allowed.',
}
save_json(phase9_summary, PHASE9_OUTPUT_DIR / 'phase9_summary.json')
print('Phase 9 code ready. Run phase9_materialize_unetpp_masks_for_fold(fold) on the training machine before file-based Phase 9 datasets.')


# ============================================================
# Phase 10 - Loss Functions
# ============================================================
# This cell defines disease-classifier losses. It does not include segmentation loss.

PHASE10_OUTPUT_DIR = ensure_dir('outputs/phase10_losses')
PHASE10_LABEL_SMOOTHING = 0.10
PHASE10_SEVERE_IMBALANCE_RATIO = 10.0
PHASE10_FOCAL_GAMMA = 2.0
PHASE10_USE_WEIGHTED_RANDOM_SAMPLER = False


class Phase10WeightedLabelSmoothingCE(nn.Module):
    def __init__(self, class_weights=None, smoothing=PHASE10_LABEL_SMOOTHING):
        super().__init__()
        self.smoothing = float(smoothing)
        if class_weights is None:
            self.register_buffer('class_weights', None)
        else:
            self.register_buffer('class_weights', torch.as_tensor(class_weights, dtype=torch.float32))

    def forward(self, logits, target):
        target = target.long()
        num_classes = logits.size(1)
        log_probs = F.log_softmax(logits, dim=1)
        with torch.no_grad():
            true_dist = torch.full_like(log_probs, self.smoothing / max(num_classes - 1, 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device).view(1, -1)
            loss = -(true_dist * log_probs * weights).sum(dim=1)
        else:
            loss = -(true_dist * log_probs).sum(dim=1)
        return loss.mean()


class Phase10FocalLoss(nn.Module):
    def __init__(self, class_weights=None, gamma=PHASE10_FOCAL_GAMMA, smoothing=PHASE10_LABEL_SMOOTHING):
        super().__init__()
        self.gamma = float(gamma)
        self.smoothing = float(smoothing)
        if class_weights is None:
            self.register_buffer('class_weights', None)
        else:
            self.register_buffer('class_weights', torch.as_tensor(class_weights, dtype=torch.float32))

    def forward(self, logits, target):
        target = target.long()
        num_classes = logits.size(1)
        log_probs = F.log_softmax(logits, dim=1)
        probs = log_probs.exp()
        with torch.no_grad():
            true_dist = torch.full_like(log_probs, self.smoothing / max(num_classes - 1, 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        focal = (1.0 - probs).pow(self.gamma)
        if self.class_weights is not None:
            focal = focal * self.class_weights.to(logits.device).view(1, -1)
        return -(true_dist * focal * log_probs).sum(dim=1).mean()


class Phase10DiseaseClassifierLoss(nn.Module):
    """Default Version 1 objective: L_total = L_cls only."""
    def __init__(self, cls_loss):
        super().__init__()
        self.cls_loss = cls_loss

    def forward(self, outputs, batch):
        logits = outputs['logits'] if isinstance(outputs, dict) else outputs
        labels = batch['label'] if isinstance(batch, dict) else batch
        loss_cls = self.cls_loss(logits, labels)
        return {'loss': loss_cls, 'loss_cls': loss_cls.detach()}


def phase10_class_counts(train_df, label_col='label'):
    counts = train_df[label_col].astype(int).value_counts().sort_index()
    num_classes = int(max(counts.index.max() + 1, counts.size)) if len(counts) else 0
    return counts.reindex(range(num_classes), fill_value=0).astype(int)


def phase10_inverse_frequency_weights(counts):
    counts = pd.Series(counts).astype(float)
    safe = counts.replace(0, np.nan)
    inv = 1.0 / safe
    inv = inv / inv.mean(skipna=True)
    inv = inv.fillna(0.0)
    return torch.tensor(inv.to_numpy(dtype=np.float32))


def phase10_imbalance_ratio(counts):
    counts = pd.Series(counts).astype(float)
    nonzero = counts[counts > 0]
    if len(nonzero) == 0:
        return np.nan
    return float(nonzero.max() / nonzero.min())


def phase10_build_loss_bundle(train_df, use_weighted_random_sampler=PHASE10_USE_WEIGHTED_RANDOM_SAMPLER, severe_ratio=PHASE10_SEVERE_IMBALANCE_RATIO):
    counts = phase10_class_counts(train_df)
    ratio = phase10_imbalance_ratio(counts)
    class_weights = None if use_weighted_random_sampler else phase10_inverse_frequency_weights(counts)
    if np.isfinite(ratio) and ratio > float(severe_ratio):
        cls_loss = Phase10FocalLoss(class_weights=class_weights)
        loss_name = 'focal_loss'
    else:
        cls_loss = Phase10WeightedLabelSmoothingCE(class_weights=class_weights)
        loss_name = 'weighted_label_smoothing_ce'
    bundle = {
        'loss_name': loss_name,
        'class_counts': counts.to_dict(),
        'imbalance_ratio': ratio,
        'class_weights': None if class_weights is None else class_weights.tolist(),
        'use_weighted_random_sampler': bool(use_weighted_random_sampler),
        'criterion': Phase10DiseaseClassifierLoss(cls_loss),
    }
    return bundle


def phase10_make_weighted_sampler(train_df, label_col='label'):
    counts = phase10_class_counts(train_df, label_col=label_col)
    weights = phase10_inverse_frequency_weights(counts).numpy()
    sample_weights = train_df[label_col].astype(int).map(lambda y: weights[int(y)]).to_numpy(dtype=np.float32)
    return torch.utils.data.WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)


def phase10_loss_plan_for_outer_folds(fold_df=None):
    if fold_df is None:
        if 'phase2_folds' not in globals():
            raise RuntimeError('phase2_folds is required. Run Phase 1/2 fast reload if needed.')
        fold_df = phase2_folds
    rows = []
    for fold in sorted(fold_df['fold'].dropna().unique()):
        train_df = fold_df[fold_df['fold'] != fold]
        bundle = phase10_build_loss_bundle(train_df)
        rows.append({
            'fold': int(fold),
            'loss_name': bundle['loss_name'],
            'class_counts': json.dumps({str(k): int(v) for k, v in bundle['class_counts'].items()}),
            'imbalance_ratio': bundle['imbalance_ratio'],
            'class_weights': json.dumps(bundle['class_weights']),
            'weighted_random_sampler_default': bundle['use_weighted_random_sampler'],
        })
    return pd.DataFrame(rows)


phase10_summary = {
    'default_loss': 'weighted_label_smoothing_ce unless severe imbalance triggers focal_loss',
    'label_smoothing': PHASE10_LABEL_SMOOTHING,
    'severe_imbalance_ratio': PHASE10_SEVERE_IMBALANCE_RATIO,
    'weighted_random_sampler_default': PHASE10_USE_WEIGHTED_RANDOM_SAMPLER,
    'segmentation_loss_in_classifier': False,
}
save_json(phase10_summary, PHASE10_OUTPUT_DIR / 'phase10_summary.json')
if 'phase2_folds' in globals():
    phase10_loss_plan_for_outer_folds(phase2_folds).to_csv(PHASE10_OUTPUT_DIR / 'phase10_outer_fold_loss_plan.csv', index=False)
print('Phase 10 loss code ready. Default classifier objective is L_total = L_cls only.')

In [ ]:
# ============================================================
# Phase 11 - Data Augmentation
# ============================================================
# Defines synchronized training-only OCTA augmentation for Phase 9+ classifier datasets.
# Geometry is shared across Sup/Deep/CC, U-Net++ student inputs, and mask tensors.
# Intensity/noise is applied only to classifier OCTA encoder inputs, not to masks.

PHASE11_OUTPUT_DIR = ensure_dir(segcomp_model_dir(SEGCOMP_ACTIVE_MODEL_ID) / 'phase11_augmentation')
PHASE11_HFLIP_P = 0.5
PHASE11_VFLIP_P = 0.3
PHASE11_ROTATION_DEGREES = 15.0
PHASE11_TRANSLATE_FRAC = 0.05
PHASE11_SCALE_RANGE = (0.95, 1.05)
PHASE11_SHEAR_DEGREES = 5.0
PHASE11_BRIGHTNESS_DELTA = 0.10
PHASE11_CONTRAST_DELTA = 0.10
PHASE11_GAUSSIAN_NOISE_SIGMA = 0.01
PHASE11_GAUSSIAN_NOISE_P = 0.20
PHASE11_CLINICAL_NOISE_SIGMA = 0.05
PHASE11_CLINICAL_NOISE_P = 0.30
PHASE11_CLINICAL_RANDOM_MASK_P = 0.10
PHASE11_RUN_SMOKE_TEST = False

if torch is None:
    raise ImportError('PyTorch is required for Phase 11 augmentation.')


def phase11_continuous_clinical_indices(clinical_preprocessor):
    return [i for i, col in enumerate(clinical_preprocessor.clinical_cols) if clinical_preprocessor.specs.get(col) == 'continuous']


def phase11_sample_params(rng, image_size=224):
    tx_max = float(PHASE11_TRANSLATE_FRAC) * int(image_size)
    ty_max = float(PHASE11_TRANSLATE_FRAC) * int(image_size)
    return {
        'hflip': bool(rng.random() < PHASE11_HFLIP_P),
        'vflip': bool(rng.random() < PHASE11_VFLIP_P),
        'angle': float(rng.uniform(-PHASE11_ROTATION_DEGREES, PHASE11_ROTATION_DEGREES)),
        'translate': (int(round(rng.uniform(-tx_max, tx_max))), int(round(rng.uniform(-ty_max, ty_max)))),
        'scale': float(rng.uniform(PHASE11_SCALE_RANGE[0], PHASE11_SCALE_RANGE[1])),
        'shear': [float(rng.uniform(-PHASE11_SHEAR_DEGREES, PHASE11_SHEAR_DEGREES)), 0.0],
        'brightness': float(rng.uniform(1.0 - PHASE11_BRIGHTNESS_DELTA, 1.0 + PHASE11_BRIGHTNESS_DELTA)),
        'contrast': float(rng.uniform(1.0 - PHASE11_CONTRAST_DELTA, 1.0 + PHASE11_CONTRAST_DELTA)),
        'add_noise': bool(rng.random() < PHASE11_GAUSSIAN_NOISE_P),
    }


def phase11_affine_tensor(x, params, interpolation='bilinear', fill=0.0):
    """Apply shared geometry to a [C,H,W] tensor."""
    if TVF is None or InterpolationMode is None:
        # Conservative fallback: still apply shared flips if torchvision is unavailable.
        y = x
        if params.get('hflip'):
            y = torch.flip(y, dims=[-1])
        if params.get('vflip'):
            y = torch.flip(y, dims=[-2])
        return y
    y = x
    if params.get('hflip'):
        y = TVF.hflip(y)
    if params.get('vflip'):
        y = TVF.vflip(y)
    interp = InterpolationMode.NEAREST if interpolation == 'nearest' else InterpolationMode.BILINEAR
    return TVF.affine(
        y,
        angle=float(params['angle']),
        translate=tuple(params['translate']),
        scale=float(params['scale']),
        shear=params['shear'],
        interpolation=interp,
        fill=fill,
        center=None,
    )


def phase11_intensity_augment_encoder_image(x, params):
    """Mild brightness/contrast/noise on standardized classifier encoder tensor [1,H,W]."""
    y = x.float()
    mean = y.mean(dim=(-2, -1), keepdim=True)
    y = (y - mean) * float(params['contrast']) + mean
    y = y + float(params['brightness'] - 1.0)
    if params.get('add_noise'):
        y = y + torch.randn_like(y) * float(PHASE11_GAUSSIAN_NOISE_SIGMA)
    return y


def phase11_augment_clinical(sample, clinical_preprocessor, rng):
    values = sample['clinical_features'].clone().float()
    mask = sample['clinical_mask'].clone().float()
    continuous_idx = phase11_continuous_clinical_indices(clinical_preprocessor)
    if continuous_idx and rng.random() < PHASE11_CLINICAL_NOISE_P:
        idx = torch.tensor(continuous_idx, dtype=torch.long)
        observed = mask[idx] > 0.5
        if bool(observed.any()):
            noise = torch.randn(int(observed.sum().item()), dtype=values.dtype) * float(PHASE11_CLINICAL_NOISE_SIGMA)
            values[idx[observed]] = values[idx[observed]] + noise
    if rng.random() < PHASE11_CLINICAL_RANDOM_MASK_P:
        observed_idx = torch.where(mask > 0.5)[0]
        if len(observed_idx):
            drop = observed_idx[int(rng.integers(0, len(observed_idx)))]
            values[drop] = 0.0
            mask[drop] = 0.0
    sample['clinical_features'] = values
    sample['clinical_mask'] = mask
    return sample


class RastaPhase11Dataset(RastaPhase9Dataset):
    """Phase 9 dataset with synchronized train-only OCTA/mask/clinical augmentation."""
    def __init__(self, df, clinical_preprocessor, layer_stats, train=False, seed=PHASE2_SEED, **kwargs):
        # Disable earlier Phase 2 augmentation/dropout in the base class. Phase 11 owns augmentation.
        super().__init__(
            df,
            clinical_preprocessor,
            layer_stats,
            train=False,
            clinical_mask_dropout=0.0,
            seed=seed,
            **kwargs,
        )
        self.phase11_train = bool(train)
        self.phase11_rng = np.random.default_rng(seed)

    def __getitem__(self, idx):
        sample = super().__getitem__(idx)
        if not self.phase11_train:
            sample['phase11_augmented'] = False
            return sample

        params = phase11_sample_params(self.phase11_rng, image_size=self.image_size)
        # Apply identical geometry to classifier OCTA tensors.
        for key in ['sup_image', 'deep_image', 'cc_image']:
            sample[key] = phase11_intensity_augment_encoder_image(
                phase11_affine_tensor(sample[key], params, interpolation='bilinear', fill=0.0),
                params,
            )
        # Apply identical geometry to U-Net++ student inputs, but no brightness/noise so the frozen segmenter sees clean OCTA contrast.
        for key in ['sup_student_image', 'deep_student_image', 'cc_student_image']:
            if key in sample:
                sample[key] = phase11_affine_tensor(sample[key], params, interpolation='bilinear', fill=0.0).clamp(0.0, 1.0)
        # Apply identical geometry to any file-based U-Net++ masks using nearest-neighbour interpolation.
        for key in PHASE8_MASK_ORDER:
            if key in sample:
                sample[key] = phase11_affine_tensor(sample[key], params, interpolation='nearest', fill=0.0).clamp(0.0, 1.0)
        sample = phase11_augment_clinical(sample, self.clinical_preprocessor, self.phase11_rng)
        sample['phase11_augmented'] = True
        sample['phase11_params'] = params
        sample['mask_source'] = PHASE9_SELECTED_MASK_SOURCE
        return sample


def phase11_make_datasets_for_fold(fold, fold_df=None, biomarker_specs=None, biomarker_stats_by_fold=None, require_unetpp_manifest=True, **dataset_kwargs):
    """Build Phase 11 train/validation datasets for one outer fold.

    Requires fold-specific U-Net++ materialized manifests from Phase 9. These manifests are produced
    by frozen checkpoint inference, not by redoing Phase 8 training.
    """
    if fold_df is None:
        if 'phase2_folds' not in globals():
            raise RuntimeError('phase2_folds is required. Run Phase 1/2 fast reload if needed.')
        fold_df = phase2_folds.copy()
    fold = int(fold)
    merged = phase9_merge_unetpp_manifest_for_fold(fold_df, fold) if require_unetpp_manifest else fold_df.copy()
    if 'mask_source' not in merged or set(merged['mask_source'].dropna().astype(str)) != {PHASE9_SELECTED_MASK_SOURCE}:
        raise ValueError('Phase 11 requires mask_source=unetpp_student only. Materialize Phase 9 U-Net++ masks first.')
    train_df = merged[merged['fold'] != fold].copy()
    val_df = merged[merged['fold'] == fold].copy()
    validate_no_patient_overlap_between_splits(train_df, val_df)
    biomarker_specs = biomarker_specs if biomarker_specs is not None else infer_phase5_biomarker_specs(merged)
    train_stats = fit_phase5_biomarker_stats(train_df, biomarker_specs)
    biomarker_stats_by_fold = biomarker_stats_by_fold or {fold: train_stats}
    stats = biomarker_stats_by_fold.get(fold, train_stats)
    train_ds = RastaPhase11Dataset(
        train_df,
        clinical_preprocessors[fold],
        layer_stats_by_fold[fold],
        train=True,
        seed=PHASE2_SEED + fold,
        biomarker_specs=biomarker_specs,
        biomarker_stats=stats,
        **dataset_kwargs,
    )
    val_ds = RastaPhase11Dataset(
        val_df,
        clinical_preprocessors[fold],
        layer_stats_by_fold[fold],
        train=False,
        seed=PHASE2_SEED + 1000 + fold,
        biomarker_specs=biomarker_specs,
        biomarker_stats=stats,
        **dataset_kwargs,
    )
    return train_ds, val_ds, biomarker_specs, stats


def phase11_override_phase9_dataset_builder():
    """Optional convenience: make Phase 9 callers use Phase 11 augmentation by default."""
    globals()['make_phase9_datasets_for_fold'] = phase11_make_datasets_for_fold
    return 'make_phase9_datasets_for_fold now points to phase11_make_datasets_for_fold'


phase11_summary = {
    'mask_source': PHASE9_SELECTED_MASK_SOURCE,
    'train_only': True,
    'geometry_shared_across_layers_student_inputs_and_masks': True,
    'encoder_intensity_jitter': {'brightness_delta': PHASE11_BRIGHTNESS_DELTA, 'contrast_delta': PHASE11_CONTRAST_DELTA, 'noise_sigma': PHASE11_GAUSSIAN_NOISE_SIGMA, 'noise_p': PHASE11_GAUSSIAN_NOISE_P},
    'clinical_augmentation': {'continuous_noise_sigma': PHASE11_CLINICAL_NOISE_SIGMA, 'continuous_noise_p': PHASE11_CLINICAL_NOISE_P, 'random_observed_mask_p': PHASE11_CLINICAL_RANDOM_MASK_P},
    'note': 'Validation/test samples are not augmented. U-Net++ student inputs receive shared geometry but not intensity noise.',
}
save_json(phase11_summary, PHASE11_OUTPUT_DIR / 'phase11_summary.json')
print('Phase 11 augmentation code ready. Use phase11_make_datasets_for_fold(fold) for Phase 12 training datasets.')

In [ ]:
# ============================================================
# Phase 12 - Training Strategy
# ============================================================
# Defines downstream disease-classifier training. Heavy training is disabled by default.
# Stage 1 reuses completed Phase 3 SimCLR backbones. It does not rerun pretraining.
# Stage 2 trains downstream modules with OCTA backbones frozen.
# Stage 3 fine-tunes the disease classifier end to end with frozen U-Net++ segmenters.

PHASE12_OUTPUT_DIR = ensure_dir(segcomp_model_dir(SEGCOMP_ACTIVE_MODEL_ID) / 'phase12_training')
PHASE12_FOLDS = [0, 1, 2, 3, 4]
PHASE12_RUN_TRAINING = False  # the six-model comparison orchestrator launches Phase 12 once per architecture
PHASE12_BATCH_SIZE = 32
PHASE12_NUM_WORKERS = 0
PHASE12_PIN_MEMORY = False
PHASE12_STAGE2_EPOCHS = 15
PHASE12_STAGE3_EPOCHS = 100
PHASE12_STAGE2_LR = 3e-4
PHASE12_STAGE3_HEAD_LR = 3e-4
PHASE12_STAGE3_BACKBONE_LR = 3e-5
PHASE12_WEIGHT_DECAY = 1e-2
PHASE12_GRAD_CLIP_NORM = 1.0
PHASE12_EARLY_STOP_PATIENCE = 15
PHASE12_INNER_VAL_FRACTION = 0.20
PHASE12_WARMUP_EPOCHS = 5
PHASE12_USE_AMP = True
PHASE12_AUTO_RESUME = True
PHASE12_CHECKPOINT_KIND = 'best'
PHASE12_AUTO_MATERIALIZE_UNETPP_MASKS = True
PHASE12_RANDOM_SEED = PHASE2_SEED if 'PHASE2_SEED' in globals() else 82

if torch is None:
    raise ImportError('PyTorch is required for Phase 12 training.')

try:
    from sklearn.metrics import f1_score as _sk_f1_score, balanced_accuracy_score as _sk_balanced_accuracy_score
except Exception:
    _sk_f1_score = None
    _sk_balanced_accuracy_score = None


def phase12_to_device(obj, device):
    if torch.is_tensor(obj):
        return obj.to(device, non_blocking=True)
    if isinstance(obj, dict):
        return {k: phase12_to_device(v, device) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return type(obj)(phase12_to_device(v, device) for v in obj)
    return obj


def phase12_labels_from_batch(batch):
    y = batch['label']
    return y.long() if torch.is_tensor(y) else torch.as_tensor(y, dtype=torch.long)


def phase12_macro_f1(labels, preds, num_classes=None):
    labels = np.asarray(labels, dtype=int)
    preds = np.asarray(preds, dtype=int)
    if _sk_f1_score is not None:
        return float(_sk_f1_score(labels, preds, average='macro', zero_division=0))
    if num_classes is None:
        num_classes = int(max(labels.max(initial=0), preds.max(initial=0)) + 1)
    scores = []
    for cls in range(int(num_classes)):
        tp = np.sum((preds == cls) & (labels == cls))
        fp = np.sum((preds == cls) & (labels != cls))
        fn = np.sum((preds != cls) & (labels == cls))
        denom = 2 * tp + fp + fn
        scores.append(0.0 if denom == 0 else float(2 * tp / denom))
    return float(np.mean(scores)) if scores else 0.0


def phase12_balanced_accuracy(labels, preds, num_classes=None):
    labels = np.asarray(labels, dtype=int)
    preds = np.asarray(preds, dtype=int)
    if _sk_balanced_accuracy_score is not None:
        return float(_sk_balanced_accuracy_score(labels, preds))
    if num_classes is None:
        num_classes = int(max(labels.max(initial=0), preds.max(initial=0)) + 1)
    recalls = []
    for cls in range(int(num_classes)):
        mask = labels == cls
        if mask.any():
            recalls.append(float(np.mean(preds[mask] == cls)))
    return float(np.mean(recalls)) if recalls else 0.0



def phase12_per_class_recall(labels, preds, num_classes=None):
    labels = np.asarray(labels, dtype=int)
    preds = np.asarray(preds, dtype=int)
    if num_classes is None:
        if labels.size == 0 and preds.size == 0:
            return {}
        num_classes = int(max(labels.max(initial=0), preds.max(initial=0)) + 1)
    out = {}
    for cls in range(int(num_classes)):
        mask = labels == cls
        out[str(cls)] = None if not mask.any() else float(np.mean(preds[mask] == cls))
    return out


def phase12_patient_grouped_inner_split(train_df, val_fraction=PHASE12_INNER_VAL_FRACTION, seed=PHASE12_RANDOM_SEED):
    df = train_df.copy()
    if 'patient_id' not in df:
        raise ValueError('patient_id is required for leakage-safe inner validation split.')
    patients = pd.Series(df['patient_id'].astype(str).dropna().unique())
    shuffled = patients.sample(frac=1.0, random_state=int(seed)).to_numpy()
    n_val = int(round(len(shuffled) * float(val_fraction)))
    n_val = min(max(n_val, 1), max(len(shuffled) - 1, 1))
    val_patients = set(shuffled[:n_val])
    val_mask = df['patient_id'].astype(str).isin(val_patients)
    inner_train = df[~val_mask].copy()
    inner_val = df[val_mask].copy()
    validate_no_patient_overlap_between_splits(inner_train, inner_val)
    return inner_train, inner_val


def phase12_make_dataframes_for_fold(fold, fold_df=None):
    if fold_df is None:
        if 'phase2_folds' not in globals():
            raise RuntimeError('phase2_folds is required. Run Phase 1/2 fast reload if needed.')
        fold_df = phase2_folds.copy()
    fold = int(fold)
    try:
        merged = phase9_merge_unetpp_manifest_for_fold(fold_df, fold)
    except FileNotFoundError:
        if not PHASE12_AUTO_MATERIALIZE_UNETPP_MASKS:
            raise
        print(f'Phase 12 auto-materializing U-Net++ masks for fold {fold} from existing Phase 8D checkpoints.')
        phase9_materialize_unetpp_masks_for_fold(fold, fold_df=fold_df, checkpoint_kind=PHASE12_CHECKPOINT_KIND)
        merged = phase9_merge_unetpp_manifest_for_fold(fold_df, fold)
    if 'mask_source' not in merged or set(merged['mask_source'].dropna().astype(str)) != {PHASE9_SELECTED_MASK_SOURCE}:
        raise ValueError('Phase 12 requires mask_source=unetpp_student only.')
    outer_train = merged[merged['fold'] != fold].copy()
    outer_test = merged[merged['fold'] == fold].copy()
    validate_no_patient_overlap_between_splits(outer_train, outer_test)
    inner_train, inner_val = phase12_patient_grouped_inner_split(outer_train, seed=PHASE12_RANDOM_SEED + fold)
    validate_no_patient_overlap_between_splits(inner_train, outer_test)
    validate_no_patient_overlap_between_splits(inner_val, outer_test)
    return inner_train, inner_val, outer_test, merged


def phase12_make_datasets_for_fold(fold, fold_df=None, biomarker_specs=None):
    inner_train, inner_val, outer_test, merged = phase12_make_dataframes_for_fold(fold, fold_df=fold_df)
    biomarker_specs = biomarker_specs if biomarker_specs is not None else infer_phase5_biomarker_specs(merged)
    biomarker_stats = fit_phase5_biomarker_stats(inner_train, biomarker_specs)
    cp = clinical_preprocessors[int(fold)]
    layer_stats = layer_stats_by_fold[int(fold)]
    train_ds = RastaPhase11Dataset(
        inner_train, cp, layer_stats, train=True, seed=PHASE12_RANDOM_SEED + int(fold),
        biomarker_specs=biomarker_specs, biomarker_stats=biomarker_stats,
    )
    val_ds = RastaPhase11Dataset(
        inner_val, cp, layer_stats, train=False, seed=PHASE12_RANDOM_SEED + 1000 + int(fold),
        biomarker_specs=biomarker_specs, biomarker_stats=biomarker_stats,
    )
    test_ds = RastaPhase11Dataset(
        outer_test, cp, layer_stats, train=False, seed=PHASE12_RANDOM_SEED + 2000 + int(fold),
        biomarker_specs=biomarker_specs, biomarker_stats=biomarker_stats,
    )
    return train_ds, val_ds, test_ds, biomarker_specs, biomarker_stats, {'inner_train': inner_train, 'inner_val': inner_val, 'outer_test': outer_test, 'merged': merged}


def phase12_make_loader(ds, train=False, batch_size=PHASE12_BATCH_SIZE):
    return torch.utils.data.DataLoader(
        ds,
        batch_size=int(batch_size),
        shuffle=bool(train),
        num_workers=PHASE12_NUM_WORKERS,
        pin_memory=PHASE12_PIN_MEMORY,
        drop_last=False,
    )


def phase12_set_octa_backbones_trainable(model, trainable):
    encoders = model.octa_encoder.image_encoders.encoders
    for encoder in encoders.values():
        for p in encoder.backbone.parameters():
            p.requires_grad = bool(trainable)


def phase12_backbone_param_ids(model):
    ids = set()
    encoders = model.octa_encoder.image_encoders.encoders
    for encoder in encoders.values():
        for p in encoder.backbone.parameters():
            ids.add(id(p))
    return ids


def phase12_optimizer(model, stage):
    if stage == 'stage2':
        params = [p for p in model.parameters() if p.requires_grad]
        return torch.optim.AdamW(params, lr=PHASE12_STAGE2_LR, weight_decay=PHASE12_WEIGHT_DECAY)
    if stage == 'stage3':
        backbone_ids = phase12_backbone_param_ids(model)
        backbone_params, head_params = [], []
        for p in model.parameters():
            if not p.requires_grad:
                continue
            if id(p) in backbone_ids:
                backbone_params.append(p)
            else:
                head_params.append(p)
        return torch.optim.AdamW(
            [
                {'params': backbone_params, 'lr': PHASE12_STAGE3_BACKBONE_LR, 'name': 'backbone'},
                {'params': head_params, 'lr': PHASE12_STAGE3_HEAD_LR, 'name': 'head'},
            ],
            weight_decay=PHASE12_WEIGHT_DECAY,
        )
    raise ValueError(f'Unknown stage: {stage}')


def phase12_set_epoch_lrs(optimizer, stage, epoch, total_epochs):
    if stage == 'stage2':
        return [group['lr'] for group in optimizer.param_groups]
    warmup = max(int(PHASE12_WARMUP_EPOCHS), 1)
    if epoch <= warmup:
        factor = float(epoch) / float(warmup)
    else:
        progress = (epoch - warmup) / max(total_epochs - warmup, 1)
        factor = 0.5 * (1.0 + np.cos(np.pi * progress))
    for group in optimizer.param_groups:
        base = PHASE12_STAGE3_BACKBONE_LR if group.get('name') == 'backbone' else PHASE12_STAGE3_HEAD_LR
        group['lr'] = float(base * factor)
    return [group['lr'] for group in optimizer.param_groups]


def phase12_checkpoint_payload(model, optimizer, scaler, epoch, history, best_metric, stale_epochs, stage, early_stopped=False, completed=False):
    return {
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scaler': None if scaler is None else scaler.state_dict(),
        'epoch': int(epoch),
        'history': history,
        'best_macro_f1': float(best_metric),
        'stale_epochs': int(stale_epochs),
        'stage': stage,
        'mask_source': PHASE9_SELECTED_MASK_SOURCE,
        'early_stopped': bool(early_stopped),
        'completed': bool(completed),
    }


def phase12_load_checkpoint(path, model, optimizer=None, scaler=None, device='cpu'):
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message=r'.*weights_only=False.*', category=FutureWarning)
        payload = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(payload['model'], strict=False)
    if optimizer is not None and payload.get('optimizer') is not None:
        optimizer.load_state_dict(payload['optimizer'])
    if scaler is not None and payload.get('scaler') is not None:
        scaler.load_state_dict(payload['scaler'])
    return payload


@torch.no_grad()
def phase12_evaluate(model, loader, criterion_bundle, device, temperature=None):
    model.eval()
    total_loss = 0.0
    n = 0
    all_logits, all_labels, all_patient_ids, all_eyes = [], [], [], []
    for batch in loader:
        batch = phase12_to_device(batch, device)
        labels = phase12_labels_from_batch(batch)
        outputs = model(batch)
        loss_dict = criterion_bundle['criterion'](outputs, batch)
        logits = outputs['logits']
        if temperature is not None:
            logits = phase9_apply_temperature(logits, temperature)
        total_loss += float(loss_dict['loss'].item()) * labels.numel()
        n += labels.numel()
        all_logits.append(logits.detach().cpu())
        all_labels.append(labels.detach().cpu())
        all_patient_ids.extend(batch.get('patient_id', [''] * labels.numel()))
        all_eyes.extend(batch.get('eye', [''] * labels.numel()))
    logits = torch.cat(all_logits, dim=0) if all_logits else torch.empty(0, 0)
    labels = torch.cat(all_labels, dim=0) if all_labels else torch.empty(0, dtype=torch.long)
    probs = torch.softmax(logits, dim=1) if logits.numel() else logits
    preds = probs.argmax(dim=1).numpy() if probs.numel() else np.array([], dtype=int)
    y = labels.numpy() if labels.numel() else np.array([], dtype=int)
    num_classes = logits.shape[1] if logits.ndim == 2 and logits.shape[1] else None
    return {
        'loss': total_loss / max(n, 1),
        'macro_f1': phase12_macro_f1(y, preds, num_classes=num_classes),
        'balanced_accuracy': phase12_balanced_accuracy(y, preds, num_classes=num_classes),
        'per_class_recall': phase12_per_class_recall(y, preds, num_classes=num_classes),
        'logits': logits,
        'labels': labels,
        'probs': probs,
        'preds': preds,
        'patient_ids': all_patient_ids,
        'eyes': all_eyes,
        'n': int(n),
    }


def phase12_train_one_stage(model, train_loader, val_loader, test_loader, criterion_bundle, stage, epochs, output_dir, device, fold=None, progress_callback=None):
    """Train one classifier stage with resumable checkpoints and train/val/test logging.

    Test metrics are monitored for visibility only. Checkpoint selection and early stopping use
    inner-validation macro-F1, never the outer test metric.
    """
    output_dir = ensure_dir(output_dir)
    phase12_set_octa_backbones_trainable(model, trainable=(stage == 'stage3'))
    optimizer = phase12_optimizer(model, stage)
    use_amp = bool(PHASE12_USE_AMP and device.type == 'cuda')
    try:
        scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    last_path = output_dir / f'{stage}_last.pt'
    best_path = output_dir / f'{stage}_best.pt'
    history_path = output_dir / f'{stage}_history.csv'
    start_epoch = 1
    best_metric = -1.0
    stale_epochs = 0
    history = []
    if PHASE12_AUTO_RESUME and last_path.exists():
        payload = phase12_load_checkpoint(last_path, model, optimizer=optimizer, scaler=scaler, device=device)
        history = payload.get('history', []) or []
        best_metric = float(payload.get('best_macro_f1', -1.0))
        stale_epochs = int(payload.get('stale_epochs', 0))
        start_epoch = int(payload.get('epoch', 0)) + 1
        if payload.get('early_stopped') or payload.get('completed') or start_epoch > int(epochs):
            reason = 'early_stopped' if payload.get('early_stopped') else 'completed'
            print(f'COMPLETE Phase 12 {stage} fold={fold}: {reason}; last_epoch={start_epoch - 1}, best_macro_f1={best_metric:.4f}')
            return pd.DataFrame(history)
        print(f'RESUME Phase 12 {stage} fold={fold}: last_epoch={start_epoch - 1}, best_macro_f1={best_metric:.4f}, stale={stale_epochs}, remaining={max(0, int(epochs)-start_epoch+1)}')
    if start_epoch > int(epochs):
        print(f'COMPLETE Phase 12 {stage} fold={fold}: already reached epoch {start_epoch - 1}')
        return pd.DataFrame(history)

    print(f'START Phase 12 {stage} fold={fold}: epochs={epochs}, start_epoch={start_epoch}, device={device}, train_batches={len(train_loader)}, val_batches={len(val_loader)}, test_batches={len(test_loader)}')
    run_start = time.time()
    epoch_times = []
    for epoch in range(start_epoch, int(epochs) + 1):
        t0 = time.time()
        lrs = phase12_set_epoch_lrs(optimizer, stage, epoch, int(epochs))
        lr_text = ','.join(f'{float(x):.2e}' for x in lrs)
        print(f'START Phase 12 {stage} fold={fold} epoch {epoch}/{epochs}: lr=[{lr_text}]')
        model.train()
        train_loss = 0.0
        n_train = 0
        train_logits, train_labels = [], []
        grad_norms = []
        for batch in train_loader:
            batch = phase12_to_device(batch, device)
            labels = phase12_labels_from_batch(batch)
            optimizer.zero_grad(set_to_none=True)
            if use_amp:
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    outputs = model(batch)
                    loss = criterion_bundle['criterion'](outputs, batch)['loss']
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], PHASE12_GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(batch)
                loss = criterion_bundle['criterion'](outputs, batch)['loss']
                loss.backward()
                grad_norm = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], PHASE12_GRAD_CLIP_NORM)
                optimizer.step()
            grad_norms.append(float(grad_norm.detach().cpu() if torch.is_tensor(grad_norm) else grad_norm))
            if 'penultimate' in outputs:
                model.classifier.update_prototypes(outputs['penultimate'].detach(), labels.detach())
            train_loss += float(loss.item()) * labels.numel()
            n_train += labels.numel()
            train_logits.append(outputs['logits'].detach().cpu())
            train_labels.append(labels.detach().cpu())
        tr_logits = torch.cat(train_logits, dim=0)
        tr_labels = torch.cat(train_labels, dim=0)
        tr_preds = tr_logits.argmax(dim=1).numpy()
        tr_y = tr_labels.numpy()
        num_classes = tr_logits.shape[1]
        train_metrics = {
            'loss': train_loss / max(n_train, 1),
            'macro_f1': phase12_macro_f1(tr_y, tr_preds, num_classes=num_classes),
            'balanced_accuracy': phase12_balanced_accuracy(tr_y, tr_preds, num_classes=num_classes),
            'per_class_recall': phase12_per_class_recall(tr_y, tr_preds, num_classes=num_classes),
            'n': int(n_train),
        }
        val_metrics = phase12_evaluate(model, val_loader, criterion_bundle, device)
        test_metrics = phase12_evaluate(model, test_loader, criterion_bundle, device)
        epoch_sec = time.time() - t0
        epoch_times.append(epoch_sec)
        avg_sec = float(np.mean(epoch_times))
        remaining = max(0, int(epochs) - epoch)
        eta_min = remaining * avg_sec / 60.0
        proto_alpha = float(F.softplus(model.classifier.proto_alpha_raw).detach().cpu()) if hasattr(model, 'classifier') else np.nan
        row = {
            'stage': stage,
            'fold': int(fold) if fold is not None else -1,
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'train_macro_f1': train_metrics['macro_f1'],
            'train_balanced_accuracy': train_metrics['balanced_accuracy'],
            'train_per_class_recall': json.dumps(train_metrics['per_class_recall']),
            'val_loss': val_metrics['loss'],
            'val_macro_f1': val_metrics['macro_f1'],
            'val_balanced_accuracy': val_metrics['balanced_accuracy'],
            'val_per_class_recall': json.dumps(val_metrics['per_class_recall']),
            'test_loss': test_metrics['loss'],
            'test_macro_f1': test_metrics['macro_f1'],
            'test_balanced_accuracy': test_metrics['balanced_accuracy'],
            'test_per_class_recall': json.dumps(test_metrics['per_class_recall']),
            'lr': json.dumps([float(x) for x in lrs]),
            'epoch_sec': epoch_sec,
            'elapsed_min': (time.time() - run_start) / 60.0,
            'eta_min': eta_min,
            'grad_norm_mean': float(np.mean(grad_norms)) if grad_norms else np.nan,
            'grad_norm_max': float(np.max(grad_norms)) if grad_norms else np.nan,
            'prototype_alpha': proto_alpha,
            'prototype_initialized_count': int(model.classifier.prototype_initialized.sum().item()),
        }
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)
        improved = row['val_macro_f1'] > best_metric
        early_stopped = False
        if improved:
            best_metric = row['val_macro_f1']
            stale_epochs = 0
            torch.save(phase12_checkpoint_payload(model, optimizer, scaler, epoch, history, best_metric, stale_epochs, stage, early_stopped=False, completed=False), best_path)
        else:
            stale_epochs += 1
        if stage == 'stage3' and stale_epochs >= PHASE12_EARLY_STOP_PATIENCE:
            early_stopped = True
        completed = epoch >= int(epochs)
        torch.save(phase12_checkpoint_payload(model, optimizer, scaler, epoch, history, best_metric, stale_epochs, stage, early_stopped=early_stopped, completed=completed), last_path)
        print(
            f"END Phase 12 {stage} fold={fold} epoch {epoch}/{epochs}: "
            f"train_loss={row['train_loss']:.4f} train_macro_f1={row['train_macro_f1']:.4f} train_bal_acc={row['train_balanced_accuracy']:.4f} | "
            f"val_loss={row['val_loss']:.4f} val_macro_f1={row['val_macro_f1']:.4f} val_bal_acc={row['val_balanced_accuracy']:.4f} | "
            f"test_loss={row['test_loss']:.4f} test_macro_f1={row['test_macro_f1']:.4f} test_bal_acc={row['test_balanced_accuracy']:.4f} | "
            f"grad_norm_mean={row['grad_norm_mean']:.3f} proto_alpha={row['prototype_alpha']:.4f} proto_init={row['prototype_initialized_count']} "
            f"time={epoch_sec/60:.1f}m avg={avg_sec/60:.1f}m ETA={eta_min:.1f}m best_val_macro_f1={best_metric:.4f} stale={stale_epochs}"
        )
        if progress_callback is not None:
            progress_callback({'stage': stage, 'fold': fold, 'epoch': epoch, 'stage_epochs': int(epochs), 'epoch_seconds': epoch_sec, 'stage_elapsed_seconds': time.time() - run_start, 'stage_eta_seconds': remaining * avg_sec, 'best_val_macro_f1': best_metric})
        if early_stopped:
            print(f'EARLY STOP Phase 12 {stage} fold={fold}: patience={PHASE12_EARLY_STOP_PATIENCE}')
            break
    if best_path.exists():
        phase12_load_checkpoint(best_path, model, device=device)
    print(f'DONE Phase 12 {stage} fold={fold}: best_val_macro_f1={best_metric:.4f}, epochs_recorded={len(history)}')
    return pd.DataFrame(history)

def phase12_predictions_dataframe(eval_metrics, fold, temperature):
    probs = eval_metrics['probs'].numpy()
    labels = eval_metrics['labels'].numpy()
    preds = eval_metrics['preds']
    rows = []
    for i in range(len(labels)):
        row = {
            'fold': int(fold),
            'patient_id': eval_metrics['patient_ids'][i] if i < len(eval_metrics['patient_ids']) else '',
            'eye': eval_metrics['eyes'][i] if i < len(eval_metrics['eyes']) else '',
            'label': int(labels[i]),
            'pred': int(preds[i]),
            'correct': int(labels[i] == preds[i]),
            'temperature': float(temperature),
            'mask_source': PHASE9_SELECTED_MASK_SOURCE,
            'segmentation_model_id': SEGCOMP_ACTIVE_MODEL_ID,
        }
        for k in range(probs.shape[1]):
            row[f'prob_class_{k}'] = float(probs[i, k])
        rows.append(row)
    return pd.DataFrame(rows)



def phase12_stage1_reuse_report(fold):
    """Summarize reused Phase 3 SimCLR backbones for Stage 1."""
    rows = []
    for layer in ['sup', 'deep', 'cc']:
        path = phase4_find_backbone_path(layer, fold=fold, scope=globals().get('PHASE3_PRETRAIN_SCOPE', 'fold_train'))
        hist_path = Path(str(path).replace('_backbone.pt', '_history.csv'))
        row = {
            'fold': int(fold),
            'layer': layer,
            'backbone_path': str(path),
            'backbone_exists': bool(path.exists()),
            'history_path': str(hist_path),
            'history_exists': bool(hist_path.exists()),
        }
        if hist_path.exists():
            hist = pd.read_csv(hist_path)
            loss_col = 'loss' if 'loss' in hist.columns else ('train_loss' if 'train_loss' in hist.columns else None)
            if loss_col is not None and len(hist):
                row.update({
                    'epochs': int(len(hist)),
                    'first_loss': float(hist[loss_col].iloc[0]),
                    'final_loss': float(hist[loss_col].iloc[-1]),
                    'best_loss': float(hist[loss_col].min()),
                    'loss_decrease_first_to_best': float(hist[loss_col].iloc[0] - hist[loss_col].min()),
                })
        rows.append(row)
    return pd.DataFrame(rows)


def phase12_train_fold(fold, fold_df=None, device=None):
    seed_everything(PHASE12_RANDOM_SEED + int(fold))
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    fold = int(fold)
    fold_dir = ensure_dir(PHASE12_OUTPUT_DIR / f'fold_{fold}')
    train_ds, val_ds, test_ds, biomarker_specs, biomarker_stats, dfs = phase12_make_datasets_for_fold(fold, fold_df=fold_df)
    loaders = {
        'train': phase12_make_loader(train_ds, train=True),
        'val': phase12_make_loader(val_ds, train=False),
        'test': phase12_make_loader(test_ds, train=False),
    }
    loss_bundle = phase10_build_loss_bundle(dfs['inner_train'])
    model = build_phase9_model_for_fold(fold, biomarker_specs=biomarker_specs, device=device, freeze_octa_backbones=True, checkpoint_kind=PHASE12_CHECKPOINT_KIND)
    print(f'Phase 12 fold {fold} split sizes: train={len(train_ds)} inner_val={len(val_ds)} outer_test={len(test_ds)} mask_source={PHASE9_SELECTED_MASK_SOURCE}')
    stage1_df = phase12_stage1_reuse_report(fold)
    stage1_df.to_csv(fold_dir / 'stage1_reuse_report.csv', index=False)
    stage1_report = {
        'fold': fold,
        'status': 'completed_by_reusing_existing_phase3_backbones_no_training_in_phase12',
        'phase3_scope': globals().get('PHASE3_PRETRAIN_SCOPE', 'fold_train'),
        'mask_source': PHASE9_SELECTED_MASK_SOURCE,
        'split_sizes': {'train': len(train_ds), 'inner_val': len(val_ds), 'outer_test': len(test_ds)},
        'layers': stage1_df.to_dict(orient='records'),
    }
    save_json(stage1_report, fold_dir / 'stage1_reuse_report.json')
    print('Stage 1 reuse audit:', stage1_df[['layer', 'backbone_exists', 'history_exists', 'best_loss']].to_dict(orient='records') if 'best_loss' in stage1_df else stage1_df.to_dict(orient='records'))
    stage2_history = phase12_train_one_stage(model, loaders['train'], loaders['val'], loaders['test'], loss_bundle, 'stage2', PHASE12_STAGE2_EPOCHS, fold_dir, device, fold=fold)
    stage2_best = fold_dir / 'stage2_best.pt'
    if stage2_best.exists():
        phase12_load_checkpoint(stage2_best, model, device=device)
    stage3_history = phase12_train_one_stage(model, loaders['train'], loaders['val'], loaders['test'], loss_bundle, 'stage3', PHASE12_STAGE3_EPOCHS, fold_dir, device, fold=fold)
    val_eval_uncal = phase12_evaluate(model, loaders['val'], loss_bundle, device)
    temperature = phase9_fit_temperature(val_eval_uncal['logits'].to(device), val_eval_uncal['labels'].to(device)) if val_eval_uncal['n'] else 1.0
    val_eval = phase12_evaluate(model, loaders['val'], loss_bundle, device, temperature=temperature)
    test_eval = phase12_evaluate(model, loaders['test'], loss_bundle, device, temperature=temperature)
    pred_df = phase12_predictions_dataframe(test_eval, fold=fold, temperature=temperature)
    pred_df.to_csv(fold_dir / 'outer_test_predictions.csv', index=False)
    summary = {
        'fold': fold,
        'mask_source': PHASE9_SELECTED_MASK_SOURCE,
        'n_train': len(train_ds),
        'n_inner_val': len(val_ds),
        'n_outer_test': len(test_ds),
        'loss_name': loss_bundle['loss_name'],
        'imbalance_ratio': loss_bundle['imbalance_ratio'],
        'temperature': temperature,
        'val_macro_f1': val_eval['macro_f1'],
        'val_balanced_accuracy': val_eval['balanced_accuracy'],
        'test_macro_f1': test_eval['macro_f1'],
        'test_balanced_accuracy': test_eval['balanced_accuracy'],
        'stage2_epochs_recorded': int(len(stage2_history)),
        'stage3_epochs_recorded': int(len(stage3_history)),
    }
    save_json(summary, fold_dir / 'fold_summary.json')
    return summary


def phase12_train_cross_validation(folds=PHASE12_FOLDS, fold_df=None, device=None):
    summaries = []
    for fold in folds:
        summaries.append(phase12_train_fold(int(fold), fold_df=fold_df, device=device))
    summary_df = pd.DataFrame(summaries)
    summary_df.to_csv(PHASE12_OUTPUT_DIR / 'phase12_cross_validation_summary.csv', index=False)
    all_preds = []
    for fold in folds:
        path = PHASE12_OUTPUT_DIR / f'fold_{int(fold)}' / 'outer_test_predictions.csv'
        if path.exists():
            all_preds.append(pd.read_csv(path))
    if all_preds:
        pd.concat(all_preds, ignore_index=True).to_csv(PHASE12_OUTPUT_DIR / 'phase12_all_outer_test_predictions.csv', index=False)
    return summary_df


phase12_summary = {
    'run_training_default': PHASE12_RUN_TRAINING,
    'folds': PHASE12_FOLDS,
    'mask_source': PHASE9_SELECTED_MASK_SOURCE,
    'stage1': 'reuse completed Phase 3 SimCLR backbones',
    'stage2_epochs': PHASE12_STAGE2_EPOCHS,
    'stage3_epochs': PHASE12_STAGE3_EPOCHS,
    'stage2_lr': PHASE12_STAGE2_LR,
    'stage3_head_lr': PHASE12_STAGE3_HEAD_LR,
    'stage3_backbone_lr': PHASE12_STAGE3_BACKBONE_LR,
    'inner_validation_fraction': PHASE12_INNER_VAL_FRACTION,
    'early_stop_patience': PHASE12_EARLY_STOP_PATIENCE,
    'auto_materialize_unetpp_masks': PHASE12_AUTO_MATERIALIZE_UNETPP_MASKS,
    'note': 'Run phase12_train_cross_validation(PHASE12_FOLDS) on the training machine. Missing Phase 9 U-Net++ manifests can be materialized automatically from existing Phase 8D checkpoints.',
}
save_json(phase12_summary, PHASE12_OUTPUT_DIR / 'phase12_summary.json')
if PHASE12_RUN_TRAINING:
    phase12_cv_summary = phase12_train_cross_validation(PHASE12_FOLDS)
else:
    print('Phase 12 training code ready. Set PHASE12_RUN_TRAINING=True or call phase12_train_cross_validation(PHASE12_FOLDS) on the training machine.')

In [ ]:
# ============================================================
# Phase 13 - Explainability
# ============================================================
# Defines post-training explainability for trained Phase 12 fold models:
#   1. Grad-CAM per OCTA layer using Phase 4 final spatial maps.
#   2. Cross-layer attention dominance from Phase 6 attention weights.
#   3. Tabular SHAP where available, with an explicit occlusion fallback.
#   4. Integrated Grad-CAM plus attention consistency summaries.
#
# Heavy explainability generation is disabled by default in this review copy.
# After Phase 12 checkpoints exist, run phase13_run_cross_validation(PHASE13_FOLDS).

PHASE13_OUTPUT_DIR = ensure_dir(segcomp_model_dir(SEGCOMP_ACTIVE_MODEL_ID) / 'phase13_explainability')
PHASE13_GRADCAM_DIR = ensure_dir(PHASE13_OUTPUT_DIR / 'gradcam')
PHASE13_ATTENTION_DIR = ensure_dir(PHASE13_OUTPUT_DIR / 'attention')
PHASE13_TABULAR_DIR = ensure_dir(PHASE13_OUTPUT_DIR / 'tabular_attribution')
PHASE13_INTEGRATED_DIR = ensure_dir(PHASE13_OUTPUT_DIR / 'integrated')

PHASE13_FOLDS = PHASE12_FOLDS if 'PHASE12_FOLDS' in globals() else [0, 1, 2, 3, 4]
PHASE13_BATCH_SIZE = 4
PHASE13_GRADCAM_BATCH_SIZE = 1  # Grad-CAM backprop is memory-heavy on 8 GB GPUs.
PHASE13_TABULAR_BATCH_SIZE = 1  # KernelSHAP loops per sample; keep image batch tiny.
PHASE13_SHAP_PREDICT_BATCH_SIZE = 4  # Chunk KernelSHAP model calls to avoid repeating images into huge GPU batches.
PHASE13_NUM_WORKERS = 0
PHASE13_CHECKPOINT_NAME = 'stage3_best.pt'
PHASE13_FALLBACK_CHECKPOINT_NAME = 'stage3_last.pt'
PHASE13_MAX_GRADCAM_SAMPLES_PER_FOLD = 24
PHASE13_MAX_ATTENTION_SAMPLES_PER_FOLD = None
PHASE13_MAX_TABULAR_SAMPLES_PER_FOLD = 12
PHASE13_CLASS_BALANCED_EXAMPLES = True
PHASE13_EXAMPLES_PER_CLASS = None  # None means derive from max_samples / n_classes.
PHASE13_GRADCAM_TARGET = 'pred'  # Legacy single-target default.
PHASE13_GRADCAM_TARGETS = ['pred', 'label', 'all_classes']  # pred, true label, then explicit per-class targets.
PHASE13_SHAP_BACKGROUND_SIZE = 24
PHASE13_SHAP_NSAMPLES = 128
PHASE13_REQUIRE_REAL_SHAP = True
PHASE13_RUN_GRADCAM = True
PHASE13_RUN_ATTENTION = True
PHASE13_RUN_TABULAR_SHAP = True
PHASE13_RUN_EXPLAINABILITY = True
PHASE13_LOG_EVERY_N_BATCHES = 1
PHASE13_LOG_EVERY_N_SAMPLES = 4
PHASE13_LAYER_NAMES = ['sup', 'deep', 'cc']
PHASE13_IMAGE_KEYS = {'sup': 'sup_image', 'deep': 'deep_image', 'cc': 'cc_image'}

if torch is None:
    raise ImportError('PyTorch is required for Phase 13 explainability.')

import math
import time
import warnings
import json
import re
from pathlib import Path
import torch.nn.functional as F
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print('Phase 13 plotting disabled because matplotlib could not be imported:', exc)



def phase13_format_seconds(seconds):
    seconds = max(float(seconds), 0.0)
    if seconds < 60:
        return f'{seconds:.1f}s'
    minutes, sec = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f'{hours:d}h {minutes:02d}m {sec:02d}s'
    return f'{minutes:d}m {sec:02d}s'


def phase13_log(message):
    print(f'[Phase 13] {message}', flush=True)


def phase13_clear_cuda_cache():
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def phase13_log_progress(task, fold, seen, total, start_time, extra=''):
    elapsed = time.time() - start_time
    total = int(total) if total is not None else None
    if total and seen > 0:
        rate = seen / max(elapsed, 1e-6)
        eta = (total - seen) / max(rate, 1e-6)
        msg = f'fold {int(fold)} {task}: {seen}/{total} samples | elapsed {phase13_format_seconds(elapsed)} | ETA {phase13_format_seconds(eta)}'
    else:
        msg = f'fold {int(fold)} {task}: {seen} samples | elapsed {phase13_format_seconds(elapsed)}'
    if extra:
        msg += f' | {extra}'
    phase13_log(msg)

def phase13_slug(value):
    if '_phase8_safe_slug' in globals():
        return _phase8_safe_slug(value)
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value))


def phase13_label_name_map(fold_df=None):
    if fold_df is None and 'phase2_folds' in globals():
        fold_df = phase2_folds
    if fold_df is None or 'label' not in fold_df:
        return {}
    if 'cohort' in fold_df:
        pairs = fold_df[['label', 'cohort']].drop_duplicates().sort_values('label')
        return {int(r['label']): str(r['cohort']) for _, r in pairs.iterrows()}
    labels = sorted(pd.Series(fold_df['label']).dropna().astype(int).unique().tolist())
    return {int(k): f'class_{int(k)}' for k in labels}


def phase13_get_temperature(fold):
    summary_path = PHASE12_OUTPUT_DIR / f'fold_{int(fold)}' / 'fold_summary.json'
    if summary_path.exists():
        try:
            with open(summary_path, 'r', encoding='utf-8') as f:
                payload = json.load(f)
            return float(payload.get('temperature', 1.0))
        except Exception:
            return 1.0
    return 1.0


def phase13_checkpoint_path(fold, checkpoint_name=PHASE13_CHECKPOINT_NAME):
    fold_dir = PHASE12_OUTPUT_DIR / f'fold_{int(fold)}'
    primary = fold_dir / checkpoint_name
    if primary.exists():
        return primary
    fallback = fold_dir / PHASE13_FALLBACK_CHECKPOINT_NAME
    if fallback.exists():
        return fallback
    raise FileNotFoundError(f'No Phase 12 checkpoint found for fold {fold}: tried {primary} and {fallback}')


def phase13_make_loader(dataset, batch_size=PHASE13_BATCH_SIZE):
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=int(batch_size),
        shuffle=False,
        num_workers=int(PHASE13_NUM_WORKERS),
        pin_memory=False,
    )


def phase13_dataset_dataframe(dataset):
    if hasattr(dataset, 'df'):
        return dataset.df.reset_index(drop=True)
    if isinstance(dataset, torch.utils.data.Subset):
        parent_df = phase13_dataset_dataframe(dataset.dataset)
        return parent_df.iloc[list(dataset.indices)].reset_index(drop=True)
    return None


def phase13_select_explainability_indices(dataset, max_samples=None, class_balanced=PHASE13_CLASS_BALANCED_EXAMPLES, examples_per_class=PHASE13_EXAMPLES_PER_CLASS):
    """Return deterministic indices for explainability examples, balanced by true class when possible."""
    n = len(dataset)
    if max_samples is None and not class_balanced and examples_per_class is None:
        return list(range(n))
    df = phase13_dataset_dataframe(dataset)
    if df is None or 'label' not in df or not class_balanced:
        limit = n if max_samples is None else min(int(max_samples), n)
        return list(range(limit))
    labels = pd.Series(df['label']).astype(int).reset_index(drop=True)
    classes = sorted(labels.dropna().unique().tolist())
    if not classes:
        limit = n if max_samples is None else min(int(max_samples), n)
        return list(range(limit))
    if examples_per_class is not None:
        quota_by_class = {cls: int(examples_per_class) for cls in classes}
    elif max_samples is None:
        quota_by_class = {cls: int((labels == cls).sum()) for cls in classes}
    else:
        max_samples = int(max_samples)
        base = max(1, max_samples // max(len(classes), 1))
        remainder = max(0, max_samples - base * len(classes))
        quota_by_class = {cls: base + (1 if i < remainder else 0) for i, cls in enumerate(classes)}
    selected = []
    for cls in classes:
        cls_indices = np.flatnonzero(labels.to_numpy() == int(cls)).tolist()
        quota = min(int(quota_by_class.get(cls, 0)), len(cls_indices))
        if quota <= 0:
            continue
        if len(cls_indices) <= quota:
            chosen = cls_indices
        else:
            positions = np.linspace(0, len(cls_indices) - 1, quota).round().astype(int).tolist()
            chosen = [cls_indices[pos] for pos in positions]
        selected.extend(chosen)
    if max_samples is not None and len(selected) > int(max_samples):
        selected = selected[:int(max_samples)]
    return selected


def phase13_make_explainability_loader(artifacts, split, max_samples=None, class_balanced=PHASE13_CLASS_BALANCED_EXAMPLES, task_name='explainability', batch_size=PHASE13_BATCH_SIZE):
    dataset = artifacts['datasets'][split]
    indices = phase13_select_explainability_indices(dataset, max_samples=max_samples, class_balanced=class_balanced)
    subset = torch.utils.data.Subset(dataset, indices)
    df = phase13_dataset_dataframe(dataset)
    if df is not None and 'label' in df and indices:
        counts = pd.Series(df.iloc[indices]['label']).astype(int).value_counts().sort_index().to_dict()
        readable = {artifacts.get('label_names', {}).get(int(k), str(k)): int(v) for k, v in counts.items()}
        phase13_log(f"fold {artifacts['fold']} {task_name}: selected {len(indices)}/{len(dataset)} {split} samples with label counts {readable}")
    else:
        phase13_log(f"fold {artifacts['fold']} {task_name}: selected {len(indices)}/{len(dataset)} {split} samples")
    return phase13_make_loader(subset, batch_size=batch_size), indices


def phase13_expand_gradcam_targets(artifacts, targets=None):
    targets = PHASE13_GRADCAM_TARGETS if targets is None else targets
    if isinstance(targets, (str, int)):
        targets = [targets]
    expanded = []
    class_ids = sorted(int(k) for k in artifacts.get('label_names', {}).keys())
    for target in targets:
        if target == 'all_classes':
            expanded.extend(class_ids)
        else:
            expanded.append(target)
    deduped = []
    seen = set()
    for target in expanded:
        key = f'{type(target).__name__}:{target}'
        if key not in seen:
            deduped.append(target)
            seen.add(key)
    return deduped


def phase13_target_slug(target, artifacts=None):
    if isinstance(target, int):
        name = None
        if artifacts is not None:
            name = artifacts.get('label_names', {}).get(int(target))
        return f'class_{int(target)}_{phase13_slug(name)}' if name else f'class_{int(target)}'
    return phase13_slug(target)


def phase13_load_fold_artifacts(fold, device=None, checkpoint_name=PHASE13_CHECKPOINT_NAME):
    """Build datasets and load the trained Phase 12 fold model for explainability."""
    fold = int(fold)
    t0 = time.time()
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    phase13_log(f'fold {fold}: loading Phase 12 artifacts on device={device}')
    train_ds, val_ds, test_ds, biomarker_specs, biomarker_stats, dfs = phase12_make_datasets_for_fold(fold)
    phase13_log(
        f'fold {fold}: datasets ready train={len(train_ds)} inner_val={len(val_ds)} test={len(test_ds)} '
        f'biomarkers={len(biomarker_specs or [])}'
    )
    phase13_log(f'fold {fold}: building trained Phase 9/12 model wrapper')
    model = build_phase9_model_for_fold(
        fold,
        biomarker_specs=biomarker_specs,
        device=device,
        freeze_octa_backbones=False,
        checkpoint_kind=PHASE12_CHECKPOINT_KIND if 'PHASE12_CHECKPOINT_KIND' in globals() else 'best',
    )
    ckpt = phase13_checkpoint_path(fold, checkpoint_name=checkpoint_name)
    phase13_log(f'fold {fold}: loading checkpoint {ckpt}')
    phase12_load_checkpoint(ckpt, model, device=device)
    model.eval()
    if hasattr(model, 'mask_generator'):
        model.mask_generator.eval()
    loaders = {
        'train': phase13_make_loader(train_ds),
        'val': phase13_make_loader(val_ds),
        'test': phase13_make_loader(test_ds),
    }
    temperature = phase13_get_temperature(fold)
    phase13_log(f'fold {fold}: artifacts loaded in {phase13_format_seconds(time.time() - t0)} | temperature={temperature:.4f}')
    return {
        'fold': fold,
        'device': device,
        'model': model,
        'checkpoint_path': ckpt,
        'temperature': temperature,
        'datasets': {'train': train_ds, 'val': val_ds, 'test': test_ds},
        'loaders': loaders,
        'biomarker_specs': biomarker_specs,
        'biomarker_stats': biomarker_stats,
        'dataframes': dfs,
        'label_names': phase13_label_name_map(phase2_folds if 'phase2_folds' in globals() else None),
    }

def phase13_apply_temperature(logits, temperature=1.0):
    if 'phase9_apply_temperature' in globals():
        return phase9_apply_temperature(logits, temperature)
    return logits / max(float(temperature), 1e-6)


def phase13_batch_value(batch, key, index, default=None):
    if key not in batch:
        return default
    value = batch[key]
    if torch.is_tensor(value):
        item = value[index]
        return item.item() if item.numel() == 1 else item.detach().cpu().numpy()
    if isinstance(value, (list, tuple)):
        return value[index]
    return value


def phase13_slice_batch(batch, index):
    sliced = {}
    for key, value in batch.items():
        if torch.is_tensor(value):
            sliced[key] = value[index:index + 1]
        elif isinstance(value, (list, tuple)):
            sliced[key] = [value[index]]
        else:
            sliced[key] = value
    return sliced


def phase13_repeat_first_sample_batch(batch, n):
    repeated = {}
    for key, value in batch.items():
        if torch.is_tensor(value):
            repeated[key] = value[:1].repeat((int(n),) + (1,) * (value.ndim - 1))
        elif isinstance(value, (list, tuple)):
            repeated[key] = [value[0]] * int(n)
        else:
            repeated[key] = value
    return repeated


def phase13_image_tensor_to_numpy(x):
    arr = x.detach().float().cpu().numpy()
    if arr.ndim == 3:
        arr = arr[0]
    arr = np.asarray(arr, dtype=np.float32)
    lo, hi = np.nanpercentile(arr, [1, 99]) if np.isfinite(arr).any() else (0.0, 1.0)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.nanmin(arr)), float(np.nanmax(arr) + 1e-6)
    return np.clip((arr - lo) / max(hi - lo, 1e-6), 0, 1)


def phase13_normalize_cam(cam):
    cam = torch.relu(cam.detach().float())
    flat = cam.flatten(1)
    cmin = flat.min(dim=1).values.view(-1, 1, 1)
    cmax = flat.max(dim=1).values.view(-1, 1, 1)
    return (cam - cmin) / (cmax - cmin + 1e-6)


def phase13_compute_gradcam_batch(model, batch, device, temperature=1.0, target='pred'):
    """Compute per-layer Grad-CAM for one batch."""
    model.eval()
    if hasattr(model, 'mask_generator'):
        model.mask_generator.eval()
    batch_dev = phase12_to_device(batch, device)
    outputs = model(batch_dev, return_attention=True, use_prototypes=True)
    logits = outputs['logits']
    probs = torch.softmax(phase13_apply_temperature(logits, temperature), dim=1)
    pred_idx = probs.argmax(dim=1)
    if target == 'pred':
        target_idx = pred_idx
    elif target == 'label':
        target_idx = phase12_labels_from_batch(batch_dev)
    elif isinstance(target, int):
        target_idx = torch.full((logits.shape[0],), int(target), dtype=torch.long, device=logits.device)
    else:
        raise ValueError("target must be 'pred', 'label', or an integer class index")
    spatial_maps = outputs.get('spatial_maps', {})
    map_tensors = [spatial_maps[layer] for layer in PHASE13_LAYER_NAMES if layer in spatial_maps]
    if not map_tensors:
        raise RuntimeError('Model output did not include Phase 4 spatial maps for Grad-CAM.')
    target_scores = logits.gather(1, target_idx.view(-1, 1)).sum()
    grads = torch.autograd.grad(target_scores, map_tensors, retain_graph=False, allow_unused=True)
    cams = {}
    for layer, fmap, grad in zip([l for l in PHASE13_LAYER_NAMES if l in spatial_maps], map_tensors, grads):
        if grad is None:
            cams[layer] = torch.zeros((fmap.shape[0], fmap.shape[2], fmap.shape[3]), device=fmap.device)
            continue
        weights = grad.mean(dim=(2, 3), keepdim=True)
        cam = (weights * fmap).sum(dim=1)
        cam = phase13_normalize_cam(cam)
        cam = F.interpolate(cam[:, None], size=(PHASE4_IMAGE_SIZE, PHASE4_IMAGE_SIZE), mode='bilinear', align_corners=False)[:, 0]
        cams[layer] = cam.detach().cpu()
    return {
        'cams': cams,
        'outputs': {k: v.detach().cpu() if torch.is_tensor(v) else v for k, v in outputs.items() if k in ['logits', 'attention_weights']},
        'probs': probs.detach().cpu(),
        'target_idx': target_idx.detach().cpu(),
        'pred_idx': pred_idx.detach().cpu(),
    }


def phase13_save_gradcam_overlay(batch, cams, sample_index, out_path, label_names=None, probs=None, target_idx=None, pred_idx=None):
    if plt is None:
        return None
    label = int(phase13_batch_value(batch, 'label', sample_index, -1))
    target_class = int(target_idx[sample_index]) if target_idx is not None else -1
    pred = int(pred_idx[sample_index]) if pred_idx is not None else target_class
    target_prob = float(probs[sample_index, target_class]) if probs is not None and target_class >= 0 else float('nan')
    pred_prob = float(probs[sample_index, pred]) if probs is not None and pred >= 0 else float('nan')
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, layer in zip(axes, PHASE13_LAYER_NAMES):
        img = phase13_image_tensor_to_numpy(batch[PHASE13_IMAGE_KEYS[layer]][sample_index])
        cam = cams[layer][sample_index].numpy()
        ax.imshow(img, cmap='gray')
        ax.imshow(cam, cmap='hot', alpha=0.45, vmin=0, vmax=1)
        ax.set_title(layer.upper())
        ax.axis('off')
    label_text = label_names.get(label, str(label)) if label_names else str(label)
    target_text = label_names.get(target_class, str(target_class)) if label_names else str(target_class)
    pred_text = label_names.get(pred, str(pred)) if label_names else str(pred)
    fig.suptitle(
        f"Patient {phase13_batch_value(batch, 'patient_id', sample_index, '')} {phase13_batch_value(batch, 'eye', sample_index, '')} | "
        f"true={label_text} target={target_text} p_target={target_prob:.3f} pred={pred_text} p_pred={pred_prob:.3f}"
    )
    fig.tight_layout()
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return out_path


def phase13_run_gradcam_for_fold(fold, artifacts=None, max_samples=PHASE13_MAX_GRADCAM_SAMPLES_PER_FOLD, split='test', target=PHASE13_GRADCAM_TARGET):
    artifacts = artifacts or phase13_load_fold_artifacts(fold)
    model = artifacts['model']
    device = artifacts['device']
    loader, selected_indices = phase13_make_explainability_loader(
        artifacts,
        split,
        max_samples=max_samples,
        class_balanced=PHASE13_CLASS_BALANCED_EXAMPLES,
        task_name=f'Grad-CAM target={target}',
        batch_size=PHASE13_GRADCAM_BATCH_SIZE,
    )
    out_dir = ensure_dir(PHASE13_GRADCAM_DIR / f'fold_{int(fold)}')
    total_target = len(loader.dataset) if hasattr(loader, 'dataset') else None
    phase13_log(f'fold {int(fold)}: Grad-CAM start split={split} target={target} samples={total_target} batches={len(loader)}')
    records = []
    seen = 0
    start_time = time.time()
    for batch_idx, batch in enumerate(loader, start=1):
        if max_samples is not None and seen >= int(max_samples):
            break
        remaining = None if max_samples is None else max(int(max_samples) - seen, 0)
        if remaining == 0:
            break
        phase13_log(f'fold {int(fold)} Grad-CAM batch {batch_idx}/{len(loader)}: computing forward/backward')
        result = phase13_compute_gradcam_batch(model, batch, device, temperature=artifacts['temperature'], target=target)
        bsz = next(iter(result['cams'].values())).shape[0]
        bsz = min(bsz, remaining) if remaining is not None else bsz
        for j in range(bsz):
            patient_id = str(phase13_batch_value(batch, 'patient_id', j, 'unknown'))
            eye = str(phase13_batch_value(batch, 'eye', j, 'eye'))
            sample_id = f'{phase13_slug(patient_id)}_{phase13_slug(eye)}'
            target_slug = phase13_target_slug(target, artifacts)
            overlay_path = out_dir / f'{sample_id}_target_{target_slug}_gradcam_overlay.png'
            phase13_save_gradcam_overlay(
                batch,
                result['cams'],
                j,
                overlay_path,
                label_names=artifacts['label_names'],
                probs=result['probs'],
                target_idx=result['target_idx'],
                pred_idx=result['pred_idx'],
            )
            for layer in PHASE13_LAYER_NAMES:
                cam_np = result['cams'][layer][j].numpy().astype(np.float32)
                cam_path = out_dir / f'{sample_id}_target_{target_slug}_{layer}_gradcam.npy'
                np.save(cam_path, cam_np)
                label_val = int(phase13_batch_value(batch, 'label', j, -1))
                pred_val = int(result['pred_idx'][j])
                target_val = int(result['target_idx'][j])
                records.append({
                    'fold': int(fold),
                    'split': split,
                    'patient_id': patient_id,
                    'eye': eye,
                    'sample_id': sample_id,
                    'layer': layer,
                    'target_mode': str(target),
                    'target_class': target_val,
                    'target_name': artifacts['label_names'].get(target_val, str(target_val)),
                    'label': label_val,
                    'label_name': artifacts['label_names'].get(label_val, str(label_val)),
                    'pred': pred_val,
                    'pred_name': artifacts['label_names'].get(pred_val, str(pred_val)),
                    'pred_probability': float(result['probs'][j, pred_val]),
                    'target_probability': float(result['probs'][j, target_val]),
                    'cam_mean': float(cam_np.mean()),
                    'cam_max': float(cam_np.max()),
                    'cam_p95': float(np.percentile(cam_np, 95)),
                    'cam_path': str(cam_path),
                    'overlay_path': str(overlay_path),
                })
        seen += bsz
        if batch_idx % int(PHASE13_LOG_EVERY_N_BATCHES) == 0 or seen >= total_target:
            phase13_log_progress('Grad-CAM', fold, seen, total_target, start_time, extra=f'last_batch={batch_idx}/{len(loader)}')
        del result
        phase13_clear_cuda_cache()
    df = pd.DataFrame(records)
    target_slug = phase13_target_slug(target, artifacts)
    target_summary_path = out_dir / f'phase13_gradcam_summary_target_{target_slug}.csv'
    df.to_csv(target_summary_path, index=False)
    summary_path = out_dir / 'phase13_gradcam_summary.csv'
    df.to_csv(summary_path, index=False)
    phase13_log(f'fold {int(fold)}: Grad-CAM done rows={len(df)} target_summary={target_summary_path}')
    return df

def phase13_run_attention_for_fold(fold, artifacts=None, max_samples=PHASE13_MAX_ATTENTION_SAMPLES_PER_FOLD, split='test'):
    artifacts = artifacts or phase13_load_fold_artifacts(fold)
    model = artifacts['model']
    device = artifacts['device']
    loader = artifacts['loaders'][split]
    out_dir = ensure_dir(PHASE13_ATTENTION_DIR / f'fold_{int(fold)}')
    total_available = len(loader.dataset) if hasattr(loader, 'dataset') else None
    total_target = total_available if max_samples is None else min(int(max_samples), int(total_available or max_samples))
    phase13_log(f'fold {int(fold)}: attention dominance start split={split} samples={total_target} batches={len(loader)}')
    records = []
    seen = 0
    start_time = time.time()
    model.eval()
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader, start=1):
            if max_samples is not None and seen >= int(max_samples):
                break
            phase13_log(f'fold {int(fold)} attention batch {batch_idx}/{len(loader)}: forward pass')
            batch_dev = phase12_to_device(batch, device)
            outputs = model(batch_dev, return_attention=True, use_prototypes=True)
            probs = torch.softmax(phase13_apply_temperature(outputs['logits'], artifacts['temperature']), dim=1).detach().cpu()
            preds = probs.argmax(dim=1).numpy()
            attn = outputs.get('attention_weights')
            if attn is None:
                raise RuntimeError('Model output did not include cross-layer attention weights.')
            dominance = phase6_layer_dominance(attn).reset_index(drop=True)
            bsz = len(dominance)
            if max_samples is not None:
                bsz = min(bsz, int(max_samples) - seen)
            for j in range(bsz):
                label = int(phase13_batch_value(batch, 'label', j, -1))
                pred = int(preds[j])
                rec = {
                    'fold': int(fold),
                    'split': split,
                    'patient_id': str(phase13_batch_value(batch, 'patient_id', j, '')),
                    'eye': str(phase13_batch_value(batch, 'eye', j, '')),
                    'label': label,
                    'label_name': artifacts['label_names'].get(label, str(label)),
                    'pred': pred,
                    'pred_name': artifacts['label_names'].get(pred, str(pred)),
                    'pred_probability': float(probs[j, pred]),
                }
                rec.update({k: float(v) for k, v in dominance.iloc[j].to_dict().items()})
                records.append(rec)
            seen += bsz
            if batch_idx % int(PHASE13_LOG_EVERY_N_BATCHES) == 0 or seen >= total_target:
                phase13_log_progress('attention', fold, seen, total_target, start_time, extra=f'last_batch={batch_idx}/{len(loader)}')
    df = pd.DataFrame(records)
    summary_path = out_dir / 'phase13_attention_dominance.csv'
    df.to_csv(summary_path, index=False)
    if len(df):
        dom_cols = [f'{name}_dominance' for name in PHASE13_LAYER_NAMES]
        aggregate = df.groupby('label_name')[dom_cols].agg(['mean', 'std', 'count'])
        aggregate.columns = ['_'.join([str(x) for x in col if str(x)]) for col in aggregate.columns]
        aggregate_path = out_dir / 'phase13_attention_dominance_by_cohort.csv'
        aggregate.reset_index().to_csv(aggregate_path, index=False)
        if plt is not None:
            mean_df = df.groupby('label_name')[dom_cols].mean().reset_index()
            plot_df = mean_df.melt(id_vars='label_name', value_vars=dom_cols, var_name='layer', value_name='dominance')
            plot_df['layer'] = plot_df['layer'].str.replace('_dominance', '', regex=False)
            fig, ax = plt.subplots(figsize=(8, 4))
            for i, layer in enumerate(PHASE13_LAYER_NAMES):
                sub = plot_df[plot_df['layer'] == layer]
                ax.bar(np.arange(len(sub)) + (i - 1) * 0.22, sub['dominance'], width=0.22, label=layer.upper())
            ax.set_xticks(np.arange(len(mean_df)))
            ax.set_xticklabels(mean_df['label_name'], rotation=30, ha='right')
            ax.set_ylabel('Mean attention received')
            ax.set_title('Phase 13 layer dominance by cohort')
            ax.legend()
            fig.tight_layout()
            fig.savefig(out_dir / 'phase13_attention_dominance_by_cohort.png', dpi=180)
            plt.close(fig)
        phase13_log(f'fold {int(fold)}: attention aggregate written {aggregate_path}')
    phase13_log(f'fold {int(fold)}: attention dominance done rows={len(df)} summary={summary_path}')
    return df

def phase13_tabular_feature_names(artifacts):
    cp = clinical_preprocessors[int(artifacts['fold'])]
    clinical_names = list(getattr(cp, 'clinical_cols', []))
    biomarker_names = phase5_biomarker_names(artifacts.get('biomarker_specs', [])) if 'phase5_biomarker_names' in globals() else []
    return clinical_names + biomarker_names, clinical_names, biomarker_names


def phase13_tabular_matrix_from_batch(batch):
    clinical = batch['clinical_features'].detach().float().cpu().numpy()
    biomarkers = batch.get('biomarkers')
    if torch.is_tensor(biomarkers) and biomarkers.numel():
        bio = biomarkers.detach().float().cpu().numpy()
        return np.concatenate([clinical, bio], axis=1)
    return clinical


def phase13_collect_tabular_background(loader, max_rows=PHASE13_SHAP_BACKGROUND_SIZE):
    arrays = []
    for batch in loader:
        arrays.append(phase13_tabular_matrix_from_batch(batch))
        if sum(a.shape[0] for a in arrays) >= int(max_rows):
            break
    if not arrays:
        raise RuntimeError('Could not collect tabular background rows for SHAP.')
    x = np.concatenate(arrays, axis=0)[:int(max_rows)]
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def phase13_tabular_predict_fn(model, base_batch, x_matrix, clinical_dim, biomarker_dim, device, temperature=1.0):
    x_matrix = np.asarray(x_matrix, dtype=np.float32)
    if x_matrix.ndim == 1:
        x_matrix = x_matrix[None, :]
    n = x_matrix.shape[0]
    model.eval()
    if hasattr(model, 'mask_generator'):
        model.mask_generator.eval()
    outputs = []
    chunk_size = max(1, int(PHASE13_SHAP_PREDICT_BATCH_SIZE))
    for start in range(0, n, chunk_size):
        stop = min(start + chunk_size, n)
        x_chunk = x_matrix[start:stop]
        work = phase13_repeat_first_sample_batch(base_batch, x_chunk.shape[0])
        work['clinical_features'] = torch.from_numpy(x_chunk[:, :clinical_dim].astype(np.float32))
        if biomarker_dim > 0:
            work['biomarkers'] = torch.from_numpy(x_chunk[:, clinical_dim:clinical_dim + biomarker_dim].astype(np.float32))
        work = phase12_to_device(work, device)
        with torch.no_grad():
            out = model(work, return_attention=False, use_prototypes=True)
            probs = torch.softmax(phase13_apply_temperature(out['logits'], temperature), dim=1)
        outputs.append(probs.detach().cpu().numpy())
        del work, out, probs
        phase13_clear_cuda_cache()
    return np.concatenate(outputs, axis=0)


def phase13_extract_shap_for_class(shap_values, target_class):
    if isinstance(shap_values, list):
        return np.asarray(shap_values[int(target_class)])[0]
    arr = np.asarray(shap_values)
    if arr.ndim == 3 and arr.shape[0] == 1:
        return arr[0, :, int(target_class)]
    if arr.ndim == 3 and arr.shape[1] == 1:
        return arr[int(target_class), 0, :]
    if arr.ndim == 2:
        return arr[0]
    raise ValueError(f'Unsupported SHAP value shape: {arr.shape}')


def phase13_occlusion_tabular_attribution(predict_fn, x, background_mean, target_class):
    base_prob = float(predict_fn(x)[0, int(target_class)])
    vals = []
    for j in range(x.shape[1]):
        x_occ = x.copy()
        x_occ[0, j] = background_mean[j]
        occ_prob = float(predict_fn(x_occ)[0, int(target_class)])
        vals.append(base_prob - occ_prob)
    return np.asarray(vals, dtype=np.float32), base_prob


def phase13_save_tabular_waterfall(rows_df, out_path, title, top_n=20):
    if plt is None or rows_df.empty:
        return None
    plot_df = rows_df.copy()
    plot_df = plot_df.reindex(plot_df['attribution'].abs().sort_values(ascending=False).index).head(int(top_n))
    plot_df = plot_df.iloc[::-1]
    colors = np.where(plot_df['attribution'] >= 0, '#B23A48', '#2A6FBB')
    fig, ax = plt.subplots(figsize=(8, max(4, 0.28 * len(plot_df))))
    ax.barh(plot_df['feature'], plot_df['attribution'], color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Attribution to target probability')
    ax.set_title(title)
    fig.tight_layout()
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return out_path


def phase13_run_tabular_attribution_for_fold(fold, artifacts=None, max_samples=PHASE13_MAX_TABULAR_SAMPLES_PER_FOLD, split='test'):
    artifacts = artifacts or phase13_load_fold_artifacts(fold)
    model = artifacts['model']
    device = artifacts['device']
    out_dir = ensure_dir(PHASE13_TABULAR_DIR / f'fold_{int(fold)}')
    feature_names, clinical_names, biomarker_names = phase13_tabular_feature_names(artifacts)
    clinical_dim = len(clinical_names)
    biomarker_dim = len(biomarker_names)
    phase13_log(f'fold {int(fold)}: tabular attribution setup features={len(feature_names)} clinical={clinical_dim} biomarkers={biomarker_dim}')
    background = phase13_collect_tabular_background(artifacts['loaders']['val'], max_rows=PHASE13_SHAP_BACKGROUND_SIZE)
    background_mean = background.mean(axis=0)
    phase13_log(f'fold {int(fold)}: tabular background collected rows={background.shape[0]}')
    try:
        import shap
        shap_available = True
    except Exception:
        shap = None
        shap_available = False
    method = 'kernel_shap' if shap_available else 'occlusion_fallback_not_shap'
    if PHASE13_REQUIRE_REAL_SHAP and not shap_available:
        raise ImportError('Phase 13 is configured with PHASE13_REQUIRE_REAL_SHAP=True, but shap is not installed. Install shap or set PHASE13_REQUIRE_REAL_SHAP=False for the explicit occlusion fallback.')
    run_loader, selected_indices = phase13_make_explainability_loader(
        artifacts,
        split,
        max_samples=max_samples,
        class_balanced=PHASE13_CLASS_BALANCED_EXAMPLES,
        task_name='tabular attribution',
        batch_size=PHASE13_TABULAR_BATCH_SIZE,
    )
    total_target = len(run_loader.dataset) if hasattr(run_loader, 'dataset') else max_samples
    phase13_log(f'fold {int(fold)}: tabular attribution method={method} max_samples={total_target} nsamples={PHASE13_SHAP_NSAMPLES}')
    records = []
    seen = 0
    start_time = time.time()
    for batch_idx, batch in enumerate(run_loader, start=1):
        bsz = batch['clinical_features'].shape[0]
        for j in range(bsz):
            if max_samples is not None and seen >= int(max_samples):
                break
            base_batch = phase13_slice_batch(batch, j)
            x = phase13_tabular_matrix_from_batch(base_batch)
            predict_fn = lambda z, bb=base_batch: phase13_tabular_predict_fn(
                model, bb, z, clinical_dim, biomarker_dim, device, temperature=artifacts['temperature']
            )
            probs = predict_fn(x)
            target_class = int(np.argmax(probs[0]))
            patient_id = str(phase13_batch_value(batch, 'patient_id', j, 'unknown'))
            eye = str(phase13_batch_value(batch, 'eye', j, 'eye'))
            phase13_log(f'fold {int(fold)} tabular sample {seen + 1}/{total_target}: patient={patient_id} eye={eye} target={target_class} method={method}')
            if shap_available:
                explainer = shap.KernelExplainer(predict_fn, background)
                with warnings.catch_warnings():
                    warnings.filterwarnings('ignore')
                    try:
                        shap_values = explainer.shap_values(x, nsamples=int(PHASE13_SHAP_NSAMPLES), silent=True)
                    except TypeError:
                        shap_values = explainer.shap_values(x, nsamples=int(PHASE13_SHAP_NSAMPLES))
                attributions = phase13_extract_shap_for_class(shap_values, target_class)
                base_prob = float(probs[0, target_class])
            else:
                attributions, base_prob = phase13_occlusion_tabular_attribution(predict_fn, x, background_mean, target_class)
            sample_id = f'{phase13_slug(patient_id)}_{phase13_slug(eye)}'
            sample_rows = []
            for k, feature in enumerate(feature_names):
                group = 'clinical' if k < clinical_dim else 'generated_biomarker'
                rec = {
                    'fold': int(fold),
                    'split': split,
                    'method': method,
                    'patient_id': patient_id,
                    'eye': eye,
                    'sample_id': sample_id,
                    'label': int(phase13_batch_value(batch, 'label', j, -1)),
                    'label_name': artifacts['label_names'].get(int(phase13_batch_value(batch, 'label', j, -1)), str(phase13_batch_value(batch, 'label', j, -1))),
                    'target_class': target_class,
                    'target_name': artifacts['label_names'].get(target_class, str(target_class)),
                    'target_probability': base_prob,
                    'feature': feature,
                    'feature_group': group,
                    'attribution': float(attributions[k]),
                    'abs_attribution': float(abs(attributions[k])),
                }
                records.append(rec)
                sample_rows.append(rec)
            sample_df = pd.DataFrame(sample_rows)
            sample_path = out_dir / f'{sample_id}_tabular_attributions.csv'
            sample_df.to_csv(sample_path, index=False)
            phase13_save_tabular_waterfall(
                sample_df,
                out_dir / f'{sample_id}_tabular_waterfall.png',
                f'{sample_id} target={artifacts["label_names"].get(target_class, target_class)} ({method})',
            )
            seen += 1
            if seen % int(PHASE13_LOG_EVERY_N_SAMPLES) == 0 or (total_target is not None and seen >= int(total_target)):
                phase13_log_progress('tabular attribution', fold, seen, total_target, start_time, extra=f'batch={batch_idx}')
            phase13_clear_cuda_cache()
        if max_samples is not None and seen >= int(max_samples):
            break
    df = pd.DataFrame(records)
    summary_path = out_dir / 'phase13_tabular_attributions.csv'
    df.to_csv(summary_path, index=False)
    if len(df):
        summary = df.groupby(['feature_group', 'feature'])['abs_attribution'].mean().reset_index().sort_values('abs_attribution', ascending=False)
        top_path = out_dir / 'phase13_tabular_mean_abs_attribution.csv'
        summary.to_csv(top_path, index=False)
        if plt is not None:
            top = summary.head(30).iloc[::-1]
            fig, ax = plt.subplots(figsize=(9, max(4, 0.25 * len(top))))
            ax.barh(top['feature'], top['abs_attribution'], color=np.where(top['feature_group'] == 'clinical', '#4C72B0', '#55A868'))
            ax.set_xlabel('Mean absolute attribution')
            ax.set_title(f'Phase 13 tabular attribution summary ({method})')
            fig.tight_layout()
            fig.savefig(out_dir / 'phase13_tabular_mean_abs_attribution_top30.png', dpi=180, bbox_inches='tight')
            plt.close(fig)
        phase13_log(f'fold {int(fold)}: tabular summary written {top_path}')
    phase13_log(f'fold {int(fold)}: tabular attribution done rows={len(df)} summary={summary_path}')
    return df

def phase13_integrated_analysis_for_fold(fold):
    grad_path = PHASE13_GRADCAM_DIR / f'fold_{int(fold)}' / 'phase13_gradcam_summary.csv'
    attn_path = PHASE13_ATTENTION_DIR / f'fold_{int(fold)}' / 'phase13_attention_dominance.csv'
    out_dir = ensure_dir(PHASE13_INTEGRATED_DIR / f'fold_{int(fold)}')
    if not grad_path.exists() or not attn_path.exists():
        return pd.DataFrame()
    grad = pd.read_csv(grad_path)
    attn = pd.read_csv(attn_path)
    if grad.empty or attn.empty:
        return pd.DataFrame()
    target_cols = [col for col in ['target_mode', 'target_class', 'target_name'] if col in grad.columns]
    pivot_index = ['fold', 'patient_id', 'eye'] + target_cols
    pivot_mean = grad.pivot_table(index=pivot_index, columns='layer', values='cam_mean', aggfunc='mean').reset_index()
    pivot_mean = pivot_mean.rename(columns={layer: f'{layer}_cam_mean' for layer in PHASE13_LAYER_NAMES})
    merged = attn.merge(pivot_mean, on=['fold', 'patient_id', 'eye'], how='inner')
    merged.to_csv(out_dir / 'phase13_gradcam_attention_merged.csv', index=False)
    rows = []
    group_cols = target_cols if target_cols else [None]
    grouped = [((), merged)] if group_cols == [None] else merged.groupby(group_cols, dropna=False)
    for keys, group in grouped:
        if group_cols != [None]:
            if not isinstance(keys, tuple):
                keys = (keys,)
            target_info = dict(zip(group_cols, keys))
        else:
            target_info = {}
        for layer in PHASE13_LAYER_NAMES:
            cam_col = f'{layer}_cam_mean'
            dom_col = f'{layer}_dominance'
            clean = group[[cam_col, dom_col]].dropna() if cam_col in group and dom_col in group else pd.DataFrame()
            if clean.shape[0] >= 3:
                corr = clean.corr(method='pearson').iloc[0, 1]
                row = {'fold': int(fold), 'layer': layer, 'n': int(clean.shape[0]), 'pearson_cam_mean_vs_dominance': float(corr)}
                row.update(target_info)
                rows.append(row)
    corr_df = pd.DataFrame(rows)
    corr_df.to_csv(out_dir / 'phase13_gradcam_attention_correlations.csv', index=False)
    return corr_df


def phase13_run_fold(fold):
    fold = int(fold)
    fold_start = time.time()
    phase13_log(f'fold {fold}: START Phase 13')
    artifacts = phase13_load_fold_artifacts(fold)
    outputs = {'fold': fold, 'checkpoint_path': str(artifacts['checkpoint_path']), 'temperature': artifacts['temperature']}
    if PHASE13_RUN_ATTENTION:
        phase13_log(f'fold {fold}: START attention dominance')
        task_start = time.time()
        attn_df = phase13_run_attention_for_fold(fold, artifacts=artifacts)
        outputs['attention_rows'] = int(len(attn_df))
        phase13_log(f'fold {fold}: END attention dominance rows={len(attn_df)} time={phase13_format_seconds(time.time() - task_start)}')
    else:
        phase13_log(f'fold {fold}: SKIP attention dominance')
    if PHASE13_RUN_GRADCAM:
        phase13_log(f'fold {fold}: START Grad-CAM targets={PHASE13_GRADCAM_TARGETS}')
        task_start = time.time()
        grad_dfs = []
        expanded_targets = phase13_expand_gradcam_targets(artifacts, PHASE13_GRADCAM_TARGETS)
        for target in expanded_targets:
            phase13_log(f'fold {fold}: START Grad-CAM target={target}')
            target_start = time.time()
            grad_dfs.append(phase13_run_gradcam_for_fold(fold, artifacts=artifacts, target=target))
            phase13_clear_cuda_cache()
            phase13_log(f'fold {fold}: END Grad-CAM target={target} rows={len(grad_dfs[-1])} time={phase13_format_seconds(time.time() - target_start)}')
        grad_df = pd.concat(grad_dfs, ignore_index=True) if grad_dfs else pd.DataFrame()
        combined_path = PHASE13_GRADCAM_DIR / f'fold_{fold}' / 'phase13_gradcam_summary.csv'
        grad_df.to_csv(combined_path, index=False)
        outputs['gradcam_rows'] = int(len(grad_df))
        outputs['gradcam_targets'] = [str(t) for t in expanded_targets]
        phase13_log(f'fold {fold}: END Grad-CAM rows={len(grad_df)} combined={combined_path} time={phase13_format_seconds(time.time() - task_start)}')
    else:
        phase13_log(f'fold {fold}: SKIP Grad-CAM')
    if PHASE13_RUN_TABULAR_SHAP:
        phase13_log(f'fold {fold}: START tabular attribution')
        task_start = time.time()
        tab_df = phase13_run_tabular_attribution_for_fold(fold, artifacts=artifacts)
        outputs['tabular_attribution_rows'] = int(len(tab_df))
        phase13_log(f'fold {fold}: END tabular attribution rows={len(tab_df)} time={phase13_format_seconds(time.time() - task_start)}')
    else:
        phase13_log(f'fold {fold}: SKIP tabular attribution because PHASE13_RUN_TABULAR_SHAP=False')
    phase13_log(f'fold {fold}: START integrated analysis')
    corr_df = phase13_integrated_analysis_for_fold(fold)
    outputs['integrated_correlation_rows'] = int(len(corr_df))
    outputs['elapsed_seconds'] = float(time.time() - fold_start)
    summary_path = PHASE13_OUTPUT_DIR / f'fold_{fold}_phase13_summary.json'
    save_json(outputs, summary_path)
    phase13_log(f'fold {fold}: END Phase 13 time={phase13_format_seconds(outputs["elapsed_seconds"])} summary={summary_path}')
    del artifacts
    phase13_clear_cuda_cache()
    return outputs

def phase13_run_cross_validation(folds=PHASE13_FOLDS):
    folds = [int(f) for f in folds]
    cv_start = time.time()
    phase13_log(f'cross-validation START folds={folds} attention={PHASE13_RUN_ATTENTION} gradcam={PHASE13_RUN_GRADCAM} tabular={PHASE13_RUN_TABULAR_SHAP}')
    summaries = []
    for i, fold in enumerate(folds, start=1):
        phase13_log(f'cross-validation fold {i}/{len(folds)} START fold={fold}')
        summaries.append(phase13_run_fold(fold))
        phase13_log(f'cross-validation fold {i}/{len(folds)} END fold={fold}')
    summary_df = pd.DataFrame(summaries)
    summary_path = PHASE13_OUTPUT_DIR / 'phase13_cross_validation_summary.csv'
    summary_df.to_csv(summary_path, index=False)
    phase13_log(f'cross-validation summaries written {summary_path}')
    all_attention, all_gradcam, all_tabular, all_corr = [], [], [], []
    for fold in folds:
        fold = int(fold)
        for path, target in [
            (PHASE13_ATTENTION_DIR / f'fold_{fold}' / 'phase13_attention_dominance.csv', all_attention),
            (PHASE13_GRADCAM_DIR / f'fold_{fold}' / 'phase13_gradcam_summary.csv', all_gradcam),
            (PHASE13_TABULAR_DIR / f'fold_{fold}' / 'phase13_tabular_attributions.csv', all_tabular),
            (PHASE13_INTEGRATED_DIR / f'fold_{fold}' / 'phase13_gradcam_attention_correlations.csv', all_corr),
        ]:
            if path.exists():
                target.append(pd.read_csv(path))
    if all_attention:
        out = PHASE13_OUTPUT_DIR / 'phase13_all_attention_dominance.csv'
        pd.concat(all_attention, ignore_index=True).to_csv(out, index=False)
        phase13_log(f'combined attention written {out}')
    if all_gradcam:
        out = PHASE13_OUTPUT_DIR / 'phase13_all_gradcam_summary.csv'
        pd.concat(all_gradcam, ignore_index=True).to_csv(out, index=False)
        phase13_log(f'combined Grad-CAM written {out}')
    if all_tabular:
        out = PHASE13_OUTPUT_DIR / 'phase13_all_tabular_attributions.csv'
        pd.concat(all_tabular, ignore_index=True).to_csv(out, index=False)
        phase13_log(f'combined tabular attribution written {out}')
    if all_corr:
        out = PHASE13_OUTPUT_DIR / 'phase13_all_gradcam_attention_correlations.csv'
        pd.concat(all_corr, ignore_index=True).to_csv(out, index=False)
        phase13_log(f'combined integrated correlations written {out}')
    phase13_log(f'cross-validation END time={phase13_format_seconds(time.time() - cv_start)}')
    return summary_df

phase13_summary = {
    'run_explainability_default': PHASE13_RUN_EXPLAINABILITY,
    'log_every_n_batches': PHASE13_LOG_EVERY_N_BATCHES,
    'log_every_n_samples': PHASE13_LOG_EVERY_N_SAMPLES,
    'gradcam_batch_size': PHASE13_GRADCAM_BATCH_SIZE,
    'tabular_batch_size': PHASE13_TABULAR_BATCH_SIZE,
    'shap_predict_batch_size': PHASE13_SHAP_PREDICT_BATCH_SIZE,
    'folds': list(map(int, PHASE13_FOLDS)),
    'checkpoint_name': PHASE13_CHECKPOINT_NAME,
    'class_balanced_examples': PHASE13_CLASS_BALANCED_EXAMPLES,
    'max_gradcam_samples_per_fold': PHASE13_MAX_GRADCAM_SAMPLES_PER_FOLD,
    'max_tabular_samples_per_fold': PHASE13_MAX_TABULAR_SAMPLES_PER_FOLD,
    'gradcam_targets': PHASE13_GRADCAM_TARGETS,
    'require_real_shap': PHASE13_REQUIRE_REAL_SHAP,
    'outputs': {
        'gradcam': str(PHASE13_GRADCAM_DIR),
        'attention': str(PHASE13_ATTENTION_DIR),
        'tabular_attribution': str(PHASE13_TABULAR_DIR),
        'integrated': str(PHASE13_INTEGRATED_DIR),
    },
    'tabular_note': 'PHASE13_RUN_TABULAR_SHAP uses KernelSHAP. With PHASE13_REQUIRE_REAL_SHAP=True, the cell raises if shap is unavailable instead of writing fallback values.',
    'gradcam_note': 'Grad-CAM uses final Phase 4 spatial maps for Sup, Deep, and CC encoders. Use PHASE13_GRADCAM_TARGET or the target argument for predicted, true-label, or per-class heatmaps.',
    'attention_note': 'Attention dominance is an explanatory proxy, not causal proof.',
}
save_json(phase13_summary, PHASE13_OUTPUT_DIR / 'phase13_summary.json')

if PHASE13_RUN_EXPLAINABILITY:
    phase13_cv_summary = phase13_run_cross_validation(PHASE13_FOLDS)
else:
    print('Phase 13 explainability code ready. Run phase13_run_cross_validation(PHASE13_FOLDS) after Phase 12 checkpoints are available.')

In [ ]:
# ============================================================
# Phase 14 - Evaluation Metrics
# ============================================================
# Scores only out-of-fold outer-test predictions from Phase 12. It reports both
# eye-level performance and patient-level performance (mean OD/OS probabilities).
# Confidence intervals are stratified patient-cluster bootstraps, so bilateral
# eyes are never treated as independent inference units.
#
# The Phase 12 OOF CSV is the source of truth for deterministic calibrated
# classification metrics. MC-dropout is generated separately from frozen Stage 3
# checkpoints and is used only for uncertainty analyses; it never replaces OOF
# probabilities silently.

from pathlib import Path
import json
import math
import time
import warnings
import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print('Phase 14 plotting disabled because matplotlib could not be imported:', exc)

try:
    from sklearn.metrics import (
        accuracy_score,
        average_precision_score,
        balanced_accuracy_score,
        confusion_matrix,
        f1_score,
        precision_recall_fscore_support,
        roc_auc_score,
    )
except Exception as exc:
    raise ImportError('Phase 14 requires scikit-learn.') from exc

PHASE14_OUTPUT_DIR = Path(segcomp_model_dir(SEGCOMP_ACTIVE_MODEL_ID) / 'phase14_evaluation')
PHASE14_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PHASE14_OOF_PATH = Path(segcomp_model_dir(SEGCOMP_ACTIVE_MODEL_ID) / 'phase12_training' / 'phase12_all_outer_test_predictions.csv')
PHASE14_MC_DROPOUT_PATH = PHASE14_OUTPUT_DIR / 'phase14_mc_dropout_outer_test_predictions.csv'
PHASE14_FOLDS = globals().get('PHASE12_FOLDS', [0, 1, 2, 3, 4])
PHASE14_BOOTSTRAP_SAMPLES = 1000
PHASE14_BOOTSTRAP_SEED = 20260710
PHASE14_ECE_BINS = 10
PHASE14_MC_DROPOUT_PASSES = 30
PHASE14_MC_DROPOUT_BATCH_SIZE = 1  # 8 GB GPU-safe; inference is checkpoint-based.
PHASE14_NUM_WORKERS = 0
PHASE14_RUN_EVALUATION = False
PHASE14_RUN_MC_DROPOUT = False
PHASE14_RUN_SMOKE_TEST = False
PHASE14_LOG_EVERY_N_BOOTSTRAPS = 10
PHASE14_LOG_EVERY_N_MC_BATCHES = 1
PHASE14_LOG_MC_PASS_MILESTONES = True  # Set True only when per-pass logging is needed.
PHASE14_PRIMARY_METRICS = [
    'accuracy', 'macro_roc_auc_ovr', 'macro_f1', 'balanced_accuracy',
    'weighted_f1', 'macro_pr_auc_ovr', 'top2_accuracy', 'ece_10', 'brier_multiclass',
]


def phase14_log(message):
    print(f'[Phase 14] {message}', flush=True)


def phase14_format_seconds(seconds):
    seconds = max(float(seconds), 0.0)
    if seconds < 60:
        return f'{seconds:.1f}s'
    minutes, seconds = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    return f'{hours}h {minutes:02d}m {seconds:02d}s' if hours else f'{minutes}m {seconds:02d}s'


def phase14_log_progress(task, completed, total, start_time, extra=''):
    elapsed = time.time() - start_time
    rate = completed / max(elapsed, 1e-6)
    eta = (total - completed) / max(rate, 1e-6) if total else np.nan
    message = f'{task}: {completed}/{total} | elapsed={phase14_format_seconds(elapsed)}'
    if np.isfinite(eta):
        message += f' | ETA={phase14_format_seconds(eta)}'
    if extra:
        message += f' | {extra}'
    phase14_log(message)


def phase14_save_json(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, default=str)


def phase14_probability_columns(df, prefix='prob_class_'):
    cols = [c for c in df.columns if c.startswith(prefix)]
    if not cols:
        raise ValueError(f'No probability columns beginning with {prefix!r} were found.')
    try:
        return sorted(cols, key=lambda c: int(c.removeprefix(prefix)))
    except ValueError:
        return sorted(cols)


def phase14_label_names(classes):
    """Use manifest cohort names when loaded, otherwise stable Version-1 names."""
    fallback = {0: 'RETINORM', 1: 'ORNET', 2: 'FAMILIPO', 3: 'MRCC'}
    if 'phase2_folds' in globals() and isinstance(phase2_folds, pd.DataFrame):
        if {'label', 'cohort'}.issubset(phase2_folds.columns):
            named = phase2_folds[['label', 'cohort']].dropna().drop_duplicates('label')
            mapping = {int(r.label): str(r.cohort) for r in named.itertuples(index=False)}
            return {int(k): mapping.get(int(k), fallback.get(int(k), f'class_{int(k)}')) for k in classes}
    return {int(k): fallback.get(int(k), f'class_{int(k)}') for k in classes}


def phase14_validate_oof_predictions(df):
    required = {'fold', 'patient_id', 'eye', 'label', 'pred', 'correct'}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f'Phase 12 OOF predictions are missing required columns: {missing}')
    prob_cols = phase14_probability_columns(df)
    out = df.copy()
    out['fold'] = out['fold'].astype(int)
    out['patient_id'] = out['patient_id'].astype(str)
    out['eye'] = out['eye'].astype(str)
    out['label'] = out['label'].astype(int)
    probs = out[prob_cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
    if not np.isfinite(probs).all() or (probs < -1e-7).any():
        raise ValueError('OOF probability columns must be finite, non-negative values.')
    row_sums = probs.sum(axis=1)
    if not np.allclose(row_sums, 1.0, atol=2e-4):
        raise ValueError('OOF probability rows must sum to one. Do not evaluate unnormalised logits here.')
    classes = np.arange(len(prob_cols), dtype=int)
    if not set(out['label'].unique()).issubset(set(classes)):
        raise ValueError('OOF labels fall outside the available probability classes.')
    duplicate_keys = out.duplicated(['fold', 'patient_id', 'eye'], keep=False)
    if duplicate_keys.any():
        raise ValueError('OOF table has duplicate fold/patient/eye rows and cannot be scored safely.')
    patient_labels = out.groupby('patient_id', sort=False)['label'].nunique()
    if (patient_labels > 1).any():
        bad = patient_labels[patient_labels > 1].index[:5].tolist()
        raise ValueError(f'Patient labels are inconsistent across eyes: {bad}')
    patient_folds = out.groupby('patient_id', sort=False)['fold'].nunique()
    if (patient_folds > 1).any():
        bad = patient_folds[patient_folds > 1].index[:5].tolist()
        raise ValueError(f'Patient leakage: patients appear in multiple outer folds: {bad}')
    out['prob_sum'] = row_sums
    recomputed_pred = probs.argmax(axis=1)
    if not np.array_equal(out['pred'].to_numpy(dtype=int), recomputed_pred):
        raise ValueError('OOF pred values disagree with argmax calibrated probabilities.')
    out['correct'] = (out['label'].to_numpy(dtype=int) == recomputed_pred).astype(int)
    return out, prob_cols, classes


def phase14_load_oof_predictions(path=PHASE14_OOF_PATH):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Phase 12 OOF prediction file not found: {path}')
    df, prob_cols, classes = phase14_validate_oof_predictions(pd.read_csv(path))
    phase14_log(f'loaded {len(df)} OOF eye predictions from {path} across {df.patient_id.nunique()} patients')
    return df, prob_cols, classes


def phase14_patient_predictions(eye_df, prob_cols):
    """Average calibrated OD/OS probabilities once per patient for inference."""
    aggregations = {c: 'mean' for c in prob_cols}
    # MC probabilities are means over stochastic passes. Patient aggregation averages
    # those eye-level means, which is the available patient inference approximation
    # without storing every pass × eye tensor.
    mc_prob_cols = [c for c in eye_df.columns if c.startswith('mc_prob_class_')]
    mc_var_cols = [c for c in eye_df.columns if c.startswith('mc_epistemic_var_class_')]
    aggregations.update({c: 'mean' for c in mc_prob_cols + mc_var_cols})
    aggregations.update({'label': 'first', 'fold': 'first'})
    if 'mask_source' in eye_df.columns:
        aggregations['mask_source'] = lambda x: '|'.join(sorted(set(x.dropna().astype(str))))
    patient_df = eye_df.groupby('patient_id', as_index=False, sort=False).agg(aggregations)
    probs = patient_df[prob_cols].to_numpy(dtype=float)
    patient_df['pred'] = probs.argmax(axis=1).astype(int)
    patient_df['correct'] = (patient_df['label'].to_numpy(dtype=int) == patient_df['pred'].to_numpy(dtype=int)).astype(int)
    patient_df['n_eyes_aggregated'] = eye_df.groupby('patient_id', sort=False).size().reindex(patient_df['patient_id']).to_numpy()
    if mc_prob_cols:
        mc_probs = patient_df[mc_prob_cols].to_numpy(dtype=float)
        if not np.allclose(mc_probs.sum(axis=1), 1.0, atol=2e-4):
            raise ValueError('Patient-averaged MC-dropout probabilities do not sum to one.')
        patient_df['mc_pred'] = mc_probs.argmax(axis=1).astype(int)
        patient_df['mc_correct'] = (patient_df['label'].to_numpy(dtype=int) == patient_df['mc_pred'].to_numpy(dtype=int)).astype(int)
        patient_df['mc_entropy'] = -(mc_probs * np.log(np.clip(mc_probs, 1e-8, 1.0))).sum(axis=1)
    if not np.allclose(probs.sum(axis=1), 1.0, atol=2e-4):
        raise ValueError('Patient-averaged probabilities do not sum to one.')
    return patient_df


def phase14_safe_auc(y, probs, classes, kind='roc'):
    try:
        onehot = np.eye(len(classes), dtype=int)[np.asarray(y, dtype=int)]
        if kind == 'roc':
            return float(roc_auc_score(onehot, probs, average='macro', multi_class='ovr', labels=classes))
        return float(average_precision_score(onehot, probs, average='macro'))
    except ValueError:
        return float('nan')


def phase14_ece(y, probs, n_bins=PHASE14_ECE_BINS):
    confidence = probs.max(axis=1)
    predicted = probs.argmax(axis=1)
    correct = (predicted == y).astype(float)
    bins = np.linspace(0.0, 1.0, int(n_bins) + 1)
    ece = 0.0
    rows = []
    for i in range(int(n_bins)):
        lo, hi = bins[i], bins[i + 1]
        mask = (confidence >= lo) & ((confidence < hi) if i < int(n_bins) - 1 else (confidence <= hi))
        count = int(mask.sum())
        if count:
            mean_conf = float(confidence[mask].mean())
            accuracy = float(correct[mask].mean())
            ece += count / max(len(y), 1) * abs(accuracy - mean_conf)
        else:
            mean_conf, accuracy = np.nan, np.nan
        rows.append({'bin': i, 'lower': lo, 'upper': hi, 'count': count, 'mean_confidence': mean_conf, 'accuracy': accuracy, 'gap': abs(accuracy - mean_conf) if count else np.nan})
    return float(ece), pd.DataFrame(rows)


def phase14_per_class_metrics(y, probs, classes, label_names):
    pred = probs.argmax(axis=1)
    precision, recall, f1, support = precision_recall_fscore_support(y, pred, labels=classes, zero_division=0)
    rows = []
    for idx, cls in enumerate(classes):
        positive = y == cls
        negative = ~positive
        tp = int(np.sum((pred == cls) & positive))
        tn = int(np.sum((pred != cls) & negative))
        fp = int(np.sum((pred == cls) & negative))
        fn = int(np.sum((pred != cls) & positive))
        try:
            auc = float(roc_auc_score(positive.astype(int), probs[:, idx]))
        except ValueError:
            auc = float('nan')
        try:
            pr_auc = float(average_precision_score(positive.astype(int), probs[:, idx]))
        except ValueError:
            pr_auc = float('nan')
        rows.append({
            'class_id': int(cls), 'class_name': label_names[int(cls)], 'support': int(support[idx]),
            'precision': float(precision[idx]), 'recall': float(recall[idx]), 'sensitivity': float(recall[idx]),
            'specificity': float(tn / max(tn + fp, 1)), 'f1': float(f1[idx]),
            'roc_auc_ovr': auc, 'pr_auc_ovr': pr_auc, 'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
        })
    return pd.DataFrame(rows)


def phase14_global_metrics(df, prob_cols, classes):
    y = df['label'].to_numpy(dtype=int)
    probs = df[prob_cols].to_numpy(dtype=float)
    pred = probs.argmax(axis=1)
    ece, _ = phase14_ece(y, probs)
    onehot = np.eye(len(classes), dtype=float)[y]
    confidence = probs.max(axis=1)
    order = np.argsort(-confidence, kind='stable')
    top_n = max(1, int(math.ceil(0.30 * len(df))))
    top_mask = np.zeros(len(df), dtype=bool)
    top_mask[order[:top_n]] = True
    return {
        'n_samples': int(len(df)),
        'n_patients': int(df['patient_id'].nunique()) if 'patient_id' in df else int(len(df)),
        'accuracy': float(accuracy_score(y, pred)),
        'macro_roc_auc_ovr': phase14_safe_auc(y, probs, classes, kind='roc'),
        'macro_f1': float(f1_score(y, pred, labels=classes, average='macro', zero_division=0)),
        'balanced_accuracy': float(balanced_accuracy_score(y, pred)),
        'weighted_f1': float(f1_score(y, pred, labels=classes, average='weighted', zero_division=0)),
        'macro_pr_auc_ovr': phase14_safe_auc(y, probs, classes, kind='pr'),
        'top2_accuracy': float(np.mean(np.any(np.argsort(-probs, axis=1)[:, :min(2, len(classes))] == y[:, None], axis=1))),
        'ece_10': float(ece),
        'brier_multiclass': float(np.mean(np.sum((probs - onehot) ** 2, axis=1))),
        'mean_confidence': float(confidence.mean()),
        'top30_confidence_coverage': float(top_mask.mean()),
        'top30_confidence_accuracy': float(np.mean(pred[top_mask] == y[top_mask])),
    }


def phase14_confusion_dataframe(df, prob_cols, classes, label_names):
    y = df['label'].to_numpy(dtype=int)
    pred = df[prob_cols].to_numpy(dtype=float).argmax(axis=1)
    matrix = confusion_matrix(y, pred, labels=classes, normalize='true')
    out = pd.DataFrame(matrix, index=[label_names[int(c)] for c in classes], columns=[label_names[int(c)] for c in classes])
    out.index.name = 'true_class'
    return out


def phase14_reliability_by_class(df, prob_cols, classes, label_names, n_bins=PHASE14_ECE_BINS):
    y = df['label'].to_numpy(dtype=int)
    probs = df[prob_cols].to_numpy(dtype=float)
    rows = []
    bins = np.linspace(0.0, 1.0, int(n_bins) + 1)
    for col_idx, cls in enumerate(classes):
        truth = (y == cls).astype(float)
        score = probs[:, col_idx]
        for i in range(int(n_bins)):
            lo, hi = bins[i], bins[i + 1]
            mask = (score >= lo) & ((score < hi) if i < int(n_bins) - 1 else (score <= hi))
            rows.append({
                'class_id': int(cls), 'class_name': label_names[int(cls)], 'bin': int(i), 'lower': float(lo), 'upper': float(hi),
                'count': int(mask.sum()), 'mean_probability': float(score[mask].mean()) if mask.any() else np.nan,
                'empirical_frequency': float(truth[mask].mean()) if mask.any() else np.nan,
            })
    out = pd.DataFrame(rows)
    out['calibration_gap'] = (out['mean_probability'] - out['empirical_frequency']).abs()
    return out


def phase14_uncertainty_summary(df, prob_cols, classes):
    """Uses MC columns when present; reports an explicit unavailable status otherwise."""
    mc_cols = [f'mc_prob_class_{int(c)}' for c in classes]
    if not set(mc_cols).issubset(df.columns):
        return pd.DataFrame([{
            'status': 'unavailable',
            'reason': 'No checkpoint-derived MC-dropout prediction artifact was merged. Run phase14_generate_mc_dropout_predictions() first.',
        }]), pd.DataFrame()
    probs = df[mc_cols].to_numpy(dtype=float)
    y = df['label'].to_numpy(dtype=int)
    pred = probs.argmax(axis=1)
    entropy = df['mc_entropy'].to_numpy(dtype=float) if 'mc_entropy' in df else -(probs * np.log(np.clip(probs, 1e-8, 1))).sum(axis=1)
    q = np.quantile(entropy, [0.0, 0.25, 0.5, 0.75, 1.0])
    rows = []
    for i in range(4):
        lo, hi = q[i], q[i + 1]
        mask = (entropy >= lo) & ((entropy < hi) if i < 3 else (entropy <= hi))
        rows.append({'entropy_quartile': int(i + 1), 'entropy_lower': float(lo), 'entropy_upper': float(hi), 'n': int(mask.sum()), 'mean_entropy': float(entropy[mask].mean()) if mask.any() else np.nan, 'accuracy': float(np.mean(pred[mask] == y[mask])) if mask.any() else np.nan})
    # Spearman correlation without a SciPy dependency. Negative is desirable here.
    entropy_rank = pd.Series(entropy).rank(method='average').to_numpy()
    correct_rank = pd.Series((pred == y).astype(int)).rank(method='average').to_numpy()
    rho = float(np.corrcoef(entropy_rank, correct_rank)[0, 1]) if len(entropy) > 1 else np.nan
    confidence = probs.max(axis=1)
    top_n = max(1, int(math.ceil(0.30 * len(df))))
    selected = np.argsort(-confidence, kind='stable')[:top_n]
    summary = pd.DataFrame([{
        'status': 'available', 'n_samples': int(len(df)), 'mean_prediction_entropy': float(entropy.mean()),
        'entropy_accuracy_spearman_rho': rho, 'mc_accuracy': float(np.mean(pred == y)),
        'top30_confidence_coverage': float(len(selected) / max(len(df), 1)),
        'top30_confidence_accuracy': float(np.mean(pred[selected] == y[selected])),
    }])
    return summary, pd.DataFrame(rows)


def phase14_cluster_bootstrap_indices(df, rng, label_col='label', patient_col='patient_id'):
    """Stratified patient-cluster resampling, retaining every sampled patient's eyes."""
    patient_labels = df[[patient_col, label_col]].drop_duplicates(patient_col)
    groups = {pid: idx.to_numpy() for pid, idx in df.groupby(patient_col, sort=False).groups.items()}
    sampled = []
    for _, block in patient_labels.groupby(label_col, sort=True):
        patients = block[patient_col].astype(str).to_numpy()
        chosen = rng.choice(patients, size=len(patients), replace=True)
        sampled.extend(groups[str(pid)] for pid in chosen)
    return np.concatenate(sampled) if sampled else np.array([], dtype=int)


def phase14_bootstrap_intervals(df, prob_cols, classes, n_bootstrap=PHASE14_BOOTSTRAP_SAMPLES, seed=PHASE14_BOOTSTRAP_SEED):
    rng = np.random.default_rng(int(seed))
    values = {metric: [] for metric in PHASE14_PRIMARY_METRICS}
    start = time.time()
    n_bootstrap = int(n_bootstrap)
    phase14_log(f'START stratified patient-cluster bootstrap: replicates={n_bootstrap}, patients={df.patient_id.nunique()}, eye_rows={len(df)}')
    for iteration in range(n_bootstrap):
        idx = phase14_cluster_bootstrap_indices(df.reset_index(drop=True), rng)
        metrics = phase14_global_metrics(df.iloc[idx].reset_index(drop=True), prob_cols, classes)
        for metric in values:
            values[metric].append(metrics.get(metric, np.nan))
        completed = iteration + 1
        if completed % max(1, int(PHASE14_LOG_EVERY_N_BOOTSTRAPS)) == 0 or completed == n_bootstrap:
            phase14_log_progress('bootstrap', completed, n_bootstrap, start)
    phase14_log(f'END stratified patient-cluster bootstrap: {n_bootstrap} replicates in {phase14_format_seconds(time.time() - start)}')
    rows = []
    point = phase14_global_metrics(df, prob_cols, classes)
    for metric, samples in values.items():
        arr = np.asarray(samples, dtype=float)
        finite = arr[np.isfinite(arr)]
        rows.append({
            'metric': metric, 'point_estimate': point.get(metric, np.nan), 'ci_95_low': float(np.quantile(finite, 0.025)) if len(finite) else np.nan,
            'ci_95_high': float(np.quantile(finite, 0.975)) if len(finite) else np.nan,
            'bootstrap_samples_requested': int(n_bootstrap), 'bootstrap_samples_valid': int(len(finite)),
            'resampling_unit': 'stratified_patient_cluster',
        })
    return pd.DataFrame(rows)


def phase14_plot_reliability(reliability_df, path, title):
    if plt is None or reliability_df.empty:
        return None
    fig, ax = plt.subplots(figsize=(6, 6))
    for class_name, sub in reliability_df.groupby('class_name', sort=False):
        sub = sub.dropna(subset=['mean_probability', 'empirical_frequency'])
        if len(sub):
            ax.plot(sub['mean_probability'], sub['empirical_frequency'], marker='o', label=str(class_name))
    ax.plot([0, 1], [0, 1], '--', color='black', linewidth=1, label='Perfect calibration')
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='Mean predicted probability', ylabel='Observed frequency', title=title)
    ax.legend(loc='best', fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    plt.close(fig)
    return Path(path)


def phase14_merge_mc_dropout(eye_df, mc_path=PHASE14_MC_DROPOUT_PATH):
    mc_path = Path(mc_path)
    if not mc_path.exists():
        return eye_df.copy(), False
    mc = pd.read_csv(mc_path)
    keys = ['fold', 'patient_id', 'eye', 'label']
    missing = sorted(set(keys) - set(mc.columns))
    if missing:
        raise ValueError(f'MC-dropout artifact is missing identity columns: {missing}')
    mc['fold'] = mc['fold'].astype(int)
    mc['patient_id'] = mc['patient_id'].astype(str)
    mc['eye'] = mc['eye'].astype(str)
    mc['label'] = mc['label'].astype(int)
    if mc.duplicated(keys).any():
        raise ValueError('MC-dropout artifact has duplicate fold/patient/eye/label rows.')
    mc_probability_cols = phase14_probability_columns(mc, prefix='mc_prob_class_')
    mc_probs = mc[mc_probability_cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
    if not np.isfinite(mc_probs).all() or not np.allclose(mc_probs.sum(axis=1), 1.0, atol=2e-4):
        raise ValueError('MC-dropout probabilities must be finite and sum to one.')
    merged = eye_df.merge(mc, on=keys, how='left', validate='one_to_one', suffixes=('', '_mc'))
    mc_prob_cols = [c for c in merged.columns if c.startswith('mc_prob_class_')]
    if len(merged) != len(eye_df) or not mc_prob_cols or merged[mc_prob_cols].isna().any().any():
        raise ValueError('MC-dropout artifact does not exactly cover the deterministic Phase 12 OOF rows.')
    return merged, True


def phase14_evaluate_predictions(oof_path=PHASE14_OOF_PATH, output_dir=PHASE14_OUTPUT_DIR, n_bootstrap=PHASE14_BOOTSTRAP_SAMPLES, mc_path=PHASE14_MC_DROPOUT_PATH):
    """Write Phase 14 evaluation bundle from deterministic OOF predictions and optional MC artifact."""
    run_start = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    phase14_log(f'START evaluation: OOF={oof_path}, bootstrap_replicates={int(n_bootstrap)}, output={output_dir}')
    phase14_log('Step 1/5: loading and validating deterministic calibrated OOF predictions')
    eye_df, prob_cols, classes = phase14_load_oof_predictions(oof_path)
    phase14_log('Step 2/5: checking optional checkpoint-derived MC-dropout artifact')
    eye_df, mc_available = phase14_merge_mc_dropout(eye_df, mc_path=mc_path)
    phase14_log(f'MC-dropout artifact merged={mc_available}')
    patient_df = phase14_patient_predictions(eye_df, prob_cols)
    label_names = phase14_label_names(classes)
    phase14_log(f'Prepared inference units: eyes={len(eye_df)}, patients={len(patient_df)}, classes={list(label_names.values())}')
    metadata = {
        'source_oof_path': str(oof_path), 'mc_dropout_path': str(mc_path), 'mc_dropout_merged': bool(mc_available),
        'classes': [int(c) for c in classes], 'label_names': label_names, 'bootstrap_samples': int(n_bootstrap),
        'bootstrap_method': 'stratified patient-cluster bootstrap; eye-level samples retain all sampled bilateral eyes',
        'metric_scope': 'out-of-fold outer-test predictions only',
    }
    phase14_save_json(metadata, output_dir / 'phase14_evaluation_manifest.json')
    all_summaries, all_per_class, all_cis, all_reliability, all_uncertainty = [], [], [], [], []
    for level_index, (level, frame) in enumerate([('eye', eye_df), ('patient', patient_df)], start=1):
        level_start = time.time()
        phase14_log(f'Step 3/5 ({level_index}/2): START {level}-level metrics for n={len(frame)}')
        metrics = phase14_global_metrics(frame, prob_cols, classes)
        all_summaries.append({'level': level, **metrics})
        phase14_log(f'{level}-level primary metrics: macro_auc={metrics["macro_roc_auc_ovr"]:.4f}, macro_f1={metrics["macro_f1"]:.4f}, balanced_accuracy={metrics["balanced_accuracy"]:.4f}, ECE={metrics["ece_10"]:.4f}')
        phase14_log(f'{level}-level: calculating per-class metrics, normalized confusion matrix, and reliability bins')
        per_class = phase14_per_class_metrics(frame['label'].to_numpy(dtype=int), frame[prob_cols].to_numpy(dtype=float), classes, label_names)
        per_class.insert(0, 'level', level)
        all_per_class.append(per_class)
        cm = phase14_confusion_dataframe(frame, prob_cols, classes, label_names)
        cm.to_csv(output_dir / f'phase14_confusion_matrix_{level}_true_normalized.csv')
        reliability = phase14_reliability_by_class(frame, prob_cols, classes, label_names)
        reliability.insert(0, 'level', level)
        all_reliability.append(reliability)
        reliability_plot = phase14_plot_reliability(reliability, output_dir / f'phase14_reliability_diagram_{level}.png', f'Phase 14 reliability - {level}-level')
        uncertainty, quartiles = phase14_uncertainty_summary(frame, prob_cols, classes)
        uncertainty.insert(0, 'level', level)
        all_uncertainty.append(uncertainty)
        if not quartiles.empty:
            quartiles.insert(0, 'level', level)
            quartiles.to_csv(output_dir / f'phase14_uncertainty_entropy_quartiles_{level}.csv', index=False)
        phase14_log(f'{level}-level: START 95% bootstrap CIs ({int(n_bootstrap)} replicates)')
        ci = phase14_bootstrap_intervals(frame, prob_cols, classes, n_bootstrap=n_bootstrap, seed=PHASE14_BOOTSTRAP_SEED + (0 if level == 'eye' else 1))
        ci.insert(0, 'level', level)
        all_cis.append(ci)
        phase14_log(f'{level}-level: END outputs complete in {phase14_format_seconds(time.time() - level_start)} | reliability_plot={reliability_plot}')
    phase14_log('Step 4/5: calculating per-fold eye and patient stability metrics')
    fold_rows = []
    fold_blocks = list(eye_df.groupby('fold', sort=True))
    for fold_index, (fold, block) in enumerate(fold_blocks, start=1):
        fold_rows.append({'level': 'eye', 'fold': int(fold), **phase14_global_metrics(block, prob_cols, classes)})
        patient_block = patient_df[patient_df['fold'] == int(fold)]
        fold_rows.append({'level': 'patient', 'fold': int(fold), **phase14_global_metrics(patient_block, prob_cols, classes)})
        phase14_log(f'per-fold metrics: fold {int(fold)} ({fold_index}/{len(fold_blocks)}) eyes={len(block)} patients={len(patient_block)}')
    phase14_log('Step 5/5: writing consolidated CSV, JSON, and prediction artifacts')
    pd.DataFrame(all_summaries).to_csv(output_dir / 'phase14_metrics_summary.csv', index=False)
    pd.concat(all_per_class, ignore_index=True).to_csv(output_dir / 'phase14_per_class_metrics.csv', index=False)
    pd.concat(all_cis, ignore_index=True).to_csv(output_dir / 'phase14_bootstrap_confidence_intervals.csv', index=False)
    pd.concat(all_reliability, ignore_index=True).to_csv(output_dir / 'phase14_reliability_bins.csv', index=False)
    pd.concat(all_uncertainty, ignore_index=True).to_csv(output_dir / 'phase14_uncertainty_summary.csv', index=False)
    pd.DataFrame(fold_rows).to_csv(output_dir / 'phase14_fold_metrics.csv', index=False)
    eye_df.to_csv(output_dir / 'phase14_eye_level_oof_predictions_with_uncertainty.csv', index=False)
    patient_df.to_csv(output_dir / 'phase14_patient_level_oof_predictions.csv', index=False)
    phase14_log(f'END evaluation: complete in {phase14_format_seconds(time.time() - run_start)} | MC-dropout merged={mc_available} | output={output_dir}')
    return {
        'eye_predictions': eye_df, 'patient_predictions': patient_df, 'metrics': pd.DataFrame(all_summaries),
        'per_class_metrics': pd.concat(all_per_class, ignore_index=True), 'bootstrap_cis': pd.concat(all_cis, ignore_index=True),
    }

# ---------- MC-dropout artifact generation (checkpoint-based, optional) ----------

def phase14_require_mc_prerequisites():
    required = ['phase13_load_fold_artifacts', 'phase12_to_device', 'phase12_labels_from_batch', 'torch']
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError('MC-dropout needs trained-model definitions. Run the Phase 13 fast-reload cell first. Missing: ' + ', '.join(missing))


def phase14_set_dropout_only(model, enabled=True):
    """Keep normalisation and frozen U-Net++ inference in eval mode while sampling dropout."""
    model.eval()
    dropout_types = tuple(t for t in [
        getattr(torch.nn, 'Dropout', None), getattr(torch.nn, 'Dropout1d', None),
        getattr(torch.nn, 'Dropout2d', None), getattr(torch.nn, 'Dropout3d', None), getattr(torch.nn, 'AlphaDropout', None),
    ] if t is not None)
    changed = []
    for module in model.modules():
        if isinstance(module, dropout_types):
            module.train(bool(enabled))
            changed.append(module)
    if hasattr(model, 'mask_generator') and model.mask_generator is not None:
        model.mask_generator.eval()
    return changed


def phase14_mc_dropout_for_fold(fold, passes=PHASE14_MC_DROPOUT_PASSES, batch_size=PHASE14_MC_DROPOUT_BATCH_SIZE, device=None):
    phase14_require_mc_prerequisites()
    fold_start = time.time()
    phase14_log(f'fold {int(fold)}: START loading trained checkpoint, datasets, and materialized U-Net++ masks')
    artifacts = phase13_load_fold_artifacts(int(fold), device=device)
    model, device = artifacts['model'], artifacts['device']
    dataset = artifacts['datasets']['test']
    loader = torch.utils.data.DataLoader(dataset, batch_size=int(batch_size), shuffle=False, num_workers=int(PHASE14_NUM_WORKERS), pin_memory=False)
    phase14_set_dropout_only(model, enabled=True)
    phase14_log(f'fold {int(fold)}: START MC-dropout inference, samples={len(dataset)}, batches={len(loader)}, passes_per_batch={int(passes)}, device={device}')
    rows = []
    try:
        with torch.no_grad():
            for batch_index, batch in enumerate(loader, start=1):
                batch_start = time.time()
                if batch_index % max(1, int(PHASE14_LOG_EVERY_N_MC_BATCHES)) == 0:
                    phase14_log(f'fold {int(fold)}: START MC batch {batch_index}/{len(loader)} ({int(passes)} stochastic passes)')
                work = phase12_to_device(batch, device)
                sampled_probs = []
                for pass_index in range(int(passes)):
                    logits = model(work, return_attention=False, use_prototypes=True)['logits']
                    sampled_probs.append(torch.softmax(phase9_apply_temperature(logits, artifacts['temperature']), dim=1))
                    if PHASE14_LOG_MC_PASS_MILESTONES and ((pass_index + 1) % 5 == 0 or pass_index + 1 == int(passes)):
                        phase14_log(f'fold {int(fold)}: batch {batch_index}/{len(loader)} MC pass {pass_index + 1}/{int(passes)}')
                sampled_probs = torch.stack(sampled_probs, dim=0)
                mean_probs = sampled_probs.mean(dim=0).detach().cpu().numpy()
                epistemic_var = sampled_probs.var(dim=0, unbiased=False).detach().cpu().numpy()
                entropy = -(mean_probs * np.log(np.clip(mean_probs, 1e-8, 1.0))).sum(axis=1)
                labels = phase12_labels_from_batch(batch).detach().cpu().numpy().astype(int)
                preds = mean_probs.argmax(axis=1)
                for i in range(len(labels)):
                    row = {
                        'fold': int(fold), 'patient_id': str(batch['patient_id'][i]), 'eye': str(batch['eye'][i]),
                        'label': int(labels[i]), 'mc_pred': int(preds[i]), 'mc_correct': int(labels[i] == preds[i]),
                        'mc_entropy': float(entropy[i]), 'mc_passes': int(passes), 'temperature': float(artifacts['temperature']),
                    }
                    for class_id in range(mean_probs.shape[1]):
                        row[f'mc_prob_class_{class_id}'] = float(mean_probs[i, class_id])
                        row[f'mc_epistemic_var_class_{class_id}'] = float(epistemic_var[i, class_id])
                    rows.append(row)
                completed = min(len(rows), len(dataset))
                if batch_index % max(1, int(PHASE14_LOG_EVERY_N_MC_BATCHES)) == 0 or batch_index == len(loader):
                    phase14_log_progress(f'fold {int(fold)} MC-dropout', completed, len(dataset), fold_start, extra=f'batch={batch_index}/{len(loader)} batch_time={phase14_format_seconds(time.time() - batch_start)}')
                del work, sampled_probs
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    finally:
        model.eval()
        if hasattr(model, 'mask_generator') and model.mask_generator is not None:
            model.mask_generator.eval()
    phase14_log(f'fold {int(fold)}: END MC-dropout inference, rows={len(rows)}, total_time={phase14_format_seconds(time.time() - fold_start)}')
    return pd.DataFrame(rows)

def phase14_generate_mc_dropout_predictions(folds=PHASE14_FOLDS, output_path=PHASE14_MC_DROPOUT_PATH, passes=PHASE14_MC_DROPOUT_PASSES, device=None):
    """Generate complete held-out MC predictions without retraining or changing Phase 12 OOF metrics."""
    run_start = time.time()
    folds = [int(fold) for fold in folds]
    phase14_log(f'START complete MC-dropout generation: folds={folds}, passes={int(passes)}, batch_size={PHASE14_MC_DROPOUT_BATCH_SIZE}')
    all_rows = []
    for fold_index, fold in enumerate(folds, start=1):
        fold_start = time.time()
        phase14_log(f'MC-dropout fold {fold_index}/{len(folds)}: fold={fold}')
        all_rows.append(phase14_mc_dropout_for_fold(fold, passes=passes, device=device))
        phase14_log_progress('MC-dropout folds', fold_index, len(folds), run_start, extra=f'fold={fold} fold_time={phase14_format_seconds(time.time() - fold_start)}')
    phase14_log('Validating complete MC artifact identities against Phase 12 deterministic OOF predictions')
    out = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
    deterministic, _, _ = phase14_load_oof_predictions()
    keys = ['fold', 'patient_id', 'eye', 'label']
    if len(out) != len(deterministic) or not out[keys].sort_values(keys).reset_index(drop=True).equals(deterministic[keys].sort_values(keys).reset_index(drop=True)):
        raise RuntimeError('MC-dropout rows do not exactly match Phase 12 OOF identities; artifact was not written.')
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(output_path, index=False)
    phase14_log(f'END complete MC-dropout generation: rows={len(out)}, time={phase14_format_seconds(time.time() - run_start)}, output={output_path}')
    return out

# ---------- Future Phase 16 model-comparison utilities ----------

def phase14_exact_mcnemar_p_value(n01, n10):
    n = int(n01 + n10)
    if n == 0:
        return 1.0
    k = min(int(n01), int(n10))
    tail = sum(math.comb(n, i) for i in range(k + 1)) / (2 ** n)
    return float(min(1.0, 2.0 * tail))


def phase14_delong_binary(y_true, scores_a, scores_b):
    """Paired DeLong test for one binary OVR class, implemented without SciPy."""
    y = np.asarray(y_true, dtype=int)
    a, b = np.asarray(scores_a, dtype=float), np.asarray(scores_b, dtype=float)
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    if len(pos) < 2 or len(neg) < 2:
        return {'auc_a': np.nan, 'auc_b': np.nan, 'auc_difference': np.nan, 'z': np.nan, 'p_value': np.nan}
    def placements(scores):
        comparisons = scores[pos, None] - scores[None, neg]
        kernel = (comparisons > 0).astype(float) + 0.5 * (comparisons == 0)
        return kernel.mean(axis=1), kernel.mean(axis=0)
    v10_a, v01_a = placements(a)
    v10_b, v01_b = placements(b)
    auc_a, auc_b = float(v10_a.mean()), float(v10_b.mean())
    cov10 = np.cov(np.vstack([v10_a, v10_b]), ddof=1) / len(pos)
    cov01 = np.cov(np.vstack([v01_a, v01_b]), ddof=1) / len(neg)
    variance = float(np.array([1.0, -1.0]) @ (cov10 + cov01) @ np.array([1.0, -1.0]))
    se = math.sqrt(max(variance, 0.0))
    delta = auc_a - auc_b
    z = delta / se if se > 0 else np.nan
    p = math.erfc(abs(z) / math.sqrt(2.0)) if np.isfinite(z) else np.nan
    return {'auc_a': auc_a, 'auc_b': auc_b, 'auc_difference': delta, 'z': float(z), 'p_value': float(p)}


def phase14_cohens_h(p_a, p_b):
    return float(2.0 * (math.asin(math.sqrt(np.clip(p_a, 0, 1))) - math.asin(math.sqrt(np.clip(p_b, 0, 1)))))


def phase14_compare_models(proposed_df, baseline_df, proposed_prob_cols=None, baseline_prob_cols=None, output_path=None):
    """Paired OOF comparison for Phase 16 baselines: McNemar, per-class DeLong, Bonferroni, Cohen's h."""
    proposed_prob_cols = proposed_prob_cols or phase14_probability_columns(proposed_df)
    baseline_prob_cols = baseline_prob_cols or phase14_probability_columns(baseline_df)
    if len(proposed_prob_cols) != len(baseline_prob_cols):
        raise ValueError('Compared models must expose the same number of class probabilities.')
    keys = ['fold', 'patient_id', 'eye', 'label']
    left = proposed_df[keys + proposed_prob_cols].copy()
    right = baseline_df[keys + baseline_prob_cols].copy()
    right = right.rename(columns={c: f'baseline_{i}' for i, c in enumerate(baseline_prob_cols)})
    paired = left.merge(right, on=keys, how='inner', validate='one_to_one')
    if len(paired) != len(proposed_df) or len(paired) != len(baseline_df):
        raise ValueError('Model comparison requires identical complete OOF sample identities.')
    y = paired['label'].to_numpy(dtype=int)
    pa = paired[proposed_prob_cols].to_numpy(dtype=float)
    pb = paired[[f'baseline_{i}' for i in range(len(baseline_prob_cols))]].to_numpy(dtype=float)
    correct_a, correct_b = pa.argmax(axis=1) == y, pb.argmax(axis=1) == y
    n01, n10 = int(np.sum(correct_a & ~correct_b)), int(np.sum(~correct_a & correct_b))
    rows = [{'test': 'mcnemar_exact', 'class_id': 'all', 'n_proposed_correct_baseline_wrong': n01, 'n_proposed_wrong_baseline_correct': n10, 'accuracy_difference': float(correct_a.mean() - correct_b.mean()), 'p_value': phase14_exact_mcnemar_p_value(n01, n10), 'effect_size': float(correct_a.mean() - correct_b.mean()), 'effect_size_name': 'accuracy_difference'}]
    classes = np.arange(pa.shape[1])
    for cls in classes:
        result = phase14_delong_binary((y == cls).astype(int), pa[:, cls], pb[:, cls])
        result.update({'test': 'delong_ovr_auc', 'class_id': int(cls), 'effect_size': phase14_cohens_h(result['auc_a'], result['auc_b']) if np.isfinite(result['auc_a']) and np.isfinite(result['auc_b']) else np.nan, 'effect_size_name': "Cohen's h"})
        rows.append(result)
    out = pd.DataFrame(rows)
    delong_mask = out['test'].eq('delong_ovr_auc')
    out.loc[delong_mask, 'p_value_bonferroni'] = np.minimum(1.0, out.loc[delong_mask, 'p_value'] * int(delong_mask.sum()))
    if output_path is not None:
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        out.to_csv(output_path, index=False)
    return out


def phase14_smoke_test():
    """Small deterministic, no-file smoke test for metric, CI, and comparison helpers."""
    rng = np.random.default_rng(7)
    rows = []
    for cls in range(8):
        for patient in range(3):
            for eye in ['OD', 'OS']:
                p = rng.dirichlet(np.ones(8) + np.eye(8)[cls] * 5.0)
                rows.append({'fold': patient % 2, 'patient_id': f'{cls}_{patient}', 'eye': eye, 'label': cls, 'pred': int(p.argmax()), 'correct': int(p.argmax() == cls), **{f'prob_class_{i}': float(p[i]) for i in range(8)}})
    frame, cols, classes = phase14_validate_oof_predictions(pd.DataFrame(rows))
    metrics = phase14_global_metrics(frame, cols, classes)
    cis = phase14_bootstrap_intervals(frame, cols, classes, n_bootstrap=8, seed=1)
    comparison = phase14_compare_models(frame, frame.assign(**{c: frame[c].to_numpy() for c in cols}), cols, cols)
    assert metrics['n_samples'] == len(frame) and len(cis) == len(PHASE14_PRIMARY_METRICS) and len(comparison) == 1 + len(classes)
    return {'metrics': metrics, 'ci_rows': len(cis), 'comparison_rows': len(comparison)}


phase14_summary = {
    'oof_source': str(PHASE14_OOF_PATH), 'output_dir': str(PHASE14_OUTPUT_DIR),
    'bootstrap_samples': PHASE14_BOOTSTRAP_SAMPLES, 'bootstrap_unit': 'stratified patient cluster',
    'eye_level_and_patient_level': True, 'patient_prediction_rule': 'mean calibrated OD/OS probabilities',
    'mc_dropout_passes': PHASE14_MC_DROPOUT_PASSES,
    'note': 'Run phase14_generate_mc_dropout_predictions() after Phase 13 fast reload, then phase14_evaluate_predictions() on the training machine.',
}
phase14_save_json(phase14_summary, PHASE14_OUTPUT_DIR / 'phase14_summary.json')
if PHASE14_RUN_MC_DROPOUT:
    phase14_generate_mc_dropout_predictions(PHASE14_FOLDS)
if PHASE14_RUN_EVALUATION:
    phase14_results = phase14_evaluate_predictions()
if PHASE14_RUN_SMOKE_TEST:
    print('Phase 14 smoke test:', phase14_smoke_test())
if not PHASE14_RUN_EVALUATION:
    print('Phase 14 evaluation code ready. Run checkpoint MC-dropout first if uncertainty artifacts are needed, then call phase14_evaluate_predictions().')

In [ ]:
# ============================================================
# Eight-class comparison orchestration for Phases 9-14
# ============================================================
# Run only after the required upstream artifacts/checkpoints exist.


def segcomp_materialize_masks(model_ids=SEGCOMP_MODEL_IDS, folds=PHASE8_COMPARISON_FOLDS, fold_df=None, device=None):
    summaries=[]
    for model_id in model_ids:
        segcomp_activate_model(model_id)
        for fold in folds:
            _,summary=phase9_materialize_unetpp_masks_for_fold(
                fold,fold_df=fold_df,output_dir=PHASE9_UNETPP_MASK_DIR,device=device)
            summary['segmentation_model_id']=model_id; summaries.append(summary)
    out=pd.DataFrame(summaries)
    out.to_csv(ensure_dir(SEGCOMP_ROOT/'comparison')/'mask_materialization_summary.csv',index=False)
    return out


def segcomp_train_all_classifiers(model_ids=SEGCOMP_MODEL_IDS, folds=PHASE12_FOLDS, fold_df=None, device=None):
    results={}
    for model_id in model_ids:
        print(f'[Classifier comparison] START {model_id}')
        segcomp_activate_model(model_id)
        results[model_id]=phase12_train_cross_validation(folds=folds,fold_df=fold_df,device=device)
    return results


def segcomp_explain_all_models(model_ids=SEGCOMP_MODEL_IDS, folds=PHASE13_FOLDS):
    results={}
    for model_id in model_ids:
        print(f'[Explainability comparison] START {model_id}')
        segcomp_activate_model(model_id)
        results[model_id]=phase13_run_cross_validation(folds=folds)
    return results


def segcomp_evaluate_all_models(model_ids=SEGCOMP_MODEL_IDS, folds=PHASE14_FOLDS, generate_mc_dropout=True, device=None):
    results={}; primary_rows=[]
    for model_id in model_ids:
        print(f'[Evaluation comparison] START {model_id}')
        segcomp_activate_model(model_id)
        mc_path=PHASE14_MC_DROPOUT_PATH
        if generate_mc_dropout:
            phase14_generate_mc_dropout_predictions(folds=folds,output_path=mc_path,passes=PHASE14_MC_DROPOUT_PASSES,device=device)
        result=phase14_evaluate_predictions(
            oof_path=PHASE14_OOF_PATH,output_dir=PHASE14_OUTPUT_DIR,
            n_bootstrap=PHASE14_BOOTSTRAP_SAMPLES,mc_path=mc_path,
        )
        results[model_id]=result
        for level in ['eye','patient']:
            metrics=result.get(level,{}).get('metrics',{}) if isinstance(result,dict) else {}
            primary_rows.append({'segmentation_model_id':model_id,'level':level,**metrics})
    comparison_dir=ensure_dir(SEGCOMP_ROOT/'comparison')
    pd.DataFrame(primary_rows).to_csv(comparison_dir/'phase14_all_model_primary_metrics.csv',index=False)
    return results


def segcomp_compare_models_to_unetpp(model_ids=SEGCOMP_MODEL_IDS):
    """Run paired patient-level comparisons against the prespecified U-Net++ reference."""
    comparison_dir=ensure_dir(SEGCOMP_ROOT/'comparison'/'paired_to_unetpp')
    reference_path=segcomp_model_dir('unetpp')/'phase12_training'/'phase12_all_outer_test_predictions.csv'
    reference_eye,reference_cols,_=phase14_validate_oof_predictions(pd.read_csv(reference_path))
    reference=phase14_patient_predictions(reference_eye,reference_cols)
    reference['eye']='patient'
    outputs=[]
    for model_id in model_ids:
        if model_id=='unetpp':
            continue
        candidate_path=segcomp_model_dir(model_id)/'phase12_training'/'phase12_all_outer_test_predictions.csv'
        candidate_eye,candidate_cols,_=phase14_validate_oof_predictions(pd.read_csv(candidate_path))
        candidate=phase14_patient_predictions(candidate_eye,candidate_cols)
        candidate['eye']='patient'
        result=phase14_compare_models(candidate,reference,candidate_cols,reference_cols,comparison_dir/f'{model_id}_vs_unetpp.csv')
        result.insert(0,'candidate_model_id',model_id)
        outputs.append(result)
    combined=pd.concat(outputs,ignore_index=True) if outputs else pd.DataFrame()
    if len(combined) and 'p_value' in combined:
        combined['p_value_five_model_bonferroni']=np.minimum(1.0,combined['p_value'].astype(float)*max(1,len(model_ids)-1))
    combined.to_csv(comparison_dir/'all_models_vs_unetpp.csv',index=False)
    return combined


def segcomp_collect_segmentation_metrics(model_ids=SEGCOMP_MODEL_IDS):
    frames=[]
    for model_id in model_ids:
        path=segcomp_model_dir(model_id)/'phase8_segmentation'/'phase8_all_test_metrics.csv'
        if not path.exists():
            raise FileNotFoundError(f'Missing segmentation metric file: {path}')
        frame=pd.read_csv(path); frame['segmentation_model_id']=model_id; frames.append(frame)
    combined=pd.concat(frames,ignore_index=True)
    combined.to_csv(ensure_dir(SEGCOMP_ROOT/'comparison')/'phase8_all_model_test_metrics.csv',index=False)
    return combined


def segcomp_run_requested_stages():
    """Honor the disabled-by-default comparison flags in dependency order."""
    outputs={}
    if SEGCOMP_RUN_MASK_MATERIALIZATION:
        outputs['materialization']=segcomp_materialize_masks()
    if SEGCOMP_RUN_CLASSIFIER_TRAINING:
        outputs['classifiers']=segcomp_train_all_classifiers()
    if SEGCOMP_RUN_EXPLAINABILITY:
        outputs['explainability']=segcomp_explain_all_models()
    if SEGCOMP_RUN_EVALUATION:
        outputs['evaluation']=segcomp_evaluate_all_models()
    if not outputs:
        print('No heavy comparison stage selected. Enable a SEGCOMP_RUN_* flag or call the stage function directly.')
    return outputs


segcomp_activate_model('unetpp')
print('Comparison orchestration ready for:', SEGCOMP_MODEL_IDS)
print('Enabled stages will now run in dependency order on the Windows GPU machine.')
segcomp_run_outputs = segcomp_run_requested_stages()
